# QuantumJam 2026 · Curva de energía de $\mathrm{H_4}$ con VQE

Cadena lineal de 4 H con espaciado uniforme $d$, base STO-3G, neutra y singlete: 4 orbitales espaciales → 8 espín-orbitales → **8 qubits** (Jordan-Wigner), 4 electrones. Calculamos con VQE la curva $E(d)$ para $d \in [0{,}6;\ 3{,}0]$ Å, con $R_e$, $D_e = E(3{,}0\ \text{Å}) - E(R_e)$ y una barra de error $\pm\sigma$ por punto, y la comparamos con CCSD y con la energía exacta (FCI). Enunciado completo en `docs/challenge/`.

### El problema: por qué la zona estirada es difícil

Cerca del equilibrio ($R_e \approx 0{,}89$ Å) el estado fundamental es casi un solo determinante (Hartree-Fock pesa 0,985 a 0,6 Å) y CCSD es prácticamente exacto (< 0,01 mHa de error). Al estirar la cadena, los 4 electrones terminan en cuatro orbitales 1s casi desacoplados, un singlete con muchos determinantes de peso parecido: el estado se vuelve **multirreferencia** (HF pesa 0,58 a 1,8 Å y 0,27 a 3,0 Å). Un método de una sola referencia como CCSD no lo describe bien, y como no es variacional, **cae por debajo de FCI**:

| $d$ (Å) | 0,9 | 1,6 | **1,8** | 2,0 | 2,4 | 2,6 | 3,0 |
|---|---|---|---|---|---|---|---|
| $E_\text{CCSD} - E_\text{FCI}$ (mHa) | +0,01 | −3,0 | **−9,1** | −18,3 | −30,6 | −30,6 | −23,0 |

(`docs/analysis/data/h4_reference.json`; CCSD(T) es peor todavía). Por eso el desafío pide, para $d \geq 1{,}8$ Å, $|E_\text{VQE} - E_\text{FCI}| + \sigma < |E_\text{CCSD} - E_\text{FCI}|$; 1,8 Å es el punto más exigente porque ahí el margen es el más chico (9,1 mHa). La meta máxima es precisión química, $< 1{,}6$ mHa. CCSD también subestima $D_e$ en ~23 mHa (≈ 290 contra 313 mHa de FCI). Un detalle más: desde ~2,4 Å el primer excitado es un triplete a pocos mHa (0,46 mHa a 3,0 Å), así que la energía sola no alcanza para saber si el estado es el correcto.

### Qué hacemos

| Paso | Sección | Qué sale |
|---|---|---|
| Referencias clásicas HF, CCSD, FCI (solo para comparar y elegir puntos de partida) | 1 | tabla `refs` |
| Hamiltoniano de 8 qubits por Jordan-Wigner, validado contra HF y FCI | 2 | `qubit_hamiltonian(d)` |
| Ansatz **2-QUpCCGSD podado**: 19 excitaciones de qubit como rotaciones de Givens, 465 `cz`, conserva $N$ y $S_z$ | 3 | `build_ansatz()` |
| Búsqueda de parámetros con statevector (permitido), *warm start* entre distancias vecinas y varios arranques | 4 | `params` |
| Energía **reportada**: medición con `SamplerV2` de `qiskit_aer` por grupos QWC, piloto + reparto óptimo de shots ($N_g \propto \sqrt{\operatorname{Var}_g}$) hasta un $\sigma$ objetivo por punto (0,8 mHa en $d \geq 1{,}8$ Å, 3 mHa en el resto); $E$ y $\sigma$ desde los bitstrings, $\sigma$ validada | 4.2 | tabla `vqe` |
| Curva $E(d) \pm \sigma$ vs. CCSD y FCI, $R_e$ (ajuste local) y $D_e$ con incertidumbre (bootstrap + sensibilidad), tabla por punto (energía, $\sigma$, shots, `cz`, ¿le gana a CCSD?) | 5 | gráficos y tablas |
| Confiabilidad en la zona estirada **sin mirar FCI** ($\langle S^2\rangle$, varianza, robustez) y recién después la comparación con CCSD y FCI | 6 | evidencia y veredicto |
| Resumen de resultados, decisiones, qué no funcionó y limitaciones | Resultados clave, 7 | |
| Extras, fuera del desafío central: comparación de ansätze, SQD, geometría cuadrada, ruido (`ruido.py` y `FakeFez`) y esquemas de medición | Extras | |

Reglas que respetamos en todo el notebook: el algoritmo es VQE; **ninguna energía reportada sale del statevector** (el statevector solo se usa para buscar parámetros y validar el ansatz); HF, CCSD y FCI son referencias, nunca resultados.

### Cómo leer y correr el notebook

- **Colab:** abrir el notebook y *Entorno de ejecución → Ejecutar todas*. La primera celda instala las dependencias con las versiones fijadas (las mismas que `requirements.txt`). Si Colab pide reiniciar el entorno después del `%pip`, reiniciar y volver a ejecutar todo. Es autocontenido: no importa nada del repo.
- **Local:** `nix develop -c jupyter lab` (mismas versiones; ver `README` del repo).
- **Tiempo:** con los valores por defecto la corrida completa tarda ~2,5–3 min. El camino del resultado corre entero en cada corrida: referencias (`RUN_REFS = True`), búsqueda de θ* (`RUN_PARAMS = True`) y la medición con `SamplerV2` de #6 (siempre). Los diagnósticos y extras caros usan resultados guardados en el propio notebook (`RUN_ANSATZ_CHECK`, `RUN_GEO`, `RUN_ANSATZ5`, `RUN_ANSATZ_COMPARISON`, `RUN_RUIDO`, `RUN_FEZ`, `RUN_MITIGACION`, en `False`); con `True` cada uno recalcula (todo en `True`: ~10 min). Para una corrida de ~1,5 min, `RUN_REFS = RUN_PARAMS = False`. El extra de `FakeFez` (#12) solo necesita `qiskit-ibm-runtime` con `RUN_FEZ = True` (en Colab lo instala su `%pip`; no está en nixpkgs). Para una prueba rápida, la variable de entorno `QJ_SHOTS_SCALE` (p. ej. `0.05`) escala todos los presupuestos de shots ($\sigma$ crece como $1/\sqrt{\text{escala}}$); los números reportados son con escala 1.
- La celda **0.1** es el contrato que comparten todas las secciones: grilla `D`, geometría, convención de qubits (`HF_BITSTRING = "00110011"`: qubit $p$ = $(p,\alpha)$, qubit $4+p$ = $(p,\beta)$), transpilación y conteo de `cz` (base `cz, sx, x, rz`, qubits en línea) y los nombres de las tablas `refs`, `params`, `vqe`.
- Cada sección termina con un markdown de **decisiones**: qué elegimos, por qué y qué probamos que no funcionó. La sección 7 los resume y analiza las limitaciones.
- Todas las energías son **totales** (incluyen la repulsión nuclear), en Hartree; los errores, en mHa.
- **Resultados clave** ($R_e$, $D_e$, ¿le ganamos a CCSD?, ¿precisión química?, `cz` y shots) en la celda *Resultados clave*, justo antes de la sección 7.

## 0. Instalación

In [ ]:
%pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 pyscf==2.14.0 numpy scipy pandas matplotlib

## 0.1 Contrato compartido

Nombres, firmas y convenciones que usa todo el notebook. Cada sección reemplaza su `NotImplementedError`; cambios a este contrato se avisan en el issue.

In [ ]:
# Contrato compartido: nombres, firmas y convenciones que usa todo el notebook.
# Cada sección implementa su parte; nadie cambia esto sin avisar en el issue correspondiente.
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap

# --- Grilla y geometría -------------------------------------------------------
BASIS = "sto-3g"
D = np.round(np.concatenate([np.arange(0.6, 1.75, 0.1), np.arange(1.8, 3.01, 0.2)]), 2)  # Å, 19 puntos
STRETCHED = 1.8    # Å: desde acá hay que ganarle a CCSD
DISSOCIATED = 3.0  # Å: D_e = E(3,0) - E(R_e)
CHEM_ACC = 1.6e-3  # Ha


def h4_geometry(d: float) -> str:
    """Cadena lineal de 4 H sobre el eje z, separados por d Å (formato PySCF)."""
    return "; ".join(f"H 0 0 {i * d:.6f}" for i in range(4))


# --- Convención de qubits (Jordan-Wigner) --------------------------------------
# Orbitales espaciales MO de RHF, p = 0..3 en orden de energía.
# qubit p     = espín-orbital (p, α)
# qubit 4 + p = espín-orbital (p, β)
# |1> = ocupado. Qiskit es little-endian: el label de un estado se lee q7 ... q0.
N_ORB, N_QUBITS, N_ALPHA, N_BETA = 4, 8, 2, 2
HF_BITSTRING = "00110011"  # q0, q1 (α 0,1) y q4, q5 (β 0,1) ocupados


def alpha_qubit(p: int) -> int:
    return p


def beta_qubit(p: int) -> int:
    return N_ORB + p


# --- Transpilación y conteo de cz (igual para todos) ---------------------------
BASIS_GATES = ["cz", "sx", "x", "rz"]


def transpile_line(qc: QuantumCircuit) -> QuantumCircuit:
    return transpile(qc, basis_gates=BASIS_GATES, coupling_map=CouplingMap.from_line(qc.num_qubits),
                     optimization_level=3, seed_transpiler=0)


def cz_count(qc: QuantumCircuit) -> int:
    return transpile_line(qc).count_ops().get("cz", 0)


# --- Firmas que implementa cada sección ----------------------------------------
# Tablas compartidas (energías totales en Ha, con repulsión nuclear):
#   refs   : DataFrame  d, e_hf, e_ccsd, e_fci           (#1)
#   params : dict       d -> np.ndarray θ*               (#5)
#   vqe    : DataFrame  d, energy, sigma, shots, cz      (#6)

def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'}."""
    raise NotImplementedError


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    """#2. (H electrónico como SparsePauliOp de 8 qubits en la convención de arriba, e_nuc)."""
    raise NotImplementedError


def build_ansatz() -> QuantumCircuit:
    """#4. Circuito parametrizado de 8 qubits que arranca en HF_BITSTRING."""
    raise NotImplementedError


## 1. Referencias clásicas (HF, CCSD, FCI)

Energías de referencia en toda la grilla `D` con PySCF. **FCI** es la respuesta exacta en la base STO-3G, contra la que se mide todo. **CCSD** es el método clásico a superar en la zona estirada. Ninguna de las dos es un resultado del VQE: solo sirven para comparar y para elegir puntos de partida.

Esta celda define `references(d)` y la tabla `refs` (`d, e_hf, e_ccsd, e_fci`), que también se guarda en `results/classical.csv`. **Se calcula en cada corrida** (`RUN_REFS = True`, ~1 min). Para una corrida rápida, `RUN_REFS = False` toma `refs` de `REFS_CACHE` (celda de abajo, a precisión completa, de una corrida de esta misma celda); `references(d)` queda definida igual y la usa la sección 4.2. Usa el mismo procedimiento de RHF que `h4_scf` de la sección 2 (`conv_tol = 1e-11`). Las energías no dependen del signo de los orbitales. Para cualquier cosa que dependa de los orbitales (vector CI, amplitudes), usar `h4_scf(d)`.

**Decisiones**

- Se usa **RHF**, no UHF. Desde ~1,2 Å RHF tiene una inestabilidad RHF → UHF: la solución UHF baja la energía pero rompe la simetría de espín. Queremos un singlete con orbitales restringidos, que es lo que usa el Hamiltoniano de qubits. El chequeo de **estabilidad interna** (RHF → RHF) pasa en todos los puntos: no hay una solución RHF más baja que la encontrada.
- Tolerancias ajustadas (`1e-11` en SCF, `1e-10` en CCSD, `1e-12` en FCI). El código verifica la convergencia de RHF y CCSD en cada punto y que el estado FCI sea singlete (⟨S²⟩ = 0). Todos convergen con el `init_guess` por defecto, así que no hizo falta arrancar desde la densidad del punto vecino.

**Qué muestran los números**

- Cerca del equilibrio (0,6–1,1 Å) CCSD está a menos de 0,01 mHa de FCI.
- Desde ~1,1 Å CCSD **deja de ser variacional** y queda por debajo de FCI. El error es −9,1 mHa a 1,8 Å (el punto más exigente de la zona juzgada), llega a −30,6 mHa a 2,6 Å y vuelve a −23,0 mHa a 3,0 Å.
- Para ganarle a CCSD hace falta `|E_VQE − E_FCI| + σ < 9,1 mHa` a 1,8 Å y < 18 mHa desde 2,0 Å.

In [ ]:
# Referencias guardadas (#1), para una corrida rápida: la celda siguiente las usa solo con RUN_REFS = False
# (por defecto recalcula con PySCF, ~1 min). Salieron de esa misma celda con True, a precisión completa (17 cifras);
# para regenerarlas: poner True, correr y pegar acá results/classical.csv.
REFS_CACHE = {
    "origen": "corrida completa del 2026-11-21 con RUN_* = True, SHOTS_SCALE = 1, qiskit 2.5.2, qiskit-aer 0.17.2, pyscf 2.14.0",
    "refs": """\
d,e_hf,e_ccsd,e_fci
0.6,-1.9295642484520141,-1.9601888879027298,-1.9601936448939266
0.7,-2.0691974228038688,-2.1069926724685675,-2.1069969150952894
0.8,-2.121386755870223,-2.167555587074489,-2.167560544134051
0.9,-2.124259738972778,-2.180309410719456,-2.1803166143238593
1.0,-2.0985459369979065,-2.166379520465788,-2.1663874486347607
1.1,-2.056024905835257,-2.137975448072445,-2.137970526843626
1.2,-2.0038674831266956,-2.102671142473027,-2.102608480955422
1.3,-1.9467416264563098,-2.0654610501215,-2.065228963271289
1.4,-1.8877903045162745,-2.0297069611005707,-2.0290704935919592
1.5,-1.829137412443024,-1.9976241531195098,-1.9961503255188089
1.6,-1.7722031019330002,-1.9705722218907205,-1.967560309920373
1.7,-1.717918015075727,-1.9492120975576395,-1.9436920386881202
1.8,-1.6668671614438375,-1.933538275685437,-1.9244306381367224
2.0,-1.5756164767018657,-1.916086099892169,-1.8977806459898732
2.2,-1.4991456250348574,-1.9091203134659334,-1.882712637431846
2.4,-1.4364585399896872,-1.9052115883073035,-1.8746515824858951
2.6,-1.3858241148118247,-1.9010992817458612,-1.8704727364209426
2.8,-1.3453650770714778,-1.8960243486635733,-1.8683497076851219
3.0,-1.3133117862215051,-1.8903043733010625,-1.8672913724013764
""",
}

In [ ]:
# 1. Referencias clásicas: HF, CCSD y FCI en toda la grilla D (#1).
# Solo sirven para comparar y elegir puntos de partida; nunca son resultados.
# Las energías no dependen del signo de los orbitales: los orbitales "oficiales" (signo fijo,
# los del Hamiltoniano) son los de h4_scf(d) en la sección 2. Acá se usa el mismo procedimiento.
import io
import os
import matplotlib.pyplot as plt
from pyscf import cc, fci, gto, scf

RUN_REFS = True  # True (por defecto): calcular con PySCF (~1 min). False: usar la tabla guardada en REFS_CACHE (celda de arriba)


def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'} en Ha (energías totales)."""
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    # Al estirar, el SCF podría caer en un punto silla. Si la solución tiene una inestabilidad
    # interna (RHF -> RHF), seguimos la dirección inestable y reconvergemos (en esta grilla no pasa).
    for _ in range(5):
        mo, _, stable, _ = mf.stability(return_status=True)
        if stable:
            break
        mf.kernel(dm0=mf.make_rdm1(mo, mf.mo_occ))
    assert mf.converged, f"RHF no convergió en d={d}"

    ccsd = cc.CCSD(mf)
    ccsd.conv_tol, ccsd.max_cycle = 1e-10, 500
    ccsd.kernel()
    assert ccsd.converged, f"CCSD no convergió en d={d}"

    solver = fci.FCI(mf)
    solver.conv_tol = 1e-12
    e_fci, ci = solver.kernel()
    s2, _ = solver.spin_square(ci, N_ORB, (N_ALPHA, N_BETA))
    assert abs(s2) < 1e-6, f"FCI no es singlete en d={d}: <S²>={s2:.3g}"
    return {"d": float(d), "e_hf": mf.e_tot, "e_ccsd": ccsd.e_tot, "e_fci": e_fci}


if RUN_REFS:
    refs = pd.DataFrame([references(d) for d in D])
else:
    refs = pd.read_csv(io.StringIO(REFS_CACHE["refs"]), float_precision="round_trip")
    assert np.allclose(refs.d, D), "REFS_CACHE es de otra grilla: correr con RUN_REFS = True"
    print(f"Referencias guardadas ({REFS_CACHE['origen']}). Para recalcular: RUN_REFS = True.")

# Chequeos de cordura: FCI es exacto en la base (variacional), CCSD es casi exacto cerca del equilibrio.
assert (refs.e_fci <= refs.e_hf + 1e-10).all()
near_eq = refs[(refs.d >= 0.8) & (refs.d <= 1.0)]
assert ((near_eq.e_ccsd - near_eq.e_fci).abs() < CHEM_ACC).all()

os.makedirs("results", exist_ok=True)
refs.to_csv("results/classical.csv", index=False)

err_ccsd = 1e3 * (refs.e_ccsd - refs.e_fci)  # mHa, con signo
print(refs.assign(ccsd_menos_fci_mHa=err_ccsd).to_string(
    index=False, formatters={"d": "{:.2f}".format, "ccsd_menos_fci_mHa": "{:+.3f}".format}, float_format="%.6f"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.plot(refs.d, refs.e_hf, "s-", ms=4, label="RHF")
ax1.plot(refs.d, refs.e_ccsd, "^-", ms=4, label="CCSD")
ax1.plot(refs.d, refs.e_fci, "o-", ms=4, color="k", label="FCI (exacta)")
ax1.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0, label=f"zona estirada (d ≥ {STRETCHED} Å)")
ax1.set(xlabel="d (Å)", ylabel="E (Ha)", title="Curva de energía de H₄ (STO-3G)")
ax1.legend()

ax2.semilogy(refs.d, err_ccsd.abs(), "^-", ms=4, color="C1", label="|E_CCSD − E_FCI|")
ax2.axhline(1e3 * CHEM_ACC, ls="--", color="C2", label="precisión química (1,6 mHa)")
ax2.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0)
ax2.set(xlabel="d (Å)", ylabel="error (mHa)", title="Error de CCSD: lo que hay que superar")
ax2.legend()
plt.tight_layout()
plt.show()

## 2. Hamiltoniano de qubits

In [ ]:
import copy
from functools import lru_cache

from pyscf import ao2mo, fci, gto, scf
from pyscf.fci import cistring


def _so(p: int, spin: int) -> int:
    return alpha_qubit(p) if spin == 0 else beta_qubit(p)


@lru_cache(maxsize=None)
def _jw_ladder(n_qubits: int) -> tuple[tuple[SparsePauliOp, ...], tuple[SparsePauliOp, ...]]:
    annihilation = []
    for j in range(n_qubits):
        z_string = list(range(j))
        annihilation.append(SparsePauliOp.from_sparse_list(
            [("Z" * j + "X", z_string + [j], 0.5), ("Z" * j + "Y", z_string + [j], 0.5j)],
            num_qubits=n_qubits))
    creation = [a.adjoint() for a in annihilation]
    return tuple(creation), tuple(annihilation)


def jordan_wigner(h1_so: np.ndarray, h2_so: np.ndarray | None = None, atol: float = 1e-12,
                  hermitian: bool = True) -> SparsePauliOp:
    n = h1_so.shape[0]
    cdag, a = _jw_ladder(n)
    terms = [SparsePauliOp("I" * n, 0.0)]
    for p, q in np.argwhere(np.abs(h1_so) > atol):
        terms.append(h1_so[p, q] * cdag[p] @ a[q])
    if h2_so is not None:
        for p, q, r, s in np.argwhere(np.abs(h2_so) > atol):
            if p == r or q == s:
                continue
            terms.append(0.5 * h2_so[p, q, r, s] * cdag[p] @ cdag[r] @ a[s] @ a[q])
    op = SparsePauliOp.sum(terms).simplify(atol=atol)
    if not hermitian:
        return op
    if np.max(np.abs(op.coeffs.imag), initial=0.0) > 1e-10:
        raise ValueError("Operador no hermítico (coeficientes complejos): revisar índices o usar hermitian=False")
    return SparsePauliOp(op.paulis, op.coeffs.real)


def spatial_to_spin_orbitals(h1: np.ndarray, eri: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n_orb = h1.shape[0]
    n_so = 2 * n_orb
    h1_so = np.zeros((n_so, n_so))
    h2_so = np.zeros((n_so, n_so, n_so, n_so))
    for s1 in (0, 1):
        i = [_so(p, s1) for p in range(n_orb)]
        h1_so[np.ix_(i, i)] = h1
        for s2 in (0, 1):
            k = [_so(p, s2) for p in range(n_orb)]
            h2_so[np.ix_(i, i, k, k)] = eri
    return h1_so, h2_so


def _distance_key(d: float) -> float:
    return round(float(d), 6)


@lru_cache(maxsize=None)
def _h4_scf_cached(d: float) -> scf.hf.RHF:
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    if not mf.converged:
        raise RuntimeError(f"RHF no convergió en d = {d}")
    c = mf.mo_coeff
    significant = np.abs(c) > 1e-4
    if not significant.any(axis=0).all():
        raise RuntimeError(f"Orbital sin coeficientes > 1e-4 en d = {d}: no se puede fijar el signo")
    first = np.argmax(significant, axis=0)
    mf.mo_coeff = c * np.sign(c[first, np.arange(c.shape[1])])
    return mf


def h4_scf(d: float) -> scf.hf.RHF:
    mf = copy.copy(_h4_scf_cached(_distance_key(d)))
    for attr in ("mo_coeff", "mo_energy", "mo_occ"):
        setattr(mf, attr, getattr(mf, attr).copy())
    return mf


@lru_cache(maxsize=None)
def _qubit_hamiltonian_cached(d: float) -> tuple[SparsePauliOp, float]:
    mf = _h4_scf_cached(d)
    mol, c = mf.mol, mf.mo_coeff
    h1 = c.T @ mf.get_hcore() @ c
    eri = ao2mo.restore(1, ao2mo.full(mol, c), c.shape[1])
    h1_so, h2_so = spatial_to_spin_orbitals(h1, eri)
    return jordan_wigner(h1_so, h2_so), mol.energy_nuc()


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    op, e_nuc = _qubit_hamiltonian_cached(_distance_key(d))
    return op.copy(), e_nuc


_STRINGS = cistring.make_strings(range(N_ORB), N_ALPHA)
SECTOR = np.array([int(a) | (int(b) << N_ORB) for a in _STRINGS for b in _STRINGS])


def sector_ground_energy(op: SparsePauliOp) -> float:
    m = op.to_matrix()
    return np.linalg.eigvalsh(m[np.ix_(SECTOR, SECTOR)])[0].real


def hf_energy_from_op(op: SparsePauliOp) -> float:
    i = int(HF_BITSTRING, 2)
    return op.to_matrix()[i, i].real


def validate_hamiltonian(d: float) -> dict:
    op, e_nuc = qubit_hamiltonian(d)
    mf = h4_scf(d)
    e_fci = fci.FCI(mf).kernel()[0]
    e_hf_op = hf_energy_from_op(op) + e_nuc
    e_gs_op = sector_ground_energy(op) + e_nuc
    return {"d": float(d), "n_paulis": len(op), "e_hf": mf.e_tot, "e_fci": e_fci,
            "err_hf": abs(e_hf_op - mf.e_tot), "err_fci": abs(e_gs_op - e_fci)}


ham_check = pd.DataFrame([validate_hamiltonian(d) for d in D])
assert (ham_check[["err_hf", "err_fci"]] < 1e-8).all().all(), ham_check
ham_check


### Decisiones (#2)

**Firma.** `qubit_hamiltonian(d) -> (op, e_nuc)`, como dice el contrato 0.1 ([ADR-0001](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0001-firma-qubit-hamiltonian.md)). `op` es solo la parte electrónica: energía total = $\langle op \rangle + e_\text{nuc}$. El estado HF es la constante `HF_BITSTRING = "00110011"` porque no depende de $d$.

**Orden de qubits.** Qubit $p$ = espín-orbital $(p, \alpha)$, qubit $4+p$ = $(p, \beta)$, con $p = 0..3$ los orbitales RHF ordenados por energía. `HF_BITSTRING` deja ocupados q0, q1, q4, q5 (Qiskit es little-endian: el label se lee q7 … q0).

**Hamiltoniano.** Integrales en base MO de RHF: $h_{pq} = C^T h^\text{core} C$ y $(pq|rs)$ con `ao2mo` (notación química). En espín-orbitales:

$$\hat H = \sum_{PQ} h_{PQ}\, a_P^\dagger a_Q + \tfrac12 \sum_{PQRS} (PQ|RS)\, a_P^\dagger a_R^\dagger a_S a_Q,$$

con el espín conservado dentro de cada par $(P,Q)$ y $(R,S)$. Jordan-Wigner a mano: $a_j = Z_0 \cdots Z_{j-1}\,(X_j + iY_j)/2$ ([ADR-0002](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0002-jordan-wigner-a-mano.md)). `jordan_wigner(h1_so, h2_so)` es genérica: sirve también para $N$, $S_z$, $S^2$ (#8) y para el tapering (#14). Por defecto exige un operador hermítico (coeficientes reales) y falla si no lo es, porque eso delata un error de índices; para operadores no hermíticos como $S^+ = \sum_p a^\dagger_{p\alpha} a_{p\beta}$ usar `hermitian=False`, que devuelve los coeficientes complejos.

**Signo de los orbitales.** PySCF puede devolver un orbital con el signo invertido entre una distancia y la siguiente. El espectro no cambia, pero los coeficientes de $\hat H$ sí, y eso rompe el *warm start* de #5. Se fija el signo para que el primer coeficiente de cada orbital con $|c| > 10^{-4}$ sea positivo (si algún orbital no tiene ninguno, se lanza un error en vez de anularlo) ([ADR-0003](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0003-signo-orbitales-y-cache.md)). Verificado: ningún término de Pauli cambia de signo de golpe entre distancias vecinas (los únicos cambios, cerca de 0,67 Å, son coeficientes que cruzan cero de forma suave).

**Orbitales RHF en toda la grilla.** RHF es inestable hacia UHF desde ~1,1 Å (`docs/analysis/state-of-the-art.md`; verificado con `mf.stability(external=True)`), pero se mantienen los orbitales RHF: solo definen la base (FCI no cambia), el estado de referencia sigue siendo un singlete y ⟨S²⟩ = 0 sigue sirviendo como diagnóstico en la sección 6 (#8). Con orbitales UHF la referencia quedaría contaminada de espín.

**Caché.** RHF y el Hamiltoniano se calculan una sola vez por distancia (`lru_cache` en funciones privadas). Las funciones públicas devuelven **copias**: `qubit_hamiltonian` devuelve `op.copy()` y `h4_scf` una copia del `mf` con sus propios `mo_coeff`, `mo_energy` y `mo_occ`, así que modificar lo devuelto (o correr `mf.kernel()` / `mf.stability()` sobre él) no altera el caché ni desincroniza orbitales y Hamiltoniano. La clave del caché es `d` redondeada a 6 decimales: `qubit_hamiltonian(0.1 * 7)` reutiliza el de `0.7`.

**Validación** (tabla `ham_check`, las 19 distancias):

| Prueba | Criterio | Resultado |
|---|---|---|
| $\langle \text{HF}\vert op \vert \text{HF}\rangle + e_\text{nuc}$ vs. $E_\text{HF}$ de PySCF | < 1e-8 Ha | máx. ≈ 2e-14 Ha |
| Mínimo de `op` en el sector $N=4$, $S_z=0$ (36 estados) + $e_\text{nuc}$ vs. FCI | < 1e-8 Ha | máx. ≈ 2e-14 Ha |
| Términos de Pauli | — | 185 (184 más la identidad) en todas las distancias, mismo conjunto de Paulis |

`SECTOR` son los 36 índices de la base computacional con 2 α y 2 β, **en el orden `ci[ia, ib]` de PySCF** (`cistring.make_strings`): un vector CI `ci` de PySCF corresponde a `psi[SECTOR] = ci.ravel()` (signos verificados contra `fci.direct_spin1.energy` en estados al azar, error < 3e-14 Ha). Es la misma definición que usa la sección 3 (#4).

La validación contra FCI se hace en el sector $N=4$, $S_z=0$: el mínimo del espectro completo (256 estados) puede tener otro número de electrones. FCI se calcula acá mismo con los orbitales de `h4_scf`; cuando esté `refs` (#1) se puede comparar también contra `refs.e_fci`.

**Qué no funcionó.** La primera versión armaba los productos con `A.compose(B)`, que en Qiskit es la matriz $B \cdot A$, no $A \cdot B$: quedaba $a_Q a_P^\dagger$ en vez de $a_P^\dagger a_Q$, o sea el Hamiltoniano con partículas y huecos intercambiados. Con 4 electrones en 8 espín-orbitales (medio llenado) ese operador tiene **el mismo espectro en el sector** $N=4$, así que la prueba contra FCI pasaba igual; solo la prueba de HF lo detectó (el estado HF aparecía en `11001100`). Se usa `@` (`A @ B` = $A \cdot B$). Por eso las dos pruebas son necesarias.

## 3. Ansatz

### Decisión (#4)

Elegimos un **2-QUpCCGSD podado**: excitaciones de pares $(p_\uparrow p_\downarrow)\to(q_\uparrow q_\downarrow)$ y simples $p\to q$ (mismo ángulo en α y β), como excitaciones de **qubit** (sin cadenas Z), en dos capas, con 19 de las 24 excitaciones.

Comparación con statevector (error vs. FCI en mHa a 0,9 / 1,8 / 3,0 Å; `cz` tras transpilar a `cz, sx, x, rz` en línea):

| Ansatz | `cz` | Parámetros | Error (mHa) |
|---|---|---|---|
| UCCSD (fermiónico) | 1277 | 26 | 0,03 / 3,3 / 0,46 |
| UCCSD de qubits | 1247 | 26 | 0,003 / 3,0 / 0,21 |
| 1-UpCCGSD | 474 | 12 | 14 / 11 / 0,29 |
| 2-UpCCGSD (fermiónico) | 982 | 24 | exacto |
| 2-QUpCCGSD (Givens) | 590 | 24 | exacto |
| **2-QUpCCGSD podado (elegido)** | **465** | **19** | **≤ 0,03 en toda la grilla** |
| 2-QUpCCGSD podado más agresivo | 411 | 17 | ≤ 0,5 (en 6 distancias) |
| QEB-ADAPT-VQE (un circuito por distancia) | 461–573 | 16–19 | ≤ 0,01 |
| Hardware-efficient (RY·RZ + CZ en línea), 6 capas | 42 | 112 | 53 / 23 / 0,67 |
| Hardware-efficient, 8 capas (optimizando fidelidad con FCI) | 56 | 144 | 1 − F = 1e-5 / 1,5e-3 / 2e-4 |
| *CCSD (referencia)* | – | – | 0,007 / −9,1 / −23 |

Por qué:

- **Llega a FCI en la zona estirada.** UCCSD falla ahí (3,3 mHa a 1,8 Å, fuera de precisión química) por la misma razón que CCSD: el estado se vuelve multirreferencia. Las excitaciones generalizadas de pares sí lo capturan.
- **Conserva N = 4 y S_z = 0**: cada excitación mueve electrones dentro del mismo bloque de espín. Lo verificamos numéricamente abajo, y sirve para el argumento de confiabilidad (#8).
- **Menos `cz`**: cada excitación es una rotación de Givens entre dos estados de la base, implementada con un multiplexor de RY en código Gray (14 CX por doble) en lugar de `PauliEvolutionGate` (837 → 590 `cz`). Después podamos de a una las excitaciones de menor ángulo mientras el error siguiera < 0,1 mHa en 6 distancias (590 → 465 `cz`).

Qué no funcionó o descartamos:

- UCCSD y 1-UpCCGSD: no llegan a precisión química en la zona estirada.
- Sintetizar las excitaciones con `PauliEvolutionGate`: correcto pero caro en `cz`.
- **Hardware-efficient** (RY·RZ por qubit + cadena de CZ): muy barato en `cz` (42–56), pero optimizando la energía queda trabado lejos de FCI (53 mHa a 0,9 Å y 23 mHa a 1,8 Å con 6 capas y 6 arranques). Con 8 capas *puede* representar el estado (fidelidad 1 − 1,5e-3 a 1,8 Å si se optimiza contra el vector FCI, cosa que en un VQE real no se puede), así que el problema es el paisaje de optimización, no la expresividad. Además no conserva N ni S_z (fuga de hasta 0,4 %), lo que complica #8. Queda como opción para los extras con ruido si alguien encuentra cómo optimizarlo.
- **QEB-ADAPT-VQE** (sugerido en #4): con un pool de 90 excitaciones de qubit converge a ≤ 0,01 mHa con 16–19 excitaciones, pero necesita 461–573 `cz` (sus dobles αβ genéricas tocan qubits lejanos en la línea) y un circuito distinto por distancia. Incluso cortado justo al llegar a precisión química necesita 467 `cz` a 1,8 Å. No mejora al ansatz elegido.
- La versión de 17 excitaciones (411 `cz`) queda como alternativa si los extras con ruido necesitan menos compuertas, a costa de errores de hasta 0,5 mHa.

Limitaciones:

- **El paisaje de energía tiene mínimos locales.** Con un solo arranque, algunas distancias quedan a 1 mHa o hasta decenas de mHa; con 30 arranques el mejor llega a ≤ 0,003 mHa. Con los 10 arranques de la validación de abajo, el peor punto queda entre 0,04 y 0,13 mHa según la corrida (diferencias numéricas mínimas cambian a qué mínimo cae cada arranque). La optimización (#5) tiene que usar multi-start.
- **Contaminación de espín.** Conservar N y S_z no garantiza S² = 0: las excitaciones de qubit no están adaptadas al espín, así que si la optimización cae en otro mínimo el estado puede mezclarse con el triplete, que en la zona estirada está casi degenerado (brecha de 0,5 mHa a 3,0 Å, así que la energía no lo delata). Con el Hamiltoniano de #2 (orbitales con signo fijo) la validación llega al singlete: ⟨S²⟩ ≤ 3e-7. Con un stand-in previo cuyos orbitales no tenían el signo fijo, el warm start entre distancias caía en otros mínimos y vimos ⟨S²⟩ de hasta 3e-2 (~1,5 % de triplete) desde 2,0 Å. Por eso la tabla de abajo reporta ⟨S²⟩ y la brecha, #8 tiene que reportarlo, y #5 puede asegurarlo con una penalización λ⟨S²⟩.
- **465 `cz` es mucho para hardware ruidoso.** Con 0,1 % de error por `cz`, la probabilidad de al menos una falla es 1 − 0,999⁴⁶⁵ ≈ 37 %. Para el desafío central (sin ruido) no importa; para los extras con ruido conviene la versión podada más agresiva o menos qubits (#14).

La validación de abajo usa el Hamiltoniano de #2 (`qubit_hamiltonian`) restringido a los 36 estados con N = 4 y S_z = 0, y toma como FCI su autovalor mínimo. Mientras #2 no estaba, la exploración usó un stand-in armado con PySCF (misma convención de signos, verificado en #4); con el Hamiltoniano de #2 los resultados son los mismos.


**Resultados guardados.** El barrido de la validación (10 arranques de BFGS por distancia) tarda ~1 min, así que por defecto la tabla `ansatz_check` sale de `ANSATZ_CHECK_CACHE` (de una corrida de esa celda con `RUN_ANSATZ_CHECK = True`); `sector_ansatz`, `sector_hamiltonian` y los chequeos de circuito vs. sector se ejecutan igual. Para recalcular: `RUN_ANSATZ_CHECK = True`.

In [ ]:
# Ansatz elegido (#4): 2-QUpCCGSD podado, 19 excitaciones de qubit como rotaciones de Givens.
import itertools
from qiskit.circuit import ParameterVector


def givens_single(qc: QuantumCircuit, t, i: int, k: int) -> None:
    """exp(θ(Q†_k Q_i − h.c.)): rota |q_i=1, q_k=0⟩ hacia |q_i=0, q_k=1⟩."""
    qc.cx(k, i)
    qc.cry(2 * t, i, k)
    qc.cx(k, i)


def _ucry(qc: QuantumCircuit, t, controls: list, target: int, pattern: tuple) -> None:
    """RY(2θ) en target solo cuando controls == pattern. Multiplexor Gray: 2^n CX."""
    n = 2 ** len(controls)
    gray = [m ^ (m >> 1) for m in range(n)]
    idx = sum(b << c for c, b in enumerate(pattern))
    for m in range(n):
        qc.ry(2 * t * (-1) ** bin(idx & gray[m]).count("1") / n, target)
        qc.cx(controls[(gray[m] ^ gray[(m + 1) % n]).bit_length() - 1], target)


def givens_double(qc: QuantumCircuit, t, i: int, j: int, k: int, l: int) -> None:
    """exp(θ(Q†_k Q†_l Q_j Q_i − h.c.)): rota |ij=11, kl=00⟩ hacia |ij=00, kl=11⟩ (14 CX)."""
    for m in (i, j, l):
        qc.cx(k, m)
    _ucry(qc, t, [i, j, l], k, (1, 1, 0))
    for m in (l, j, i):
        qc.cx(k, m)


# (capa, tipo, p, q). "D": par (p↑, p↓) → (q↑, q↓). "S": simple p → q, en α y en β con el mismo θ.
ANSATZ_OPS = [
    (0, "D", 0, 2), (0, "D", 1, 2), (0, "D", 1, 3), (0, "D", 2, 3),
    (0, "S", 0, 1), (0, "S", 0, 3), (0, "S", 1, 2), (0, "S", 1, 3), (0, "S", 2, 3),
    (1, "D", 0, 1), (1, "D", 0, 3), (1, "D", 1, 2), (1, "D", 1, 3), (1, "D", 2, 3),
    (1, "S", 0, 2), (1, "S", 0, 3), (1, "S", 1, 2), (1, "S", 1, 3), (1, "S", 2, 3),
]


def build_ansatz() -> QuantumCircuit:
    """#4. Circuito de 8 qubits que arranca en HF_BITSTRING; un parámetro por excitación."""
    th = ParameterVector("θ", len(ANSATZ_OPS))
    qc = QuantumCircuit(N_QUBITS, name="QUpCCGSD-19")
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for t, (_, kind, p, q) in zip(th, ANSATZ_OPS):
        if kind == "D":
            givens_double(qc, t, alpha_qubit(p), beta_qubit(p), alpha_qubit(q), beta_qubit(q))
        else:
            givens_single(qc, t, alpha_qubit(p), alpha_qubit(q))
            givens_single(qc, t, beta_qubit(p), beta_qubit(q))
    return qc


ansatz = build_ansatz()
print(f"{ansatz.name}: {ansatz.num_parameters} parámetros, {cz_count(ansatz)} cz (transpilado a cz/sx/x/rz en línea)")


In [ ]:
# Resultados guardados de la validación del ansatz (#4). Recalcularlos tarda ~1 min, así que la celda siguiente
# los usa por defecto (RUN_ANSATZ_CHECK = False). Salieron de esa misma celda con True;
# para regenerarlos: poner True, correr y pegar acá results/ansatz_check.csv.
ANSATZ_CHECK_CACHE = {
    "origen": "corrida completa del 2026-11-21 con RUN_* = True, SHOTS_SCALE = 1, qiskit 2.5.2, qiskit-aer 0.17.2, pyscf 2.14.0",
    "ansatz_check": """\
d,e_ansatz,e_fci,err_mHa,S2,gap_mHa
0.6,-1.960190887,-1.960193645,0.00275789,2.7077e-07,576.921
0.7,-2.106996209,-2.106996915,0.000706403,6.34904e-12,452.526
0.8,-2.16756011,-2.167560544,0.000434433,9.87256e-09,359.648
0.9,-2.180316614,-2.180316614,1.33227e-12,3.66491e-17,288.706
1,-2.166385619,-2.166387449,0.00183001,4.7863e-08,232.63
1.1,-2.13794541,-2.137970527,0.0251172,3.258e-07,186.941
1.2,-2.102608325,-2.102608481,0.000155959,6.21512e-08,149.092
1.3,-2.065228963,-2.065228963,-1.33227e-12,3.87544e-21,117.656
1.4,-2.029070494,-2.029070494,-1.77636e-12,1.30773e-20,91.7246
1.5,-1.996150326,-1.996150326,-1.77636e-12,1.26912e-15,70.5918
1.6,-1.96756031,-1.96756031,-4.44089e-13,2.46152e-18,53.6259
1.7,-1.943692039,-1.943692039,-2.88658e-12,4.48929e-21,40.2277
1.8,-1.924430638,-1.924430638,-8.88178e-13,2.88081e-21,29.825
2,-1.897780646,-1.897780646,0,5.21854e-17,15.905
2.2,-1.882712637,-1.882712637,1.55431e-12,3.23582e-15,8.21447
2.4,-1.874651582,-1.874651582,2.22045e-13,2.66018e-16,4.13769
2.6,-1.870472736,-1.870472736,-6.66134e-13,5.01704e-14,2.03701
2.8,-1.868349708,-1.868349708,-1.33227e-12,6.6575e-20,0.978889
3,-1.867291372,-1.867291372,-1.33227e-12,1.63894e-15,0.458232
""",
}

In [ ]:
# Validación del ansatz en toda la grilla, con statevector (permitido para elegir parámetros y ansatz).
# Usa el Hamiltoniano de #2 (`qubit_hamiltonian`) restringido al sector N=4, S_z=0 (36 estados):
# el ansatz conserva N y S_z, así que simular ahí es exacto y mucho más rápido que con 256 amplitudes.
import io

from pyscf.fci import cistring, spin_op
from scipy.optimize import minimize
from qiskit.quantum_info import Statevector

_STRINGS = cistring.make_strings(range(N_ORB), N_ALPHA)
SECTOR = np.array([int(a) | (int(b) << N_ORB) for a in _STRINGS for b in _STRINGS])  # índices de qubits


def sector_hamiltonian(d: float) -> tuple[np.ndarray, float]:
    """H total (con e_nuc) de #2 en el sector de 36 estados, y su autovalor mínimo (= E_FCI)."""
    op, e_nuc = qubit_hamiltonian(d)
    M = op.to_matrix(sparse=True)[SECTOR][:, SECTOR].toarray()
    assert np.abs(M.imag).max() < 1e-12
    H = M.real + e_nuc * np.eye(len(SECTOR))
    return H, np.linalg.eigvalsh(H)[0]


def _qubit_excitation(create: tuple, annihilate: tuple) -> np.ndarray:
    """K = T − T† en el sector, con T = Q†...Q... (operadores de qubit, sin cadenas Z)."""
    def q(j, dag):
        lab = lambda p: "I" * (N_QUBITS - j - 1) + p + "I" * j
        return SparsePauliOp([lab("X"), lab("Y")], [0.5, -0.5j if dag else 0.5j])
    T = SparsePauliOp("I" * N_QUBITS)
    for c in create:
        T = T @ q(c, True)
    for a in annihilate:
        T = T @ q(a, False)
    M = (T - T.adjoint()).simplify().to_matrix(sparse=True)[SECTOR][:, SECTOR].toarray()
    return M.real


class SectorAnsatz:
    """El mismo ansatz simulado en el sector de 36 estados, con gradiente adjunto exacto (rápido)."""

    def __init__(self, ops):
        self.groups = []
        for _, kind, p, q in ops:
            if kind == "D":
                self.groups.append([_qubit_excitation((alpha_qubit(q), beta_qubit(q)), (beta_qubit(p), alpha_qubit(p)))])
            else:
                self.groups.append([_qubit_excitation((alpha_qubit(q),), (alpha_qubit(p),)),
                                    _qubit_excitation((beta_qubit(q),), (beta_qubit(p),))])
        self.n = len(self.groups)
        self.hf = (SECTOR == int(HF_BITSTRING, 2)).astype(float)

    def _U(self, k, t):  # cada generador cumple K³ = −K, así que exp(tK) = 1 + sin(t)K + (1 − cos(t))K²
        I = np.eye(len(SECTOR))
        U = I
        for K in self.groups[k]:
            U = (I + np.sin(t) * K + (1 - np.cos(t)) * K @ K) @ U
        return U

    def state(self, x):
        v = self.hf
        for k in range(self.n):
            v = self._U(k, x[k]) @ v
        return v

    def energy_grad(self, x, H):
        Us = [self._U(k, x[k]) for k in range(self.n)]
        v = self.hf
        for U in Us:
            v = U @ v
        lam = H @ v
        e, g = v @ lam, np.empty(self.n)
        for k in reversed(range(self.n)):
            g[k] = 2 * lam @ (sum(self.groups[k]) @ v)
            v, lam = Us[k].T @ v, Us[k].T @ lam
        return e, g


sector_ansatz = SectorAnsatz(ANSATZ_OPS)

# El circuito real y la simulación en el sector tienen que dar el mismo estado.
_x = np.random.default_rng(0).normal(size=ansatz.num_parameters)
_full = Statevector(ansatz.assign_parameters(_x)).data
assert np.isclose(np.linalg.norm(_full[SECTOR]), 1), "el ansatz se sale del sector N=4, S_z=0"
assert np.isclose(abs(np.vdot(_full[SECTOR], sector_ansatz.state(_x))), 1), "circuito ≠ simulación en el sector"

RUN_ANSATZ_CHECK = False  # True: recalcular el barrido (~1 min). False: usar ANSATZ_CHECK_CACHE (celda de arriba)
if RUN_ANSATZ_CHECK:
    # El paisaje de energía tiene mínimos locales (algunos arranques quedan a decenas de mHa): por eso
    # probamos varios arranques por distancia (cero, el óptimo de la distancia anterior y 8 al azar).
    rng = np.random.default_rng(0)
    rows, x0 = [], np.zeros(sector_ansatz.n)
    for d in D:
        H, e_fci = sector_hamiltonian(d)
        starts = [np.zeros(sector_ansatz.n), x0] + [rng.normal(scale=0.7, size=sector_ansatz.n) for _ in range(8)]
        best = min((minimize(sector_ansatz.energy_grad, s, args=(H,), jac=True, method="BFGS", options={"gtol": 1e-10})
                    for s in starts), key=lambda r: r.fun)
        x0 = best.x
        s2 = spin_op.spin_square0(sector_ansatz.state(best.x).reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0]
        gap = np.diff(np.linalg.eigvalsh(H)[:2])[0]
        rows.append({"d": d, "e_ansatz": best.fun, "e_fci": e_fci, "err_mHa": (best.fun - e_fci) * 1e3,
                     "S2": s2, "gap_mHa": gap * 1e3})
    ansatz_check = pd.DataFrame(rows)
    os.makedirs("results", exist_ok=True)
    ansatz_check.to_csv("results/ansatz_check.csv", index=False)
else:
    ansatz_check = pd.read_csv(io.StringIO(ANSATZ_CHECK_CACHE["ansatz_check"]))
    print(f"Resultados guardados ({ANSATZ_CHECK_CACHE['origen']}). Para recalcular: RUN_ANSATZ_CHECK = True.")
print(f"error máximo vs. FCI en la grilla: {ansatz_check.err_mHa.abs().max():.3f} mHa "
      f"(zona estirada: {ansatz_check[ansatz_check.d >= STRETCHED].err_mHa.abs().max():.3f} mHa); "
      f"⟨S²⟩ máximo: {ansatz_check.S2.max():.1e} (singlete puro = 0, triplete = 2)")
ansatz_check

## 4. VQE

In [ ]:
# θ* y diagnósticos guardados de la búsqueda de parámetros (#5, que permite "cachear params y documentarlo").
# La celda siguiente repite la búsqueda por defecto (RUN_PARAMS = True, ~1 min); estos θ* se usan solo con
# RUN_PARAMS = False, para una corrida rápida.
# Salieron de esa misma celda con True; params a precisión completa (17 cifras): con las mismas semillas, la
# medición de #6 da exactamente las mismas energías. Para regenerarlos: poner True, correr y pegar acá
# results/params.csv, results/opt.csv, results/starts_log.csv y results/multistart.csv.
PARAMS_CACHE = {
    "origen": "corrida completa del 2026-11-21 con RUN_* = True, SHOTS_SCALE = 1, qiskit 2.5.2, qiskit-aer 0.17.2, pyscf 2.14.0",
    "params": """\
d,theta_0,theta_1,theta_2,theta_3,theta_4,theta_5,theta_6,theta_7,theta_8,theta_9,theta_10,theta_11,theta_12,theta_13,theta_14,theta_15,theta_16,theta_17,theta_18
0.6,-0.06240501355565214,1.4299001321715783,0.4294200492487267,-1.6265889533956994,0.18909690291956988,-0.5819240388602359,-0.7853967687068858,1.5707954940584816,-0.7853979785183036,-1.7781383463857614,-0.07039176940794516,0.14937814843148137,0.0008421280367860563,2.825376398358706,0.16758285578918075,3.5990731272906793e-07,6.516267826129747e-07,0.11286466465861736,-2.220592076520084e-06
0.7,-0.07877408187650164,1.3916945544978758,0.47528614397067215,-1.6479711490212001,0.1978158118378639,-0.6476729796989932,-0.7853977456779397,1.5707960878970435,-0.7853982319112083,-1.7932086036962362,-0.07719377817183797,0.1922987821066018,0.006139295448808966,2.766133306464724,0.1703204822903588,9.551989760663149e-08,1.9534942199389753e-07,0.13184377908187597,-4.764016069055819e-07
0.8,-0.09776540313688603,1.34575997040173,0.5125095075141479,-1.6736308824058974,0.20697260274650592,-0.7080305895473169,-0.7854005868768155,1.5707985527287909,-0.7853973410324039,-1.7954588524883095,-0.08464974680049737,0.24300152143159204,0.014718351387103545,2.715518612462179,0.17300022849874958,-5.329163432071767e-07,-1.7612210505975505e-06,0.15117148592888055,2.2922696587351788e-06
0.9,-0.11948963814468237,1.2907857561691645,0.5423733260384742,-1.7040867919372369,0.21724327237834018,-0.7629593685425213,-0.7853984814062104,1.5707970163522575,-0.7853979659310516,-1.7899874444364021,-0.0929915362188814,0.3027413446752859,0.027084809831616347,2.672174706721251,0.17625748681712636,-5.536609813565612e-08,-3.8662853401747026e-07,0.17118863888633057,1.6120323857331748e-06
1.0,-0.14401114146143854,1.225181171398354,0.5658404203078966,-1.739976923824856,0.22928020449422845,-0.8101976592926039,-0.7853996734225837,1.5707989879381172,-0.7853974156522932,-1.7799510631362938,-0.10271730479536219,0.3729700068761373,0.043230969508981754,2.6382494107208316,0.1810354783625541,-4.0862894587882353e-07,-1.5349054490761755e-06,0.19196479940735625,4.862954015973888e-06
1.1,-0.17087993522511863,1.1480947738741363,0.5840927329498101,-1.7818392750344423,0.24344143095283388,-0.8468396123866332,-0.7853978894040206,1.5707949101430698,-0.7853986631752842,-1.7679031035234765,-0.11392932431341808,0.454537577084513,0.06227162371276369,2.618078261988071,0.1882395026029577,1.0545856059028912e-07,8.758608116656042e-07,0.21323002695621102,-7.002742115854476e-07
1.2,-0.19871985954238097,1.0603899658319091,0.5986977581070089,-1.8298539027793685,0.25976600324380134,-0.8706564315464902,-0.7853979312137346,1.5707957977197422,-0.7853983138808702,-1.7557771169388896,-0.12577285673536753,0.546827059847068,0.08207764680272427,2.617247606604455,0.19856598540898268,1.1136116575289789e-07,3.0972892618152327e-07,0.23440396530331245,-3.8991636391366525e-07
1.3,-0.22541826919058122,0.9646926539677239,0.611560808741069,-1.8840827657284551,0.2779284589702517,-0.8807891429245749,-0.7853982795794565,1.5707961784442739,-0.7853980188631741,-1.7446490899908462,-0.13644533534470638,0.6472205146075389,0.09928051500415891,2.639674840472457,0.21238857842175618,2.839825530602383e-08,9.153506375742504e-08,0.2547456463185906,6.732987964530409e-07
1.4,-0.24868746155217783,0.8646489059082396,0.6246172990439505,-1.9447787873147446,0.2971711456775349,-0.8776652767563379,-0.785407684988388,1.5708200317617107,-0.7853885974476782,-1.7344942512171906,-0.1438792007522124,0.7509476416386986,0.10971543583005176,2.685982435330604,0.22972135952107242,-5.9399383935465585e-06,-1.1961152340973349e-05,0.2734321347904362,2.263501200731794e-05
1.5,-0.2669534620927614,0.7633916180575379,0.6393292963108524,-2.0128394182626383,0.3163242989313856,-0.8629725004070605,-0.785397921659803,1.5707953520889795,-0.7853984930963747,-1.7242527280231141,-0.14675343291934115,0.8519525438066423,0.10948051408051773,2.7526533751904663,0.250123969370168,1.9245335761090172e-07,2.2220196359877753e-07,0.2898487687710785,-2.2759209836727357e-07
1.6,-0.2796358148130655,0.6628288558215701,0.6562276057609199,-2.0893176017976596,0.33400036810553596,-0.838405687393417,-0.7853984116539009,1.570796637388456,-0.7853979450234027,-1.71168673609838,-0.14563212407986453,0.9434677659042918,0.09535368745467597,2.8353520874695572,0.2728420324983797,-1.163178009974203e-07,-4.4240908559289306e-08,0.30332075778023837,3.0070581859492404e-07
1.7,-0.28763188892603125,0.5628485843963267,0.6739533462141916,-2.1748942260299953,0.3492193733379317,-0.8057999450486121,-0.7853981413665898,1.5707966044730517,-0.7853981020769947,-1.6934699860659943,-0.14357233434785047,1.020234892647565,0.06578944994111487,2.9295038238811872,0.2968080477173052,-7.094880670538219e-08,-1.1858919729992071e-07,0.3135561361067362,2.0002752622198605e-07
1.8,-0.29301424583409885,0.46230243897014406,0.6885829239881193,-2.2680794949181435,0.3621311594299363,-0.7664477148008908,-0.7853981750817526,1.5707964762701407,-0.7853979235861176,-1.6653189637172021,-0.14653679799170372,1.0797153394166419,0.02102782132065355,3.0330206554644934,0.3208699670281229,-2.5195276089074e-07,-7.760146121164263e-09,0.32048670985550426,3.708959584703384e-07
2.0,-0.3081404708385435,0.25870073705731506,0.6779963440062249,-2.4484050654223966,0.39232701304213374,-0.6707461055225236,-0.7853981650407561,1.5707961671111337,-0.7853981919656636,-1.5636666820128449,-0.207770028684754,1.156154340774508,-0.10430158164247885,3.2817656253775254,0.36539821948693557,-5.8499122111809056e-08,5.722933755616074e-08,0.325415072416129,-6.419630097970126e-08
2.2,-0.35868623077919926,0.03986668646464891,0.5401987104503252,-2.528718086189674,0.5148230527945056,-0.5591201390310002,-0.7853981894776837,1.5707964106078134,-0.7853981052046338,-1.4030889679631025,-0.4725261083352164,1.214780123340506,-0.2501454113849586,3.683670646354396,0.4015501548959025,8.521519262731598e-09,-9.730037884615108e-08,0.3215603052716824,-2.7379863562388005e-09
2.4,-0.4074188212929699,-0.17667159108950103,0.9692844037719567,-1.9009514660855615,0.1022445302123831,-0.8554720978562393,-1.6879267848212471,2.4827770396086724,-0.8849034207947316,-1.6712877835034572,-0.3379492658273585,1.6673762125142613,-0.30665256980508937,4.57394881276445,0.6634192972196461,-0.2551440789685863,-0.10216644898492616,0.17532734774004077,0.3625418397965311
2.6,-0.44435067652340393,0.24220524434432456,1.056594086224431,-1.7141854215433312,0.11319155181360406,-0.8675682678200212,-1.641375942692166,2.5262825091314203,-0.8327588239979635,-1.8174236583578018,0.023456843051200698,1.6320681064692755,-0.485148958674455,4.259796003551107,0.8006311698698658,-0.3326120471043157,-0.11890533705288396,0.1785789897860071,0.42658979780581346
2.8,-0.24588435151233848,0.5881525238233827,0.9817866213024106,-1.1219957865636019,-0.3659627680130971,-0.6828756283826911,-1.4107282111135804,2.3865353785728285,-0.8246147907556781,-1.3545255399929104,-0.29992825572100196,2.453434942012389,0.38225911130664664,4.810166181673148,0.2891688161649811,-0.4259513395678066,-0.26020506555777273,-0.21967602941867628,0.20824898125531324
3.0,-0.26824739323719654,0.7299032643836145,0.9419864608120838,-0.9964778745732386,-0.3891702501807084,-0.6834496996905363,-1.404750775293906,2.3729461844465827,-0.8083053075071133,-1.4207645803518185,-0.2817289286860655,2.4964690763558894,0.43820483383221726,4.8133449399055745,0.2741160310460972,-0.4415476825332304,-0.26180128985583284,-0.2506112480687556,0.20856674997128394
""",
    "opt": """\
d,e_sv,err_mHa,S2,e_fci,origin,nit,nfev,converged,grad_norm
0.6,-1.9601936448878103,6.11422e-09,3.17297e-16,-1.9601936448939246,pulido desde 0.7,66,72,True,1.08005e-06
0.7,-2.1069969150949586,3.20188e-10,1.88636e-14,-2.1069969150952788,pulido desde 0.8,61,67,True,1.31164e-07
0.8,-2.16756054413048,3.57669e-09,2.32058e-15,-2.167560544134057,pulido desde 0.9,59,64,True,9.36024e-07
0.9,-2.1803166143235164,3.31735e-10,1.20626e-13,-2.180316614323848,pulido desde 1.0,55,60,True,5.12783e-07
1.0,-2.166387448631914,2.85016e-09,6.62318e-14,-2.1663874486347643,pulido desde 1.1,54,57,True,1.53278e-06
1.1,-2.137970526843003,6.31051e-10,2.59262e-13,-2.137970526843634,pulido desde 1.2,53,57,True,9.49782e-07
1.2,-2.102608480955341,7.59393e-11,1.86851e-15,-2.102608480955417,pulido desde 1.3,49,53,True,2.22699e-07
1.3,-2.0652289632709615,3.13971e-10,1.73453e-15,-2.0652289632712755,pulido desde 1.4,48,51,True,6.05489e-07
1.4,-2.0290704935337476,5.82117e-08,7.55591e-12,-2.0290704935919592,pulido desde 1.5,45,48,True,2.3481e-06
1.5,-1.9961503255181583,6.48814e-10,1.93328e-12,-1.9961503255188071,"0,6→3,0 azar",141,154,True,1.21053e-06
1.6,-1.9675603099202164,1.62537e-10,9.96117e-13,-1.967560309920379,"0,6→3,0 warm+ruido",94,106,True,8.36423e-07
1.7,-1.9436920386879222,1.94067e-10,1.22421e-13,-1.9436920386881162,"0,6→3,0 warm+ruido",62,64,True,6.57358e-07
1.8,-1.9244306381364402,2.77112e-10,1.19046e-15,-1.9244306381367173,"0,6→3,0 warm+ruido",69,74,True,1.05335e-06
2.0,-1.8977806459897515,1.34115e-10,8.40382e-15,-1.8977806459898856,"0,6→3,0 warm+ruido",51,55,True,6.59928e-07
2.2,-1.8827126374316439,2.05391e-10,3.58207e-13,-1.8827126374318492,"0,6→3,0 warm+ruido",70,73,True,7.5495e-07
2.4,-1.8746505643875,0.0010181,2.12345e-05,-1.874651582485912,"0,6→3,0 warm+ruido",70,73,True,6.11007e-07
2.6,-1.8704721356967757,0.000600724,6.63896e-06,-1.8704727364209426,"0,6→3,0 warm",83,92,True,6.4618e-07
2.8,-1.868346696130265,0.00301155,5.75118e-08,-1.8683497076851177,"0,6→3,0 warm+ruido",63,66,True,9.95281e-07
3.0,-1.86728915716973,0.00221523,2.19116e-09,-1.8672913724013522,"0,6→3,0 warm+ruido",52,59,True,5.43021e-07
""",
    "starts_log": """\
d,direction,start,e_sv,err_mHa,S2,nit,converged
0.6,"0,6→3,0",warm,-1.950005538,10.1881,4.88838e-09,72,True
0.6,"0,6→3,0",warm+ruido,-1.95913843,1.05521,0.00140094,166,True
0.6,"0,6→3,0",warm+ruido,-1.959115145,1.0785,0.00129067,281,True
0.6,"0,6→3,0",azar,-1.958164414,2.02923,0.0014954,67,True
0.6,"0,6→3,0",azar,-1.960168203,0.0254422,2.74822e-05,168,True
0.6,"0,6→3,0",azar,-1.950907014,9.28663,0.000945051,106,True
0.7,"0,6→3,0",warm,-2.106968592,0.0283236,4.34501e-05,67,True
0.7,"0,6→3,0",warm+ruido,-2.106980156,0.0167588,3.5126e-05,158,True
0.7,"0,6→3,0",warm+ruido,-2.106848723,0.148192,5.00885e-06,101,True
0.7,"0,6→3,0",azar,-2.106993148,0.0037667,1.68055e-06,211,True
0.7,"0,6→3,0",azar,-2.106992641,0.00427453,4.61241e-09,690,True
0.7,"0,6→3,0",azar,-2.106718323,0.278592,3.24435e-05,299,True
0.8,"0,6→3,0",warm,-2.167522221,0.0383232,1.97072e-05,54,True
0.8,"0,6→3,0",warm+ruido,-2.16752222,0.0383246,1.96803e-05,82,True
0.8,"0,6→3,0",warm+ruido,-2.167519499,0.0410453,2.71183e-06,216,True
0.8,"0,6→3,0",azar,-2.167294854,0.26569,5.82418e-07,104,True
0.8,"0,6→3,0",azar,-2.167559379,0.0011648,7.80719e-08,139,True
0.8,"0,6→3,0",azar,-2.167527293,0.0332509,2.6839e-07,202,True
0.9,"0,6→3,0",warm,-2.180309723,0.00689182,2.04841e-06,59,True
0.9,"0,6→3,0",warm+ruido,-2.180309722,0.00689184,2.04808e-06,92,True
0.9,"0,6→3,0",warm+ruido,-2.179044774,1.27184,0.0033891,118,True
0.9,"0,6→3,0",azar,-2.179610075,0.706539,0.000219145,280,True
0.9,"0,6→3,0",azar,-2.180008335,0.30828,0.000105348,150,True
0.9,"0,6→3,0",azar,-2.178018651,2.29796,0.00367147,111,True
1,"0,6→3,0",warm,-2.166338195,0.0492535,3.37728e-05,54,True
1,"0,6→3,0",warm+ruido,-2.166366459,0.0209898,4.72994e-07,161,True
1,"0,6→3,0",warm+ruido,-2.166338195,0.0492541,3.37603e-05,68,True
1,"0,6→3,0",azar,-2.164113843,2.27361,0.00407891,237,True
1,"0,6→3,0",azar,-2.166188758,0.19869,4.6602e-05,178,True
1,"0,6→3,0",azar,-2.166239464,0.147985,1.29916e-06,63,True
1.1,"0,6→3,0",warm,-2.137912422,0.0581052,1.36462e-06,79,True
1.1,"0,6→3,0",warm+ruido,-2.137912422,0.0581053,1.36353e-06,106,True
1.1,"0,6→3,0",warm+ruido,-2.137912422,0.0581052,1.36455e-06,166,True
1.1,"0,6→3,0",azar,-2.137212387,0.75814,7.21292e-05,256,True
1.1,"0,6→3,0",azar,-2.137896688,0.0738387,7.62928e-06,141,True
1.1,"0,6→3,0",azar,-2.137139108,0.831419,0.000717276,132,True
1.2,"0,6→3,0",warm,-2.102482593,0.125888,2.85402e-06,56,True
1.2,"0,6→3,0",warm+ruido,-2.102604221,0.00425991,7.92179e-07,173,True
1.2,"0,6→3,0",warm+ruido,-2.102482593,0.125888,2.86762e-06,89,True
1.2,"0,6→3,0",azar,-2.10259326,0.0152208,8.41046e-08,167,True
1.2,"0,6→3,0",azar,-2.101623013,0.985468,0.000865471,67,True
1.2,"0,6→3,0",azar,-2.102524057,0.084424,6.87768e-06,149,True
1.3,"0,6→3,0",warm,-2.065217668,0.0112951,1.27325e-07,94,True
1.3,"0,6→3,0",warm+ruido,-2.065227422,0.00154151,1.81244e-08,514,True
1.3,"0,6→3,0",warm+ruido,-2.064820852,0.408112,4.13867e-05,84,True
1.3,"0,6→3,0",azar,-2.065197903,0.0310603,6.02824e-06,228,True
1.3,"0,6→3,0",azar,-2.065228955,8.29849e-06,1.77177e-10,366,True
1.3,"0,6→3,0",azar,-2.061369584,3.85938,0.00521891,103,True
1.4,"0,6→3,0",warm,-2.029062821,0.00767224,2.25998e-06,74,True
1.4,"0,6→3,0",warm+ruido,-2.029062821,0.00767243,2.25606e-06,99,True
1.4,"0,6→3,0",warm+ruido,-2.029062821,0.00767243,2.25619e-06,98,True
1.4,"0,6→3,0",azar,-2.023196432,5.87406,0.00209331,56,True
1.4,"0,6→3,0",azar,-2.027114408,1.95609,0.000415751,102,True
1.4,"0,6→3,0",azar,-2.028083392,0.987102,0.00500335,142,True
1.5,"0,6→3,0",warm,-1.9961499,0.000425573,3.33202e-08,78,True
1.5,"0,6→3,0",warm+ruido,-1.995893747,0.256579,0.000387296,92,True
1.5,"0,6→3,0",warm+ruido,-1.9961499,0.000425575,3.32704e-08,93,True
1.5,"0,6→3,0",azar,-1.993264979,2.88535,0.0116534,121,True
1.5,"0,6→3,0",azar,-1.996016736,0.133589,7.4978e-05,233,True
1.5,"0,6→3,0",azar,-1.996150326,6.48814e-10,1.93328e-12,141,True
1.6,"0,6→3,0",warm,-1.96756031,1.09512e-09,4.86908e-13,45,True
1.6,"0,6→3,0",warm+ruido,-1.96756031,1.62537e-10,9.96117e-13,94,True
1.6,"0,6→3,0",warm+ruido,-1.96756031,1.49236e-09,3.035e-13,65,True
1.6,"0,6→3,0",azar,-1.964949948,2.61036,0.0117471,167,True
1.6,"0,6→3,0",azar,-1.967542221,0.0180888,2.15135e-05,70,True
1.6,"0,6→3,0",azar,-1.966452862,1.10745,0.00180697,124,True
1.7,"0,6→3,0",warm,-1.943692039,1.55507e-08,2.08766e-11,42,True
1.7,"0,6→3,0",warm+ruido,-1.943692039,4.69513e-09,2.88987e-12,52,True
1.7,"0,6→3,0",warm+ruido,-1.943692039,1.94067e-10,1.22421e-13,62,True
1.7,"0,6→3,0",azar,-1.943455637,0.236402,0.00014628,245,True
1.7,"0,6→3,0",azar,-1.943551537,0.140502,0.000207007,85,True
1.7,"0,6→3,0",azar,-1.942020687,1.67135,0.00278961,287,True
1.8,"0,6→3,0",warm,-1.924430638,2.27796e-09,1.2994e-13,43,True
1.8,"0,6→3,0",warm+ruido,-1.924430638,7.16094e-10,1.46248e-12,88,True
1.8,"0,6→3,0",warm+ruido,-1.924430638,2.77112e-10,1.19046e-15,69,True
1.8,"0,6→3,0",azar,-1.924393445,0.0371933,6.34252e-05,101,True
1.8,"0,6→3,0",azar,-1.924328283,0.102355,0.00021912,179,True
1.8,"0,6→3,0",azar,-1.897250331,27.1803,-2.04924e-18,53,True
2,"0,6→3,0",warm,-1.897780646,2.94809e-09,9.05585e-13,46,True
2,"0,6→3,0",warm+ruido,-1.897780646,1.34115e-10,8.40382e-15,51,True
2,"0,6→3,0",warm+ruido,-1.897780646,9.49241e-10,2.20281e-15,56,True
2,"0,6→3,0",azar,-1.895334466,2.44618,0.011247,188,True
2,"0,6→3,0",azar,-1.8977709,0.00974604,5.61292e-06,162,True
2,"0,6→3,0",azar,-1.89775084,0.0298055,6.28043e-07,139,True
2.2,"0,6→3,0",warm,-1.882712637,9.53682e-10,8.95292e-14,53,True
2.2,"0,6→3,0",warm+ruido,-1.882712637,2.05391e-10,3.58207e-13,70,True
2.2,"0,6→3,0",warm+ruido,-1.882712637,3.888e-10,5.24612e-12,91,True
2.2,"0,6→3,0",azar,-1.881076886,1.63575,0.00121229,200,True
2.2,"0,6→3,0",azar,-1.882252848,0.459789,1.2036e-05,149,True
2.2,"0,6→3,0",azar,-1.879486491,3.22615,0.00129464,168,True
2.4,"0,6→3,0",warm,-1.874644938,0.00664478,3.14793e-06,104,True
2.4,"0,6→3,0",warm+ruido,-1.873664732,0.986851,0.001824,136,True
2.4,"0,6→3,0",warm+ruido,-1.874650564,0.0010181,2.12345e-05,70,True
2.4,"0,6→3,0",azar,-1.86395874,10.6928,8.4807e-05,109,True
2.4,"0,6→3,0",azar,-1.873720375,0.931208,0.00889027,158,True
2.4,"0,6→3,0",azar,-1.874563283,0.088299,0.000638577,219,True
2.6,"0,6→3,0",warm,-1.870472136,0.000600724,6.63896e-06,83,True
2.6,"0,6→3,0",warm+ruido,-1.870309813,0.162923,0.000159559,73,True
2.6,"0,6→3,0",warm+ruido,-1.870102289,0.370448,3.23997e-05,107,True
2.6,"0,6→3,0",azar,-1.870048784,0.423952,4.72813e-05,108,True
2.6,"0,6→3,0",azar,-1.866434381,4.03836,9.00917e-05,81,True
2.6,"0,6→3,0",azar,-1.867749462,2.72327,0.000775342,181,True
2.8,"0,6→3,0",warm,-1.86833728,0.0124272,5.83334e-05,61,True
2.8,"0,6→3,0",warm+ruido,-1.86833728,0.0124277,5.83223e-05,77,True
2.8,"0,6→3,0",warm+ruido,-1.868346696,0.00301155,5.75118e-08,63,True
2.8,"0,6→3,0",azar,-1.868275965,0.0737429,0.00023968,117,True
2.8,"0,6→3,0",azar,-1.866979923,1.36978,0.000260783,80,True
2.8,"0,6→3,0",azar,-1.868277115,0.0725922,0.000160617,103,True
3,"0,6→3,0",warm,-1.867289157,0.00221523,2.38848e-09,41,True
3,"0,6→3,0",warm+ruido,-1.867289157,0.00221523,2.29622e-09,54,True
3,"0,6→3,0",warm+ruido,-1.867289157,0.00221523,2.19116e-09,52,True
3,"0,6→3,0",azar,-1.867210265,0.0811075,1.9565e-05,180,True
3,"0,6→3,0",azar,-1.867258031,0.0333418,1.27204e-05,149,True
3,"0,6→3,0",azar,-1.866204846,1.08653,7.64013e-06,294,True
3,"3,0→0,6",warm,-1.580700096,286.591,0.00482069,39,True
3,"3,0→0,6",warm+ruido,-1.866160802,1.13057,0.000300691,360,True
3,"3,0→0,6",warm+ruido,-1.866031003,1.26037,0.00430011,138,True
3,"3,0→0,6",azar,-1.86720992,0.0814521,1.2241e-09,186,True
3,"3,0→0,6",azar,-1.866153683,1.13769,0.000618895,188,True
3,"3,0→0,6",azar,-1.866536989,0.754383,3.48307e-05,260,True
2.8,"3,0→0,6",warm,-1.868176941,0.172766,3.30475e-09,72,True
2.8,"3,0→0,6",warm+ruido,-1.86709669,1.25302,0.00021891,61,True
2.8,"3,0→0,6",warm+ruido,-1.868287698,0.06201,2.12915e-06,296,True
2.8,"3,0→0,6",azar,-1.86641124,1.93847,0.000266212,121,True
2.8,"3,0→0,6",azar,-1.865396961,2.95275,0.00466623,85,True
2.8,"3,0→0,6",azar,-1.865960325,2.38938,1.37064e-05,86,True
2.6,"3,0→0,6",warm,-1.870360372,0.112365,3.19819e-06,110,True
2.6,"3,0→0,6",warm+ruido,-1.870360371,0.112365,3.19386e-06,139,True
2.6,"3,0→0,6",warm+ruido,-1.870354644,0.118092,2.63432e-07,211,True
2.6,"3,0→0,6",azar,-1.869489036,0.9837,6.85441e-05,86,True
2.6,"3,0→0,6",azar,-1.865129748,5.34299,3.98186e-05,97,True
2.6,"3,0→0,6",azar,-1.870111854,0.360882,6.01796e-05,174,True
2.4,"3,0→0,6",warm,-1.874464952,0.18663,1.97325e-06,87,True
2.4,"3,0→0,6",warm+ruido,-1.870143921,4.50766,3.57794e-05,70,True
2.4,"3,0→0,6",warm+ruido,-1.874597197,0.0543855,1.78791e-06,132,True
2.4,"3,0→0,6",azar,-1.864830348,9.82123,3.81898e-05,64,True
2.4,"3,0→0,6",azar,-1.873293592,1.35799,0.00121783,77,True
2.4,"3,0→0,6",azar,-1.873373057,1.27853,0.000232272,122,True
2.2,"3,0→0,6",warm,-1.882661705,0.0509329,0.000118852,110,True
2.2,"3,0→0,6",warm+ruido,-1.882661703,0.0509347,0.000118815,114,True
2.2,"3,0→0,6",warm+ruido,-1.882204358,0.50828,2.25588e-05,116,True
2.2,"3,0→0,6",azar,-1.881187818,1.52482,0.00462907,117,True
2.2,"3,0→0,6",azar,-1.86539828,17.3144,0.000553973,88,True
2.2,"3,0→0,6",azar,-1.882706345,0.0062926,2.44395e-05,313,True
2,"3,0→0,6",warm,-1.897703824,0.0768221,0.000515094,111,True
2,"3,0→0,6",warm+ruido,-1.896898235,0.882411,5.62239e-06,226,True
2,"3,0→0,6",warm+ruido,-1.897703824,0.0768223,0.000515091,82,True
2,"3,0→0,6",azar,-1.897736574,0.0440722,7.95307e-05,178,True
2,"3,0→0,6",azar,-1.897616873,0.163773,0.000310707,221,True
2,"3,0→0,6",azar,-1.895399546,2.3811,0.000363448,93,True
1.8,"3,0→0,6",warm,-1.924399412,0.0312262,4.34676e-05,73,True
1.8,"3,0→0,6",warm+ruido,-1.924399412,0.031226,4.3471e-05,68,True
1.8,"3,0→0,6",warm+ruido,-1.92272809,1.70255,0.00652156,91,True
1.8,"3,0→0,6",azar,-1.923485904,0.944735,0.00157459,228,True
1.8,"3,0→0,6",azar,-1.922522825,1.90781,0.000478061,86,True
1.8,"3,0→0,6",azar,-1.922932695,1.49794,0.00140052,92,True
1.7,"3,0→0,6",warm,-1.943648975,0.0430636,5.42109e-05,59,True
1.7,"3,0→0,6",warm+ruido,-1.943648976,0.0430626,5.42301e-05,81,True
1.7,"3,0→0,6",warm+ruido,-1.943648976,0.043063,5.42234e-05,120,True
1.7,"3,0→0,6",azar,-1.943594607,0.097432,2.41576e-05,121,True
1.7,"3,0→0,6",azar,-1.943202321,0.489717,0.00448993,159,True
1.7,"3,0→0,6",azar,-1.940736565,2.95547,0.000354944,206,True
1.6,"3,0→0,6",warm,-1.967517424,0.0428861,3.79854e-05,60,True
1.6,"3,0→0,6",warm+ruido,-1.967459566,0.100744,0.000160826,123,True
1.6,"3,0→0,6",warm+ruido,-1.967517424,0.0428859,3.79901e-05,84,True
1.6,"3,0→0,6",azar,-1.967543418,0.016892,4.81047e-08,229,True
1.6,"3,0→0,6",azar,-1.967532256,0.0280534,0.000189755,188,True
1.6,"3,0→0,6",azar,-1.966795549,0.764761,4.83016e-05,118,True
1.5,"3,0→0,6",warm,-1.996046315,0.104011,5.8908e-07,81,True
1.5,"3,0→0,6",warm+ruido,-1.996131992,0.0183339,8.19621e-12,162,True
1.5,"3,0→0,6",warm+ruido,-1.996046315,0.104011,5.8843e-07,105,True
1.5,"3,0→0,6",azar,-1.995067482,1.08284,3.0086e-05,108,True
1.5,"3,0→0,6",azar,-1.990733323,5.417,5.49542e-05,126,True
1.5,"3,0→0,6",azar,-1.996149821,0.000504783,4.54503e-11,203,True
1.4,"3,0→0,6",warm,-2.028882762,0.187732,5.068e-08,48,True
1.4,"3,0→0,6",warm+ruido,-2.028882762,0.187732,5.15245e-08,57,True
1.4,"3,0→0,6",warm+ruido,-2.028958204,0.112289,5.1527e-07,57,True
1.4,"3,0→0,6",azar,-2.028792763,0.277731,0.000167653,203,True
1.4,"3,0→0,6",azar,-2.028675763,0.394731,0.000392105,118,True
1.4,"3,0→0,6",azar,-2.028876739,0.193755,3.89509e-07,115,True
1.3,"3,0→0,6",warm,-2.065213087,0.0158761,4.84662e-07,47,True
1.3,"3,0→0,6",warm+ruido,-2.065213087,0.015876,4.86486e-07,61,True
1.3,"3,0→0,6",warm+ruido,-2.065213087,0.0158761,4.84867e-07,63,True
1.3,"3,0→0,6",azar,-2.065018586,0.210377,0.00115184,81,True
1.3,"3,0→0,6",azar,-2.065174779,0.0541838,0.000114017,191,True
1.3,"3,0→0,6",azar,-2.065147016,0.081947,0.000352959,182,True
1.2,"3,0→0,6",warm,-2.102578862,0.0296186,9.60133e-06,49,True
1.2,"3,0→0,6",warm+ruido,-2.102578863,0.0296177,9.61909e-06,58,True
1.2,"3,0→0,6",warm+ruido,-2.102578863,0.0296182,9.60906e-06,65,True
1.2,"3,0→0,6",azar,-2.101621814,0.986667,0.00188041,122,True
1.2,"3,0→0,6",azar,-2.100382466,2.22602,0.0117765,138,True
1.2,"3,0→0,6",azar,-2.102598165,0.0103158,8.33268e-07,212,True
1.1,"3,0→0,6",warm,-2.137962552,0.0079752,3.55791e-07,77,True
1.1,"3,0→0,6",warm+ruido,-2.137962552,0.00797515,3.56719e-07,91,True
1.1,"3,0→0,6",warm+ruido,-2.137962552,0.00797524,3.57014e-07,76,True
1.1,"3,0→0,6",azar,-2.13790308,0.0674468,1.66919e-06,163,True
1.1,"3,0→0,6",azar,-2.137792129,0.178398,2.31667e-07,205,True
1.1,"3,0→0,6",azar,-2.137761818,0.208709,0.000328499,108,True
1,"3,0→0,6",warm,-2.166380774,0.00667485,1.88154e-07,62,True
1,"3,0→0,6",warm+ruido,-2.166295962,0.0914869,8.35594e-06,114,True
1,"3,0→0,6",warm+ruido,-2.166295962,0.0914867,8.36076e-06,148,True
1,"3,0→0,6",azar,-2.165619507,0.767942,3.3198e-05,338,True
1,"3,0→0,6",azar,-2.16610983,0.277618,0.000527217,88,True
1,"3,0→0,6",azar,-2.166386418,0.00103059,4.75732e-08,92,True
0.9,"3,0→0,6",warm,-2.180316071,0.00054335,6.64556e-09,40,True
0.9,"3,0→0,6",warm+ruido,-2.179306281,1.01033,0.00147242,85,True
0.9,"3,0→0,6",warm+ruido,-2.179632111,0.684504,0.000232642,103,True
0.9,"3,0→0,6",azar,-2.180293999,0.0226158,1.15842e-06,162,True
0.9,"3,0→0,6",azar,-2.179769255,0.54736,8.48171e-06,97,True
0.9,"3,0→0,6",azar,-2.180313432,0.00318265,8.92055e-08,206,True
0.8,"3,0→0,6",warm,-2.167549471,0.0110733,2.79827e-06,43,True
0.8,"3,0→0,6",warm+ruido,-2.167550816,0.00972846,7.26816e-07,140,True
0.8,"3,0→0,6",warm+ruido,-2.167341498,0.219046,0.000121116,161,True
0.8,"3,0→0,6",azar,-2.165337295,2.22325,0.00280986,276,True
0.8,"3,0→0,6",azar,-2.167553152,0.00739246,9.77555e-07,110,True
0.8,"3,0→0,6",azar,-2.167452428,0.108117,1.36896e-06,79,True
0.7,"3,0→0,6",warm,-2.10699665,0.000265092,2.42935e-08,58,True
0.7,"3,0→0,6",warm+ruido,-2.106741955,0.25496,3.0453e-06,173,True
0.7,"3,0→0,6",warm+ruido,-2.106624572,0.372343,4.03779e-06,368,True
0.7,"3,0→0,6",azar,-2.105933284,1.06363,0.000112834,141,True
0.7,"3,0→0,6",azar,-2.10684205,0.154865,1.10579e-05,231,True
0.7,"3,0→0,6",azar,-2.106969131,0.0277839,2.93253e-06,195,True
0.6,"3,0→0,6",warm,-1.960186181,0.00746399,5.50956e-07,48,True
0.6,"3,0→0,6",warm+ruido,-1.958564299,1.62935,0.000735821,199,True
0.6,"3,0→0,6",warm+ruido,-1.960186181,0.00746351,5.58369e-07,108,True
0.6,"3,0→0,6",azar,-1.959387384,0.806261,0.000626227,135,True
0.6,"3,0→0,6",azar,-1.959298999,0.894646,0.00102549,180,True
0.6,"3,0→0,6",azar,-1.958985378,1.20827,8.97807e-05,327,True
""",
    "multistart": """\
d,start,e_sv,err_mHa,S2,nit,converged
1.8,HF (θ=0),-1.771346667,153.084,0.00834049,33,True
1.8,warm desde 1.7,-1.924430638,2.27796e-09,1.2994e-13,43,True
1.8,azar 0,-1.924165192,0.265446,0.000171282,95,True
1.8,azar 1,-1.924424386,0.00625171,1.52246e-06,188,True
1.8,azar 2,-1.924393413,0.0372249,1.27055e-05,147,True
1.8,azar 3,-1.924348204,0.0824344,0.000169747,237,True
1.8,azar 4,-1.861171219,63.2594,0.0197161,227,True
1.8,azar 5,-1.923897349,0.533289,7.33418e-05,101,True
1.8,azar 6,-1.924035253,0.395385,0.00173835,185,True
1.8,azar 7,-1.923517552,0.913086,0.000935976,80,True
2.4,HF (θ=0),-1.645103301,229.548,0.00263973,37,True
2.4,warm desde 2.2,-1.874644938,0.00664478,3.14793e-06,104,True
2.4,azar 0,-1.864150705,10.5009,1.93573e-06,66,True
2.4,azar 1,-1.873178164,1.47342,0.00802033,162,True
2.4,azar 2,-1.870805785,3.8458,0.000358939,134,True
2.4,azar 3,-1.874629036,0.0225469,1.54999e-05,116,True
2.4,azar 4,-1.870084704,4.56688,0.000684532,116,True
2.4,azar 5,-1.869811392,4.84019,0.00376783,103,True
2.4,azar 6,-1.874618966,0.0326161,7.88502e-08,486,True
2.4,azar 7,-1.874576068,0.0755148,7.98108e-07,196,True
3,HF (θ=0),-1.580700096,286.591,0.00482069,39,True
3,warm desde 2.8,-1.867289157,0.00221523,2.38848e-09,41,True
3,azar 0,-1.867141039,0.150334,1.39931e-05,174,True
3,azar 1,-1.867151397,0.139975,0.00586129,201,True
3,azar 2,-1.867197502,0.0938702,1.62942e-05,187,True
3,azar 3,-1.866024265,1.26711,9.79164e-05,70,True
3,azar 4,-1.86722175,0.0696226,1.36494e-08,67,True
3,azar 5,-1.866168661,1.12271,5.14397e-05,274,True
3,azar 6,-1.867205386,0.0859868,8.20752e-09,323,True
3,azar 7,-1.867291146,0.00022605,4.25402e-08,564,True
""",
}

In [ ]:
# VQE, parte 1 (#5): optimización de θ*(d) en toda la grilla, con statevector (permitido para buscar
# parámetros). Las energías de esta celda son solo diagnóstico: la energía reportada la mide #6.
# Reutiliza de la sección 3 `sector_hamiltonian`, `sector_ansatz` (gradiente adjunto exacto), `ansatz`,
# `minimize`, `Statevector` y `spin_op`.
import io
import time

RUN_PARAMS = True  # True (por defecto): buscar θ* en esta corrida (~1 min). False: usar los θ* guardados en PARAMS_CACHE (celda de arriba)

_t0 = time.perf_counter()
H_SECTOR = {float(d): sector_hamiltonian(d) for d in D}  # d -> (H con e_nuc, E_FCI); E_FCI solo para comparar

# S² en el sector (mismo orden ci[ia, ib] que SECTOR). Se minimiza E(θ) + λ⟨S²⟩(θ): ⟨S²⟩ es otro valor
# esperado, así que el gradiente adjunto sirve igual con H + λS². En el singlete la penalización vale 0
# y no mueve el mínimo; sin ella, en la zona estirada hay mínimos locales contaminados con triplete a
# solo ~0,02 mHa de FCI (la brecha singlete-triplete es de 0,5 mHa a 3,0 Å), que la energía no delata.
S2_SECTOR = np.column_stack([spin_op.contract_ss(e.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA)).ravel()
                             for e in np.eye(len(SECTOR))])
_v = sector_ansatz.state(np.random.default_rng(2).normal(size=sector_ansatz.n))
assert np.allclose(S2_SECTOR, S2_SECTOR.T)
assert np.isclose(_v @ S2_SECTOR @ _v, spin_op.spin_square0(_v.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0])
LAMBDA_S2 = 0.05  # Ha: ⟨S²⟩ = 0,03 (1,5 % de triplete) pasa a costar 1,5 mHa
COST = {d: H + LAMBDA_S2 * S2_SECTOR for d, (H, _) in H_SECTOR.items()}

# El gradiente adjunto de la sección 3 coincide con diferencias finitas centradas.
_x, _C = np.random.default_rng(1).normal(size=sector_ansatz.n), COST[STRETCHED]
_fd = [(sector_ansatz.energy_grad(_x + h, _C)[0] - sector_ansatz.energy_grad(_x - h, _C)[0]) / 2e-6
       for h in 1e-6 * np.eye(sector_ansatz.n)]
assert np.allclose(sector_ansatz.energy_grad(_x, _C)[1], _fd, atol=1e-7), "gradiente adjunto ≠ diferencias finitas"


def optimize(d: float, x0: np.ndarray):
    """BFGS con gradiente adjunto exacto sobre E + λ⟨S²⟩. |∇| < 1e-6 deja el costo a ~1e-12 Ha del mínimo local."""
    return minimize(sector_ansatz.energy_grad, x0, args=(COST[d],), jac=True, method="BFGS", options={"gtol": 1e-6})


def diagnostics(d: float, x: np.ndarray) -> dict:
    """Energía statevector (sin penalización), error vs. FCI y ⟨S²⟩ (0 = singlete, 2 = triplete)."""
    v = sector_ansatz.state(x)
    H, e_fci = H_SECTOR[d]
    return {"e_sv": v @ H @ v, "err_mHa": (v @ H @ v - e_fci) * 1e3, "S2": v @ S2_SECTOR @ v}


grid = list(map(float, D))
MULTISTART_D = (1.8, 2.4, 3.0)
if RUN_PARAMS:
    # 1) Warm start en las dos direcciones + multi-start: en cada d se arranca desde θ* de la distancia vecina,
    #    desde N_PERTURB perturbaciones de ese θ* y desde N_RANDOM puntos al azar; se queda el mejor.
    N_PERTURB, N_RANDOM = 2, 3
    rng = np.random.default_rng(5)
    best, starts_log = {}, []  # best: d -> (resultado de minimize, origen)
    for direction, order in (("0,6→3,0", D), ("3,0→0,6", D[::-1])):
        x = np.zeros(sector_ansatz.n)  # θ = 0 es HF: el primer punto se salva con los arranques al azar
        for d in map(float, order):
            starts = ([("warm", x)] + [("warm+ruido", x + rng.normal(scale=0.3, size=x.size)) for _ in range(N_PERTURB)]
                      + [("azar", rng.normal(scale=0.7, size=x.size)) for _ in range(N_RANDOM)])
            runs = []
            for kind, x0 in starts:
                r = optimize(d, x0)
                runs.append((r, f"{direction} {kind}"))
                starts_log.append({"d": d, "direction": direction, "start": kind, **diagnostics(d, r.x),
                                   "nit": r.nit, "converged": r.success})
            r, origin = min(runs, key=lambda t: t[0].fun)
            x = r.x
            if d not in best or r.fun < best[d][0].fun:
                best[d] = (r, origin)
    starts_log = pd.DataFrame(starts_log)

    # 2) Pulido: cada d se reoptimiza desde el θ* final de sus dos vecinos, hasta que nada mejore.
    #    Corrige los puntos donde ninguna de las dos pasadas encontró el mismo mínimo que sus vecinos.
    #    Cada ronda recorre la grilla en el sentido opuesto a la anterior, así una mejora se propaga entera.
    for n_polish in range(1, 21):
        improved = 0
        for i in (range(len(grid)) if n_polish % 2 else reversed(range(len(grid)))):
            for j in (i - 1, i + 1):
                if 0 <= j < len(grid):
                    r = optimize(grid[i], best[grid[j]][0].x)
                    if r.fun < best[grid[i]][0].fun - 1e-9:
                        best[grid[i]], improved = (r, f"pulido desde {grid[j]}"), improved + 1
        if not improved:
            break
    assert not improved, "el pulido no terminó en 20 rondas"

    params = {d: best[d][0].x for d in grid}  # contrato 0.1: d -> θ* (np.ndarray, en el orden de build_ansatz)
    opt = pd.DataFrame([{"d": d, **diagnostics(d, r.x), "e_fci": H_SECTOR[d][1], "origin": origin, "nit": r.nit,
                         "nfev": r.nfev, "converged": r.success, "grad_norm": np.linalg.norm(r.jac)}
                        for d, (r, origin) in best.items()]).sort_values("d", ignore_index=True)
    t_opt = time.perf_counter() - _t0

    # 3) Paisaje en la zona estirada (para #8): resultado desde 10 arranques distintos (HF, warm desde la
    #    distancia vecina ya optimizada y 8 al azar), cada uno por separado.
    ms_rows = []
    for d in MULTISTART_D:
        neighbor = grid[grid.index(d) - 1]
        starts = ([("HF (θ=0)", np.zeros(sector_ansatz.n)), (f"warm desde {neighbor}", params[neighbor])]
                  + [(f"azar {i}", rng.normal(scale=0.7, size=sector_ansatz.n)) for i in range(8)])
        for kind, x0 in starts:
            r = optimize(d, x0)
            ms_rows.append({"d": d, "start": kind, **diagnostics(d, r.x), "nit": r.nit, "converged": r.success})
    multistart = pd.DataFrame(ms_rows)
    os.makedirs("results", exist_ok=True)
    pd.DataFrame([{"d": d, **{f"theta_{k}": v for k, v in enumerate(params[d])}} for d in grid]).to_csv(
        "results/params.csv", index=False, float_format="%.17g")
    opt.to_csv("results/opt.csv", index=False)
    starts_log.to_csv("results/starts_log.csv", index=False)
    multistart.to_csv("results/multistart.csv", index=False)
else:
    _p = pd.read_csv(io.StringIO(PARAMS_CACHE["params"]), float_precision="round_trip")
    params = {float(r.d): np.array(r.iloc[1:], dtype=float) for _, r in _p.iterrows()}
    assert list(params) == grid and all(len(v) == sector_ansatz.n for v in params.values()), \
        "PARAMS_CACHE es de otra grilla u otro ansatz: correr con RUN_PARAMS = True"
    opt = pd.read_csv(io.StringIO(PARAMS_CACHE["opt"]), float_precision="round_trip")
    starts_log = pd.read_csv(io.StringIO(PARAMS_CACHE["starts_log"]))
    multistart = pd.read_csv(io.StringIO(PARAMS_CACHE["multistart"]))
    # Los θ* guardados reproducen la energía statevector guardada (si el Hamiltoniano o el ansatz cambian, salta acá).
    for d in grid:
        assert abs(diagnostics(d, params[d])["e_sv"] - opt.set_index("d").e_sv[d]) < 1e-9, \
            f"θ* guardado no reproduce e_sv en d={d}: correr con RUN_PARAMS = True"
    print(f"θ* guardados ({PARAMS_CACHE['origen']}). Para repetir la búsqueda: RUN_PARAMS = True.")


def vqe_state(d: float) -> Statevector:
    """Statevector de 8 qubits del circuito real en θ*(d) (para ⟨S²⟩ y la varianza en #8)."""
    return Statevector(ansatz.assign_parameters(params[float(d)]))


# El circuito real con el Hamiltoniano completo de 8 qubits da la misma energía que la simulación en el sector.
for d in (0.9, STRETCHED, 2.4, DISSOCIATED):
    op, e_nuc = qubit_hamiltonian(d)
    e_full = vqe_state(d).expectation_value(op).real + e_nuc
    assert abs(e_full - opt.set_index("d").e_sv[d]) < 1e-9, (d, e_full)

stretched = opt[opt.d >= STRETCHED]
assert len(params) == len(D) and opt.converged.all()
assert (opt.S2 < 1e-3).all(), opt  # singlete en todas las distancias (#8)
assert (stretched.err_mHa.abs() < CHEM_ACC * 1e3).all(), stretched
jump = np.abs(np.diff(opt.err_mHa)).max()  # suavidad: un salto del error entre vecinos delata un mínimo local
print((f"{time.perf_counter() - _t0:.0f} s ({t_opt:.0f} s optimización, {n_polish} rondas de pulido). " if RUN_PARAMS else
       f"{time.perf_counter() - _t0:.0f} s (θ* guardados). ") +
      f"Error máx. vs. FCI: {opt.err_mHa.abs().max():.4f} mHa (zona estirada: {stretched.err_mHa.abs().max():.4f} mHa); "
      f"salto máx. del error entre vecinos: {jump:.4f} mHa; ⟨S²⟩ máx.: {opt.S2.max():.1e}")
print("Multi-start en la zona estirada (10 arranques por distancia, sin elegir):")
print(multistart.groupby("d").agg(err_min=("err_mHa", "min"), err_mediana=("err_mHa", "median"),
                                  err_max=("err_mHa", "max"), S2_max=("S2", "max")).to_string(float_format="%.3g"))
opt

### Decisiones (#5)

**Qué deja esta celda.** `params = {d: θ*}` (contrato 0.1, un `np.ndarray` de 19 ángulos por distancia, en el orden de `build_ansatz`), la tabla `opt` (energía statevector, error vs. FCI, ⟨S²⟩, iteraciones, convergencia y de qué arranque salió cada θ*), `starts_log` (todos los arranques del barrido), `multistart` (10 arranques sin elegir a 1,8 / 2,4 / 3,0 Å) y `vqe_state(d)`, que regenera el statevector de 8 qubits del circuito real en θ*(d) para ⟨S²⟩ y la varianza (#8). Las energías de `opt` son **solo diagnóstico**: la tabla `vqe` del contrato (energía, σ, shots, `cz`) la arma #6 midiendo con `qiskit_aer`, nunca desde el statevector.

**El VQE completo corre en cada corrida.** Por defecto (`RUN_PARAMS = True`) la celda hace la búsqueda de θ* (~1 min) y #6 mide con esos θ*: optimización y medición en la misma corrida. Para una corrida rápida, `RUN_PARAMS = False` (como permite #5) carga `params` a precisión completa (17 cifras, así las energías que mide #6 son idénticas a las de una corrida completa), `opt`, `starts_log` y `multistart` desde `PARAMS_CACHE` (de una corrida de esa misma celda con `RUN_PARAMS = True`) y comprueba que los θ* guardados reproducen la energía statevector de `opt` (< 1e-9 Ha) y la del circuito real. La medición de #6 se hace igual en los dos casos.

**Resultado** (una corrida, semilla fija):

| | Valor |
|---|---|
| Error máximo vs. FCI, toda la grilla | 0,003 mHa (exacto a < 1e-7 mHa hasta 2,2 Å) |
| Error máximo vs. FCI, $d \geq 1{,}8$ Å | 0,003 mHa (criterio: 1,6 mHa) |
| Salto máximo del error entre distancias vecinas | 0,002 mHa (curva suave) |
| ⟨S²⟩ máximo | 2e-5 (singlete) |
| Convergencia | 19/19 (BFGS, $\lvert\nabla\rvert < 10^{-6}$), 45–140 iteraciones por punto |
| Tiempo de la celda | 60–75 s local (`RUN_PARAMS = True`, por defecto); < 1 s con los θ* guardados (`RUN_PARAMS = False`) |

Verificado: el circuito real (`ansatz` con θ*) y el Hamiltoniano de 8 qubits de #2 dan la misma energía que la simulación en el sector (< 1e-9 Ha, a 0,9 / 1,8 / 2,4 / 3,0 Å).

**Simulador.** La búsqueda usa `sector_ansatz` de la sección 3: el ansatz conserva N y S_z, así que simular en los 36 estados del sector es exacto y mucho más rápido que con 256 amplitudes.

**Optimizador: BFGS con gradiente adjunto exacto.** El gradiente completo sale de una pasada hacia adelante y otra hacia atrás sobre el estado (verificado contra diferencias finitas). La regla de *parameter-shift* de 2 términos no sirve acá: el generador de cada excitación tiene autovalores {−1, 0, 1} (frecuencias 1 y 2: 4 evaluaciones con la regla generalizada) y las simples comparten ángulo en α y β (autovalores −2…2: 8 evaluaciones), o sea 9·4 + 10·8 = 116 circuitos por gradiente. Además cada θ aparece en varias compuertas (8 `ry` por doble). Con statevector, el gradiente adjunto es exacto y cuesta del orden de una evaluación de la energía.

**Warm start en las dos direcciones + multi-start + pulido.**

1. Se recorre la grilla 0,6 → 3,0 y 3,0 → 0,6. En cada $d$ se arranca desde θ* de la distancia anterior (*warm*), desde 2 perturbaciones de ese θ* (ruido gaussiano σ = 0,3) y desde 3 puntos al azar (σ = 0,7), y se queda el de menor costo. Entre las dos pasadas se guarda el mejor por punto.
2. Pulido: cada $d$ se reoptimiza desde el θ* final de sus dos vecinos, alternando el sentido del recorrido, hasta que ningún punto mejore (> 1e-9 Ha; 3 rondas).

Cada pieza aporta: el mejor por punto de una sola dirección queda a 0,06 mHa (ida) o 0,11 mHa (vuelta); combinando las dos y puliendo, a 0,003 mHa. Por tipo de arranque, en todo el barrido: desde *warm* el 95 % de las corridas llega a precisión química y el 47 % a < 0,01 mHa; desde el azar, el 75 % y el 10 %.

**Penalización de espín: se minimiza $E(\theta) + \lambda \langle S^2\rangle(\theta)$, con λ = 0,05 Ha.** ⟨S²⟩ es otro valor esperado, así que el gradiente adjunto se aplica a $H + \lambda S^2$ sin cambios. En el singlete exacto la penalización vale 0 y el mínimo no se mueve; `opt.e_sv` es la energía sin penalizar. Sin ella, con otra semilla el barrido terminó con ⟨S²⟩ = 3e-2 a 2,8 Å (≈ 1,5 % de triplete) y un error de solo 0,02 mHa: el triplete está a 0,5 mHa a 3,0 Å, así que la energía no lo delata (ver la limitación de la sección 3). Con la penalización, en tres semillas ⟨S²⟩ ≤ 2e-4 y el error ≤ 0,03 mHa.

**Multi-start en la zona estirada** (`multistart`, error vs. FCI en mHa, 10 arranques por distancia):

| $d$ (Å) | HF (θ = 0) | *warm* desde el vecino | mejor de 8 al azar | mediana de 8 al azar | peor de 8 al azar |
|---|---|---|---|---|---|
| 1,8 | 153 | < 1e-8 | 0,006 | 0,3 | 63 |
| 2,4 | 230 | 0,007 | 0,02 | 2,7 | 10,5 |
| 3,0 | 287 | 0,002 | 0,0002 | 0,12 | 1,3 |

**Qué no funcionó.**

- **Arrancar desde HF (θ = 0)**: cae en un punto estacionario a 10 mHa (0,6 Å) y hasta 287 mHa (3,0 Å). Por eso el primer punto de cada pasada depende de los arranques al azar.
- **Un solo arranque**: uno al azar queda fuera de precisión química en ~25 % de las corridas (hasta decenas de mHa) e incluso uno *warm* en ~5 %, como ya había visto #4.
- **L-BFGS-B**: llega a los mismos mínimos, pero tarda 2–4× más (con `ftol` relativo de 1e-13 necesita 300–900 iteraciones). **Ajustar la tolerancia** (`gtol` de 1e-6 a 1e-10, o L-BFGS-B con `ftol = 1e-15`) desde θ* no baja el error ni un µHa: los errores residuales de ~1e-3 mHa son mínimos locales reales, no falta de convergencia.
- **Reescribir el gradiente con productos matriz-vector** (en vez de armar cada $e^{\theta K}$): solo 15 % más rápido, porque domina el costo de Python; se reutiliza el de la sección 3.

**Limitaciones.** El paisaje tiene mínimos locales hasta el nivel del µHa, así que los últimos dígitos de `params` dependen de la semilla (el error se mantiene ≤ 0,04 mHa en todas las semillas probadas, con y sin penalización). Diferencias numéricas mínimas (del orden de 1e-13 Ha) cambian a qué mínimo cae un arranque al azar: entre dos corridas del mismo código, el mejor de los 8 al azar a 3,0 Å pasó de 0,0002 a 0,002 mHa. Elegir por $E + \lambda\langle S^2\rangle$ evita que uno de esos empates elija un mínimo contaminado (queda λ⟨S²⟩ por encima); entre mínimos singlete solo cambian los µHa. Por lo mismo, θ* cambia entre corridas aunque la energía casi no: para comparar corridas usar energías y ⟨S²⟩, no los ángulos.

### 4.2 Medición de la energía con `SamplerV2`: energía, σ, shots y `cz` (#6)

Con los parámetros óptimos $\theta^*(d)$ fijos (de la optimización de arriba, #5), acá producimos la **energía reportada** de
cada punto de la grilla. La regla es dura: el número que va a la tabla sale de **medir circuitos** con
`qiskit_aer.primitives.SamplerV2` con una cantidad explícita de shots, nunca del statevector.

Receta, en una línea: partimos $H = c_I + \sum_k c_k P_k$ en grupos de Paulis que conmutan qubit a qubit (QWC), medimos cada
grupo en su base con `SamplerV2`, calculamos el valor **por shot** del grupo entero y combinamos
$E = e_\text{nuc} + c_I + \sum_g \bar h_g$, $\;\sigma^2 = \sum_g \operatorname{Var}_g / N_g$. Los shots se reparten entre grupos
como $N_g \propto \sqrt{\operatorname{Var}_g}$ (reparto óptimo) usando una corrida piloto, y el total por punto se dimensiona
a partir del margen de CCSD.

Salida: la tabla del contrato `vqe` con columnas `d, energy, sigma, shots, cz`, y la función reutilizable `measure_energy(...)`.

In [ ]:
# --- 4.2.0 Perillas de la medición ---------------------------------------------------------------
import os
import time
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter, ParameterVector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2, EstimatorV2

# SHOTS_SCALE multiplica TODOS los presupuestos de shots de esta sección (tabla y validaciones).
# 1.0 = corrida completa (lo reportado). Para una prueba rápida: SHOTS_SCALE = 0.02 (σ crece como 1/sqrt(escala)).
# También se puede fijar desde afuera con la variable de entorno QJ_SHOTS_SCALE.
SHOTS_SCALE = float(os.environ.get("QJ_SHOTS_SCALE", 1.0))

SEED = 2026                    # semilla base: todo lo de esta sección es reproducible
SIGMA_DEFAULT = 3.0e-3         # Ha: σ objetivo "sano" en toda la curva
SIGMA_MARGIN_FRACTION = 0.2    # en d >= STRETCHED: σ objetivo <= 1/5 del margen |E_CCSD - E_FCI|
SIGMA_CHEM_ACC = 0.8e-3        # Ha: en d >= STRETCHED, σ <= 0,8 mHa para apuntar a precisión química (2σ < 1,6 mHa)
PILOT_SHOTS_PER_GROUP = 500    # corrida piloto (solo para estimar Var_g y repartir; no entra en E ni σ)
MIN_GROUP_SHOTS = 100          # piso de shots por grupo en la corrida principal
MAX_SHOTS_PER_POINT = 6_000_000  # tope de seguridad por punto (si se alcanza, se avisa); σ = 0,8 mHa pide ~3-4 M

_T0_MEDICION = time.time()
print(f"SHOTS_SCALE = {SHOTS_SCALE}  (1.0 = corrida reportada)")

#### Piezas que todavía no están en `main`: stand-ins que se apagan solos

Esta sección necesita `references(d)` (#1), `qubit_hamiltonian(d)` (#2), `build_ansatz()` (#4) y la tabla `params` (#5).
Para no esperar, cada pieza se pide primero a la función real del contrato; si levanta `NotImplementedError` (o la
tabla `params` no existe), se usa un **stand-in** definido acá (`_standin_*`) y se imprime un aviso de una línea.
Cuando los compañeros suban su código, se usa la pieza real automáticamente, sin tocar esta sección.

- **Referencias (stand-in de #1):** RHF, CCSD y FCI con PySCF. Solo se usan para dimensionar shots (margen de CCSD) y para
  validar el Hamiltoniano; nunca como resultado.
- **Hamiltoniano (stand-in de #2):** Jordan-Wigner escrito a mano en la convención del contrato (adaptado de
  `scripts/h4_reference.py` de la rama de análisis), validado contra FCI diagonalizando en el sector $N=4$, $S_z=0$.
- **Ansatz y parámetros (stand-in de #4 y #5):** UCCSD de un paso (26 excitaciones de espín-orbital, conserva $N_\alpha$ y
  $N_\beta$) que arranca en `HF_BITSTRING`, con $\theta$ ajustado por statevector (permitido para *buscar* parámetros).
  **No llega a FCI** en la zona estirada (UCCSD de un paso se traba a unos mHa); alcanza para construir y validar la medición.

In [ ]:
# --- 4.2.1 Stand-ins de #1 y #2 (se apagan solos cuando existan references / qubit_hamiltonian) ---
from pyscf import ao2mo, cc, fci, gto, scf
from pyscf import lib as pyscf_lib

# Reproducibilidad bit a bit: PySCF reparte sumas entre hilos de OpenMP y el orden de suma cambia de corrida en corrida
# (diferencias de ~1e-14 que el ajuste de θ amplifica a ~1e-9 Ha y que terminan moviendo el reparto de shots). Con 1 hilo
# el resultado es idéntico. Para H4 en STO-3G los cálculos son diminutos: un hilo no es más lento (los hilos solo agregan costo).
pyscf_lib.num_threads(1)

STANDIN_USED = {}  # qué stand-ins se usaron en esta corrida (queda en el resumen final)


def _notice_standin(key, msg):
    if key not in STANDIN_USED:
        STANDIN_USED[key] = msg
        print(f"[stand-in] {msg}")


_STANDIN_SCF_CACHE = {}


def _standin_scf(d):
    """RHF de PySCF para H4 a distancia d (cacheado: lo comparten los stand-ins de #1 y #2)."""
    d = float(round(d, 4))
    if d not in _STANDIN_SCF_CACHE:
        mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", verbose=0)
        mf = scf.RHF(mol)
        mf.conv_tol = 1e-10
        mf.run()
        _STANDIN_SCF_CACHE[d] = (mol, mf)
    return _STANDIN_SCF_CACHE[d]


def _standin_references(d):
    """STAND-IN de #1: {'d','e_hf','e_ccsd','e_fci'} con PySCF (solo para comparar y dimensionar shots)."""
    mol, mf = _standin_scf(d)
    mycc = cc.CCSD(mf)
    mycc.max_cycle = 500
    mycc.run()
    e_fci = fci.FCI(mf).kernel()[0]
    return {"d": float(d), "e_hf": float(mf.e_tot), "e_ccsd": float(mycc.e_tot), "e_fci": float(e_fci)}


# Jordan-Wigner en la convención del contrato: qubit p = (p, α), qubit 4+p = (p, β); labels de Qiskit se leen q7...q0.
def _standin_annihilation(j, n=N_QUBITS):
    def label(p):
        s = ["I"] * n
        for k in range(j):
            s[k] = "Z"
        s[j] = p
        return "".join(reversed(s))
    return SparsePauliOp([label("X"), label("Y")], [0.5, 0.5j])


_STANDIN_A = [_standin_annihilation(j) for j in range(N_QUBITS)]
_STANDIN_AD = [a.adjoint() for a in _STANDIN_A]
_STANDIN_OPCACHE = {}  # productos de operadores de JW: no dependen de d, se calculan una vez


def _standin_fop(*idx):
    """a†_{i0} a†_{i1} ... a_{..}: idx = (('+', p), ('-', q), ...) -> SparsePauliOp (cacheado)."""
    if idx not in _STANDIN_OPCACHE:
        op = None
        for kind, p in idx:
            f = _STANDIN_AD[p] if kind == "+" else _STANDIN_A[p]
            op = f if op is None else op @ f
        _STANDIN_OPCACHE[idx] = op.simplify(atol=1e-12)
    return _STANDIN_OPCACHE[idx]


def _standin_qubit_hamiltonian(d):
    """STAND-IN de #2: (H electrónico JW de 8 qubits en la base MO de RHF, e_nuc)."""
    mol, mf = _standin_scf(d)
    C = mf.mo_coeff
    h1 = C.T @ mf.get_hcore() @ C
    eri = ao2mo.restore(1, ao2mo.kernel(mol, C), C.shape[1])  # (pq|rs), notación de químicos
    so = lambda p, s: p + N_ORB * s
    acc = []
    for p, q in itertools.product(range(N_ORB), repeat=2):
        if abs(h1[p, q]) > 1e-12:
            for s in (0, 1):
                acc.append(h1[p, q] * _standin_fop(("+", so(p, s)), ("-", so(q, s))))
    # 1/2 Σ (pq|rs) a†_{pσ} a†_{rτ} a_{sτ} a_{qσ}
    for p, q, r, s_ in itertools.product(range(N_ORB), repeat=4):
        v = eri[p, q, r, s_]
        if abs(v) < 1e-12:
            continue
        for si, ti in itertools.product((0, 1), repeat=2):
            P, Q, R, S = so(p, si), so(q, si), so(r, ti), so(s_, ti)
            if P != R and Q != S:
                acc.append(0.5 * v * _standin_fop(("+", P), ("+", R), ("-", S), ("-", Q)))
    op = SparsePauliOp.sum(acc).simplify(atol=1e-10)
    # PySCF usa hilos y los coeficientes pueden cambiar en ~1e-14 entre corridas; redondear a 1e-12 Ha deja el
    # Hamiltoniano idéntico bit a bit y hace reproducible todo lo que sigue (ajuste de θ, reparto de shots).
    op = SparsePauliOp(op.paulis, np.round(op.coeffs.real, 12))
    return op, float(mol.energy_nuc())


def get_references(d):
    """references(d) del contrato (#1); si todavía no existe, el stand-in de PySCF."""
    try:
        return references(d)
    except NotImplementedError:
        _notice_standin("references", "references(d) de #1 no está: uso _standin_references (PySCF RHF/CCSD/FCI).")
        return _standin_references(d)


def get_hamiltonian(d):
    """qubit_hamiltonian(d) del contrato (#2); si todavía no existe, el stand-in de Jordan-Wigner."""
    try:
        return qubit_hamiltonian(d)
    except NotImplementedError:
        _notice_standin("qubit_hamiltonian", "qubit_hamiltonian(d) de #2 no está: uso _standin_qubit_hamiltonian (JW propio).")
        return _standin_qubit_hamiltonian(d)


# Índices del sector N=4, Sz=0 (2 α en q0-q3, 2 β en q4-q7): 36 determinantes.
_SECTOR = [b for b in range(2 ** N_QUBITS)
           if bin(b).count("1") == N_ALPHA + N_BETA and bin(b & 0b1111).count("1") == N_ALPHA]

_t = time.time()
REF, HAM = {}, {}
for d in D:
    REF[d] = get_references(d)
    HAM[d] = get_hamiltonian(d)
    # Control (barato) de que el Hamiltoniano usado reproduce FCI en el sector físico: vale para el real y para el stand-in.
    op, e_nuc = HAM[d]
    e0 = np.linalg.eigvalsh(op.to_matrix()[np.ix_(_SECTOR, _SECTOR)].real)[0] + e_nuc
    assert abs(e0 - REF[d]["e_fci"]) < 1e-6, f"el Hamiltoniano no reproduce FCI en d={d}: {e0} vs {REF[d]['e_fci']}"

_n_terms = {len(HAM[d][0]) for d in D}
print(f"Hamiltonianos y referencias para {len(D)} distancias en {time.time() - _t:.1f} s; "
      f"términos de Pauli por punto: {sorted(_n_terms)}; FCI reproducida en el sector N=4, Sz=0 (|ΔE| < 1e-6 Ha).")

In [ ]:
# --- 4.2.2 Ansatz y θ*: build_ansatz() de #4 y params de #5, o stand-in ---------------------------
# ===================== BLOQUE STAND-IN (#4 y #5) =====================================================
# UCCSD de un paso sobre HF_BITSTRING. Cada excitación es exp(θ_k G_k) con G_k = T_k - T_k† (antihermítico, real).
# Como G_k^3 = -G_k, exp(θ G) = I + sin θ G + (1 - cos θ) G^2: eso permite simular rapidísimo en numpy y sacar el
# gradiente exacto en modo reverso para ajustar θ por statevector (permitido: es BUSCAR parámetros, no reportar).
_OCC_A, _VIR_A = [alpha_qubit(p) for p in (0, 1)], [alpha_qubit(p) for p in (2, 3)]
_OCC_B, _VIR_B = [beta_qubit(p) for p in (0, 1)], [beta_qubit(p) for p in (2, 3)]


def _standin_excitations():
    singles = [((i,), (a,)) for occ, vir in ((_OCC_A, _VIR_A), (_OCC_B, _VIR_B)) for i in occ for a in vir]
    doubles = [(tuple(_OCC_A), tuple(_VIR_A)), (tuple(_OCC_B), tuple(_VIR_B))]
    doubles += [((i, j), (a, b)) for i in _OCC_A for j in _OCC_B for a in _VIR_A for b in _VIR_B]
    return doubles + singles  # 18 dobles + 8 simples = 26


def _standin_generators():
    """Lista de H_k = i (T_k - T_k†) (hermíticos) como SparsePauliOp: exp(-i θ H_k) = exp(θ G_k)."""
    gens = []
    for occ, vir in _standin_excitations():
        T = _standin_fop(*([("+", a) for a in vir] + [("-", i) for i in reversed(occ)]))
        Hk = (1j * (T - T.adjoint())).simplify(atol=1e-12)
        assert np.allclose(Hk.coeffs.imag, 0)
        gens.append(SparsePauliOp(Hk.paulis, Hk.coeffs.real))
    return gens


_STANDIN_GENS = _standin_generators()


def _standin_build_ansatz():
    """STAND-IN de #4: UCCSD de un paso (26 parámetros), conserva N_α y N_β, arranca en HF_BITSTRING."""
    theta = ParameterVector("θ", len(_STANDIN_GENS))
    qc = QuantumCircuit(N_QUBITS, name="uccsd_standin")
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for k, Hk in enumerate(_STANDIN_GENS):
        qc.append(PauliEvolutionGate(Hk, time=theta[k]), range(N_QUBITS))
    return qc


def _standin_fit_uccsd(ops, e_nucs):
    """STAND-IN de #5: θ* por punto con statevector numpy + gradiente analítico (L-BFGS-B), arranque en caliente."""
    from scipy.optimize import minimize
    # Todo conserva N_α y N_β: alcanza con simular en el sector de 36 determinantes (matrices 36x36).
    sec = np.array(_SECTOR)
    G = [Hk.to_matrix().imag[np.ix_(sec, sec)] for Hk in _STANDIN_GENS]  # H_k = i G_k, G_k real antisim. -> G_k = Im(H_k)
    G2 = [g @ g for g in G]
    psi0 = (sec == int(HF_BITSTRING, 2)).astype(float)

    def energy_grad(th, Hm):
        states = [psi0]
        for k, t in enumerate(th):
            s = states[-1]
            states.append(s + np.sin(t) * (G[k] @ s) + (1 - np.cos(t)) * (G2[k] @ s))
        psi = states[-1]
        lam = Hm @ psi
        e = psi @ lam
        grad = np.zeros(len(th))
        for k in range(len(th) - 1, -1, -1):
            s_prev = states[k]
            dU = np.cos(th[k]) * (G[k] @ s_prev) + np.sin(th[k]) * (G2[k] @ s_prev)
            grad[k] = 2 * lam @ dU
            # lam <- U_k^T lam = U_k(-θ) lam
            lam = lam - np.sin(th[k]) * (G[k] @ lam) + (1 - np.cos(th[k])) * (G2[k] @ lam)
        return e, grad

    out, e_fit, prev = {}, {}, np.zeros(len(G))
    for d in D:
        Hm = ops[d].to_matrix().real[np.ix_(sec, sec)]
        best = None
        for x0 in (prev, np.zeros(len(G))):
            r = minimize(energy_grad, x0, args=(Hm,), jac=True, method="L-BFGS-B", options={"maxiter": 2000, "gtol": 1e-9})
            if best is None or r.fun < best.fun:
                best = r
        # θ* redondeado a 1e-6 rad: el ajuste en caliente arrastra diferencias de ~1e-12 entre máquinas/corridas, que
        # pueden cambiar el redondeo del reparto de shots y romper la reproducibilidad bit a bit (en un mínimo, el cambio
        # de energía por redondear es ~1e-12 Ha).
        theta = np.round(best.x, 6)
        out[d], e_fit[d] = theta, float(energy_grad(theta, Hm)[0]) + e_nucs[d]
        prev = best.x
    return out, e_fit
# =================== FIN BLOQUE STAND-IN (#4 y #5) ===================================================


def _fit_generic_statevector(qc, ops):
    """STAND-IN de #5 para un ansatz real sin params todavía: θ* por statevector (genérico, más lento)."""
    from scipy.optimize import minimize
    out, prev = {}, np.zeros(qc.num_parameters)
    for d in D:
        Hm = ops[d].to_matrix()

        def f(x):
            sv = Statevector(qc.assign_parameters(x)).data
            return float(np.real(np.vdot(sv, Hm @ sv)))

        r = minimize(f, prev, method="L-BFGS-B", options={"maxiter": 500})
        out[d] = prev = r.x
    return out


try:
    ansatz = build_ansatz()
except NotImplementedError:
    _notice_standin("build_ansatz", "build_ansatz() de #4 no está: uso _standin_build_ansatz (UCCSD de un paso, 26 parámetros).")
    ansatz = _standin_build_ansatz()

_params_ok = ("params" in globals() and isinstance(globals()["params"], dict)
              and all(d in globals()["params"] and len(globals()["params"][d]) == ansatz.num_parameters for d in D))
_t = time.time()
if _params_ok:
    THETA = {d: np.asarray(globals()["params"][d], dtype=float) for d in D}
elif "build_ansatz" in STANDIN_USED:
    _notice_standin("params", "params de #5 no está: θ* ajustado por statevector dentro del bloque stand-in (no llega a FCI).")
    THETA, _E_FIT = _standin_fit_uccsd({d: HAM[d][0] for d in D}, {d: HAM[d][1] for d in D})
else:
    _notice_standin("params", "params de #5 no está: θ* ajustado por statevector (genérico) sobre el ansatz de #4.")
    THETA = _fit_generic_statevector(ansatz, {d: HAM[d][0] for d in D})

# Circuito que se mide en cada punto: ansatz con θ* asignado, transpilado UNA vez a la base de Aer (más liviano que
# transpile_line, que se usa solo para contar cz). Su energía statevector (lo que #5 reporta como óptimo) sirve solo
# para VALIDAR la medición: nunca va a la tabla.
_AER = AerSimulator()
CIRCUIT, E_SV = {}, {}
for d in D:
    op, e_nuc = HAM[d]
    CIRCUIT[d] = transpile(ansatz.assign_parameters(THETA[d]), _AER, optimization_level=1, seed_transpiler=0)
    E_SV[d] = float(Statevector(CIRCUIT[d]).expectation_value(op).real) + e_nuc
if "_E_FIT" in globals() and "params" in STANDIN_USED and "build_ansatz" in STANDIN_USED:
    # el simulador numpy del ajuste y el circuito de Qiskit tienen que ser el mismo estado
    assert max(abs(_E_FIT[d] - E_SV[d]) for d in D) < 1e-8, "el circuito stand-in no reproduce el ajuste numpy"

CZ_ANSATZ = cz_count(ansatz)  # contrato: transpile_line (cz, sx, x, rz; acoplamiento lineal) sobre el ansatz
print(f"Ansatz: {ansatz.num_parameters} parámetros, {CZ_ANSATZ} cz (transpile_line). θ* listo en {time.time() - _t:.1f} s.")
print(pd.DataFrame({"d": D,
                    "E_sv - E_FCI (mHa)": [(E_SV[d] - REF[d]["e_fci"]) * 1e3 for d in D],
                    "E_CCSD - E_FCI (mHa)": [(REF[d]["e_ccsd"] - REF[d]["e_fci"]) * 1e3 for d in D]}).round(3).to_string(index=False))

#### Cómo medimos: grupos QWC, valor por shot del grupo y reparto óptimo

1. **Grupos.** Los términos de Pauli no identidad se ordenan por $|c_k|$ decreciente y se insertan en el primer grupo con
   el que conmutan qubit a qubit (*sorted insertion*). Un grupo QWC tiene una base de medición por qubit (X, Y o Z): se rota
   con `H` (X) o `S†H` (Y) y se mide todo en Z con un solo circuito.
2. **Valor por shot del grupo.** Cada bitstring $x$ da un número $h_g(x) = \sum_{k\in g} c_k (-1)^{\text{paridad}(x\,\&\,\text{soporte}_k)}$.
   Promediamos $h_g$ sobre los shots y tomamos su varianza muestral $\operatorname{Var}_g$. Como es la varianza del grupo
   **entero**, ya incluye las covarianzas entre los términos que se miden juntos (sumarlas término a término las ignoraría).
3. **Error.** Los grupos se miden con shots independientes, así que $\sigma^2 = \sum_g \operatorname{Var}_g / N_g$.
4. **Reparto.** A total $N$ fijo, $\sigma^2$ es mínima con $N_g \propto \sqrt{\operatorname{Var}_g}$, y entonces
   $\sigma = K/\sqrt{N}$ con $K = \sum_g \sqrt{\operatorname{Var}_g}$. Las $\operatorname{Var}_g$ salen de una **corrida piloto**
   corta (500 shots por grupo) que solo decide el reparto: la energía y la σ salen de la corrida principal, con shots
   nuevos, para que el reparto no sesgue la estimación. Los shots del piloto **sí** se cuentan en `shots` (costo real).
5. **Cómo se corre.** Un solo circuito por punto: ansatz con $\theta^*$ + una rotación $U(\theta_q, 0, \lambda_q)$ parametrizada en cada
   qubit + medición. Cada grupo es una *asignación* de esas rotaciones dentro de **un** PUB de `SamplerV2`, repetida
   $m_g$ veces con el mismo número de shots, así $N_g = m_g \cdot n_\text{paquete}$. Razón principal: Aer **corre en jobs
   separados los PUBs con distinto número de shots, todos con la misma semilla**, y los grupos quedaban correlacionados y la σ
   subestimada (lo vimos: dispersión entre semillas ≈ 1,4 σ). Dentro de un PUB cada asignación usa su propia corriente
   aleatoria. Con esto mismo, las **repeticiones** de la validación son bindings extra del mismo PUB.
6. **Costo.** Lo que cuesta en Aer es el *binding* (~2 ms: reescribe y fusiona las ~1000 compuertas del ansatz en cada uno), no
   los shots ni el job. Por eso usamos `method="statevector"` explícito (con `automatic` es 3 veces más lento), bindings en
   paralelo y ~2 paquetes por grupo. Aer se queda sin memoria con miles de bindings en un job (`bad allocation`), así que se
   parten en jobs de a lo sumo 600.

In [ ]:
# --- 4.2.3 measure_energy: la función reutilizable ------------------------------------------------
_PAULI_CODE = {(False, False): 0, (True, False): 1, (True, True): 2, (False, True): 3}  # (x, z) -> I, X, Y, Z
_GROUP_CACHE = {}


def qwc_groups(op):
    """Agrupa los términos no identidad de `op` en conjuntos QWC (sorted insertion por |c| decreciente).

    Devuelve (c_identidad, grupos) con cada grupo = dict(basis: array (n,) con 0=I,1=X,2=Y,3=Z,
    terms: índices, table: array (2^n,) con el valor h_g(x) para cada bitstring entero x).
    """
    key = (tuple(op.paulis.to_labels()), np.asarray(op.coeffs).tobytes())
    if key in _GROUP_CACHE:
        return _GROUP_CACHE[key]
    coeffs = np.asarray(op.coeffs)
    assert np.allclose(coeffs.imag, 0, atol=1e-10), "el observable tiene que ser hermítico (coeficientes reales)"
    coeffs = coeffs.real
    n = op.num_qubits
    xs, zs = op.paulis.x, op.paulis.z
    codes = np.array([[_PAULI_CODE[(bool(x), bool(z))] for x, z in zip(xr, zr)] for xr, zr in zip(xs, zs)])
    c_id = float(coeffs[(codes == 0).all(axis=1)].sum())
    groups = []
    for k in np.argsort(-np.abs(coeffs), kind="stable"):
        if (codes[k] == 0).all():
            continue
        for g in groups:
            if np.all((codes[k] == 0) | (g["basis"] == 0) | (codes[k] == g["basis"])):
                g["basis"] = np.where(g["basis"] == 0, codes[k], g["basis"])
                g["terms"].append(int(k))
                break
        else:
            groups.append({"basis": codes[k].copy(), "terms": [int(k)]})
    xs_all = np.arange(2 ** n)
    for g in groups:
        table = np.zeros(2 ** n)
        for k in g["terms"]:
            mask = int(sum(1 << q for q in range(n) if codes[k, q] != 0))
            parity = np.array([bin(v).count("1") & 1 for v in (xs_all & mask)])
            table += coeffs[k] * (1 - 2 * parity)
        g["table"] = table
    _GROUP_CACHE[key] = (c_id, groups)
    return c_id, groups


_ROTATION = {0: (0.0, 0.0), 3: (0.0, 0.0), 1: (np.pi / 2, np.pi), 2: (np.pi / 2, np.pi / 2)}  # U(θ,0,λ): I, H, H·S†
UNITS_PER_GROUP = 2  # granularidad del reparto: en promedio cada grupo recibe ~2 "paquetes" de shots (cada paquete = 1 binding)
# Costo de Aer = ~2 ms por binding (reescribe y fusiona las ~1000 compuertas del ansatz en cada uno), casi independiente de los
# shots. Por eso: método statevector explícito (el "automatic" gasta 3x en elegir método por binding) y bindings en paralelo.
# Las semillas de Aer son por binding: el resultado es idéntico bit a bit con cualquier número de hilos (lo verificamos).
MAX_BINDINGS_PER_JOB = 600
_AER_OPTIONS = {"method": "statevector", "max_parallel_experiments": os.cpu_count() or 1}


def _measurement_template(circuit_bound):
    """circuito + U(θ_q, 0, λ_q) en cada qubit + medición de todo en Z.

    Cada grupo QWC es una ASIGNACIÓN de (θ, λ): I/Z -> U(0,0,0), X -> U(π/2,0,π) = H, Y -> U(π/2,0,π/2) = H·S†.
    Así Aer compila el circuito una sola vez por job y cada grupo es un binding del mismo PUB.
    """
    names = set(_AER.target.operation_names)
    base = circuit_bound
    if any(inst.operation.name not in names for inst in base.data):
        base = transpile(base, _AER, optimization_level=1, seed_transpiler=0)
    n = base.num_qubits
    th, la = ParameterVector("_rot_theta", n), ParameterVector("_rot_lambda", n)
    qc = base.copy()
    for q in range(n):
        qc.u(th[q], 0.0, la[q], q)
    qc.measure_all()
    order = {p: i for i, p in enumerate(qc.parameters)}  # orden en que SamplerV2 espera los valores
    return qc, [order[p] for p in th], [order[p] for p in la]


def _sample(template, groups, bindings_per_group, shots_per_binding, seed):
    """UN job de SamplerV2: un PUB con sum(bindings) asignaciones y `shots_per_binding` shots explícitos cada una.

    bindings_per_group : (G,) o (R, G): R estimaciones independientes (repeticiones) dentro del MISMO job.
    Aer usa una corriente aleatoria distinta para cada binding del PUB, así que grupos y repeticiones son independientes.
    (Ojo: PUBs con distinto número de shots van en jobs separados con la MISMA semilla y quedan correlacionados;
    por eso no usamos un PUB por grupo ni uno por repetición con shots distintos.)
    Devuelve (medias, varianzas por shot, shots) con la forma de bindings_per_group.
    """
    qc, i_th, i_la = template
    n = qc.num_qubits
    one_d = np.ndim(bindings_per_group) == 1
    mm = np.atleast_2d(np.asarray(bindings_per_group, dtype=int))  # (R, G)
    R, G = mm.shape
    base = []
    for g in groups:
        v = np.zeros(qc.num_parameters)
        for q, b in enumerate(g["basis"]):
            v[i_th[q]], v[i_la[q]] = _ROTATION[int(b)]
        base.append(v)
    idx = np.repeat(np.arange(R * G), mm.ravel())     # (repetición, grupo) de cada binding
    rows = np.array(base)[idx % G]
    # Aer expande cada binding en un experimento completo en memoria: con miles de bindings en un job se queda sin memoria
    # ("bad allocation"). Por eso los bindings se parten en jobs de a lo sumo MAX_BINDINGS_PER_JOB, cada uno con su semilla.
    ints = []
    for k, lo in enumerate(range(0, len(rows), MAX_BINDINGS_PER_JOB)):
        seed_k = int(np.random.SeedSequence([int(seed), k]).generate_state(1)[0]) % (2 ** 31 - 2 ** 20)
        sampler = SamplerV2(seed=seed_k, options={"backend_options": _AER_OPTIONS})
        res = sampler.run([(qc, rows[lo:lo + MAX_BINDINGS_PER_JOB], int(shots_per_binding))]).result()[0]
        arr = res.data.meas.array.astype(np.int64)  # (bindings, shots, bytes), big-endian por byte
        x = np.zeros(arr.shape[:2], dtype=np.int64)
        for byte in range(arr.shape[2]):
            x = (x << 8) | arr[:, :, byte]
        ints.append(x)
    ints = np.concatenate(ints)
    counts = np.zeros((R * G, 2 ** n))
    for i, o in enumerate(idx):
        counts[o] += np.bincount(ints[i], minlength=2 ** n)
    counts = counts.reshape(R, G, 2 ** n)
    shots = counts.sum(axis=2)
    tables = np.array([g["table"] for g in groups])
    means = (counts * tables).sum(axis=2) / shots
    vars_ = (counts * (tables[None] - means[:, :, None]) ** 2).sum(axis=2) / (shots - 1)
    shots = shots.astype(int)
    return (means[0], vars_[0], shots[0]) if one_d else (means, vars_, shots)


def measure_energy_many(circuit_bound, op_electronic, e_nuc=0.0, total_shots=None, seed=0, *, n_repeats=1, target_sigma=None,
                        pilot_shots_per_group=PILOT_SHOTS_PER_GROUP, min_group_shots=MIN_GROUP_SHOTS,
                        max_shots=MAX_SHOTS_PER_POINT, return_details=False):
    """`n_repeats` estimaciones independientes (piloto + principal cada una) con solo DOS jobs de SamplerV2 en total.

    Todas las repeticiones comparten circuito, observable y presupuesto: las repeticiones son bindings extra del mismo PUB.
    Con n_repeats > 1 hay que pasar total_shots (con target_sigma cada repetición tendría un n_unit distinto). Devuelve una lista.
    """
    if circuit_bound.num_parameters:
        raise ValueError("measure_energy necesita un circuito con todos los parámetros asignados")
    if (total_shots is None) == (target_sigma is None):
        raise ValueError("pasar exactamente uno de total_shots o target_sigma")
    if n_repeats > 1 and total_shots is None:
        raise ValueError("con n_repeats > 1 hay que pasar total_shots")
    R = int(n_repeats)
    c_id, groups = qwc_groups(op_electronic)
    template = _measurement_template(circuit_bound)
    G = len(groups)
    seed_pilot, seed_main = (int(x) % (2 ** 31 - 2 ** 20) for x in np.random.SeedSequence(int(seed)).generate_state(2))

    # Piloto: mismo número de shots por grupo, SOLO para estimar Var_g (no entra en E ni en σ).
    _, var_pilot, n_pilot = _sample(template, groups, np.ones((R, G), int), max(int(pilot_shots_per_group), 20), seed_pilot)
    sq = np.sqrt(np.maximum(var_pilot, 0.0))                 # (R, G)
    sq[sq.sum(axis=1) == 0] = 1.0

    capped = False
    if total_shots is None:
        total_shots = int(np.ceil((sq[0].sum() / target_sigma) ** 2))
    if max_shots is not None and total_shots > max_shots:
        total_shots, capped = int(max_shots), True
    # Reparto N_g ∝ sqrt(Var_g) en paquetes de n_unit shots (n_unit >= min_group_shots: ningún grupo queda sin medir).
    n_unit = max(int(min_group_shots), int(total_shots) // (UNITS_PER_GROUP * G))
    m = np.maximum(1, np.round(total_shots * sq / sq.sum(axis=1, keepdims=True) / n_unit)).astype(int)

    means, var_main, n_main = _sample(template, groups, m, n_unit, seed_main)
    outs = []
    for r in range(R):
        out = dict(energy=float(e_nuc + c_id + means[r].sum()), sigma=float(np.sqrt(np.sum(var_main[r] / n_main[r]))),
                   shots=int(n_pilot[r].sum() + n_main[r].sum()), shots_pilot=int(n_pilot[r].sum()),
                   shots_main=int(n_main[r].sum()), groups=G, K=float(np.sqrt(var_main[r]).sum()), capped=capped)
        if return_details:
            out["details"] = pd.DataFrame({"basis (q7..q0)": ["".join("IXYZ"[b] for b in g["basis"][::-1]) for g in groups],
                                           "n_terms": [len(g["terms"]) for g in groups], "shots": n_main[r],
                                           "mean": means[r], "var_per_shot": var_main[r], "var_pilot": var_pilot[r]})
        outs.append(out)
    return outs


def measure_energy(circuit_bound, op_electronic, e_nuc=0.0, total_shots=None, seed=0, *, target_sigma=None,
                   pilot_shots_per_group=PILOT_SHOTS_PER_GROUP, min_group_shots=MIN_GROUP_SHOTS,
                   max_shots=MAX_SHOTS_PER_POINT, return_details=False):
    """Estima <H> = e_nuc + <op_electronic> midiendo `circuit_bound` con qiskit_aer SamplerV2 (shots explícitos).

    circuit_bound : circuito SIN parámetros libres y sin mediciones (si hace falta, se transpila para Aer adentro).
    total_shots   : shots de la corrida principal, repartidos N_g ∝ sqrt(Var_g de la piloto). O bien...
    target_sigma  : ...σ objetivo en Ha: la corrida principal usa N = (K_piloto / target_sigma)^2 (con tope max_shots).
    seed          : semilla entera; de ella salen las semillas de SamplerV2 de la piloto y de la principal (reproducible).

    Devuelve dict(energy, sigma, shots, shots_pilot, shots_main, groups, K, capped[, details]).
    `shots` cuenta TODOS los shots ejecutados (piloto + principal); `sigma` es el error estándar de `energy`.
    """
    return measure_energy_many(circuit_bound, op_electronic, e_nuc, total_shots, seed, n_repeats=1, target_sigma=target_sigma,
                               pilot_shots_per_group=pilot_shots_per_group, min_group_shots=min_group_shots,
                               max_shots=max_shots, return_details=return_details)[0]


_c_id, _groups18 = qwc_groups(HAM[STRETCHED][0])
print(f"d = {STRETCHED} Å: {len(HAM[STRETCHED][0])} términos -> c_I + {len(_groups18)} grupos QWC "
      f"(group_commuting de Qiskit da {len(HAM[STRETCHED][0].group_commuting(qubit_wise=True))}).")

#### Cuántos shots por punto

El criterio central pide $|E_\text{VQE} - E_\text{FCI}| + \sigma < |E_\text{CCSD} - E_\text{FCI}|$ en $d \ge 1{,}8$ Å. El margen más
chico es justo 1,8 Å (≈ 9,1 mHa), y desde 2,2 Å supera los 20 mHa. Regla:

$$\sigma_\text{obj}(d) = \begin{cases} \min\big(0{,}8\ \text{mHa},\ \tfrac15\,|E_\text{CCSD}-E_\text{FCI}|,\ 3\ \text{mHa}\big) & d \ge 1{,}8\ \text{Å} \\ 3\ \text{mHa} & d < 1{,}8\ \text{Å}\end{cases}$$

Así σ se come como mucho un quinto del margen y deja el resto para el sesgo del ansatz. El tope de 0,8 mHa apunta a la **meta
máxima** (precisión química, $|E-E_\text{FCI}| < 1{,}6$ mHa): con el ansatz exacto el error que queda es estadístico y $2\sigma < 1{,}6$ mHa.
En la práctica: 0,8 mHa en toda la zona estirada y 3 mHa en el resto. El total por punto sale de la piloto: $N = (K/\sigma_\text{obj})^2$ con $K = \sum_g\sqrt{\operatorname{Var}_g}$,
así que cada punto recibe los shots que **su** estado necesita (con $K \approx 1{,}5$ Ha son ~$3{,}5\times10^6$ shots por punto en $d \ge 1{,}8$ Å y
~$3\times10^5$ en el resto). `SHOTS_SCALE` escala todo para pruebas rápidas.

In [ ]:
# --- 4.2.4 Medición de toda la curva -> tabla del contrato `vqe` ------------------------------------


def sigma_target(d):
    """σ objetivo (Ha) en el punto d, a partir del margen de CCSD (ver la regla arriba)."""
    if d >= STRETCHED - 1e-9:
        margin = abs(REF[d]["e_ccsd"] - REF[d]["e_fci"])
        # Precisión química (meta máxima del enunciado): con un ansatz exacto el error que queda es estadístico,
        # así que σ <= 0,8 mHa deja |E - E_FCI| < 1,6 mHa con ~95 % de probabilidad. Cuesta ~3-4 M shots por punto.
        return min(SIGMA_CHEM_ACC, SIGMA_MARGIN_FRACTION * margin, SIGMA_DEFAULT)
    return SIGMA_DEFAULT


def _scaled(n, floor=1_000):
    """Presupuesto escalado por SHOTS_SCALE (con piso, para que las pruebas rápidas sigan teniendo sentido)."""
    return max(int(round(n * SHOTS_SCALE)), floor)


PILOT_SHOTS = _scaled(PILOT_SHOTS_PER_GROUP, floor=200)  # con escala chica la piloto no baja de 200 shots por grupo


_t = time.time()
_rows, MEAS = [], {}
for i, d in enumerate(D):
    op, e_nuc = HAM[d]
    # Con SHOTS_SCALE < 1 el σ objetivo se relaja en 1/sqrt(escala): los shots bajan en proporción.
    m = measure_energy(CIRCUIT[d], op, e_nuc, target_sigma=sigma_target(d) / np.sqrt(SHOTS_SCALE), seed=SEED + 100 * i,
                       pilot_shots_per_group=PILOT_SHOTS)
    MEAS[d] = m
    _rows.append(dict(d=float(d), energy=m["energy"], sigma=m["sigma"], shots=m["shots"], cz=CZ_ANSATZ))
    if m["capped"]:
        print(f"  aviso: d={d} llegó al tope de {MAX_SHOTS_PER_POINT} shots; σ queda por encima del objetivo")

vqe = pd.DataFrame(_rows, columns=["d", "energy", "sigma", "shots", "cz"])  # tabla del contrato (#6)
print(f"Curva medida con SamplerV2 en {time.time() - _t:.1f} s; shots totales = {vqe.shots.sum():,} "
      f"(piloto {sum(MEAS[d]['shots_pilot'] for d in D):,}).")

_show = vqe.assign(
    sigma_mHa=vqe.sigma * 1e3,
    sigma_obj_mHa=[sigma_target(d) * 1e3 / np.sqrt(SHOTS_SCALE) for d in D],
    grupos=[MEAS[d]["groups"] for d in D],
    K_Ha=[MEAS[d]["K"] for d in D],
    E_sv=[E_SV[d] for d in D],
    z=[(MEAS[d]["energy"] - E_SV[d]) / MEAS[d]["sigma"] for d in D],
)
print(_show.drop(columns="sigma").to_string(index=False, float_format=lambda x: f"{x:.6f}" if abs(x) > 50 or abs(x) < 1 else f"{x:.2f}"))

#### Validación de la medición

Tres pruebas, todas con `SamplerV2`:

1. **Medido vs. statevector en cada punto.** $z = (E_\text{medida} - E_\text{sv})/\sigma$ tiene que ser $|z| \lesssim 2$. Con 19
   puntos independientes es esperable que alguno caiga entre 2 y 3 (la probabilidad de que los 19 queden dentro de 2σ es
   ≈ 0,41), por eso además miramos $\chi^2 = \sum z^2$ contra 19 grados de libertad.
2. **σ es la dispersión real.** Repetimos la estimación completa (piloto + principal) 40 veces en tres distancias y
   comparamos el desvío estándar de las 40 energías con la σ que reporta cada corrida (si σ es correcta, el cociente cae en
   [0,82, 1,28] con 95 % de probabilidad). También miramos si la media de las 40 se aleja de $E_\text{sv}$.
3. **σ ∝ 1/√shots.** A 1,8 Å barremos el presupuesto en dos órdenes de magnitud y ajustamos la pendiente en log-log.

In [ ]:
# --- 4.2.5 Validación 1: medido vs. statevector en todos los puntos ----------------------------------
from scipy import stats

z = np.array([(MEAS[d]["energy"] - E_SV[d]) / MEAS[d]["sigma"] for d in D])
chi2 = float(np.sum(z ** 2))
p_chi2 = float(stats.chi2.sf(chi2, len(D)))
print(f"|z| <= 2 en {np.sum(np.abs(z) <= 2)}/{len(D)} puntos; |z| <= 3 en {np.sum(np.abs(z) <= 3)}/{len(D)}; "
      f"max |z| = {np.abs(z).max():.2f}; chi2 = {chi2:.1f} con {len(D)} gl (p = {p_chi2:.2f}).")
assert np.all(np.abs(z) < 3.5), "una energía medida está a más de 3.5σ del statevector: revisar la medición"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.errorbar(D, [(MEAS[d]["energy"] - E_SV[d]) * 1e3 for d in D], yerr=[MEAS[d]["sigma"] * 1e3 for d in D],
             fmt="o", ms=4, capsize=3, label="SamplerV2 − statevector (±σ)")
ax1.fill_between(D, [-2 * MEAS[d]["sigma"] * 1e3 for d in D], [2 * MEAS[d]["sigma"] * 1e3 for d in D],
                 color="0.85", label="±2σ")
ax1.axhline(0, color="k", lw=0.8)
ax1.axvline(STRETCHED, color="tab:red", ls=":", lw=1)
ax1.set(xlabel="d (Å)", ylabel="mHa", title="Energía medida − energía statevector")
ax1.legend(fontsize=8, loc="lower left")
ax2.plot(D, [MEAS[d]["sigma"] * 1e3 for d in D], "o-", ms=4, label="σ medida")
ax2.plot(D, [abs(REF[d]["e_ccsd"] - REF[d]["e_fci"]) * 1e3 for d in D], "s--", ms=4, label="|E_CCSD − E_FCI|")
ax2.axvline(STRETCHED, color="tab:red", ls=":", lw=1)
ax2.set(xlabel="d (Å)", ylabel="mHa", yscale="log", title="σ por punto vs. margen de CCSD")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.2.6 Validación 2: 50 repeticiones -> la dispersión coincide con σ ------------------------------
N_REPEATS = 40
REPEAT_SHOTS = _scaled(20_000, floor=10_000)  # el test es dispersión/σ: no necesita σ chica
_t = time.time()
rep_rows, REPEATS = [], {}
for j, d in enumerate((0.9, STRETCHED, DISSOCIATED)):
    d = float(D[np.argmin(np.abs(D - d))])
    op, e_nuc = HAM[d]
    # las 40 estimaciones completas (piloto + principal) salen de DOS jobs de SamplerV2: son bindings extra del mismo PUB
    runs = measure_energy_many(CIRCUIT[d], op, e_nuc, total_shots=REPEAT_SHOTS, seed=SEED + 10_000 + j, n_repeats=N_REPEATS,
                               pilot_shots_per_group=PILOT_SHOTS)
    E = np.array([r["energy"] for r in runs])
    S = np.array([r["sigma"] for r in runs])
    REPEATS[d] = (E, S)
    spread = E.std(ddof=1)
    lo, hi = np.sqrt((N_REPEATS - 1) / stats.chi2.ppf([0.975, 0.025], N_REPEATS - 1))  # IC 95% de spread/σ si σ es correcta
    rep_rows.append({"d": d, "shots por corrida": runs[0]["shots"], "σ media (mHa)": S.mean() * 1e3,
                     "dispersión (mHa)": spread * 1e3, "dispersión/σ": spread / S.mean(),
                     "IC95 esperado": f"[{lo:.2f}, {hi:.2f}]",
                     "(media − E_sv)/(σ/√N)": (E.mean() - E_SV[d]) / (S.mean() / np.sqrt(N_REPEATS))})
rep = pd.DataFrame(rep_rows)
print(f"{N_REPEATS} repeticiones x 3 distancias (2 jobs por distancia) en {time.time() - _t:.1f} s")
print(rep.round(3).to_string(index=False))
print("La última columna es ~N(0, 1) si no hay sesgo: con 3 puntos es normal que alguna llegue a ±2-3.")
assert all(0.6 < r < 1.6 for r in rep["dispersión/σ"]), "la dispersión entre semillas no coincide con σ"

fig, axs = plt.subplots(1, len(REPEATS), figsize=(11, 3.2))
for ax, (d, (E, S)) in zip(axs, REPEATS.items()):
    x = (E - E_SV[d]) * 1e3
    ax.hist(x, bins=10, density=True, color="tab:blue", alpha=0.6, label=f"{N_REPEATS} semillas")
    grid = np.linspace(-4, 4, 200) * S.mean() * 1e3
    ax.plot(grid, stats.norm.pdf(grid, 0, S.mean() * 1e3), "k-", label="N(0, σ reportada)")
    ax.set(title=f"d = {d} Å", xlabel="E − E_sv (mHa)")
axs[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.2.7 Validación 3: σ ∝ 1/sqrt(shots) a 1,8 Å ------------------------------------------------
d = float(STRETCHED)
op, e_nuc = HAM[d]
_k = max(SHOTS_SCALE, 0.1)  # el barrido no baja de 1/10 para que los pisos por grupo no lo distorsionen
budgets = [int(n * _k) for n in (20_000, 60_000, 200_000, 600_000)]
scan = [measure_energy(CIRCUIT[d], op, e_nuc, total_shots=n, seed=SEED + 20_000 + j, pilot_shots_per_group=PILOT_SHOTS)
        for j, n in enumerate(budgets)]
Nm = np.array([r["shots_main"] for r in scan])
Sg = np.array([r["sigma"] for r in scan])
slope = np.polyfit(np.log(Nm), np.log(Sg), 1)[0]
print(pd.DataFrame({"shots principal": Nm, "σ (mHa)": Sg * 1e3, "σ·sqrt(N) = K (Ha)": Sg * np.sqrt(Nm),
                    "E − E_sv (mHa)": [(r["energy"] - E_SV[d]) * 1e3 for r in scan]}).round(3).to_string(index=False))
print(f"pendiente log σ vs log N = {slope:.3f} (esperado −0.5)")
assert abs(slope + 0.5) < 0.08

plt.figure(figsize=(5, 3.5))
plt.loglog(Nm, Sg * 1e3, "o", label="σ medida (SamplerV2)")
plt.loglog(Nm, Sg[0] * np.sqrt(Nm[0] / Nm) * 1e3, "k--", lw=1, label="∝ 1/√N")
plt.errorbar(Nm, np.abs([(r["energy"] - E_SV[d]) * 1e3 for r in scan]), fmt="x", color="tab:orange", label="|E − E_sv|")
plt.axhline(abs(REF[d]["e_ccsd"] - REF[d]["e_fci"]) * 1e3, color="tab:red", ls=":", label="margen CCSD a 1,8 Å")
plt.xlabel("shots (corrida principal)")
plt.ylabel("mHa")
plt.title("σ ∝ 1/√shots a d = 1,8 Å")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.2.8 Control cruzado con EstimatorV2 (NO es un resultado) ---------------------------------
# EstimatorV2 de qiskit-aer NO muestrea: calcula el valor exacto y, si precision > 0, le suma ruido gaussiano de ese ancho;
# `stds` devuelve exactamente `precision`. Lo mostramos solo para confirmar que el circuito y el observable son los mismos.
est = EstimatorV2(options={"run_options": {"seed_simulator": SEED}})
_x = []
for d in (0.9, STRETCHED, DISSOCIATED):
    d = float(D[np.argmin(np.abs(D - d))])
    op, e_nuc = HAM[d]
    r0 = est.run([(CIRCUIT[d], op)], precision=0.0).result()[0]
    r1 = est.run([(CIRCUIT[d], op)], precision=MEAS[d]["sigma"]).result()[0]
    _x.append({"d": d, "SamplerV2 (reportada)": MEAS[d]["energy"], "σ SamplerV2": MEAS[d]["sigma"],
               "Estimator precision=0": float(r0.data.evs) + e_nuc, "statevector": E_SV[d],
               "Estimator precision=σ": float(r1.data.evs) + e_nuc, "stds devuelto": float(r1.data.stds)})
print("CONTROL CRUZADO (no reportado): EstimatorV2 no usa shots; con precision=0 es el statevector exacto.")
print(pd.DataFrame(_x).to_string(index=False, float_format=lambda v: f"{v:.6f}"))
assert all(abs(r["Estimator precision=0"] - r["statevector"]) < 1e-8 for r in _x)

print(f"\nStand-ins usados en esta corrida: {list(STANDIN_USED) or 'ninguno (todas las piezas reales)'}")
print(f"Tiempo total de la sección 4.2: {time.time() - _T0_MEDICION:.0f} s")
vqe

#### Qué decidimos, por qué y qué no funcionó (#6)

**Qué decidimos**

- **Las energías reportadas salen de `SamplerV2` con shots explícitos**, y E y σ se calculan desde los bitstrings.
  `EstimatorV2` de qiskit-aer aparece solo como control cruzado: no muestrea (valor exacto + ruido gaussiano de ancho
  `precision`, `stds = precision`). Con `precision=0` coincide con el statevector hasta $10^{-8}$ Ha, o sea que el chequeo
  "medido ≈ exacto" hecho con Estimator no prueba nada.
- **Grupos QWC por *sorted insertion*** (términos ordenados por $|c_k|$): 37–41 grupos para los 184 términos no identidad
  (`group_commuting(qubit_wise=True)` de Qiskit da 46). El greedy propio es determinista y nos deja precomputar la tabla
  $h_g(x)$ de cada grupo para los $2^8$ bitstrings.
- **Valor por shot del grupo entero**: su varianza muestral ya incluye las covarianzas entre los términos del grupo; grupos
  distintos usan shots independientes, así que $\sigma^2 = \sum_g \operatorname{Var}_g/N_g$.
- **Reparto óptimo $N_g \propto \sqrt{\operatorname{Var}_g}$ (con $\operatorname{Var}_g$ la varianza del grupo entero, que incluye covarianzas; no $\sum|c_k|\sigma_k$ por término) con piloto separada** (500 shots por grupo, 3–8 % del total por
  punto). La piloto solo decide el reparto y el total; E y σ salen de shots nuevos, así el reparto no se correlaciona con la
  estimación. Los shots de la piloto se cuentan en `shots` porque son costo real. Piso de 100 shots por grupo.
- **Shots por punto desde el margen de CCSD**: $\sigma_\text{obj} = \min(0{,}8\text{ mHa},\ \tfrac15|E_\text{CCSD}-E_\text{FCI}|,\ 3\text{ mHa})$ en
  $d\ge1{,}8$ Å y 3 mHa en el resto; el total sale de $N=(K/\sigma_\text{obj})^2$ con $K$ de la piloto. El tope de 0,8 mHa en la zona estirada apunta a la precisión química (meta máxima). Estimado a escala 1 (no medido localmente;
  se corre en Colab): ~3,5 × 10⁶ shots por punto en $d \ge 1{,}8$ Å y σ ≈ 3 mHa con 2,4–3,4 × 10⁵ shots en el resto; **≈ 2,8 × 10⁷ shots
  en toda la curva** (antes ≈ 6,0 × 10⁶ con σ ≈ 1,85 mHa a 1,8 Å). $K$ medido: 1,41–1,67 Ha, igual que lo que predice `docs/analysis/` (1,49 Ha a 1,8 Å).
- **Un PUB por estimación, cada grupo como asignación de rotaciones parametrizadas** $U(\theta_q,0,\lambda_q)$, repetida $m_g$
  veces con el mismo número de shots. Motivo principal: `SamplerV2` de Aer agrupa los PUBs por número de shots y corre cada
  grupo en un job con **la misma semilla**, así que PUBs con distinto número de shots salen correlacionados (medimos
  correlación 0,40 entre las medias de dos PUBs con 1000 y 1001 shots a lo largo de 200 semillas; 0,01 con los mismos shots).
  Dentro de un PUB cada asignación usa su propia corriente aleatoria (correlación ≈ 0). Bonus: Aer compila el circuito una
  sola vez por job.
- **`cz`** = `cz_count(ansatz)` del contrato (transpile_line) sobre el ansatz sin asignar, igual para todos los puntos. Para
  simular, el circuito se transpila a la base de Aer (más liviano que `transpile_line`).
- **Semillas**: cada estimación recibe una semilla entera (`SEED + 100·i` en la curva); de ella salen con `SeedSequence` dos
  semillas grandes, una para la piloto y otra para la principal (y una por cada job de 600 bindings). Todo es reproducible bit a bit
  (ver Reproducibilidad abajo: hubo que fijar PySCF a un hilo).
- **Stand-ins que se apagan solos** para #1, #2, #4 y #5 (avisan con una línea `[stand-in]`). El ansatz stand-in es UCCSD de un
  paso (26 parámetros, 1274 cz con transpile_line) ajustado por statevector en el sector de 36 determinantes: **no llega a FCI**
  (3,3 mHa arriba a 1,8 Å y 3,5 mHa a 2,0 Å). Sirve para construir y validar la medición, no es el resultado. El control
  "el Hamiltoniano reproduce FCI en el sector N=4, Sz=0" corre igual con el Hamiltoniano real.

**Validación** (corrida con `SHOTS_SCALE = 0,05`, la única que corrimos: 4,2 × 10⁵ shots en la curva, σ ≈ 13 mHa; a escala 1
σ baja ≈ 4,5 veces y el chequeo es más exigente; no la corrimos): $|z| \le 2$ en los 19 puntos, $\chi^2 = 13{,}4$ con 19 gl
($p = 0{,}82$). Con 40 repeticiones por punto, dispersión/σ = 0,91 (0,9 Å), 0,98 (1,8 Å) y 0,819 (3,0 Å): las dos
primeras dentro del intervalo del 95 % [0,82, 1,28] y la tercera pegada al borde inferior (una de tres a ese borde es esperable;
sin señal de σ subestimada, que daría > 1). Con 30 repeticiones y semillas fijas, 1,8 Å había salido en 0,76: fue una fluctuación, no
un defecto de σ. Pendiente de log σ vs. log N = −0,54 con $\sigma\sqrt N$ ≈ 1,54 Ha.

**Sesgo.** La columna $(\bar E - E_\text{sv})/(\sigma/\sqrt N)$ de la tabla de repeticiones salió alta algunas veces (hasta
2,8 con 40 repeticiones; 2,9 y 3,7 en corridas anteriores con 30). Para descartar un sesgo del estimador lo medimos aparte
(fuera del notebook) a 0,9 Å con 150 repeticiones y 4 semillas distintas: z = +0,6, −1,3, +0,7 y −0,8, sin signo
sistemático y con sesgo máximo de 1,1 mHa (σ ≈ 17 mHa por corrida; error estándar ≈ 1,4 mHa), compatible con 0. Con 3
distancias por corrida es esperable que alguna pase 2σ; la conclusión «sin sesgo» se apoya en esa prueba extendida y en el
$\chi^2$ de la curva, no solo en este último.

**Reproducibilidad.** Dos corridas seguidas con las mismas semillas dan tablas idénticas bit a bit (incluidos shots totales,
energías y σ). Antes no: el reparto de shots cambiaba entre corridas. La causa era PySCF: sus sumas en paralelo (OpenMP)
cambian de orden y los coeficientes de $H$ varían ~1e-14; el ajuste de θ lo amplifica a ~1e-9 Ha y eso mueve el redondeo del
reparto y el piloto. Se arregló con `pyscf.lib.num_threads(1)` (para H4 en STO-3G un hilo no es más lento). Los
redondeos de $H$ y θ* que ya había quedan como segunda defensa. Aer no es fuente de aleatoriedad: sus semillas son por
binding y da el mismo resultado con cualquier número de hilos (verificamos el hash de los bitstrings). En `references`,
`qubit_hamiltonian` y `params` reales de los compañeros hay que mantener esto (un hilo, o redondear) si quieren la misma garantía.
**Velocidad:** la sección tarda ≈ 50 s a escala 0,05 en una máquina sin carga (96–140 s con otros procesos corriendo; antes
≈ 116–162 s). Quedan fijos ≈ 9 s de PySCF, ≈ 9 s del ajuste de θ y ≈ 6 s de la curva. A escala 1 estimamos un tiempo parecido
porque el costo es por binding y no por shot (estimación, no medición).

**Qué no funcionó o descartamos**

- *Un PUB por grupo con shots distintos en un mismo `run`*: es lo natural, pero por lo de la semilla compartida la dispersión
  entre semillas daba ≈ 1,3–1,4 σ y $\chi^2$ salía mal ($p < 0{,}01$). Lo detectó la validación con repeticiones; por eso la
  dejamos en el notebook.
- *Un `run` de `SamplerV2` separado por grupo con semillas distintas*: arregla la correlación, pero cada `run` recompila el
  circuito (~35 ms por cada 1000 compuertas) y una estimación tardaba ~3–4 s.
- *Una repetición por llamada de `measure_energy` (30 × 3 distancias, 180 jobs)*: ~100 s. Lo que pesa es el binding, no el job,
  así que juntar las repeticiones en un PUB no alcanzó por sí solo: lo que bajó el costo fue `method="statevector"` (3×),
  bindings en paralelo (2,4×) y 2 paquetes por grupo en vez de 4. Con 100 repeticiones de golpe Aer falló por memoria
  (`bad allocation`); de ahí el corte en jobs de 600 bindings.
- *EstimatorV2 con `precision`*: descartado como fuente de números reportados (ver arriba).
- *Sumar varianzas término a término*: ignora covarianzas dentro del grupo y da una σ incorrecta.
- *Reusar los shots de la piloto en la estimación*: ahorra 3–8 %, pero acopla reparto y estimación; preferimos la versión limpia.
- *Mediciones más baratas* (rotación de base / factorización doble, ~6× menos shots a 1,8 Å según `docs/analysis/`): quedan
  como mejora posible; con QWC y reparto óptimo ya llegamos a σ = 1,8 mHa a 1,8 Å con pocos minutos de simulación.

## 5. Resultados: curva, $R_e$, $D_e$ y tabla (energía, $\sigma$, shots, `cz`)

Todo lo de esta sección se **recalcula** desde las tablas del contrato: `vqe` (energía medida con `SamplerV2`, σ, shots, cz; #6) y las
referencias CCSD/FCI de #1 (vía `get_references`). No hay números pegados a mano. Convención: energías **totales** en Ha, errores en mHa,
$\text{err} = |E - E_\text{FCI}|$. FCI y CCSD son solo para comparar; el resultado reportado es siempre el de VQE.

In [ ]:
# --- 5.1 Tabla por punto: refs + vqe -> `resultados` ------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MHA = 1e3  # Ha -> mHa
_refs_df = pd.DataFrame([get_references(float(d)) for d in vqe.d])[["d", "e_ccsd", "e_fci"]]
resultados = vqe.merge(_refs_df, on="d", how="left", validate="one_to_one")
assert resultados[["e_ccsd", "e_fci"]].notna().all().all(), "faltan referencias para algún d de vqe"
resultados["err_vqe"] = (resultados.energy - resultados.e_fci).abs() * MHA
resultados["err_ccsd"] = (resultados.e_ccsd - resultados.e_fci).abs() * MHA
resultados["sigma_mHa"] = resultados.sigma * MHA
# gana_a_ccsd = err_vqe + sigma < err_ccsd, solo para d >= 1,8 Å (en el resto queda <NA>)
_g = (resultados.err_vqe + resultados.sigma_mHa < resultados.err_ccsd)
resultados["gana_a_ccsd"] = pd.array(np.where(resultados.d >= STRETCHED - 1e-9, _g, pd.NA), dtype="boolean")

_cols = ["d", "energy", "sigma", "shots", "cz", "e_ccsd", "e_fci", "err_vqe", "err_ccsd", "gana_a_ccsd"]
tabla_resultados = resultados[_cols]
print(tabla_resultados.to_string(index=False, float_format=lambda x: f"{x:.6f}" if abs(x) < 1e-1 or abs(x) > 5 else f"{x:.3f}"))
_est = resultados[resultados.d >= STRETCHED - 1e-9]
print(f"\nPuntos con d >= {STRETCHED} Å donde VQE le gana a CCSD (err_vqe + sigma < err_ccsd): "
      f"{int(_est.gana_a_ccsd.sum())} de {len(_est)}")

In [ ]:
# --- 5.2 Gráfico: E(d) ± σ vs CCSD y FCI, y panel de errores en escala log ----------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6), constrained_layout=True)
ax1.plot(resultados.d, resultados.e_fci, "-", color="k", lw=1.6, label="FCI")
ax1.plot(resultados.d, resultados.e_ccsd, "--", color="tab:orange", lw=1.6, label="CCSD")
ax1.errorbar(resultados.d, resultados.energy, yerr=resultados.sigma, fmt="o", ms=4, capsize=3, color="tab:blue",
             label=r"VQE ($\pm\sigma$, SamplerV2)")
ax1.axvline(STRETCHED, color="gray", ls=":", lw=1)
ax1.set(xlabel="d (Å)", ylabel="E total (Ha)", title="H$_4$ lineal, STO-3G: energía total")
ax1.legend()

_floor = 1e-3  # mHa: piso para que el log no se rompa si err_vqe ~ 0
ax2.errorbar(resultados.d, np.maximum(resultados.err_vqe, _floor), yerr=resultados.sigma_mHa, fmt="o-", ms=4, capsize=3,
             color="tab:blue", label=r"VQE: $|E-E_{FCI}|\ \pm\ \sigma$")
ax2.plot(resultados.d, np.maximum(resultados.err_ccsd, _floor), "s--", ms=4, color="tab:orange", label=r"CCSD: $|E-E_{FCI}|$")
ax2.axhline(1.6, color="crimson", lw=1, label="1,6 mHa (precisión química)")
ax2.axvline(STRETCHED, color="gray", ls=":", lw=1, label=f"d = {STRETCHED} Å")
ax2.set(xlabel="d (Å)", ylabel="error (mHa)", yscale="log", title="Error respecto de FCI")
ax2.legend(fontsize=8)
plt.show()

### 5.3 `R_e` y `D_e`

**`R_e`**: ajustamos un polinomio a las energías cerca del mínimo (con la grilla de 0,1 Å cae entre 0,8 y 1,0 Å) y tomamos el mínimo del ajuste.
Los ajustes de VQE van ponderados por $1/\sigma$. **Incertidumbre**: bootstrap paramétrico, re-muestreando cada energía como
$\mathcal N(E_i,\sigma_i^2)$ y repitiendo el ajuste; además reportamos la **sensibilidad** (rango al variar grado y ventana), que es la que domina para FCI/CCSD,
que no tienen ruido de medición (su incertidumbre estadística es 0 por construcción).

**`D_e = E(3,0 Å) − E(R_e)`**, con $E(R_e)$ el valor del ajuste en su mínimo y $E(3{,}0)$ el punto medido (con su σ). Para FCI y CCSD el mismo cálculo con sus energías.

*Aclaración sobre `D_e`*: la definición es la del enunciado, pero a 3,0 Å la cadena **todavía no está del todo disociada**. Abajo calculamos la energía de 4 átomos de H aislados
(UHF STO-3G) y verificamos que la FCI a 3,0 Å queda ~1 mHa por debajo de ella, es decir, este $D_e$ **subestima** la energía de atomización completa en esa cantidad.

In [ ]:
# --- 5.3 R_e por ajuste local + bootstrap + sensibilidad; D_e con incertidumbre ------------------------
D_FIN = 3.0
d_all = resultados.d.to_numpy()
BOOT_N = 2000
_rng = np.random.default_rng(SEED + 7)

# Configuración por defecto (la de las cifras reportadas) y barrido de sensibilidad
DEG0, WIN0 = 4, (0.6, 1.4)
DEGS = (2, 3, 4, 5)
WINS = ((0.7, 1.2), (0.6, 1.3), (0.6, 1.4), (0.6, 1.6))


def _min_fit(x, y, w, deg, win):
    """Mínimo del polinomio de grado `deg` ajustado dentro de `win` (acotado a la ventana). Devuelve (R_e, E(R_e))."""
    xs = np.linspace(win[0], win[1], 2001)
    c = np.polyfit(x, y, deg, w=w)
    ys = np.polyval(c, xs)
    i = int(np.argmin(ys))
    return xs[i], ys[i]


def _fit_batch(E, sig, deg, win, nboot=0, rng=None):
    """Ajuste en `win` a E (vector sobre toda la grilla). Con nboot>0 re-muestrea E ~ N(E, sig^2). Devuelve R_e, E(R_e), E(3.0)."""
    m = (d_all >= win[0] - 1e-9) & (d_all <= win[1] + 1e-9)
    x = d_all[m]
    if len(x) < deg + 2:
        return None
    w = None if sig is None else 1.0 / sig[m]
    i3 = int(np.argmin(np.abs(d_all - D_FIN)))
    if nboot == 0:
        r, e = _min_fit(x, E[m], w, deg, win)
        return r, e, E[i3]
    out = []
    for _ in range(nboot):
        Eb = E + rng.normal(0.0, 1.0, size=E.shape) * sig
        r, e = _min_fit(x, Eb[m], w, deg, win)
        out.append((r, e, Eb[i3]))
    return np.array(out)


SERIES = {  # nombre -> (energías, σ o None)
    "VQE": (resultados.energy.to_numpy(), resultados.sigma.to_numpy()),
    "FCI": (resultados.e_fci.to_numpy(), None),
    "CCSD": (resultados.e_ccsd.to_numpy(), None),
}

# Barrido de sensibilidad (sin bootstrap): R_e y D_e para cada (grado, ventana)
sens_rows = []
for name, (E, sig) in SERIES.items():
    for deg in DEGS:
        for win in WINS:
            f = _fit_batch(E, sig, deg, win)
            if f is None:
                continue
            r, e, e3 = f
            sens_rows.append(dict(metodo=name, grado=deg, ventana=f"{win[0]}-{win[1]}", R_e=r, D_e_mHa=(e3 - e) * MHA))
sens = pd.DataFrame(sens_rows)

# Resultados por defecto + bootstrap (solo VQE tiene σ de medición)
res_rows = []
for name, (E, sig) in SERIES.items():
    r0, e0, e3 = _fit_batch(E, sig, DEG0, WIN0)
    De0 = (e3 - e0) * MHA
    if sig is not None:
        bs = _fit_batch(E, sig, DEG0, WIN0, nboot=BOOT_N, rng=_rng)
        r_boot, De_boot = bs[:, 0].std(ddof=1), ((bs[:, 2] - bs[:, 1]) * MHA).std(ddof=1)
    else:
        r_boot = De_boot = 0.0
    s = sens[sens.metodo == name]
    r_sens = (s.R_e.max() - s.R_e.min()) / 2
    D_sens = (s.D_e_mHa.max() - s.D_e_mHa.min()) / 2
    res_rows.append(dict(metodo=name, R_e_A=r0, R_e_boot=r_boot, R_e_sens=r_sens,
                         D_e_mHa=De0, D_e_boot=De_boot, D_e_sens=D_sens))
RED = pd.DataFrame(res_rows)
RED["R_e_tot"] = np.hypot(RED.R_e_boot, RED.R_e_sens)   # incertidumbre combinada en cuadratura
RED["D_e_tot"] = np.hypot(RED.D_e_boot, RED.D_e_sens)
print(f"Ajuste por defecto: grado {DEG0}, ventana {WIN0[0]}-{WIN0[1]} Å, bootstrap de {BOOT_N} muestras.")
print(RED.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSensibilidad de R_e (Å) al grado y a la ventana:")
print(sens.pivot_table(index=["metodo", "grado"], columns="ventana", values="R_e").to_string(float_format=lambda x: f"{x:.4f}"))
print("\nSensibilidad de D_e (mHa):")
print(sens.pivot_table(index=["metodo", "grado"], columns="ventana", values="D_e_mHa").to_string(float_format=lambda x: f"{x:.2f}"))

In [ ]:
# --- 5.4 Tablita final R_e / D_e y chequeo de la disociación a 3,0 Å ----------------------------------
tablita = pd.DataFrame({
    "metodo": RED.metodo,
    "R_e (Å)": [f"{r:.3f} ± {u:.3f}" for r, u in zip(RED.R_e_A, RED.R_e_tot)],
    "D_e (mHa)": [f"{x:.1f} ± {u:.1f}" for x, u in zip(RED.D_e_mHa, RED.D_e_tot)],
    "D_e (eV)": [f"{x * 0.0272114:.3f} ± {u * 0.0272114:.3f}" for x, u in zip(RED.D_e_mHa, RED.D_e_tot)],
    "± estad. R_e": [f"{u:.3f}" for u in RED.R_e_boot], "± sens. R_e": [f"{u:.3f}" for u in RED.R_e_sens],
    "± estad. D_e": [f"{u:.2f}" for u in RED.D_e_boot], "± sens. D_e": [f"{u:.2f}" for u in RED.D_e_sens],
})
print(tablita.to_string(index=False))
_ref = RED.set_index("metodo")
print(f"\nVQE - FCI: R_e = {_ref.R_e_A['VQE'] - _ref.R_e_A['FCI']:+.3f} Å, D_e = {_ref.D_e_mHa['VQE'] - _ref.D_e_mHa['FCI']:+.2f} mHa;  "
      f"CCSD - FCI: R_e = {_ref.R_e_A['CCSD'] - _ref.R_e_A['FCI']:+.3f} Å, D_e = {_ref.D_e_mHa['CCSD'] - _ref.D_e_mHa['FCI']:+.2f} mHa")

# Cuatro H aislados (UHF STO-3G, doblete): cota del límite de disociación completa
_h = gto.M(atom="H 0 0 0", basis="sto-3g", spin=1, verbose=0)
_e_h = scf.UHF(_h).run().e_tot
_gap = (resultados.set_index("d").e_fci[D_FIN] - 4 * _e_h) * MHA
print(f"\nCuatro H aislados (UHF STO-3G): {4 * _e_h:.6f} Ha; FCI(3,0 Å) queda {abs(_gap):.2f} mHa "
      f"{'por debajo' if _gap < 0 else 'por encima'} de ese límite -> a 3,0 Å la cadena no está del todo disociada "
      f"y D_e(3,0) subestima la energía de atomización completa en ~{abs(_gap):.1f} mHa.")

#### Qué decidimos, por qué y qué no funcionó (#7)

**Qué decidimos**

- **Todo sale de `vqe` + referencias** (`get_references`, que usa `references(d)` de #1 y cae al stand-in de PySCF solo si falta). `err_vqe = |E_medida - E_FCI|` usa la
  energía *medida* (con su ruido); por eso la columna `gana_a_ccsd = err_vqe + σ < err_ccsd` es conservadora: exige ganar aun sumando una σ. Se evalúa solo para $d \ge 1{,}8$ Å (`<NA>` antes).
- **`R_e`**: polinomio de grado 4 en 0,6–1,4 Å, mínimo buscado en una malla fina dentro de la ventana. VQE ajustado con pesos $1/\sigma$. Se eligió una ventana que incluye varios puntos a cada lado del mínimo
  (que cae entre 0,8 y 1,0 Å) porque la curva es muy asimétrica y una parábola sola lo sesga.
- **Incertidumbres en dos partes**: *estadística* (bootstrap paramétrico $\mathcal N(E_i,\sigma_i^2)$, 2000 muestras, semilla fija) y *de sensibilidad* (semi-rango al variar grado 2–5 y cuatro ventanas), combinadas en cuadratura.
  Para FCI y CCSD no hay ruido de medición, así que solo cuenta la sensibilidad, que además mide el error sistemático del método de ajuste con la grilla de 0,1 Å (se ve mejor comparando FCI con VQE).
- **`D_e = E(3,0) − E(R_e)`** con $E(R_e)$ el mínimo del ajuste y $E(3{,}0)$ el punto medido; en el bootstrap ambos se re-muestrean juntos, así se propaga la correlación entre ellos. En FCI/CCSD, mismo cálculo.
- **Disociación incompleta**: la FCI a 3,0 Å se compara con cuatro H aislados (UHF STO-3G) calculados en la propia celda; la diferencia (~1 mHa) se imprime y se aclara en el texto.
- **Gráfico**: el panel de errores usa un piso de $10^{-3}$ mHa para que la escala log no falle si un error sale ~0, y barras $\pm\sigma$ sobre el error de VQE.

**Qué no funcionó o descartamos**

- *Parábola sola*: con una ventana ancha da un `R_e` sesgado (la curva es asimétrica); queda en la tabla de sensibilidad como contraste, no como resultado.
- *Spline cúbico*: con σ de ~1–3 mHa por punto interpola el ruido y el mínimo salta; preferimos un ajuste ponderado de grado bajo, que lo suaviza.
- *Ajustar con todos los puntos hasta 3,0 Å*: los puntos lejanos (casi planos) tiran del polinomio y ensanchan la dependencia con el grado.
- *Reportar solo la incertidumbre del bootstrap*: subestima el error, porque no ve el sesgo del modelo de ajuste; por eso se suma la sensibilidad.
- Mientras #4/#5 sean stand-ins, el ansatz no llega a FCI (3–4 mHa arriba a 1,8–2,0 Å): `gana_a_ccsd` no es un resultado final, solo muestra que el pipeline funciona.

## 6. Zona estirada ($d \geq 1{,}8$ Å) vs. CCSD

El orden es a propósito: primero (6.1) las evidencias de que el resultado es confiable **sin mirar FCI**, después (6.2) el argumento, escrito antes de comparar, y recién en 6.3 la comparación con CCSD y FCI. Así el argumento sirve también para un sistema donde FCI no se puede calcular.

**De dónde sale cada número.** $\langle N\rangle$, $\langle S_z\rangle$, $\langle S^2\rangle$, la varianza y la cota de Temple salen del statevector del circuito real en θ* (`vqe_state`, #5): son **diagnósticos del estado**, no energías reportadas (permitido). La fracción de shots en el sector $N_\alpha = N_\beta = 2$ se mide con `SamplerV2`. La energía reportada es la de `vqe` (#6), medida con `SamplerV2`.

### 6.1 Evidencias sin FCI

In [ ]:
import matplotlib.pyplot as plt
from qiskit.circuit.library import efficient_su2
from qiskit_aer.primitives import SamplerV2

STRETCHED_D = [float(d) for d in D if d >= STRETCHED]

_sz = np.zeros((N_QUBITS, N_QUBITS))
_sp = np.zeros((N_QUBITS, N_QUBITS))
for _p in range(N_ORB):
    _sz[alpha_qubit(_p), alpha_qubit(_p)], _sz[beta_qubit(_p), beta_qubit(_p)] = 0.5, -0.5
    _sp[alpha_qubit(_p), beta_qubit(_p)] = 1.0
N_OP = jordan_wigner(np.eye(N_QUBITS))
SZ_OP = jordan_wigner(_sz)
SP_OP = jordan_wigner(_sp, hermitian=False)
_s2 = (SP_OP @ SP_OP.adjoint() + SZ_OP @ SZ_OP - SZ_OP).simplify()
assert np.abs(_s2.coeffs.imag).max() < 1e-12
S2_OP = SparsePauliOp(_s2.paulis, _s2.coeffs.real)
SYMMETRY_OPS = {"N": N_OP, "S_z": SZ_OP, "S2": S2_OP}
_SYM_M = {k: o.to_matrix(sparse=True) for k, o in SYMMETRY_OPS.items()}


def moments(m, v: np.ndarray) -> tuple[float, float]:
    mv = m @ v
    mean = np.vdot(v, mv).real
    return mean, np.linalg.norm(mv - mean * v) ** 2


def singlet_part(v: np.ndarray) -> np.ndarray:
    a = _SYM_M["S2"] @ v - 2 * v
    return (_SYM_M["S2"] @ a - 6 * a) / 12


_hf = Statevector.from_label(HF_BITSTRING).data
for _k, _target in (("N", 4), ("S_z", 0), ("S2", 0)):
    assert np.allclose(moments(_SYM_M[_k], _hf), (_target, 0), atol=1e-12), _k
_qc = efficient_su2(N_QUBITS, reps=2)
_rand = Statevector(_qc.assign_parameters(np.random.default_rng(8).normal(size=_qc.num_parameters))).data
_rand_sym = {k: moments(m, _rand) for k, m in _SYM_M.items()}
assert _rand_sym["N"][1] > 0.1 and _rand_sym["S_z"][1] > 0.1 and _rand_sym["S2"][0] > 0.1, _rand_sym
_v = sector_ansatz.state(params[DISSOCIATED])
assert np.isclose(moments(_SYM_M["S2"], vqe_state(DISSOCIATED).data)[0], _v @ S2_SECTOR @ _v, atol=1e-10)
_r4 = np.zeros(2 ** N_QUBITS)
_r4[SECTOR] = np.random.default_rng(8).normal(size=len(SECTOR))
_r4s = singlet_part(_r4)
assert np.allclose(singlet_part(_r4s), _r4s) and abs(moments(_SYM_M["S2"], _r4s / np.linalg.norm(_r4s))[0]) < 1e-10

VQD_BETA = 1.0
VQD_STARTS = 10


def excited_singlet(d: float, x_prev: np.ndarray | None, rng: np.random.Generator):
    H = H_SECTOR[d][0]
    psi0 = sector_ansatz.state(params[d])
    cost = H + LAMBDA_S2 * S2_SECTOR + VQD_BETA * np.outer(psi0, psi0)
    starts = ([] if x_prev is None else [x_prev]) + [rng.normal(scale=0.7, size=sector_ansatz.n)
                                                     for _ in range(VQD_STARTS)]
    r = min((optimize_cost(cost, x0) for x0 in starts), key=lambda r: r.fun)
    u = sector_ansatz.state(r.x)
    return r.x, {"e1": u @ H @ u, "S2_exc": u @ S2_SECTOR @ u, "overlap": (u @ psi0) ** 2}


def optimize_cost(cost: np.ndarray, x0: np.ndarray):
    return minimize(sector_ansatz.energy_grad, x0, args=(cost,), jac=True, method="BFGS", options={"gtol": 1e-6})


def z_basis_sector_fraction(d: float, shots: int = 4_000, seed: int = 8) -> float:
    qc = ansatz.assign_parameters(params[d])
    qc.measure_all()
    bits = SamplerV2(seed=seed).run([qc], shots=shots).result()[0].data.meas.get_bitstrings()
    occ = np.array([[b[N_QUBITS - 1 - q] == "1" for q in range(N_QUBITS)] for b in bits])
    n_a = occ[:, [alpha_qubit(p) for p in range(N_ORB)]].sum(axis=1)
    n_b = occ[:, [beta_qubit(p) for p in range(N_ORB)]].sum(axis=1)
    return float(np.mean((n_a == N_ALPHA) & (n_b == N_BETA)))


_t0 = time.perf_counter()
_rng = np.random.default_rng(8)
_x_exc, rel_rows = None, []
for d in STRETCHED_D:
    op, e_nuc = qubit_hamiltonian(d)
    v = vqe_state(d).data
    sym = {k: moments(m, v)[0] for k, m in _SYM_M.items()}
    ps = singlet_part(v)
    w_singlet = np.linalg.norm(ps) ** 2
    e_el, var = moments(op.to_matrix(sparse=True), ps / np.sqrt(w_singlet))
    e = e_el + e_nuc
    _x_exc, exc = excited_singlet(d, _x_exc, _rng)
    gap = exc["e1"] - e
    assert gap > 0, (d, gap)
    rel_rows.append({"d": d, "e_sv": e, "N": sym["N"], "S_z": sym["S_z"], "S2": sym["S2"],
                     "w_triplete": 1 - w_singlet, "sqrt_var_mHa": np.sqrt(var) * 1e3,
                     "gap_vqd_mHa": gap * 1e3, "S2_exc": exc["S2_exc"], "overlap_exc": exc["overlap"],
                     "temple": e - var / gap, "temple_gap_mitad": e - var / (gap / 2),
                     "frac_sector_Z": z_basis_sector_fraction(d)})
rel = pd.DataFrame(rel_rows)
rel["ventana_mHa"] = (rel.e_sv - rel.temple) * 1e3
rel["ventana_gap_mitad_mHa"] = (rel.e_sv - rel.temple_gap_mitad) * 1e3
rel = rel.merge(refs[["d", "e_ccsd"]].assign(d=lambda t: t.d.astype(float).round(2)), on="d")
rel["ccsd_bajo_temple_mHa"] = (rel.temple - rel.e_ccsd) * 1e3

_both = starts_log[starts_log.d >= STRETCHED].groupby(["d", "direction"]).e_sv.min().unstack()
rel["dif_direcciones_mHa"] = (_both.max(axis=1) - _both.min(axis=1)).reindex(rel.d).to_numpy() * 1e3
_ms = multistart.merge(rel[["d", "e_sv"]], on="d", suffixes=("", "_vqe"))
_ms["sobre_vqe_mHa"] = (_ms.e_sv - _ms.e_sv_vqe) * 1e3
robust = pd.DataFrame([{"d": d, "arranques": len(grp), "debajo_de_vqe": int((grp.sobre_vqe_mHa < -0.01).sum()),
                        "a_menos_de_0,01_mHa": int((grp.sobre_vqe_mHa < 0.01).sum()),
                        "en_precision_quimica": int((grp.sobre_vqe_mHa < CHEM_ACC * 1e3).sum()),
                        "mediana_sobre_vqe_mHa": grp.sobre_vqe_mHa.median(),
                        "S2_max_en_precision": grp.S2[grp.sobre_vqe_mHa < CHEM_ACC * 1e3].max()}
                       for d, grp in _ms.groupby("d")]).set_index("d")

assert np.allclose(rel.N, N_ALPHA + N_BETA) and np.allclose(rel.S_z, 0, atol=1e-10)
assert (rel.S2 < 1e-3).all() and (rel.frac_sector_Z == 1).all()
assert (rel.S2_exc < 1e-2).all() and (rel.overlap_exc < 1e-4).all()
assert (robust.debajo_de_vqe == 0).all(), robust
assert (rel.ccsd_bajo_temple_mHa > rel.ventana_gap_mitad_mHa).all(), rel
t_rel = time.perf_counter() - _t0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.semilogy(rel.d, rel.sqrt_var_mHa, "o-", label="√Var(H) (Weinstein)")
ax1.semilogy(rel.d, rel.ventana_mHa, "s-", label="E − cota de Temple (gap VQD)")
ax1.semilogy(rel.d, rel.ventana_gap_mitad_mHa, "s--", alpha=0.6, label="E − cota de Temple (gap / 2)")
ax1.semilogy(rel.d, rel.gap_vqd_mHa, "^:", color="0.4", label="gap al siguiente singlete (VQD)")
ax1.axhline(CHEM_ACC * 1e3, ls="--", color="C2", label="precisión química")
ax1.set(xlabel="d (Å)", ylabel="mHa", title="Evidencia 1: varianza y cota inferior de Temple (sin FCI)")
ax1.legend(fontsize=8)
for i, (d, grp) in enumerate(_ms.groupby("d")):
    ax2.semilogy(np.full(len(grp), d) + np.linspace(-0.04, 0.04, len(grp)), np.maximum(grp.sobre_vqe_mHa, 1e-4),
                 "o", color=f"C{i}", label=f"{d} Å")
ax2.axhline(CHEM_ACC * 1e3, ls="--", color="C2")
ax2.set(xlabel="d (Å)", ylabel="E(arranque) − E_VQE (mHa, piso 1e-4)",
        title="Evidencia 3: 10 arranques por distancia (#5)")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"{t_rel:.0f} s. Ventana de Temple máx.: {rel.ventana_mHa.max():.3f} mHa "
      f"({rel.ventana_gap_mitad_mHa.max():.3f} con el gap a la mitad); √Var máx.: {rel.sqrt_var_mHa.max():.3f} mHa; "
      f"⟨S²⟩ máx.: {rel.S2.max():.1e}; peso de triplete máx.: {rel.w_triplete.max():.1e}")
print(f"Sin FCI: E_VQE(statevector) − E0 ≤ {rel.ventana_mHa.max():.3f} mHa, y CCSD queda por debajo de la cota "
      f"inferior de Temple en todos los puntos: |E_CCSD − E0| ≥ {rel.ccsd_bajo_temple_mHa.min():.1f} mHa")
print(robust.to_string(float_format="%.3g"))

vqe_t = globals().get("vqe")
if isinstance(vqe_t, pd.DataFrame) and {"d", "energy", "sigma"} <= set(vqe_t.columns) \
        and not getattr(vqe_t, "attrs", {}).get("mock"):
    _m = vqe_t[["d", "energy", "sigma"]].assign(d=lambda t: t.d.astype(float).round(2))
    rel = rel.drop(columns=["energy", "sigma"], errors="ignore").merge(_m, on="d", how="left")
    rel["z_vs_statevector"] = (rel.energy - rel.e_sv) / rel.sigma
    rel["sigmas_bajo_temple"] = (rel.temple - rel.energy) / rel.sigma
    rel["cota_err_vqe_mHa"] = rel.ventana_mHa + (rel.energy - rel.e_sv).abs() * 1e3 + rel.sigma * 1e3
    rel["gana_a_ccsd_sin_fci"] = rel.cota_err_vqe_mHa < rel.ccsd_bajo_temple_mHa
    print(f"Evidencia 4: |E_medida − E_sv| / σ máx. = {rel.z_vs_statevector.abs().max():.2f}; "
          f"evidencia 5: puntos más de 2σ por debajo de Temple = {int((rel.sigmas_bajo_temple > 2).sum())}; "
          f"le gana a CCSD sin mirar FCI en {int(rel.gana_a_ccsd_sin_fci.sum())}/{len(rel)} puntos")
else:
    print("Evidencias 4 y 5 pendientes: falta la tabla `vqe` de #6 (energía medida y σ).")
rel.round(6)

### 6.2 Argumento de confiabilidad (escrito antes de mirar FCI)

Para cada uno de los 7 puntos con $d \geq 1{,}8$ Å (una corrida, semilla fija; tabla `rel`):

1. **Es el estado correcto: $N = 4$, $S_z = 0$ y singlete.** $\langle N\rangle = 4$ y $\langle S_z\rangle = 0$ exactos (el ansatz los conserva), y el 100 % de 4000 shots medidos en base Z con `SamplerV2` cae en $N_\alpha = N_\beta = 2$. $\langle S^2\rangle \leq 2\cdot 10^{-5}$: el peso del triplete es $\leq 4\cdot 10^{-6}$. Esta es la prueba que más importa en la zona estirada: a 3,0 Å el triplete está a 0,46 mHa, así que una mezcla con él casi no cambia la energía ni la varianza, y solo $\langle S^2\rangle$ la delata.
2. **Está muy cerca de un autoestado: varianza y cota de Temple.** $\sqrt{\mathrm{Var}(H)}$ es $\leq 4\cdot 10^{-4}$ mHa hasta 2,2 Å y 0,76 / 0,62 / 0,19 / 0,11 mHa a 2,4 / 2,6 / 2,8 / 3,0 Å. Por Weinstein hay un autovalor a menos de $\sqrt{\mathrm{Var}}$, pero esa cota es floja. La de **Temple** es mucho más fuerte: dentro de los singletes, $E_0 \geq \langle H\rangle - \mathrm{Var}/(E_1 - \langle H\rangle)$, donde $E_1$ es el siguiente singlete. El gap $E_1 - \langle H\rangle$ se estima con un VQE de estado excitado (VQD) con el mismo ansatz, ortogonal al fundamental y penalizando $S^2$: 76 / 41 / 21 / 10,8 / 5,3 / 2,6 / 1,2 mHa. El resultado es que **$E_0$ queda atrapada en $[\text{Temple}, E_\text{VQE}]$, una ventana de $\leq 0{,}071$ mHa** (< 1e-6 mHa hasta 2,2 Å). Si el VQD sobreestimara el gap, la cota sería optimista; con el gap a la mitad, la ventana sigue siendo $\leq 0{,}14$ mHa.
3. **No es un mínimo local.** En 1,8 / 2,4 / 3,0 Å, de los 10 arranques independientes de #5 (HF, *warm* y 8 al azar), **ninguno encuentra una energía más baja** que la del VQE; 8 / 5 / 9 quedan a menos de 1,6 mHa, y los otros quedan en mínimos locales claramente más altos. Las dos pasadas de *warm start* (0,6→3,0 y 3,0→0,6) coinciden a $\leq 0{,}11$ mHa antes del pulido, y $\sqrt{\mathrm{Var}}$ y la ventana de Temple varían suave con $d$.
4. **El error estadístico es el que dice $\sigma$.** Con la tabla `vqe` de #6 ($\sigma$ = 0,79–0,83 mHa, 3,1–3,4 × 10⁶ shots por punto), la energía medida queda a $\leq 1{,}8\sigma$ de la del statevector en los 7 puntos (`z_vs_statevector`), como corresponde a ruido estadístico. #6 además valida $\sigma$ con repeticiones.
5. **Es consistente con ser variacional.** Sin ruido, la energía del ansatz es una cota superior de $E_0$. Ningún punto medido queda más de $2\sigma$ por debajo de la cota de Temple: el máximo es 1,6σ, a 1,8 Å (`sigmas_bajo_temple`). **Que CCSD quede por debajo del VQE es lo esperable**, no una señal en contra: CCSD no es variacional. De hecho, CCSD queda por debajo de la cota *inferior* de Temple en los 7 puntos, así que su error es **al menos** 9,1 / 18,3 / 26,4 / 30,5 / 30,6 / 27,7 / 23,0 mHa.

**Conclusión antes de comparar.** El estado preparado es un singlete con $N = 4$ y $S_z = 0$, y su energía está a $\leq 0{,}07$ mHa del fundamental ($\leq 0{,}14$ con un gap la mitad de chico). CCSD, en cambio, erra por al menos 9 mHa. Con la energía medida, la cota del error del VQE es ventana + $|E_\text{medida} - E_\text{sv}|$ + $\sigma$ (`cota_err_vqe_mHa`): entre 1,0 y 2,3 mHa, contra $\geq 9{,}1$ mHa de CCSD. Así, **le ganamos a CCSD en los 7 puntos sin usar FCI** (`gana_a_ccsd_sin_fci`). La precisión química (< 1,6 mHa) no se puede certificar de esta forma en todos los puntos: la cota pasa 1,6 mHa a 1,8, 2,4 y 2,6 Å. Ahí manda $\sigma$, que es estadística.

### 6.3 Comparación con CCSD y FCI

Criterio del desafío en cada punto con $d \geq 1{,}8$ Å: $|E_\text{VQE} - E_\text{FCI}| + \sigma < |E_\text{CCSD} - E_\text{FCI}|$, y como objetivo extra, $|E_\text{VQE} - E_\text{FCI}| < 1{,}6$ mHa. Además se verifica el argumento de 6.2: $E_\text{FCI}$ tiene que caer dentro de la ventana $[\text{Temple}, E_\text{sv}]$, CCSD por debajo de la cota de Temple, y el gap del VQD tiene que coincidir con el exacto. Mientras falte `vqe` (#6), la tabla usa la energía del statevector, marcada como **provisoria** (no es la energía reportada) y sin $\sigma$.

In [ ]:
cmp = refs[refs.d >= STRETCHED][["d", "e_ccsd", "e_fci"]].assign(d=lambda t: t.d.astype(float).round(2))
cmp = cmp.merge(rel.drop(columns=["energy", "sigma", "e_ccsd"], errors="ignore"), on="d")
vqe_measured = isinstance(globals().get("vqe"), pd.DataFrame) and "energy" in rel.columns and rel.energy.notna().all()
if vqe_measured:
    cmp = cmp.merge(rel[["d", "energy", "sigma"]], on="d")
    vqe_source = "medida (#6, SamplerV2)"
else:
    cmp = cmp.assign(energy=cmp.e_sv, sigma=np.nan)
    vqe_source = "statevector (PROVISORIA: falta `vqe` de #6, no es la energía reportada)"

cmp["ccsd_menos_fci_mHa"] = (cmp.e_ccsd - cmp.e_fci) * 1e3
cmp["err_ccsd_mHa"] = cmp.ccsd_menos_fci_mHa.abs()
cmp["err_vqe_mHa"] = (cmp.energy - cmp.e_fci).abs() * 1e3
cmp["sigma_mHa"] = cmp.sigma * 1e3
cmp["err_vqe_mas_sigma_mHa"] = cmp.err_vqe_mHa + cmp.sigma_mHa.fillna(0)
cmp["gana_a_ccsd"] = cmp.err_vqe_mas_sigma_mHa < cmp.err_ccsd_mHa
cmp["precision_quimica"] = cmp.err_vqe_mHa < CHEM_ACC * 1e3
cmp["fci_en_ventana"] = (cmp.temple - 1e-9 <= cmp.e_fci) & (cmp.e_fci <= cmp.e_sv + 1e-9)
cmp["ventana_cubre_error_sv"] = cmp.ventana_mHa >= (cmp.e_sv - cmp.e_fci) * 1e3 - 1e-6


def next_singlet_gap(d: float) -> float:
    H, _ = H_SECTOR[d]
    w, U = np.linalg.eigh(H)
    s2 = np.einsum("ji,jk,ki->i", U, S2_SECTOR, U)
    singlets = w[np.abs(s2) < 1e-6]
    return (singlets[1] - singlets[0]) * 1e3


cmp["gap_exacto_mHa"] = [next_singlet_gap(d) for d in cmp.d]
cmp["gap_vqd_menos_exacto_mHa"] = cmp.gap_vqd_mHa - cmp.gap_exacto_mHa
cmp["ccsd_bajo_temple_ok"] = cmp.e_ccsd < cmp.temple
assert cmp.fci_en_ventana.all() and cmp.ccsd_bajo_temple_ok.all() and cmp.ventana_cubre_error_sv.all(), cmp
assert (cmp.gap_vqd_menos_exacto_mHa.abs() < 0.1).all(), cmp

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.semilogy(cmp.d, cmp.err_ccsd_mHa, "^-", color="C1", label="|E_CCSD − E_FCI|")
if vqe_measured:
    ax.errorbar(cmp.d, cmp.err_vqe_mHa, yerr=cmp.sigma_mHa, fmt="o-", color="C0", capsize=3,
                label="|E_VQE − E_FCI| ± σ (medida)")
else:
    ax.semilogy(cmp.d, np.maximum(cmp.err_vqe_mHa, 1e-6), "o-", color="C0", label="|E_VQE − E_FCI| (statevector, provisoria)")
ax.semilogy(cmp.d, np.maximum(cmp.ventana_mHa, 1e-6), "s--", color="C4", label="cota sin FCI: E − Temple")
ax.axhline(CHEM_ACC * 1e3, ls="--", color="C2", label="precisión química (1,6 mHa)")
ax.set(xlabel="d (Å)", ylabel="mHa (piso 1e-6)", title="Zona estirada: VQE vs. CCSD (contra FCI)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Energía VQE: {vqe_source}.")
print(f"¿Le gana a CCSD en los {len(cmp)} puntos con d ≥ {STRETCHED} Å? {'sí' if cmp.gana_a_ccsd.all() else 'no'} "
      f"({int(cmp.gana_a_ccsd.sum())}/{len(cmp)}); ¿precisión química? {int(cmp.precision_quimica.sum())}/{len(cmp)}; "
      f"E_FCI dentro de [Temple, E_sv] en {int(cmp.fci_en_ventana.sum())}/{len(cmp)}; "
      f"gap VQD vs. exacto: máx. {cmp.gap_vqd_menos_exacto_mHa.abs().max():.3f} mHa")
cmp[["d", "ccsd_menos_fci_mHa", "err_vqe_mHa", "sigma_mHa", "err_vqe_mas_sigma_mHa", "err_ccsd_mHa", "gana_a_ccsd",
     "precision_quimica", "ventana_mHa", "fci_en_ventana", "gap_vqd_mHa", "gap_exacto_mHa"]].round(4)

### Decisiones (#8)

**Qué deja esta sección.**
- `SYMMETRY_OPS`: $N$, $S_z$ y $S^2$ como `SparsePauliOp` de 8 qubits, armados con `jordan_wigner` de la sección 2. $S^2 = S^+S^- + S_z^2 - S_z$, con $S^+$ armado con `hermitian=False`.
- `rel`: una fila por punto estirado, con las evidencias sin FCI.
- `robust`: los arranques de `multistart` de #5, medidos contra la energía del VQE.
- `cmp`: la comparación final con CCSD y FCI.

Verificado:
- en HF da $N = 4$, $S_z = 0$, $S^2 = 0$ con varianza 0;
- en un estado al azar de `efficient_su2` da varianza de $N$ y $S_z$ > 0,1 y $\langle S^2\rangle$ > 0,1, o sea que detecta la ruptura de simetría;
- $\langle S^2\rangle$ coincide con el `S2_SECTOR` de PySCF que usa #5;
- el proyector al singlete es idempotente.

**Temple sobre la parte singlete, con el gap al siguiente *singlete*.** Usar el siguiente autoestado sin más no sirve: en la zona estirada es el triplete (0,46 mHa a 3,0 Å) y la cota sale inútil. Como $H$ conmuta con $S^2$, se proyecta el estado al singlete con $P_0 = (S^2 - 2)(S^2 - 6)/12$ (en $N = 4$ solo hay $S = 0, 1, 2$) y Temple se aplica dentro de los singletes. El peso que se descarta es el del triplete ($\leq 4\cdot 10^{-6}$) y se reporta (`w_triplete`).

**El gap sale de un VQD, no de diagonalizar.** Diagonalizar $H$ para obtener $E_1$ es, en la práctica, hacer FCI. En su lugar, el VQD minimiza $\langle H\rangle + 0{,}05\,\langle S^2\rangle + \beta\,|\langle\psi_0|\psi\rangle|^2$ con el mismo `sector_ansatz`, β = 1 Ha y 10 arranques al azar más el *warm start* desde la distancia anterior. La penalidad de solapamiento es el valor esperado de $\beta|\psi_0\rangle\langle\psi_0|$, así que el gradiente adjunto de #5 sirve sin cambios. Comparado después con el exacto (6.3), el gap del VQD coincide a $\leq 0{,}013$ mHa. Como el VQD da una cota superior de $E_1$, también se reporta la ventana con el gap a la mitad.

**CCSD contra la cota inferior.** Temple es una cota *inferior* de $E_0$, así que si $E_\text{CCSD}$ queda por debajo, su error es al menos $\text{Temple} - E_\text{CCSD}$. Junto con la cota superior del error del VQE (ventana + desvío medido + σ), el criterio del desafío se puede decidir sin FCI. Detalle en [ADR-0028](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0028-confiabilidad-sin-fci.md).

**Resultado** (una corrida con todos los shots, `SHOTS_SCALE = 1`, notebook hasta la sección 7 en ~7 min local):

| | Valor |
|---|---|
| $\langle S^2\rangle$ máximo / peso de triplete máximo | 2e-5 / 4e-6 |
| Shots en base Z dentro de $N_\alpha = N_\beta = 2$ | 100 % (4000 por punto) |
| $\sqrt{\mathrm{Var}(H)}$ máximo | 0,76 mHa (2,4 Å) |
| Ventana de Temple máxima (gap VQD / gap a la mitad) | 0,071 / 0,14 mHa |
| $E_\text{FCI}$ dentro de $[\text{Temple}, E_\text{sv}]$ | 7/7 |
| Error mínimo de CCSD garantizado sin FCI | 9,1 mHa (1,8 Å) |
| Gap VQD vs. exacto | ≤ 0,013 mHa |
| $\sigma$ medida ($d \geq 1{,}8$ Å, `vqe` de #6) | 0,79–0,83 mHa (3,1–3,4 × 10⁶ shots por punto) |
| Medida vs. statevector / puntos > 2σ por debajo de Temple | ≤ 1,8σ / 0 |
| Cota del error del VQE sin FCI (ventana + desvío + σ) | 1,0–2,3 mHa: le gana a CCSD sin FCI en 7/7 |
| $\lvert E_\text{VQE} - E_\text{FCI}\rvert$ medido | 0,23–1,45 mHa |
| Le gana a CCSD (con FCI) / precisión química | 7/7 / 7/7 |
| Tiempo de la sección | ~25 s local |

**Qué no funcionó.**
- **Temple con el siguiente autoestado de cualquier espín.** El triplete casi degenerado deja una cota inútil a 3,0 Å y, además, no es válida si el estado tiene algo de triplete. Se reemplazó por la proyección al singlete.
- **Detectar la ruptura de simetría con $\langle N\rangle$.** Un estado al azar de `efficient_su2` dio $\langle N\rangle = 3{,}98$ de casualidad. Lo que delata que no es autoestado de $N$ es la varianza ($\mathrm{Var}(N)$ > 0,1), no el valor medio.
- **$\sqrt{\mathrm{Var}}$ (Weinstein) como cota del error.** Es correcta pero floja: a 2,4 Å da 0,76 mHa cuando la ventana de Temple da 0,05 mHa. Sirve como señal (se dispara en un mínimo local), no como número final.

**Limitaciones.**
- La cota de Temple supone que el VQD encontró el siguiente singlete. Si encontrara uno más alto, la cota sería optimista. Lo cubre en parte la versión con el gap a la mitad, y la comparación con el exacto confirma que acá no pasa.
- El gap al siguiente singlete se cierra al estirar (1,2 mHa a 3,0 Å). Más allá de 3,0 Å, o en sistemas más grandes, Temple pierde fuerza y la ventana crece como $\mathrm{Var}/\text{gap}$.
- $\langle S^2\rangle$, la varianza y Temple se calculan con statevector. En hardware habría que medirlos: $S^2$ tiene 77 términos de Pauli y $H^2$ muchos más.
- La precisión química se cumple en los 7 puntos, pero con poco margen: el error máximo es 1,45 mHa, a 2,4 Å. Ese error es estadístico ($\sigma \approx 0{,}8$ mHa), así que con otra semilla algún punto podría quedar apenas por encima de 1,6 mHa. Sin FCI se puede certificar que le ganamos a CCSD, pero no la precisión química.

## Resultados clave

Resumen de una pantalla, recalculado desde las tablas del contrato `refs` (#1) y `vqe` (#6), sin números pegados a mano: $R_e$ y $D_e$ (VQE contra FCI y CCSD), error máximo en la zona estirada, si le ganamos a CCSD con la barra de error incluida, precisión química, `cz` por circuito y shots totales.

- $R_e$ sale de un ajuste cúbico local (5 puntos de la grilla alrededor del mínimo; para VQE, pesado con $1/\sigma$) y $D_e = E(3{,}0\ \text{Å}) - E(R_e)$ con $E(R_e)$ del mismo ajuste. Es una estimación rápida, igual para los tres métodos; los valores con incertidumbre (ajuste de grado 4, bootstrap + sensibilidad) son los de la sección 5 y se muestran al lado para comparar: la estimación rápida tiene que caer dentro de esa incertidumbre.
- "Le ganamos a CCSD" se evalúa punto por punto para $d \geq 1{,}8$ Å con el criterio del enunciado: $|E_\text{VQE} - E_\text{FCI}| + \sigma < |E_\text{CCSD} - E_\text{FCI}|$.
- Si `refs` o `vqe` todavía no existen, están incompletas o son un *stand-in* (marcado con `tabla.attrs["mock"] = True`), la celda lo dice y no calcula nada: nunca muestra números de mentira como si fueran resultados.

In [ ]:
# Resultados clave, recalculados desde `refs` (#1) y `vqe` (#6). Si faltan, avisa en vez de fallar.
from IPython.display import Markdown, display

_COLS = {"refs": ["d", "e_hf", "e_ccsd", "e_fci"], "vqe": ["d", "energy", "sigma", "shots", "cz"]}


def contract_table(name: str) -> tuple[pd.DataFrame | None, str | None]:
    """(tabla, None) si `name` cumple el contrato 0.1 en toda la grilla D; si no, (None, motivo)."""
    t = globals().get(name)
    if t is None:
        return None, f"`{name}` todavía no está definida"
    if getattr(t, "attrs", {}).get("mock"):
        return None, f"`{name}` es un stand-in (`attrs['mock']`), no un resultado"
    try:
        t = pd.DataFrame(t)
    except (TypeError, ValueError):
        return None, f"`{name}` no se puede leer como tabla"
    missing = [c for c in _COLS[name] if c not in t.columns]
    if missing:
        return None, f"a `{name}` le faltan las columnas {missing}"
    t = t[_COLS[name]].assign(d=lambda x: x.d.astype(float).round(2))
    absent = sorted(set(D) - set(t.d))
    if absent:
        return None, f"a `{name}` le faltan las distancias {', '.join(f'{x:g}' for x in absent)} Å"
    t = t[t.d.isin(D)].drop_duplicates("d").sort_values("d").reset_index(drop=True)
    if t.isna().any().any():
        return None, f"`{name}` tiene valores vacíos (NaN)"
    return t, None


def local_minimum(d: np.ndarray, e: np.ndarray, sigma: np.ndarray | None = None) -> tuple[float, float]:
    """(R_e, E(R_e)): mínimo de un ajuste cúbico a los 5 puntos de la grilla más cercanos al mínimo."""
    near = np.argsort(np.abs(d - d[np.argmin(e)]))[:5]
    w = None if sigma is None else 1 / np.maximum(sigma[near], 1e-12)
    poly = np.poly1d(np.polyfit(d[near], e[near], 3, w=w))
    x = np.linspace(d[near].min(), d[near].max(), 4001)
    return x[np.argmin(poly(x))], poly(x).min()


def section5_re_de() -> pd.DataFrame:
    """R_e y D_e con incertidumbre de la sección 5 (tabla `RED`) para comparar; vacía si la sección 5 no corrió."""
    try:
        red = globals()["RED"].set_index("metodo")
        return pd.DataFrame({
            "R_e sección 5 (Å)": [f"{r:.3f} ± {u:.3f}" for r, u in zip(red.R_e_A, red.R_e_tot)],
            "D_e sección 5 (mHa)": [f"{x:.1f} ± {u:.1f}" for x, u in zip(red.D_e_mHa, red.D_e_tot)],
        }, index=red.index)
    except (KeyError, AttributeError, TypeError):
        return pd.DataFrame()


def key_results(refs: pd.DataFrame, vqe: pd.DataFrame) -> None:
    t = refs.merge(vqe, on="d")
    t["err_vqe"] = (t.energy - t.e_fci).abs() * 1e3   # mHa
    t["err_ccsd"] = (t.e_ccsd - t.e_fci).abs() * 1e3  # mHa
    t["sigma_mHa"] = t.sigma * 1e3
    t["gana_a_ccsd"] = t.err_vqe + t.sigma_mHa < t.err_ccsd
    s = t[t.d >= STRETCHED]

    d = t.d.to_numpy()
    e_end = t.set_index("d")
    curves = {"VQE": (t.energy.to_numpy(), t.sigma.to_numpy()), "FCI": (t.e_fci.to_numpy(), None),
              "CCSD": (t.e_ccsd.to_numpy(), None)}
    col = {"VQE": "energy", "FCI": "e_fci", "CCSD": "e_ccsd"}
    rows = {}
    for name, (e, sig) in curves.items():
        r_e, e_min = local_minimum(d, e, sig)
        rows[name] = {"R_e (Å)": r_e, "D_e (mHa)": (e_end.loc[DISSOCIATED, col[name]] - e_min) * 1e3}
    quick = pd.DataFrame(rows).T.round({"R_e (Å)": 3, "D_e (mHa)": 1})
    display(quick.join(section5_re_de(), how="left"))

    worst = s.loc[s.err_vqe.idxmax()]
    margin = s.loc[(s.err_ccsd - s.err_vqe - s.sigma_mHa).idxmin()]
    yes = lambda ok: "**sí**" if ok else "**no**"
    cz = sorted(t.cz.astype(int).unique())
    lo, hi = int(t.shots.min()), int(t.shots.max())
    shots_range = f"{lo:,}" if lo == hi else f"{lo:,}–{hi:,}"
    display(Markdown("\n".join([
        f"- **Error máximo en la zona estirada** ($d \\geq {STRETCHED}$ Å): {worst.err_vqe:.2f} mHa a {worst.d} Å "
        f"($\\sigma$ = {worst.sigma_mHa:.2f} mHa); CCSD ahí erra {worst.err_ccsd:.1f} mHa.",
        f"- **¿Le ganamos a CCSD?** {yes(s.gana_a_ccsd.all())}: en {int(s.gana_a_ccsd.sum())} de {len(s)} puntos "
        f"$|E_\\text{{VQE}} - E_\\text{{FCI}}| + \\sigma < |E_\\text{{CCSD}} - E_\\text{{FCI}}|$. Margen más chico: "
        f"{margin.err_ccsd - margin.err_vqe - margin.sigma_mHa:.2f} mHa a {margin.d} Å.",
        f"- **¿Precisión química** ($< {CHEM_ACC * 1e3:g}$ mHa)? Zona estirada: {yes((s.err_vqe < CHEM_ACC * 1e3).all())} "
        f"({int((s.err_vqe < CHEM_ACC * 1e3).sum())} de {len(s)} puntos); toda la curva: "
        f"{int((t.err_vqe < CHEM_ACC * 1e3).sum())} de {len(t)} puntos.",
        f"- **`cz` por circuito:** {', '.join(map(str, cz))} (transpilado a `cz, sx, x, rz` en línea).",
        (f"- **Shots:** {int(t.shots.sum()):,} en total para {len(t)} puntos ({shots_range} por punto)."
         .replace(",", ".")),
    ])))


_tables = {name: contract_table(name) for name in _COLS}
_pending = [why for _, why in _tables.values() if why]
if _pending:
    display(Markdown("**Resultados pendientes.** " + "; ".join(_pending) + ". "
                     "Esta celda se completa sola cuando las secciones 1 y 4 llenen `refs` y `vqe`."))
else:
    key_results(_tables["refs"][0], _tables["vqe"][0])

## 7. Decisiones, qué no funcionó y limitaciones

Esta sección junta lo que documentó cada sección (el detalle, con tablas y pruebas, está en el markdown de decisiones de cada una; acá va el resumen y el enlace) y analiza hasta dónde funciona el método, dónde falla y por qué. Las decisiones de infraestructura también están como ADR en `docs/adr/` del repo. Los números citados acá son de la corrida completa (`SHOTS_SCALE = 1`, semillas fijas); los que se recalculan solos están en *Resultados clave* y en la sección 5.

### 7.1 Decisiones

| Etapa | Decisión | Por qué | Detalle |
|---|---|---|---|
| Referencias (#1) | RHF (no UHF) con chequeo de estabilidad interna, CCSD y FCI singlete con PySCF sobre la grilla `D` → `refs`. | Queremos un singlete con orbitales restringidos, como el Hamiltoniano de qubits; HF/CCSD/FCI solo sirven para comparar y elegir puntos de partida (regla del desafío). | Sección 1 |
| Hamiltoniano (#2) | `qubit_hamiltonian(d) -> (op, e_nuc)`; Jordan-Wigner escrito a mano, genérico; orbitales RHF con signo fijo; caché que devuelve copias. | Sin dependencias nuevas y con control exacto del orden de qubits; el signo fijo hace que los coeficientes de $\hat H$ varíen suave con $d$ (si no, el *warm start* arranca lejos); RHF en toda la grilla para que la referencia siga siendo un singlete y $\langle S^2\rangle$ sirva de diagnóstico. | Sección 2, *Decisiones (#2)*; [ADR-0001](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0001-firma-qubit-hamiltonian.md), [ADR-0002](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0002-jordan-wigner-a-mano.md), [ADR-0003](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0003-signo-orbitales-y-cache.md) |
| Ansatz (#4) | 2-QUpCCGSD podado: 19 excitaciones de qubit (pares generalizados + simples con el mismo ángulo en α y β) en 2 capas, como rotaciones de Givens; **465 `cz`**, 19 parámetros. | Llega a FCI en toda la grilla (≤ 0,03 mHa con statevector), donde UCCSD se traba (3,3 mHa a 1,8 Å) por la misma razón que CCSD; conserva $N$ y $S_z$; Givens con multiplexor Gray en vez de `PauliEvolutionGate` (837 → 590 `cz`) y poda de las excitaciones de menor ángulo (590 → 465). | Sección 3, *Decisión (#4)* |
| Optimización (#5) | BFGS con gradiente adjunto exacto en el sector de 36 estados; *warm start* en las dos direcciones con varios arranques por punto y rondas de pulido; se minimiza $E + 0{,}05\,\langle S^2\rangle$. | La regla de *parameter-shift* de 2 términos no vale para excitaciones dobles; los mínimos locales obligan a multi-start; la penalidad evita elegir estados con triplete casi degenerados en energía. Error ≤ 0,003 mHa en toda la grilla. | Sección 4 |
| Medición (#6) | `SamplerV2` de `qiskit_aer` con shots explícitos; $E$ y $\sigma$ desde los bitstrings. Grupos QWC por *sorted insertion* (37–41 grupos); valor por shot del grupo entero; piloto de 500 shots por grupo y reparto $N_g \propto \sqrt{\operatorname{Var}_g}$; $\sigma$ objetivo de **0,8 mHa en $d \ge 1{,}8$ Å** (≤ 1/5 del margen de CCSD y $2\sigma < 1{,}6$ mHa) y 3 mHa en el resto; todos los grupos de un punto como *bindings* de un solo PUB. `EstimatorV2` solo como control cruzado. | `EstimatorV2` de Aer no muestrea (equivale a leer el statevector). La varianza del grupo entero ya incluye las covarianzas; el reparto óptimo minimiza $\sigma$ a shots fijos; el objetivo por punto pone los shots donde se juzga el desafío (≈ 86 % de los 26,4 M shots van a la zona estirada). Un solo PUB evita que Aer correlacione grupos con la misma semilla. $\sigma$ validada con tres pruebas (medido vs. statevector, repeticiones, $\sigma \propto 1/\sqrt N$). | Sección 4.2, *Qué decidimos (#6)*; [ADR-0005](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0005-energias-con-samplerv2-no-estimatorv2.md), [ADR-0009](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0009-medicion-qwc-optima-rotacion-extra.md), [ADR-0014](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0014-presupuesto-de-shots-en-dos-etapas.md), [ADR-0015](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0015-pyscf-un-hilo-para-reproducibilidad.md), [ADR-0016](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0016-velocidad-aer-bindings-y-bloques.md) |
| Resultados (#7) | Todo se recalcula desde `vqe` + referencias. $R_e$: polinomio de grado 4 en 0,6–1,4 Å (VQE pesado con $1/\sigma$). Incertidumbre = bootstrap paramétrico ⊕ sensibilidad (grado 2–5, cuatro ventanas). $D_e = E(3{,}0) - E(R_e)$, con los dos términos re-muestreados juntos. `gana_a_ccsd` = $\text{err}_\text{VQE} + \sigma < \text{err}_\text{CCSD}$ con la energía *medida*, solo en $d \ge 1{,}8$ Å. | La curva es asimétrica (una parábola sesga $R_e$); el bootstrap solo no ve el sesgo del modelo de ajuste; el criterio exige ganar aun sumando una $\sigma$. La disociación incompleta a 3,0 Å se cuantifica (~1 mHa) contra cuatro H aislados. | Sección 5, *Qué decidimos (#7)*; [ADR-0018](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0018-ajuste-de-re.md), [ADR-0019](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0019-incertidumbre-en-dos-componentes-y-de.md), [ADR-0020](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0020-criterio-gana-a-ccsd.md) |
| Confiabilidad (#8) | Evidencias sin FCI sobre el estado en θ*: $N$, $S_z$ y $S^2$ con el mismo Jordan-Wigner; cota de Temple sobre la parte singlete con el gap al siguiente singlete estimado por VQD; arranques de #5; CCSD comparado contra la cota inferior de Temple. | El triplete casi degenerado hace inútil a la energía (y a Temple con cualquier espín) para detectar contaminación; diagonalizar para obtener $E_1$ sería FCI. Resultado: $E_0$ en una ventana de ≤ 0,07 mHa bajo el VQE y error de CCSD ≥ 9,1 mHa, sin usar FCI. | Sección 6, *Decisiones (#8)*; [ADR-0028](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0028-confiabilidad-sin-fci.md) |

### 7.2 Qué no funcionó

- **`A.compose(B)` para multiplicar operadores (#2).** En Qiskit es $B \cdot A$: daba el Hamiltoniano con partículas y huecos intercambiados, que a medio llenado tiene *el mismo espectro* en el sector $N = 4$. La prueba contra FCI pasaba igual; solo la prueba de HF lo detectó. Lección: hacen falta las dos pruebas. Se usa `@`.
- **Orbitales sin signo fijo (#2, #4).** Entre 76 y 84 términos de Pauli cambiaban de signo de golpe entre distancias vecinas; con un Hamiltoniano provisorio así, el *warm start* del ansatz caía en otros mínimos y aparecía contaminación de triplete ($\langle S^2\rangle$ hasta 3e-2) desde 2,0 Å.
- **UCCSD y 1-UpCCGSD (#4, #15):** no llegan a precisión química en la zona estirada (3,3–3,5 y 11 mHa a 1,8 Å).
- **Hardware-efficient (#4, #15):** 28–56 `cz`, pero optimizando la energía queda a 23–44 mHa de FCI a 1,8 Å y con $\langle S^2\rangle \approx 2$; puede representar el estado (optimizando contra el vector FCI, cosa imposible en un VQE real), así que el problema es el paisaje de optimización. Más capas no ayudan (#15).
- **QEB-ADAPT-VQE (#4):** recomendado por el análisis previo; llega a FCI pero necesita 461–573 `cz` en línea (sus dobles tocan qubits lejanos) y un circuito distinto por distancia. No mejora al elegido.
- **`EstimatorV2` de `qiskit_aer` para reportar:** no muestrea; devuelve el valor exacto más ruido gaussiano opcional. Equivale a leer el statevector, así que se descartó para los números reportados (#6). En #12 además esconde el error de lectura.
- **Elegir el mejor arranque solo por energía (#5, #16).** En la zona estirada a veces ganaba un estado con $\langle S^2\rangle \approx 3\cdot 10^{-2}$ a solo 0,02 mHa de FCI, y cuál ganaba cambiaba entre corridas por diferencias numéricas de ~1e-13 Ha. Se agregó la penalidad en $\langle S^2\rangle$.
- **Arrancar desde HF o al azar (#5).** Desde $\theta = 0$ el optimizador queda a 150–290 mHa en la zona estirada; con arranques al azar la mediana queda a 0,1–3 mHa. Hace falta el *warm start*; un solo sentido del barrido deja hasta 0,1 mHa, los dos más el pulido bajan a 0,003 mHa.
- **Un PUB por grupo con distinto número de shots (#6; lo vieron por separado #15, #17 y #19).** Es lo natural, pero `SamplerV2` de Aer corre en jobs separados los PUBs con distinto número de shots **con la misma semilla**: los grupos salen correlacionados, la dispersión real era ≈ 1,3–1,4 $\sigma$ y el $\chi^2$ fallaba. Lo detectó la validación con repeticiones. Un `run` por grupo con semilla propia lo arregla pero recompila el circuito cada vez (3–4 s por estimación); la solución fue un solo PUB con un *binding* por grupo.
- **Otros atajos de medición descartados (#6):** sumar varianzas término a término (ignora las covarianzas dentro del grupo), reusar los shots de la piloto en la estimación (acopla reparto y estimación) y PySCF con varios hilos (las sumas en paralelo cambian $\hat H$ en ~1e-14, eso mueve el reparto de shots y la corrida deja de ser reproducible bit a bit; se fija a un hilo).
- **Ajustes de $R_e$ descartados (#7):** parábola sola (sesgada por la asimetría de la curva), spline cúbico (con $\sigma$ de 1–3 mHa interpola el ruido y el mínimo salta), ajustar con todos los puntos hasta 3,0 Å (los puntos planos tiran del polinomio) y reportar solo el bootstrap (no ve el sesgo del modelo de ajuste).
- **Temple con el siguiente autoestado de cualquier espín (#8).** En la zona estirada es el triplete (0,46 mHa a 3,0 Å): la cota sale inútil y ni siquiera es válida si el estado tiene algo de triplete. Se proyecta al singlete y se usa el siguiente singlete.

### 7.3 Limitaciones: hasta dónde funciona y dónde falla

**Dónde funciona.** Sin ruido, en este sistema, el ansatz representa el fundamental en toda la grilla (≤ 0,03 mHa con statevector, singlete con $\langle S^2\rangle \leq 3\cdot 10^{-7}$) y el error que queda en la energía reportada es estadístico: lo fija el presupuesto de shots, no el método. En la corrida completa VQE **le gana a CCSD en los 7 puntos** con $d \ge 1{,}8$ Å (margen más chico: ~7 mHa a 1,8 Å) y queda **dentro de precisión química en los 7** (error máximo ~1,3–1,5 mHa, con $\sigma \approx 0{,}8$ mHa). $D_e$ sale ~3 mHa por encima de FCI, dentro de su incertidumbre, mientras que CCSD lo subestima en 23 mHa.

**Dónde falla o deja de valer, y por qué:**

- **Precisión química solo donde pusimos los shots.** Con $\sigma = 3$ mHa en $d < 1{,}8$ Å (decisión de presupuesto de #6), solo 11 de los 19 puntos de la curva quedan dentro de 1,6 mHa de FCI. Llevar toda la curva a 0,8 mHa costaría ~14 veces más shots en esos 12 puntos (≈ 5 × 10⁷ shots más).
- **$R_e$ está limitado por la medición cerca del mínimo.** Con el mismo ajuste, el $R_e$ de VQE queda ~0,013 Å por debajo del de FCI, unas 2,5–3 veces su incertidumbre estadística (~0,005 Å): lo arrastra el punto de 0,8 Å, que sale 9–10 mHa abajo ($z \approx -3$). La incertidumbre total que reporta la sección 5 (±0,08 Å) lo cubre, pero está dominada por la sensibilidad al modelo de ajuste, que incluye la parábola que #7 mismo descarta: es una cota conservadora, no el error típico. Con la grilla de 0,1 Å el ajuste de FCI da 0,885–0,889 Å según el método (sección 5 y *Resultados clave*), así que ~0,005 Å es la resolución del ajuste.
- **La validación de $\sigma$ es buena pero no perfecta.** A escala completa, el $\chi^2$ de medido vs. statevector da 30–32 con 19 grados de libertad ($p \approx 0{,}03$–$0{,}06$; máximo $|z| \approx 3$–$3{,}3$, siempre a 0,8 Å), mientras que las repeticiones (dispersión/$\sigma$ = 0,83–1,21) y la pendiente $\sigma \propto N^{-0{,}50}$ no muestran problemas. Puede ser una fluctuación o una $\sigma$ levemente subestimada en algunos puntos; #6 midió aparte que no hay sesgo del estimador. Lo retoma la sección 6.
- **Reproducibilidad a escala completa.** Dos corridas seguidas con las mismas semillas no dieron tablas idénticas: los shots por punto cambian hasta ~2 % y las energías una fracción de $\sigma$ en algunos puntos (p. ej. el error máximo en la zona estirada pasó de 1,45 a 1,32 mHa), aunque #6 había verificado reproducibilidad bit a bit a escala 0,05. Las conclusiones no cambian (7 de 7 contra CCSD, 7 de 7 en precisión química, $R_e$ y $D_e$ muy dentro de su incertidumbre); la causa probable es que $\theta^*$ o $\hat H$ varían en la última cifra entre corridas y eso mueve el reparto de shots. Por eso las cifras de esta sección se dan con rangos.
- **Base mínima (STO-3G).** La curva es la del *modelo* $\mathrm{H_4}$/STO-3G: "exacto" quiere decir FCI en esta base, no el $\mathrm{H_4}$ real. $R_e$ y $D_e$ cambian bastante en una base más grande, que además necesitaría muchos más qubits. Y a 3,0 Å la cadena todavía no está del todo disociada: FCI queda ~1 mHa por debajo de cuatro átomos de H aislados, así que $D_e$ (definida por el enunciado con 3,0 Å) está levemente subestimada respecto del límite de disociación.
- **Solo 8 qubits.** El sector relevante ($N = 4$, $S_z = 0$) tiene 36 determinantes: FCI es trivial clásicamente y por eso podemos validar todo contra él. El método no demuestra ventaja cuántica; demuestra el procedimiento. Al crecer el sistema crecen la cantidad de excitaciones (y de `cz`), los términos del Hamiltoniano y los shots necesarios para un $\sigma$ dado. El extra de SQD (#16) muestra el mismo límite desde otro lado: con 36 determinantes cualquier muestreo, incluso al azar, termina siendo exacto.
- **Búsqueda de parámetros con statevector.** Está permitida, pero cuesta exponencial en el número de qubits y aprovecha que el ansatz vive en el sector de 36 estados. En hardware real habría que optimizar con energías y gradientes medidos (con shots y ruido), mucho más lento e inestable.
- **El ansatz se eligió y se podó mirando FCI.** Comparar contra FCI está permitido, pero en un sistema donde FCI no se puede calcular la poda tendría que guiarse por otro criterio (gradientes al estilo ADAPT, varianza de la energía). Por eso la sección 6 arma el argumento de confiabilidad sin FCI. La poda se ajustó en la cadena lineal; en el cuadrado (#17) sigue llegando a FCI, pero menos arranques lo logran.
- **Mínimos locales.** El paisaje tiene mínimos locales: con un solo arranque algunas distancias quedan a 1 mHa o a decenas de mHa de FCI. Usamos varios arranques y *warm start*, pero no hay garantía de mínimo global; la curva suave y los arranques coincidentes son evidencia, no prueba.
- **Contaminación de espín.** El ansatz conserva $N$ y $S_z$ pero no $S^2$ (las excitaciones de qubit no están adaptadas al espín). En la zona estirada el triplete está casi degenerado (0,46 mHa a 3,0 Å), así que una mezcla singlete-triplete casi no cambia la energía ni la varianza: solo $\langle S^2\rangle$ la delata. Hay que medirlo en cada punto juzgado (sección 6).
- **Hartree-Fock deja de ser buena referencia.** RHF es inestable hacia UHF desde ~1,1 Å y su peso en el estado exacto cae a 0,27 a 3,0 Å. Mantenemos orbitales RHF (solo definen la base), pero el estado inicial `HF_BITSTRING` está lejos del fundamental estirado y el ansatz tiene que hacer todo el trabajo multirreferencia.
- **Costo de medición.** $\sigma \propto 1/\sqrt{N_\text{shots}}$: con grupos QWC y reparto óptimo, $\sigma = 0{,}8$ mHa cuesta ~3,1–3,4 × 10⁶ shots por punto (26,4 M en toda la curva). Bajar $\sigma$ a la mitad cuesta 4 veces más. La rotación de base por factorización doble (#19) pediría ~6,7 veces menos shots a 1,8 Å y cientos de veces menos a 3,0 Å, a cambio de hasta 24 `cz` extra por circuito de medición; no la adoptamos para el resultado principal porque llegó tarde, suma `cz` y en esta corrida su construcción falló a 2,4 Å (las integrales de PySCF no reprodujeron la convención de signos de #2).
- **Ruido: con este ansatz el circuito no sobrevive sin mitigación.** Con el modelo común de `ruido.py` (0,1 % por `cz`, 0,005 % por compuerta de un qubit), el ansatz completo (389 `cz` al transpilar con $\theta^*$ ya asignados; los 465 de la tabla son del circuito parametrizado) queda a 260–510 mHa de FCI, y la mejor versión corta (12 operadores, 300 `cz`) a 210–235 mHa: ningún circuito le gana a CCSD con ruido (#11). Con el ruido calibrado de `FakeFez` a 2,4 Å (`cz` ~4 veces peor, SWAPs de la conectividad heavy-hex y error de lectura), ningún circuito supera al propio estado HF medido con el mismo ruido (#12). Los resultados del desafío central son **sin ruido**; con ruido harían falta un ansatz mucho más corto, menos qubits por simetrías y mitigación (extrapolación a ruido cero, corrección de lectura), que no probamos.
- **La cota de Temple depende del gap (#8).** El siguiente singlete se acerca al estirar (1,2 mHa a 3,0 Å) y se estima con un VQD, que da una cota superior de $E_1$: si encontrara un singlete más alto, la cota sería optimista (por eso también se reporta con el gap a la mitad). Más allá de 3,0 Å la ventana Var/gap crece. Además $\langle S^2\rangle$, la varianza y Temple son diagnósticos de statevector: en hardware habría que medirlos.

### 7.4 Extras: qué aportan al análisis

Ninguno entra en el desafío central (`docs/challenge/05-extras.md`); cada uno tiene su propio markdown de decisiones en la sección *Extras*.

| Extra | Qué probamos | Qué aprendimos | Detalle |
|---|---|---|---|
| Comparación de ansätze (#15) | HEA, UCCSD y k-UpCCGSD (k = 1, 2, 3) en toda la grilla, mismo presupuesto de shots. **Por defecto usa resultados guardados** (`ANSATZ_CACHE`, `RUN_ANSATZ_COMPARISON = False`): recalcular tarda ~8 min. | 2-UpCCGSD llega a FCI y conserva $S^2$; compilado con `PauliEvolutionGate` cuesta 982 `cz`, contra 465 de la versión de Givens podada de #4. HEA es barato pero queda en mínimos locales y con $\langle S^2\rangle \approx 2$. Con ruido la elección se invierte. | *Comparación de ansätze (#15)* |
| VQE vs. SQD (#16) | SQD a mano sobre muestras de `SamplerV2`, con tope $k$ en el subespacio, contra determinantes al azar. | Con tope, las muestras del ansatz llegan a precisión química con $k$ = 16 / 12 / 6 contra ~35 / 34 / 28 del azar; sin tope, SQD es exacto hasta con muestras al azar: $\mathrm{H_4}$ es demasiado chico para distinguir. | *VQE vs. SQD (#16)* |
| Otra geometría (#17) | Cuadrado de lado $d$ con el mismo pipeline (medición propia con `SamplerV2`). | CCSD falla en toda la curva del cuadrado (casi degeneración, $\max\lvert t_2\rvert \approx 1$) y no converge a 3,0 Å; el ansatz de #4 sigue llegando a FCI. $R_e$ 1,28 vs. 0,89 Å, $D_e$ 103 vs. 313 mHa. | *Otra geometría (#17)* |
| Ruido común (#11) | La medición de #6 con `ruido.py`, ansatz completo vs. versiones cortas. | El ruido cuesta ~0,7 mHa por `cz` en la zona estirada; ninguna versión le gana a CCSD con ruido. | *Ruido (ruido.py) (#11)* |
| Ruido de hardware (#12) | Un punto (2,4 Å) con `FakeFez`, transpilado al backend. Necesita `qiskit-ibm-runtime` (su propio `%pip`). | El error de lectura aparece aun con 0 `cz` y solo se ve midiendo con `SamplerV2`; HF gana a cualquier circuito variacional. | *Ruido de hardware (FakeFez) (#12)*; [ADR-0011](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0011-fakefez-via-qiskit-ibm-runtime.md) |
| Esquemas de medición (#19) | QWC mejorado, grupos conmutantes con Clifford y factorización doble, sobre el estado de #6. | Factorización doble: ~6,7× menos shots a 1,8 Å con ≤ 24 `cz` extra; conmutantes: 2–2,7× con hasta 77 `cz`; QWC mejorado no gana nada (*sorted insertion* ya está cerca del óptimo). | *Esquemas de medición (#19)*; [ADR-0027](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0027-esquemas-de-medicion-avanzados.md) |

## Extras

Nada de esto entra en el desafío central (ver `docs/challenge/05-extras.md`).

### VQE vs. SQD (#16)

**Fuera del desafío central.** SQD (*Sample-based Quantum Diagonalization*, también QSCI) usa el circuito solo para **muestrear determinantes**; después proyecta el Hamiltoniano en el subespacio que generan esas muestras y lo diagonaliza en forma clásica. El enunciado lo deja afuera del desafío porque en $\mathrm{H_4}$/STO-3G el sector $N = 4$, $S_z = 0$ tiene solo **36 determinantes** (6 cadenas α × 6 cadenas β): con un puñado de shots de cualquier circuito, incluso al azar, se cubre todo y SQD *es* FCI. Lo interesante, entonces, es qué pasa cuando se pone un **tope $k$** al tamaño del subespacio: ¿las muestras cuánticas eligen mejores determinantes que el azar?

Cómo lo hicimos:

- **Muestras:** `SamplerV2` de `qiskit_aer`, 2000 shots por circuito, sobre el circuito transpilado a `cz, sx, x, rz` en línea. Tres fuentes en $d = 1{,}8$, $2{,}4$ y $3{,}0$ Å: el ansatz de la sección 3 (19 excitaciones, 465 `cz`), el mismo con el ruido de `05-extras` (despolarizante 0,1 % por `cz` y 0,005 % por compuerta de un qubit; `ruido.py` no está en el snapshot, así que lo armamos a mano con esos números) y una versión **recortada** a la primera capa (9 excitaciones, 191 `cz`), que como VQE queda lejos de FCI.
- **Parámetros:** si #5 ya dejó `params[d]`, se usa; si no (stand-in), se optimiza el ansatz con statevector en el sector de 36 estados con 20 arranques, minimizando $E + \lambda\langle S^2\rangle$ con $\lambda = 0{,}1$ Ha (el paisaje tiene mínimos locales y el triplete está casi degenerado, ver sección 3 y abajo). El recortado se optimiza igual, siempre.
- **Postselección y recuperación de configuraciones:** los bitstrings con $N_\alpha \neq 2$ o $N_\beta \neq 2$ (solo aparecen con ruido) se corrigen con una versión simplificada de la recuperación de configuraciones de Robledo-Moreno et al.: con las ocupaciones medias $n_q$ de las muestras buenas, a la mitad α o β que tiene electrones de más se le saca el del orbital menos ocupado, y a la que tiene de menos se le agrega en el más ocupado.
- **Tope $k$:** el subespacio son los $k$ determinantes **más frecuentes** (criterio de QSCI). Se proyecta $H$ (el de #2, `sector_hamiltonian`) en esos $k$ y se toma el menor autovalor.
- **Azar:** $k$ determinantes distintos elegidos uniformemente entre los 36 (es lo mismo que quedarse con los $k$ más frecuentes de bitstrings uniformes con el mismo $N$ y $S_z$), 300 repeticiones.
- **Sin tope, estilo `qiskit-addon-sqd`:** subespacio (cadenas α ∪ cadenas β) × (las mismas), simétrico ante α ↔ β.
- **VQE para comparar:** si #6 ya dejó la tabla `vqe`, se usa; si no (stand-in), se mide $\langle H \rangle$ con `SamplerV2` agrupando los términos de Pauli que conmutan qubit a qubit (46 grupos × 100 000 shots), con los mismos $\theta$.

**Por qué SQD a mano y no `qiskit-addon-sqd`:** el issue pedía agregarlo siguiendo la regla de dependencias, pero **no está en el nixpkgs fijado** en `flake.lock`, así que no se puede fijar la misma versión en `requirements.txt`, en el `%pip` y en nix (`nix flake check` fallaría). Con 36 determinantes no hace falta: la proyección es indexar la matriz de 36 × 36 de la sección 3 y diagonalizar con numpy. Lo que reimplementamos (postselección, recuperación de configuraciones, subespacio α × β) es una versión mínima de lo que hace la librería, no una copia exacta (ella, por ejemplo, recupera configuraciones de forma probabilística e iterativa).

Reutiliza de las secciones anteriores: `sector_hamiltonian`, `SECTOR`, `SectorAnsatz`, `sector_ansatz`, `givens_single`, `givens_double`, `ANSATZ_OPS`, `transpile_line`. Tarda alrededor de un minuto.

In [ ]:
# Extras · VQE vs. SQD (#16): muestreo con SamplerV2 y SQD implementado a mano (sin qiskit-addon-sqd).
from qiskit import ClassicalRegister
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import SamplerV2
from pyscf.fci import spin_op

SQD_D = [1.8, 2.4, 3.0]  # Å, zona estirada
SQD_SHOTS = 2_000        # shots por circuito para SQD
SQD_SEED = 16
SQD_STARTS = 20       # arranques del multi-start (stand-in de params)


def circuit_from_ops(ops, name: str) -> QuantumCircuit:
    """Igual que build_ansatz() pero con una lista de excitaciones arbitraria (para la versión recortada)."""
    th = ParameterVector("θ", len(ops))
    qc = QuantumCircuit(N_QUBITS, name=name)
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for t, (_, kind, p, q) in zip(th, ops):
        if kind == "D":
            givens_double(qc, t, alpha_qubit(p), beta_qubit(p), alpha_qubit(q), beta_qubit(q))
        else:
            givens_single(qc, t, alpha_qubit(p), alpha_qubit(q))
            givens_single(qc, t, beta_qubit(p), beta_qubit(q))
    return qc


# Dos circuitos: el ansatz elegido (19 excitaciones) y uno recortado a la primera capa (9), más barato y lejos de FCI.
OPS_SHALLOW = [op for op in ANSATZ_OPS if op[0] == 0]
SQD_CIRCUITS = {"ansatz (19 exc.)": (ANSATZ_OPS, transpile_line(build_ansatz())),
                "recortado (9 exc.)": (OPS_SHALLOW, transpile_line(circuit_from_ops(OPS_SHALLOW, "QUpCCGSD-9")))}
SQD_SECTOR_ANSATZ = {"ansatz (19 exc.)": sector_ansatz, "recortado (9 exc.)": SectorAnsatz(OPS_SHALLOW)}

# Ruido de 05-extras (ruido.py no está en el snapshot): despolarizante 0,1 % por cz y 0,005 % por compuerta de 1 qubit.
NOISE = NoiseModel(basis_gates=BASIS_GATES)
NOISE.add_all_qubit_quantum_error(depolarizing_error(1e-3, 2), ["cz"])
NOISE.add_all_qubit_quantum_error(depolarizing_error(5e-5, 1), ["sx", "x", "rz"])


# S² en el sector de 36 estados (columna j = S²|det_j⟩), para penalizar la mezcla con el triplete.
S2_SECTOR = np.array([spin_op.contract_ss(e.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA)).ravel()
                      for e in np.eye(len(SECTOR))]).T
SPIN_PENALTY = 0.1  # Ha por unidad de ⟨S²⟩


def theta_for(label: str, d: float) -> np.ndarray:
    """θ*: usa `params[d]` de #5 si existe (solo para el ansatz completo); si no, multi-start con statevector
    minimizando E + λ⟨S²⟩ (sin la penalización, en 3,0 Å el mejor arranque salía con ⟨S²⟩ = 0,08)."""
    table = globals().get("params")
    if label == "ansatz (19 exc.)" and isinstance(table, dict):
        for key, x in table.items():
            if abs(float(key) - d) < 1e-6 and len(x) == ansatz.num_parameters:
                return np.asarray(x, dtype=float)
    sa, (H, _) = SQD_SECTOR_ANSATZ[label], sector_hamiltonian(d)
    H_pen = H + SPIN_PENALTY * S2_SECTOR
    rng = np.random.default_rng(SQD_SEED)
    starts = [np.zeros(sa.n)] + [rng.normal(scale=0.7, size=sa.n) for _ in range(SQD_STARTS - 1)]
    return min((minimize(sa.energy_grad, s, args=(H_pen,), jac=True, method="BFGS", options={"gtol": 1e-10})
                for s in starts), key=lambda r: r.fun).x


def measured(qc_t: QuantumCircuit, x: np.ndarray, basis: dict | None = None) -> QuantumCircuit:
    """Asigna θ y mide el qubit lógico v en el clbit v. Ojo: el circuito transpilado permuta qubits al final
    (final_layout); medir con measure_all() sobre el transpilado mezcla α y β (nos pasó: errores de 800 mHa)."""
    qc = qc_t.assign_parameters(x)
    phys = qc_t.layout.final_index_layout()
    for v, b in (basis or {}).items():  # rotación de base para medir X o Y (la usa el VQE de la tabla)
        if b == "Y":
            qc.sdg(phys[v])
        qc.h(phys[v])
    cr = ClassicalRegister(N_QUBITS, "meas")
    qc.add_register(cr)
    for v in range(N_QUBITS):
        qc.measure(phys[v], cr[v])
    return qc


def sample_bitstrings(qc_t, x, shots=SQD_SHOTS, noisy=False) -> np.ndarray:
    """Bitstrings (enteros, q0 = bit menos significativo) en el orden en que salieron, con SamplerV2 de Aer."""
    opts = {"backend_options": {"noise_model": NOISE}} if noisy else {}
    res = SamplerV2(seed=SQD_SEED, options=opts).run([measured(qc_t, x)], shots=shots).result()
    return np.array([int(b, 2) for b in res[0].data.meas.get_bitstrings()])


# --- SQD a mano -----------------------------------------------------------------------------------
# Un determinante es un entero de 8 bits: α = bits 0..3, β = bits 4..7 (convención 0.1). SECTOR (sección 3)
# lista los 36 con N_α = N_β = 2, en el mismo orden que la matriz sector_hamiltonian(d).
POS = {int(s): i for i, s in enumerate(SECTOR)}
_POP = np.array([bin(m).count("1") for m in range(16)])


def in_sector(b: int) -> bool:
    return _POP[b & 15] == N_ALPHA and _POP[b >> N_ORB] == N_BETA


def recover_configurations(shots: np.ndarray) -> tuple[np.ndarray, int]:
    """Postselección + recuperación de configuraciones simplificada (una pasada, determinista).
    Las ocupaciones n_q se estiman con las muestras que ya están en el sector; a cada mitad (α o β) con el
    número equivocado de electrones se le saca el electrón del orbital menos ocupado o se le agrega al más ocupado."""
    good = np.array([in_sector(b) for b in shots])
    occ = np.array([(shots[good] >> q) & 1 for q in range(N_QUBITS)]).mean(axis=1)
    fixed = shots.copy()
    for i in np.flatnonzero(~good):
        out = 0
        for half, n_el in ((0, N_ALPHA), (1, N_BETA)):
            n = occ[N_ORB * half:N_ORB * (half + 1)]
            bits = [(int(shots[i]) >> (N_ORB * half + p)) & 1 for p in range(N_ORB)]
            while sum(bits) > n_el:
                bits[min((p for p in range(N_ORB) if bits[p]), key=lambda p: n[p])] = 0
            while sum(bits) < n_el:
                bits[max((p for p in range(N_ORB) if not bits[p]), key=lambda p: n[p])] = 1
            out |= sum(bt << p for p, bt in enumerate(bits)) << (N_ORB * half)
        fixed[i] = out
    return fixed, int((~good).sum())


def subspace_ground(H: np.ndarray, dets) -> tuple[float, np.ndarray]:
    """Proyecta H en el subespacio de determinantes `dets` y diagonaliza. Devuelve (E, vector en los 36)."""
    idx = [POS[int(b)] for b in dets]
    w, v = np.linalg.eigh(H[np.ix_(idx, idx)])
    psi = np.zeros(len(SECTOR))
    psi[idx] = v[:, 0]
    return w[0], psi


def ranked_dets(shots: np.ndarray) -> list[int]:
    """Determinantes distintos ordenados por frecuencia (el tope k se queda con los k más muestreados)."""
    vals, counts = np.unique(shots, return_counts=True)
    return [int(b) for b in vals[np.argsort(-counts, kind="stable")]]


def spin_square(psi: np.ndarray) -> float:
    return spin_op.spin_square0(psi.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0]


def product_subspace(shots: np.ndarray) -> list[int]:
    """Subespacio estilo qiskit-addon-sqd: (cadenas α ∪ cadenas β) × (las mismas), simétrico en α ↔ β."""
    strings = sorted(set((shots & 15).tolist()) | set((shots >> N_ORB).tolist()))
    return [a | (b << N_ORB) for a in strings for b in strings]


for label, (ops, qc_t) in SQD_CIRCUITS.items():
    print(f"{label}: {qc_t.count_ops().get('cz', 0)} cz")

In [ ]:
# Experimento clave: error de SQD vs. tope k del subespacio, muestras cuánticas vs. al azar (mismo N y S_z).
import matplotlib.pyplot as plt

N_RANDOM_TRIALS = 300
K = np.arange(1, len(SECTOR) + 1)
rng_sqd = np.random.default_rng(SQD_SEED)

SOURCES = [("ansatz (19 exc.)", False), ("ansatz (19 exc.) + ruido", True), ("recortado (9 exc.)", False)]
sqd_curves, sqd_samples, sqd_theta = {}, {}, {}
for d in SQD_D:
    H, e_fci = sector_hamiltonian(d)
    for name, noisy in SOURCES:
        label = name.replace(" + ruido", "")
        if (label, d) not in sqd_theta:
            sqd_theta[label, d] = theta_for(label, d)
        raw = sample_bitstrings(SQD_CIRCUITS[label][1], sqd_theta[label, d], noisy=noisy)
        shots, n_bad = recover_configurations(raw)
        dets = ranked_dets(shots)
        err = np.array([subspace_ground(H, dets[:k])[0] for k in range(1, len(dets) + 1)]) - e_fci
        sqd_curves[name, d] = err
        sqd_samples[name, d] = {"raw": raw, "shots": shots, "n_bad": n_bad, "dets": dets}
    # Azar: k determinantes distintos uniformes entre los 36 (= los k más frecuentes de bitstrings uniformes).
    rand = np.empty((N_RANDOM_TRIALS, len(K)))
    for t in range(N_RANDOM_TRIALS):
        perm = SECTOR[rng_sqd.permutation(len(SECTOR))]
        rand[t] = [subspace_ground(H, perm[:k])[0] for k in K]
    sqd_curves["azar", d] = rand - e_fci

FLOOR = 1e-3  # mHa: el error 0 (subespacio completo) se dibuja en el piso del eje log
COLORS = {"ansatz (19 exc.)": "#2a78d6", "ansatz (19 exc.) + ruido": "#eb6834", "recortado (9 exc.)": "#1baf7a"}
STYLES = {"ansatz (19 exc.)": "-o", "ansatz (19 exc.) + ruido": "--s", "recortado (9 exc.)": ":^"}
fig, axes = plt.subplots(1, len(SQD_D), figsize=(15, 4.3), sharey=True)
for ax, d in zip(axes, SQD_D):
    r = sqd_curves["azar", d] * 1e3
    ax.fill_between(K, np.maximum(np.percentile(r, 10, axis=0), FLOOR), np.maximum(np.percentile(r, 90, axis=0), FLOOR),
                    color="#9a9890", alpha=0.25, lw=0, label="azar (percentiles 10–90)")
    ax.plot(K, np.maximum(np.median(r, axis=0), FLOOR), color="#6b6a65", lw=2, label="azar (mediana)")
    for name, _ in SOURCES:
        e = sqd_curves[name, d] * 1e3
        ax.plot(np.arange(1, len(e) + 1), np.maximum(e, FLOOR), STYLES[name], color=COLORS[name], lw=2, ms=4, label=name, zorder=3 if "recortado" in name else 2)
    ax.axhline(CHEM_ACC * 1e3, color="#52514e", lw=1, ls="-.", label="precisión química")
    ax.set(yscale="log", xlabel="tope k (determinantes en el subespacio)", title=f"d = {d} Å", xlim=(0.5, 36.5))
    ax.grid(alpha=0.3, which="major")
axes[0].set_ylabel("E_SQD − E_FCI (mHa)")
axes[0].legend(fontsize=8, loc="lower left")
fig.suptitle(f"SQD con tope k: muestras cuánticas ({SQD_SHOTS} shots, SamplerV2) vs. determinantes al azar (N = 4, S_z = 0)")
fig.tight_layout()
plt.show()


def k_chem(err: np.ndarray) -> float:
    """Menor k con error < precisión química (inf si no se llega con los determinantes muestreados)."""
    ok = np.flatnonzero(err < CHEM_ACC)
    return ok[0] + 1 if len(ok) else np.inf


rows = []
for d in SQD_D:
    row = {"d": d}
    for name, _ in SOURCES:
        s = sqd_samples[name, d]
        row[f"k_quím {name}"] = k_chem(sqd_curves[name, d])
        row[f"dets {name}"] = len(s["dets"])
    kr = [k_chem(e) for e in sqd_curves["azar", d]]
    row["k_quím azar (mediana)"], row["k_quím azar (p90)"] = np.median(kr), np.percentile(kr, 90)
    row["fuera del sector (ruido)"] = f"{sqd_samples['ansatz (19 exc.) + ruido', d]['n_bad']}/{SQD_SHOTS}"
    # Un subespacio de k determinantes sueltos no es cerrado bajo S²: el autovector puede mezclar singlete y triplete.
    H, _ = sector_hamiltonian(d)
    smp = sqd_samples["ansatz (19 exc.)", d]
    row["⟨S²⟩ ansatz, k = 10"] = round(spin_square(subspace_ground(H, smp["dets"][:10])[1]), 4)
    row["⟨S²⟩ ansatz, α×β"] = round(spin_square(subspace_ground(H, product_subspace(smp["shots"]))[1]), 4)
    rows.append(row)
sqd_kchem = pd.DataFrame(rows).set_index("d")
sqd_kchem.T

In [ ]:
# Tabla SQD vs. VQE en la zona estirada: error, shots y cz.
# VQE: si #6 ya dejó la tabla `vqe`, se usa; si no (stand-in), se mide ⟨H⟩ con SamplerV2 agrupando los Pauli
# que conmutan qubit a qubit (QWC), con los mismos θ que usa SQD.
VQE_SHOTS_PER_GROUP = 100_000
_PARITY8 = np.array([bin(m).count("1") & 1 for m in range(2 ** N_QUBITS)])


def vqe_sampled_energy(qc_t, x, d, shots=VQE_SHOTS_PER_GROUP) -> tuple[float, float, int]:
    op, e_nuc = qubit_hamiltonian(d)
    groups = op.group_commuting(qubit_wise=True)
    pubs, terms = [], []
    for g in groups:
        px, pz = g.paulis.x, g.paulis.z  # (términos, qubits); qubit v = columna v
        basis = {v: ("Y" if (px[:, v] & pz[:, v]).any() else "X") for v in range(N_QUBITS) if px[:, v].any()}
        masks = ((px | pz) * (1 << np.arange(N_QUBITS))).sum(axis=1)
        pubs.append(measured(qc_t, x, basis))
        terms.append((masks, g.coeffs.real))
    res = SamplerV2(seed=SQD_SEED).run(pubs, shots=shots).result()
    energy, var = e_nuc, 0.0
    for r, (masks, coeffs) in zip(res, terms):
        b = np.array([int(s, 2) for s in r.data.meas.get_bitstrings()])
        f = coeffs @ (1 - 2 * _PARITY8[b[None, :] & masks[:, None]])  # estimador por shot del grupo
        energy += f.mean()
        var += f.var(ddof=1) / shots
    return energy, np.sqrt(var), shots * len(groups)


def first_full_product(shots: np.ndarray) -> int:
    """Shots hasta que el subespacio α×β cubre los 36 determinantes (ahí SQD sin tope = FCI exacto)."""
    for n in range(1, len(shots) + 1):
        if len(product_subspace(shots[:n])) == len(SECTOR):
            return n
    return -1


cz_of = {name: SQD_CIRCUITS[name.replace(" + ruido", "")][1].count_ops()["cz"] for name, _ in SOURCES}
cz_of["azar"] = 0
vqe_table = globals().get("vqe")
vs_rows = {}


def put(method, d, cz, shots, dim, err, sigma=None):
    r = vs_rows.setdefault(method, {"método": method, "cz": cz, "shots": [], "dim": []})
    r["shots"].append(shots)
    r["dim"].append(dim)
    r[f"error {d} Å (mHa)"] = f"{round(err * 1e3, 3) + 0.0:.3f}" + ("" if sigma is None else f" ± {sigma * 1e3:.2f}")


def _join(vals):  # "2000" si es igual en las tres distancias, "31 / 29 / 27" si no
    return str(vals[0]) if len(set(vals)) == 1 else " / ".join(map(str, vals))


for d in SQD_D:
    H, e_fci = sector_hamiltonian(d)
    if isinstance(vqe_table, pd.DataFrame) and np.isclose(vqe_table.d, d).any():
        v = vqe_table[np.isclose(vqe_table.d, d)].iloc[0]
        e, s, n, cz, src = v.energy, v.sigma, int(v.shots), int(v.cz), "VQE (#6)"
    else:
        e, s, n = vqe_sampled_energy(SQD_CIRCUITS["ansatz (19 exc.)"][1], sqd_theta["ansatz (19 exc.)", d], d)
        cz, src = cz_of["ansatz (19 exc.)"], "VQE (stand-in: QWC con SamplerV2)"
    put(src, d, cz, n, "–", e - e_fci, sigma=s)
    for name, _ in SOURCES:
        smp = sqd_samples[name, d]
        put(f"SQD {name}, tope k = 10", d, cz_of[name], SQD_SHOTS, 10, sqd_curves[name, d][9])
        put(f"SQD {name}, sin tope", d, cz_of[name], SQD_SHOTS, len(smp["dets"]), sqd_curves[name, d][-1])
    put("SQD azar, tope k = 10 (mediana de 300)", d, 0, "–", 10, np.median(sqd_curves["azar", d][:, 9]))
    # α×β: con el primer puñado de shots ya aparecen las 6 cadenas y el subespacio es todo el sector (= FCI).
    for name in ("ansatz (19 exc.)", "recortado (9 exc.)", "azar"):
        shots = sqd_samples[name, d]["shots"] if name != "azar" else SECTOR[rng_sqd.integers(len(SECTOR), size=SQD_SHOTS)]
        n = first_full_product(shots)
        put(f"SQD {name}, α×β (shots hasta cubrir el sector)", d, cz_of[name], n, len(SECTOR),
            subspace_ground(H, product_subspace(shots[:n]))[0] - e_fci)
for r in vs_rows.values():
    r["shots"], r["dim"] = _join(r["shots"]), _join(r["dim"])
sqd_vs_vqe = pd.DataFrame(vs_rows.values()).set_index("método")
sqd_vs_vqe

#### Resultados y decisiones (#16)

**Error vs. tope $k$ (gráfico).** Con el subespacio limitado, las muestras cuánticas le ganan claramente al azar:

| | 1,8 Å | 2,4 Å | 3,0 Å |
|---|---|---|---|
| $k$ mínimo para precisión química, ansatz (19 exc.) | 16 | 12 | 6 |
| ídem, ansatz + ruido | 24 | 30 | 6 |
| ídem, recortado (9 exc.) | no llega | no llega | 7 |
| ídem, azar (mediana / percentil 90) | 35 / 36 | 34 / 36 | 28 / 30 |
| error con $k = 10$, ansatz (mHa) | 5,1 | 2,3 | 0,39 |
| error con $k = 10$, azar, mediana (mHa) | 318 | 318 | 332 |

- **Por qué gana lo cuántico:** el fundamental FCI tiene solo ~16 determinantes con peso apreciable (los otros 20 son cero por la simetría espacial de la cadena). El ansatz los muestrea con frecuencia proporcional a $|c_i|^2$, así que los $k$ más frecuentes son casi los $k$ más importantes. El azar gasta más de la mitad del tope en determinantes de peso cero y necesita casi todo el sector para llegar a precisión química.
- **El ruido hace las dos cosas.** Con ~27 % de shots fuera del sector, la recuperación de configuraciones mete determinantes poco relevantes entre los más frecuentes: con tope, empeora (a 2,4 Å necesita $k = 30$ en lugar de 12). Sin tope, esos mismos determinantes "basura" terminan cubriendo los 36 y el resultado es exacto: el ruido *ayuda* a SQD sin tope, que es justamente la trampa del enunciado.
- **El recortado** muestrea solo los determinantes que puede alcanzar y a 1,8 y 2,4 Å se estanca por encima de precisión química. Igual mejora mucho sobre su propio VQE: con statevector, ese circuito queda a 19,9 / 8,9 / 1,0 mHa de FCI; SQD con sus muestras baja a 4,9 / 2,3 / 0,25 mHa.
- **Espín:** un subespacio de $k$ determinantes sueltos no es cerrado ante $S^2$, y el autovector mezcla singlete con triplete: con $k = 10$, $\langle S^2\rangle$ = 0,13 / 0,44 / 0,60. El subespacio α × β (simétrico) da $\langle S^2\rangle = 0$. En la zona estirada el triplete está casi degenerado, así que la energía sola no lo delata (igual que en la sección 3).

**SQD vs. VQE (tabla).** Sin tope, SQD con el ansatz llega a 0,60 / 0,13 / 0,09 mHa de FCI con 2000 shots (16–18 determinantes distintos), y con el subespacio α × β es exacto con **9–31 shots** del ansatz o **6–8 bitstrings al azar** (lo cuántico tarda igual o más, porque concentra las muestras en pocos determinantes). El VQE stand-in, con 4,6 millones de shots, queda en 1,0 ± 1,2 / 0,3 ± 1,1 / 0,04 ± 1,1 mHa: el costo de VQE es la medición, el de SQD es la diagonalización clásica, que acá es trivial (36 × 36) y en sistemas grandes es el cuello de botella. Los dos usan el mismo circuito (465 `cz`); sin tope, SQD también llega a FCI con el recortado (191 `cz`). Por eso este resultado **no** dice que SQD sea mejor para $\mathrm{H_4}$: dice que $\mathrm{H_4}$ es demasiado chico para distinguir un buen muestreo de uno malo salvo que se limite el subespacio.

Qué no funcionó o descartamos:

- **`qiskit-addon-sqd`**: no está en el nixpkgs fijado (ver arriba).
- **Medir después de transpilar con `measure_all()`**: el transpilador permuta los qubits al final (`final_layout`), así que los bits medidos no correspondían a la convención 0.1 y el determinante más frecuente "no era" HF (errores de ~800 mHa con $k = 1$). Se mide cada qubit lógico en su qubit físico con `layout.final_index_layout()`.
- **Multi-start eligiendo solo por energía**: en 2,4 y 3,0 Å el mínimo elegido cambiaba según qué celdas se hubieran corrido antes (el Hamiltoniano difería en ~1e-13 Ha por la convergencia de RHF, y eso alcanza para que BFGS caiga en otro mínimo de un paisaje casi plano). En 3,0 Å el mejor arranque tenía $\langle S^2\rangle = 0{,}08$ y, en una corrida, el recortado convergió a un estado casi triplete ($\langle S^2\rangle = 1{,}98$). Con la penalización $\lambda\langle S^2\rangle$ y 20 arranques el resultado es el mismo en las dos situaciones y todos los estados son singlete.
- **Tope sobre el subespacio α × β** (quedarse con las $m$ cadenas más frecuentes, dimensión $m^2$): lo descartamos de entrada, porque con solo 6 cadenas la curva tendría 6 puntos (dimensión 1, 4, 9, 16, 25, 36) y a $m = 6$ ya es FCI.

Limitaciones:

- Los $\theta$ del ansatz son los `params` de #5 (sección 4). El VQE de la tabla es un stand-in (QWC, sin optimizar la cantidad de shots) hasta que llegue `vqe` de #6. El código toma los reales apenas existan.
- El ruido es el modelo de `05-extras` armado a mano, sin errores de lectura ni conectividad de `FakeFez`.
- Los números dependen de la semilla (`SQD_SEED`); el $k$ del azar se reporta con mediana y percentil 90 de 300 sorteos.

### Otra geometría (#17)

Mismo pipeline, cambiando solo la geometría: **cuadrado de lado $d$** (plano xy) frente a la cadena lineal, en una grilla reducida (11 distancias). El cuadrado es el caso multirreferencia por excelencia: con simetría $D_{4h}$ los dos orbitales del medio son degenerados, así que con 4 electrones el determinante de HF está mal definido en todas las distancias, no solo en la zona estirada.

- `h4_geometry_shape(d, shape)` agrega la forma **sin tocar el contrato 0.1**: con `shape="linear"` devuelve exactamente `h4_geometry(d)`, y para la cadena lineal se reutilizan `h4_scf`/`qubit_hamiltonian` de la sección 2 tal cual.
- Referencias HF/CCSD/FCI con PySCF, Hamiltoniano con la maquinaria de la sección 2, parámetros con el simulador del sector de la sección 3 (statevector: permitido para buscar θ) y **energías reportadas medidas con `SamplerV2`** de `qiskit_aer`, con σ desde los bitstrings.
- Decisiones, resultados y limitaciones al final de la subsección.
- **Resultados guardados.** La búsqueda de θ (2/4, ~1 min) y la medición con `SamplerV2` (3/4, ~1,5 min) usan por defecto las tablas guardadas en `GEO_CACHE` (de una corrida de esas mismas celdas con `RUN_GEO = True`): `geo_opt` (con los θ a precisión completa), `geo_vqe` y las 20 repeticiones de la validación de σ. Las referencias (1/4), los controles del Hamiltoniano, la tabla de R_e/D_e y los gráficos (4/4) se recalculan igual. Para recalcular: `RUN_GEO = True` en 2/4.

In [ ]:
# Extra #17 (1/4): geometría con forma y referencias clásicas (HF, CCSD, FCI) con PySCF.
# No toca el contrato: `h4_geometry_shape(d, "linear")` devuelve exactamente `h4_geometry(d)`, y para la
# cadena lineal se reutilizan `h4_scf` / `qubit_hamiltonian` de la sección 2 tal cual.
from pyscf import cc, lib as pyscf_lib

GEO_SHAPES = ("linear", "square", "rectangle")
GEO_D = np.array([0.8, 0.9, 1.0, 1.2, 1.3, 1.4, 1.6, 1.8, 2.0, 2.4, 3.0])  # Å, grilla reducida común


def h4_geometry_shape(d: float, shape: str = "linear", aspect: float = 1.5) -> str:
    """H4 con forma: cadena lineal (contrato 0.1), cuadrado de lado d o rectángulo d × aspect·d (plano xy)."""
    if shape == "linear":
        return h4_geometry(d)
    if shape == "square":
        aspect = 1.0
    elif shape != "rectangle":
        raise ValueError(f"forma desconocida: {shape!r} (opciones: {GEO_SHAPES})")
    pts = [(0, 0), (d, 0), (d, aspect * d), (0, aspect * d)]
    return "; ".join(f"H {x:.6f} {y:.6f} 0.000000" for x, y in pts)


def _fix_orbital_signs(mf: scf.hf.RHF, d: float) -> scf.hf.RHF:
    """Mismo criterio que la sección 2 (ADR-0003): primer coeficiente con |c| > 1e-4 positivo."""
    c = mf.mo_coeff
    significant = np.abs(c) > 1e-4
    if not significant.any(axis=0).all():
        raise RuntimeError(f"Orbital sin coeficientes > 1e-4 en d = {d}: no se puede fijar el signo")
    first = np.argmax(significant, axis=0)
    mf.mo_coeff = c * np.sign(c[first, np.arange(c.shape[1])])
    return mf


@lru_cache(maxsize=None)
def _scf_shape_cached(d: float, shape: str, aspect: float) -> scf.hf.RHF:
    if shape == "linear":
        return _h4_scf_cached(d)  # el RHF de la sección 2, sin cambios
    mol = gto.M(atom=h4_geometry_shape(d, shape, aspect), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    # En el cuadrado el RHF cae en soluciones distintas según d (no todas mínimas): seguimos la
    # inestabilidad interna (RHF → RHF) hasta un mínimo, así todas las distancias usan la misma clase de solución.
    for _ in range(10):
        mo, _, stable, _ = mf.stability(return_status=True)
        if stable:
            break
        mf.kernel(mf.make_rdm1(mo, mf.mo_occ))
    if not (mf.converged and stable):
        raise RuntimeError(f"RHF no llegó a un mínimo estable en d = {d}, forma {shape}")
    return _fix_orbital_signs(mf, d)


def scf_shape(d: float, shape: str = "linear", aspect: float = 1.5) -> scf.hf.RHF:
    mf = copy.copy(_scf_shape_cached(_distance_key(d), shape, float(aspect)))
    for attr in ("mo_coeff", "mo_energy", "mo_occ"):
        setattr(mf, attr, getattr(mf, attr).copy())
    return mf


def references_shape(d: float, shape: str = "linear", aspect: float = 1.5) -> dict:
    """Como `references` (#1) pero para cualquier forma; agrega diagnósticos de CCSD y del espín."""
    mf = scf_shape(d, shape, aspect)
    mycc = cc.CCSD(mf)
    mycc.max_cycle = 100
    mycc.kernel()
    solver = fci.FCI(mf)
    solver.nroots = 2
    e, ci = solver.kernel()
    s2 = [spin_op.spin_square0(v, N_ORB, (N_ALPHA, N_BETA))[0] for v in ci]
    return {"shape": shape, "d": float(d), "e_hf": mf.e_tot, "e_ccsd": mycc.e_tot, "ccsd_conv": mycc.converged,
            "e_fci": e[0], "S2_fci": s2[0], "gap_mHa": (e[1] - e[0]) * 1e3, "S2_exc": s2[1],
            "t2_max": np.abs(mycc.t2).max(), "err_ccsd_mHa": (mycc.e_tot - e[0]) * 1e3}


# Con matrices de 4×4 los hilos de OpenMP de PySCF solo agregan overhead (~10× más lento): un hilo acá.
with pyscf_lib.with_omp_threads(1):
    geo_refs = pd.DataFrame([references_shape(d, s) for s in ("linear", "square") for d in GEO_D])
    geo_rect = pd.DataFrame([references_shape(1.2, "rectangle", a) | {"aspect": a}
                             for a in (1.0, 1.05, 1.1, 1.2, 1.4, 1.7, 2.0)])
print("Error de CCSD vs. FCI (mHa); negativo = CCSD por debajo de FCI (no variacional):")
print(geo_refs.pivot(index="d", columns="shape", values="err_ccsd_mHa").round(2).to_string())
print("\nCCSD sin converger:", geo_refs.loc[~geo_refs.ccsd_conv, ["shape", "d"]].values.tolist())
print("Cuadrado: estado fundamental de FCI (S_z = 0) con ⟨S²⟩ ≈ 2 (triplete) en d =",
      geo_refs.query("shape == 'square' and S2_fci > 1").d.tolist())

# Rectángulo d × a·d a d = 1,2 Å: de cuadrado (a = 1) a dos H2 separados. Muestra que el problema de CCSD
# viene de la casi degeneración del cuadrado, no del plano.
print("\nRectángulo a d = 1,2 Å:")
print(geo_rect[["aspect", "e_fci", "gap_mHa", "t2_max", "err_ccsd_mHa"]].round(4).to_string(index=False))
geo_refs.round(6)

In [ ]:
# Resultados guardados del extra #17 (celdas 2/4 y 3/4). Recalcularlos tarda ~2,5 min, así que esas celdas los
# usan por defecto (RUN_GEO = False). Salieron de esas mismas celdas con True (θ a precisión completa);
# para regenerarlos: poner True, correr y pegar acá results/geo_opt.csv, results/geo_vqe.csv y
# results/geo_sigma_rep.csv.
GEO_CACHE = {
    "origen": "corrida completa del 2026-11-21 con RUN_* = True, SHOTS_SCALE = 1, qiskit 2.5.2, qiskit-aer 0.17.2, pyscf 2.14.0",
    "opt": """\
shape,ansatz,d,e_sv,e_fci,err_sv_mHa,ok_starts,S2,theta
linear,podado-19,0.8,-2.167543336,-2.167560544,0.0172079,6/10,2.98224e-07,0.9238988373739473 -1.8622281663583808 -2.328571425553911 1.640038885226882 -1.8628430247865229 0.42511683924827215 -1.1561992252424858 -0.017463020080318335 -1.498286857667655 2.278489674098045 0.6115142624377782 0.008432696614462139 -0.7077031026917598 1.600887401501008 0.3676782586542515 0.1073553561159908 -0.6456385049757817 1.6038976929198392 -0.7893102625232711
linear,podado-19,0.9,-2.180311684,-2.180316614,0.00493016,8/10,3.16451e-07,1.4499545860914507 0.4247800117677813 0.04318014249523813 -1.8190536928077092 1.5008828516712858 0.09632354507175991 -0.7544264727579018 -1.5498820077970612 -0.8065734369893158 0.9428344393455329 -0.09528099478795855 -1.522218015976321 3.14375540187197 1.5420289315402116 0.019507672610665095 -0.7280729081085867 0.9556906845683152 1.562347782982433 0.8533090811869936
linear,podado-19,1,-2.166387193,-2.166387449,0.00025551,9/10,2.07166e-08,1.4136989462727312 -0.1592792895407365 -0.0939001135646424 -0.016900695000192468 0.307432259314249 0.9759462737925965 0.8088469047581773 1.5499708842228708 0.8182941313495813 -0.6719537547865876 0.04739688625702525 1.3939015393055048 -0.24539489197615771 -0.08347167812200461 -0.007743728676532322 1.4047608109281648 0.23162523916419667 0.0076541820440850445 -0.006715207170263433
linear,podado-19,1.2,-2.102604414,-2.102608481,0.00406692,7/10,6.04022e-07,1.3686444770614892 -0.16970572643126322 -0.11972458570624443 -0.03604811508166341 0.36104175657555104 0.8634740423571052 0.8193995855295177 1.5330097739633906 0.8381492698779794 -0.7784217041532474 0.08022174584183081 1.3119578133240015 -0.2098788366017971 -0.05906055534876289 -0.032251577926311774 1.352102559019969 0.248403241283955 0.007589931498136802 -0.002133734009834815
linear,podado-19,1.3,-2.065219353,-2.065228963,0.00961052,6/10,2.40865e-06,1.3419817543280452 -0.15826236638361957 -0.13302729752596024 -0.05021207628202623 0.3898719345110635 0.8196504753732267 0.8256359482952734 1.5219011125585786 0.8461484409039578 -0.8318377744957769 0.10304117117636069 1.2640936340234221 -0.19874684610954116 -0.04007052369763194 -0.04232965790414288 1.3288807390497201 0.2570059799183311 0.008241851998175379 0.001881649779164586
linear,podado-19,1.4,-2.029069254,-2.029070494,0.00123919,6/10,1.35687e-07,0.15777950394187207 -0.29573485489643736 -0.052420764533073566 2.0884928401402796 -0.35628277603764485 0.3812555215418779 -1.463348016145252 -0.027072838679646723 1.4112586924514674 -0.5617491806841423 -1.8304648261064738 1.6711327877253825 -0.1883325727003333 -0.3058779064519197 -0.2802853531451858 -0.030227871427825143 0.7497705647899532 1.4730329330342267 -2.356097415289418
linear,podado-19,1.6,-1.967559511,-1.96756031,0.000798663,8/10,1.67344e-07,0.43935550680510216 0.03193991215035029 -0.278817078168492 -0.3207742322197303 -0.37817618415789284 0.6390290842883788 -1.496694047505593 0.0020143330200462326 1.333554606352974 -0.922925271325485 -2.5034365364554496 2.9625869697075675 0.08028274831084614 0.051826758947399285 -0.5488460761338796 -0.09509179214264503 0.7352722368210214 1.5433188822096213 -2.36119297193856
linear,podado-19,1.8,-1.924423292,-1.924430638,0.00734589,3/10,1.68398e-08,0.507328717680939 0.05893685462370909 -0.43212793564961377 -0.31792558399323495 -0.3919482004798598 0.6225301932820404 -1.466196004510141 0.014855266952290707 1.3152771615318335 -0.9914063033476676 -2.436609667726712 2.956296807871262 0.023657660549161248 0.013249950537452397 -0.4746687806300907 -0.1338812510256668 0.7204269753145424 1.52797174355945 -2.349516355598697
linear,podado-19,2,-1.897779543,-1.897780646,0.00110271,7/10,8.53314e-06,-0.5287826504985198 4.785223210066252 -0.47322001407612485 3.8900229552293224 -1.2574913008345558 -0.26639484049512724 1.447998008008705 1.058565953841673 -0.060229069670836806 -0.8333688418824009 -2.106103259938056 0.8607710126926768 -0.2288098952353901 -1.4315321318668912 1.4828076256071911 0.7519947505291971 1.2457415644367578 1.5257179808425083 0.8842431152517168
linear,podado-19,2.4,-1.87465146,-1.874651582,0.00012237,9/10,4.55717e-05,-0.9027842321488181 0.8476825173258399 -0.37238131314915934 -1.0801039215053254 0.48487241979657947 -0.2537783854214768 0.7985686948122812 -0.13659039394590955 0.3284381517811553 1.033976532948074 -1.4216872049365108 -2.266601217708072 1.1457464497520062 0.7579183522820158 -0.40078581958642906 -0.11777819908461565 -0.242826071172047 0.025510874928746364 0.564197226514498
linear,podado-19,3,-1.867290443,-1.867291372,0.000928925,9/10,0.00335337,-1.0417571879546368 0.6918009112607951 -0.3355183860905227 -1.0199770509967034 0.48629493731146717 -0.3320949267483409 0.7721920189154227 -0.20734934437115943 0.3123339330561819 1.309830528484514 -1.59687417107855 -2.472659794460987 1.1295538199451174 0.60487350015856 -0.3306710081104258 -0.11762666607529085 -0.24731917107220186 0.07476923798300221 0.5665279490126409
square,podado-19,0.9,-1.846639,-1.846740573,0.101573,4/10,0.000416016,2.646924633048599 1.2626571083048763 1.8220171197588264 1.2227336263438282 1.168061996684162 -1.4050346559028764 1.0047126699133453 -0.3735147205453577 0.5522058579308743 -0.18324859052961404 0.11401465467847191 2.3089323372238986 0.19444900473137444 -1.856471726691294 -0.017155864339774816 -0.13297820928849158 0.35536259397065845 0.4613697598207544 -0.7568089109219949
square,podado-19,1,-1.915096837,-1.91510655,0.00971285,7/10,6.70873e-06,-0.13822780460457953 1.3990319054426184 3.2851603499817963 1.7188876394443149 -0.28522537264546743 -2.498771955508245 0.6526330824173993 -1.2392421365245008 -0.9027923499526409 0.018898986924293343 -0.0233857605119791 -0.9277435805187864 0.7318282367082899 0.13270383057682075 -0.3264153905165155 -0.013088190888005991 -0.2622289322730318 -0.7327279398547372 0.014333372126654862
square,podado-19,1.2,-1.967549879,-1.96754988,1.57337e-06,6/10,3.22577e-13,1.175802325907563 0.3393714276264999 0.04815794078876088 0.04377112470790331 0.6193774042275424 0.9913907637425702 0.7667622719131703 1.605685505930712 0.7845466322672916 0.4951747275814046 0.055067957461218855 0.8033612703880316 -0.3032212853361163 -1.071975253276873 0.026663201157398477 1.337354681178116 0.24797356724415287 0.006852860368838247 -0.036809538213152616
square,podado-19,1.3,-1.970376004,-1.970377538,0.00153405,6/10,3.96393e-07,1.1691466190424178 0.32648380206377536 0.04876401941909304 0.043955118378381446 0.646815955341443 0.8696754912916231 0.757184884412725 1.6176196327308638 0.7872253536867163 0.669996699184305 0.06451917486664108 0.8083229748411875 -0.2919024947438589 -1.0692872969090776 0.02257179584453129 1.292431914472726 0.2348054132964794 0.007633310366968693 -0.042947796011968874
square,podado-19,1.4,-1.965080561,-1.965080561,-2.22045e-13,6/10,1.24575e-16,1.1220074658156807 0.21670013320652246 -1.2726416544649283e-08 -7.082301614892004e-09 0.694010164956942 0.510528697527273 0.7853981290930473 1.5707963440818677 0.7853981520045804 1.266855549649556 3.567932000450159e-09 0.7853981684957422 -0.27538048895368444 -1.2531531071511484 -1.6596904510900266e-09 1.1959689442366548 0.17660609839525188 -1.0195918284263769e-08 -1.0322877185893103e-08
square,podado-19,1.6,-1.942917329,-1.94293896,0.0216308,6/10,1.94203e-06,0.9242150286200994 -0.3844256999419891 0.045825211844320125 0.026952495255211626 1.0753450839755416 0.4473129082638711 0.7829977447897742 1.565723366131543 0.7868966519157687 1.3170679255932685 0.01875770203560544 0.7857590411494824 0.3291936846576578 -2.335771062521477 -0.0023216679093149228 1.154343835796228 0.18237943021947187 -0.0016382018119266407 0.0012362451714156854
square,podado-19,1.8,-1.918048654,-1.91805379,0.00513604,5/10,2.86475e-05,0.14934710942222776 1.0384154519624489 0.30956045557243833 -2.8584172348910775 -0.9479199890750308 -0.3556552118609958 -1.3038969762460726 -1.2411270025831647 2.5142915761312707 1.6252909551936578 -0.17454903539369296 0.7023501885458594 -3.018597968784587 0.9584522173436731 -1.3230679791941757 -0.7243047580916022 0.28127540410291924 -0.15474982727630515 -1.8693861075140452
square,podado-19,2,-1.897849018,-1.897849389,0.00037052,5/10,6.75704e-06,0.17450466484512678 1.1849394941249178 0.054723164678007905 -2.674090240904908 -0.9125344422155893 -0.4202063812620477 -1.2108454422193236 -1.1847290635435268 2.4638222277781683 1.6230912330106158 -0.23091579310661448 0.9324528045700784 -2.944853233724206 1.1001773494420828 -1.3260005013089298 -0.6788343183304482 0.34916284471796105 -0.18700808179914089 -1.9126362746267547
square,podado-19,2.4,-1.875841079,-1.875841079,4.15223e-11,6/10,1.76897e-16,0.8721959536607011 -0.08027753849411537 2.616501849725528 1.0865958101715876 -0.5050715170579939 -0.20720772376946087 0.7853978619246919 -1.5707961940991781 0.7853979840043754 -2.2988971497913293 0.957560542028013 1.1885281457647678 -0.9133498100628007 -0.18327999988988067 -5.799578525108992e-08 0.14676974435719103 2.250458471620593 -1.570796250824897 -0.7540979030141397
square,podado-19,3,-1.867495184,-1.867495184,9.09939e-10,9/10,6.30883e-15,0.8559397751169949 0.05411116561112147 2.661186667527362 1.1655666318657167 -0.41568691033733546 -0.24409070919070736 0.7853955964156465 -1.5707958217337425 0.7853967247261876 -2.410813261571229 1.067989876706093 1.1548034326736376 -0.9391045340424802 -0.1056230944079218 6.255160924419297e-07 0.24954010395930604 2.2457930440856573 -1.5707951097751973 -0.7608877417264435
square,completo-24,0.9,-1.846740573,-1.846740573,1.11022e-12,7/10,2.3537e-20,0.22358995417666644 0.40564947092154646 -2.5390921336965584 -0.7351841877720867 0.39789876188842244 -0.40086354909471783 -0.5712327861033892 0.3225082552672419 0.3765172526478171 -1.6079024535405755 -0.122846999993495 -1.4849173906850173 -0.37281312816000084 0.7679454568499386 0.2800689746841653 -0.8374005613118607 1.8423242599049592 -0.8861132249860113 -0.3399857895495108 -0.9275924080775868 1.4378285340316808 -0.12381395484695816 -0.11971410158080402 0.8194719388063036
square,completo-24,1,-1.91510655,-1.91510655,-3.55271e-12,7/10,1.76098e-17,-0.44818368377326234 0.6485782767019742 -0.5297184857182243 -0.2795692562585632 1.0935553639581133 0.4438917537662126 -0.6266490980171572 0.5608005926961581 0.2097995417248113 0.23075397252871052 -0.2246013590950999 -0.058554808526350303 0.3431771624320068 1.1925919007659316 0.1273111567063582 -3.2115611967195825 -0.3074144796276 0.3551183941763511 -0.8215614464116419 -0.9180304821405298 -0.09942836967365667 0.532468371435977 -1.361243991281056 -0.6618270492826666
square,completo-24,1.2,-1.96754988,-1.96754988,-6.66134e-13,9/10,6.45957e-16,-0.6242126282995896 1.0745938933499493 -0.6095216090740887 0.24156247262315314 0.28293578805516484 -2.404363017947857 -1.1179249827346747 0.17721234432158114 1.360479942334839 -0.08470427193007275 -1.8485477224325735 -0.4413855546528483 0.7845099117704737 -0.6193925779265228 -1.477033730213452 -0.8300309690167158 0.27784953732927514 2.3511346327993823 1.2561902249895662 1.168218698485775 -0.4113547179006658 -1.3005479045053663 -2.2179156450035573 -0.31720763110941014
square,completo-24,1.3,-1.970377538,-1.970377538,-8.88178e-13,7/10,2.06075e-16,-0.1372587807985583 0.13278844084495608 0.08302667382820954 -1.2665945846338296 0.6479648471867353 -0.9715057533347532 -0.3297336168211678 -0.00997605274777947 -0.2751279396614539 0.824351522147844 -1.010247101679925 -0.31761294806774193 -0.2735825538919997 -0.076059861618019 0.26142546870221456 -0.7689474604758222 1.9357582401482725 -1.1152108017767086 0.16139666340699452 0.3089185448387137 -0.023779630360176815 0.1014973039143259 -1.2634377470017322 -1.3378344752384244
square,completo-24,1.4,-1.965080561,-1.965080561,-1.77636e-12,9/10,5.47674e-18,-0.18982985990603912 -0.8012870299042976 0.3830453562138071 0.437204728475752 0.39827335308506057 0.9866426354170109 0.7999649047071965 0.6403590266689457 1.073453916917922 -1.2777590249278785 -1.097101788402771 -0.13503581428111963 -1.6922217468304126 0.1559527687367072 -1.3478336546688021 -1.3139928684203404 -2.045737369784648 -1.3343121718728366 -1.267982741975587 0.5607905869489249 -0.054583139645041086 0.1862430577515418 0.34800899450244416 0.9562515170395739
square,completo-24,1.6,-1.94293896,-1.94293896,-8.88178e-13,8/10,5.94612e-17,-0.3807986912824516 -0.809199385733652 -0.127772841363916 -0.005090982931241125 0.43972136033485754 -0.17363692830101543 -0.906267658691238 -0.17690986487570848 0.9612837251488433 0.8420941427940223 1.010868456518391 2.8821778750650666 -1.1079072689083513 0.4377942576925666 -0.9910877328297253 0.6014675646553468 -0.9624899800434392 1.7246605666557027 -1.3857510159934445 -0.6373827722671737 0.4193061908348897 -1.3294803984749277 -0.8553952631542788 -0.20325771120066827
square,completo-24,1.8,-1.91805379,-1.91805379,-1.33227e-12,8/10,1.85228e-18,-0.07193255967929404 0.6358390344049376 0.48093076898395 0.36405184027251314 -0.15502529721204802 -0.6339403063108887 -0.31527792638584184 0.7071725957720085 -0.21477180829487377 0.9799232204984636 0.43702110065187194 0.1327343082439694 0.7289726788935521 -0.04091764884059425 1.776676456712673 -1.1113462478944187 -1.8414988610461134 -1.2347084587961261 -0.441083965508836 0.6261558651794628 1.160192611246561 -0.17067685672351243 0.0006843353109152024 -0.7568949019544692
square,completo-24,2,-1.897849389,-1.897849389,-6.66134e-13,8/10,3.78938e-17,0.08638168769479693 1.4884691930396172 0.4433993908901836 -0.7850730655466338 0.358275044730366 -0.7711328771338002 -0.41816077577354915 -0.732740689724558 -0.4938060306678823 -0.21098068445485924 -0.2770946850994775 -1.0753294132007907 -2.3020515759508497 0.5564104460467069 -0.2948297182103651 0.8599674579737587 0.5828151729716851 -0.12593776150449795 1.9001157223325125 -0.24483500974308814 0.3935366702992389 -0.08511404744539677 1.0527587670677063 -1.4059670057766953
square,completo-24,2.4,-1.875841079,-1.875841079,-1.33227e-12,9/10,5.40486e-16,0.2376566301057911 -0.09579616585139233 1.6719937718171087 -1.5604546618390167 2.129251490638392 0.7761581079150304 -0.8151527001259868 1.28459798164526 -0.764513702838087 1.5675187908506043 0.19238086354458825 -0.47234949986924746 0.8962624142382056 -0.8519913527335589 -1.5777580320667555 -0.32250227457492386 1.0521312266493335 -1.0447610473881412 -0.9633916826470462 -1.5611298089531414 -0.16060555519869305 0.6218672460749247 0.006893609145987935 0.00027663070379473434
square,completo-24,3,-1.867495184,-1.867495184,-2.22045e-12,10/10,3.01973e-14,0.3157230714184117 0.21107325558572268 0.5268761735166946 -0.8280058684893546 1.1994068648314828 0.7355155604416403 0.19805229280005032 0.5405273128150858 -0.7824227553561185 -0.33227546385556617 0.6227886915226416 -0.6365579612271111 2.082321985151752 -0.04743869025372192 0.7598251150162892 0.3549318538925311 0.47836805132190474 0.08760405242295288 -0.28974016944737135 0.7988743164547092 -0.17280947471282823 -1.0089208180510307 0.6067255345578699 -0.4106204293205772
""",
    "vqe": """\
shape,d,energy,sigma,shots,groups,cz,e_sv,e_fci
linear,0.8,-2.166438101,0.00169115,3000002,46,465,-2.167543336,-2.167560544
linear,0.9,-2.1787575,0.0016235,3000002,46,465,-2.180311684,-2.180316614
linear,1,-2.166988712,0.00157286,2999996,46,465,-2.166387193,-2.166387449
linear,1.2,-2.102325569,0.0014983,2999996,46,465,-2.102604414,-2.102608481
linear,1.3,-2.067530397,0.00146651,2999997,46,465,-2.065219353,-2.065228963
linear,1.4,-2.029696842,0.00144062,2999998,46,465,-2.029069254,-2.029070494
linear,1.6,-1.968099708,0.00138758,3000003,46,465,-1.967559511,-1.96756031
linear,1.8,-1.921296625,0.00134038,3000001,46,465,-1.924423292,-1.924430638
linear,2,-1.894457073,0.00129212,2999997,46,465,-1.897779543,-1.897780646
linear,2.4,-1.874537261,0.00123101,3000000,46,465,-1.87465146,-1.874651582
linear,3,-1.867873011,0.00119943,2999998,46,465,-1.867290443,-1.867291372
square,0.9,-1.845989113,0.00122806,2999999,33,465,-1.846639,-1.846740573
square,1,-1.914865795,0.00127815,3000002,35,465,-1.915096837,-1.91510655
square,1.2,-1.967729806,0.00119314,2999997,33,465,-1.967549879,-1.96754988
square,1.3,-1.971983523,0.00125694,2999999,35,465,-1.970376004,-1.970377538
square,1.4,-1.965325946,0.00125548,2999998,35,465,-1.965080561,-1.965080561
square,1.6,-1.945177073,0.00124882,3000000,35,465,-1.942917329,-1.94293896
square,1.8,-1.916315702,0.00123825,2999997,35,465,-1.918048654,-1.91805379
square,2,-1.897182863,0.00121476,2999998,35,465,-1.897849018,-1.897849389
square,2.4,-1.877158875,0.00116693,3000003,35,465,-1.875841079,-1.875841079
square,3,-1.866937442,0.00112805,3000000,35,465,-1.867495184,-1.867495184
""",
    "rep": """\
energy,sigma
-1.91880614,0.00480489
-1.908482199,0.00480347
-1.918801689,0.00479561
-1.920633983,0.00478635
-1.922300224,0.00477703
-1.920200471,0.00478336
-1.918397219,0.00479934
-1.919419465,0.00477582
-1.920034278,0.00478819
-1.916714423,0.00479013
-1.92584073,0.00479767
-1.917019907,0.00479486
-1.917665851,0.00479129
-1.908929566,0.00480605
-1.923463817,0.00478417
-1.916124,0.00476996
-1.926276906,0.00478211
-1.912123739,0.0048043
-1.918033572,0.00477921
-1.921321271,0.00478375
""",
}

In [ ]:
# Extra #17 (2/4): Hamiltoniano de qubits y búsqueda de parámetros (statevector, permitido) para el cuadrado.
# Misma maquinaria de la sección 2 (integrales en base MO → `spatial_to_spin_orbitals` → `jordan_wigner`)
# y el mismo simulador en el sector de 36 estados de la sección 3 (`SectorAnsatz`).
import io

@lru_cache(maxsize=None)
def _qubit_hamiltonian_shape_cached(d: float, shape: str, aspect: float) -> tuple[SparsePauliOp, float]:
    if shape == "linear":
        return _qubit_hamiltonian_cached(d)  # el Hamiltoniano de la sección 2, sin cambios
    mf = _scf_shape_cached(d, shape, aspect)
    c = mf.mo_coeff
    h1 = c.T @ mf.get_hcore() @ c
    eri = ao2mo.restore(1, ao2mo.full(mf.mol, c), c.shape[1])
    return jordan_wigner(*spatial_to_spin_orbitals(h1, eri)), mf.mol.energy_nuc()


def qubit_hamiltonian_shape(d: float, shape: str = "linear", aspect: float = 1.5) -> tuple[SparsePauliOp, float]:
    """Como `qubit_hamiltonian(d)` (#2) para cualquier forma: (H electrónico de 8 qubits, e_nuc)."""
    op, e_nuc = _qubit_hamiltonian_shape_cached(_distance_key(d), shape, float(aspect))
    return op.copy(), e_nuc


def sector_hamiltonian_shape(d: float, shape: str) -> tuple[np.ndarray, float]:
    op, e_nuc = qubit_hamiltonian_shape(d, shape)
    M = op.to_matrix(sparse=True)[SECTOR][:, SECTOR].toarray()
    H = M.real + e_nuc * np.eye(len(SECTOR))
    return H, np.linalg.eigvalsh(H)[0]


# Control: el Hamiltoniano de 8 qubits reproduce HF y FCI de PySCF también en el cuadrado.
for _d in (1.3, 3.0):
    _op, _en = qubit_hamiltonian_shape(_d, "square")
    _r = geo_refs.query("shape == 'square' and d == @_d").iloc[0]
    assert abs(hf_energy_from_op(_op) + _en - _r.e_hf) < 1e-8 and abs(sector_ground_energy(_op) + _en - _r.e_fci) < 1e-8
assert np.allclose(qubit_hamiltonian_shape(1.8)[0].coeffs, qubit_hamiltonian(1.8)[0].coeffs)  # lineal = sección 2

# El ansatz elegido (podado sobre la cadena lineal) y el 2-QUpCCGSD completo (12 excitaciones × 2 capas).
GEO_FULL_OPS = [(l, k, p, q) for l in (0, 1) for k in ("D", "S") for p, q in itertools.combinations(range(N_ORB), 2)]
geo_ansatz = {"podado-19": sector_ansatz if "sector_ansatz" in globals() else SectorAnsatz(ANSATZ_OPS),
              "completo-24": SectorAnsatz(GEO_FULL_OPS)}
GEO_STARTS = 10  # cero, el óptimo de la distancia anterior y 8 al azar (como la sección 3)


def geo_optimize(shape: str, name: str, grid, seed: int = 0) -> pd.DataFrame:
    sa, rng = geo_ansatz[name], np.random.default_rng(seed)
    rows, x_prev = [], np.zeros(sa.n)
    for d in grid:
        H, e_fci = sector_hamiltonian_shape(d, shape)
        starts = [np.zeros(sa.n), x_prev] + [rng.normal(scale=0.7, size=sa.n) for _ in range(GEO_STARTS - 2)]
        runs = [minimize(sa.energy_grad, s, args=(H,), jac=True, method="BFGS", options={"gtol": 1e-8}) for s in starts]
        errs = np.array([r.fun - e_fci for r in runs]) * 1e3
        best = runs[int(np.argmin(errs))]
        x_prev = best.x
        s2 = spin_op.spin_square0(sa.state(best.x).reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0]
        rows.append({"shape": shape, "ansatz": name, "d": float(d), "e_sv": best.fun, "e_fci": e_fci,
                     "err_sv_mHa": errs.min(), "ok_starts": f"{(errs < CHEM_ACC * 1e3).sum()}/{len(runs)}",
                     "S2": s2, "theta": best.x})
    return pd.DataFrame(rows)


GEO_VQE_D = {"linear": GEO_D, "square": GEO_D[GEO_D >= 0.9]}  # cuadrado: en 0,8 Å el fundamental es triplete
RUN_GEO = False  # True: repetir búsqueda (~1 min) y medición de 3/4 (~1,5 min). False: usar GEO_CACHE (celda de arriba)
if RUN_GEO:
    geo_opt = pd.concat([geo_optimize("linear", "podado-19", GEO_VQE_D["linear"]),
                         geo_optimize("square", "podado-19", GEO_VQE_D["square"]),
                         geo_optimize("square", "completo-24", GEO_VQE_D["square"])], ignore_index=True)
    os.makedirs("results", exist_ok=True)
    geo_opt.assign(theta=geo_opt.theta.map(lambda t: " ".join(f"{v:.17g}" for v in t))).to_csv("results/geo_opt.csv", index=False)
else:
    geo_opt = pd.read_csv(io.StringIO(GEO_CACHE["opt"]), float_precision="round_trip")
    geo_opt["theta"] = geo_opt.theta.map(lambda t: np.array(t.split(), dtype=float))
    print(f"Resultados guardados ({GEO_CACHE['origen']}). Para recalcular: RUN_GEO = True.")
print("Statevector (solo para elegir θ; no son energías reportadas). Error vs. FCI en mHa y arranques "
      f"que llegan a precisión química ({CHEM_ACC * 1e3} mHa):")
print(geo_opt.pivot(index="d", columns=["shape", "ansatz"], values=["err_sv_mHa", "ok_starts"]).to_string(
    float_format=lambda v: f"{v:.4f}"))
print(f"⟨S²⟩ máximo: " + ", ".join(f"{s}/{a}: {g.S2.max():.1e}" for (s, a), g in geo_opt.groupby(["shape", "ansatz"])))

In [ ]:
# Extra #17 (3/4): energía MEDIDA con SamplerV2 de qiskit_aer (lo único que se reporta).
# Stand-in compacto de la medición de #6 (todavía no está en main): grupos QWC del SparsePauliOp, una
# rotación de base por grupo, shots por grupo ∝ Σ|c_k| del grupo, y E y σ desde los bitstrings.
from qiskit_aer.primitives import SamplerV2

GEO_SHOTS = 3_000_000  # por punto (suma sobre grupos)


def qwc_measurement_plan(op: SparsePauliOp) -> tuple[float, list]:
    """(término constante, [(base por qubit 'Z'/'X'/'Y', máscaras de bits, coeficientes)] por grupo QWC)."""
    is_id = ~(op.paulis.x.any(axis=1) | op.paulis.z.any(axis=1))
    const = op.coeffs[is_id].real.sum()
    plan = []
    for g in SparsePauliOp(op.paulis[~is_id], op.coeffs[~is_id]).group_commuting(qubit_wise=True):
        x, z = g.paulis.x, g.paulis.z
        basis = np.where((x & z).any(axis=0), "Y", np.where(x.any(axis=0), "X", "Z"))
        masks = ((x | z) * (1 << np.arange(op.num_qubits))).sum(axis=1)
        plan.append((tuple(basis), masks, g.coeffs.real))
    return const, plan


def qwc_sampler_energy(qc: QuantumCircuit, theta: np.ndarray, op: SparsePauliOp, e_nuc: float,
                       shots: int = GEO_SHOTS, seed: int = 0) -> dict:
    const, plan = qwc_measurement_plan(op)
    weights = np.array([np.abs(c).sum() for _, _, c in plan])
    n_shots = np.maximum(np.round(shots * weights / weights.sum()).astype(int), 100)
    energy, var = const + e_nuc, 0.0
    for g, ((basis, masks, coeffs), n) in enumerate(zip(plan, n_shots)):
        circ = qc.copy()
        for q, b in enumerate(basis):
            if b == "Y":
                circ.sdg(q)
            if b != "Z":
                circ.h(q)
        circ.measure_all()
        # Un job por grupo con su propia semilla: en qiskit-aer 0.17.2, PUBs del mismo job con distinta
        # cantidad de shots comparten el flujo aleatorio y sus resultados quedan correlacionados (ver markdown).
        sub_seed = int(np.random.SeedSequence([seed, g]).generate_state(1)[0])
        meas = SamplerV2(seed=sub_seed).run([(circ, theta, int(n))]).result()[0].data.meas
        freq = np.bincount(meas.array[:, -1], minlength=2 ** op.num_qubits).astype(float)  # 8 qubits: 1 byte por shot
        parity = np.array([[(b & m).bit_count() & 1 for m in masks] for b in range(len(freq))])
        vals = (1 - 2 * parity) @ coeffs  # valor del grupo completo en cada bitstring (incluye covarianzas)
        mean = freq @ vals / n
        energy += mean
        var += freq @ (vals - mean) ** 2 / (n - 1) / n
    return {"energy": energy, "sigma": np.sqrt(var), "shots": int(n_shots.sum()), "groups": len(plan)}


geo_circuit = transpile(build_ansatz(), basis_gates=["cx", "u"], optimization_level=1)  # para Aer (más rápido)
geo_cz = cz_count(build_ansatz())
if RUN_GEO:
    _rows = []
    for _k, _r in geo_opt.query("ansatz == 'podado-19'").iterrows():
        _op, _en = qubit_hamiltonian_shape(_r.d, _r["shape"])
        _m = qwc_sampler_energy(geo_circuit, _r.theta, _op, _en, seed=_k)
        _rows.append({"shape": _r["shape"], "d": _r.d, **_m, "cz": geo_cz, "e_sv": _r.e_sv, "e_fci": _r.e_fci})
    geo_vqe = pd.DataFrame(_rows)
    geo_vqe.to_csv("results/geo_vqe.csv", index=False)
else:
    geo_vqe = pd.read_csv(io.StringIO(GEO_CACHE["vqe"]))
    print(f"Resultados guardados ({GEO_CACHE['origen']}). Para volver a medir: RUN_GEO = True.")
geo_vqe["err_mHa"] = (geo_vqe.energy - geo_vqe.e_fci) * 1e3
geo_vqe["sigma_mHa"] = geo_vqe.sigma * 1e3
geo_vqe["dev_sv_in_sigma"] = (geo_vqe.energy - geo_vqe.e_sv) / geo_vqe.sigma
print(f"Grupos QWC por punto: {sorted(int(g) for g in geo_vqe.groups.unique())}; shots por punto: {GEO_SHOTS:.0e}; "
      f"medida vs. statevector: máx. {geo_vqe.dev_sv_in_sigma.abs().max():.1f} σ")

# Validación de σ: repetir la medición con otras semillas en un punto del cuadrado y comparar la dispersión.
_r = geo_opt.query("shape == 'square' and ansatz == 'podado-19' and d == 1.8").iloc[0]
if RUN_GEO:
    _op, _en = qubit_hamiltonian_shape(1.8, "square")
    _rep = [qwc_sampler_energy(geo_circuit, _r.theta, _op, _en, shots=200_000, seed=1000 + s) for s in range(20)]
    pd.DataFrame(_rep)[["energy", "sigma"]].to_csv("results/geo_sigma_rep.csv", index=False)
else:
    _rep = pd.read_csv(io.StringIO(GEO_CACHE["rep"])).to_dict("records")
_e = np.array([x["energy"] for x in _rep])
print(f"Validación de σ (cuadrado, 1,8 Å, 2e5 shots, 20 semillas): dispersión {_e.std(ddof=1) * 1e3:.2f} mHa, "
      f"σ calculada {np.mean([x['sigma'] for x in _rep]) * 1e3:.2f} mHa, media − statevector "
      f"{(_e.mean() - _r.e_sv) * 1e3:+.2f} ± {_e.std(ddof=1) / np.sqrt(len(_e)) * 1e3:.2f} mHa")
geo_vqe[["shape", "d", "energy", "sigma_mHa", "shots", "cz", "err_mHa", "dev_sv_in_sigma"]].round(6)

In [ ]:
# Extra #17 (4/4): curva del cuadrado vs. cadena lineal, error vs. d, R_e y D_e.
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline


def re_de(d: np.ndarray, e: np.ndarray) -> tuple[float, float]:
    """R_e = mínimo del spline cúbico de E(d); D_e = E(3,0 Å) − E(R_e) (definición de la celda 0.1)."""
    spline, fine = CubicSpline(d, e), np.linspace(d.min(), d.max(), 4001)
    i = np.argmin(spline(fine))
    return fine[i], spline(DISSOCIATED) - spline(fine[i])


def re_de_sampled(d, e, sigma, n: int = 500, seed: int = 0):
    """R_e, D_e de la curva medida y su incertidumbre (bootstrap paramétrico: E_i ~ N(energy_i, σ_i))."""
    rng = np.random.default_rng(seed)
    boot = np.array([re_de(d, e + sigma * rng.standard_normal(len(e))) for _ in range(n)])
    return re_de(d, e), boot.std(axis=0)


_summary = []
for shape in ("linear", "square"):
    rf = geo_refs.query("shape == @shape")
    vq = geo_vqe.query("shape == @shape")
    rf_v = rf[rf.d.isin(vq.d)]
    (re_v, de_v), (dre, dde) = re_de_sampled(vq.d.values, vq.energy.values, vq.sigma.values)
    re_f, de_f = re_de(rf_v.d.values, rf_v.e_fci.values)
    cc_ok = rf_v[rf_v.ccsd_conv]
    re_c, de_c = re_de(cc_ok.d.values, cc_ok.e_ccsd.values)
    if cc_ok.d.max() < DISSOCIATED:  # sin CCSD convergido a 3,0 Å no hay D_e (el spline extrapolaría)
        de_c = np.nan
    _summary.append({"forma": shape, "R_e FCI (Å)": re_f, "R_e VQE (Å)": f"{re_v:.3f} ± {dre:.3f}",
                    "R_e CCSD (Å)": re_c, "D_e FCI (mHa)": de_f * 1e3, "D_e VQE (mHa)": f"{de_v * 1e3:.1f} ± {dde * 1e3:.1f}",
                    "D_e CCSD (mHa)": de_c * 1e3,
                    "máx |err CCSD| (mHa)": rf_v.err_ccsd_mHa.abs().max(),
                    "máx |err VQE| (mHa)": vq.err_mHa.abs().max(), "máx σ (mHa)": vq.sigma_mHa.max()})
geo_summary = pd.DataFrame(_summary).set_index("forma")
print("CCSD del cuadrado no converge a 3,0 Å: su D_e queda sin definir (NaN) y no se grafica ese punto.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
colors = {"linear": "tab:blue", "square": "tab:red"}
names = {"linear": "lineal", "square": "cuadrado"}
for shape, col in colors.items():
    rf, vq = geo_refs.query("shape == @shape"), geo_vqe.query("shape == @shape")
    rf = rf[rf.d.isin(vq.d)]  # cuadrado sin 0,8 Å: ahí el fundamental del sector es triplete
    ok = rf.ccsd_conv
    axes[0].plot(rf.d, rf.e_fci, "-", color=col, lw=1.2, label=f"FCI ({names[shape]})")
    axes[0].plot(rf.d[ok], rf.e_ccsd[ok], "--", color=col, lw=1, alpha=0.7, label=f"CCSD ({names[shape]})")
    axes[0].errorbar(vq.d, vq.energy, yerr=vq.sigma, fmt="o", ms=4, color=col, capsize=2,
                     label=f"VQE medido ± σ ({names[shape]})")
    axes[1].plot(rf.d[ok], rf.err_ccsd_mHa[ok], "--s", ms=3, color=col, alpha=0.7, label=f"CCSD − FCI ({names[shape]})")
    axes[1].errorbar(vq.d, vq.err_mHa, yerr=vq.sigma_mHa, fmt="o", ms=4, color=col, capsize=2,
                     label=f"VQE − FCI ± σ ({names[shape]})")
axes[0].set(xlabel="d (Å)", ylabel="Energía total (Ha)", title="$\\mathrm{H_4}$ lineal vs. cuadrado (STO-3G)")
axes[1].axhspan(-CHEM_ACC * 1e3, CHEM_ACC * 1e3, color="0.85", label="precisión química")
axes[1].axhline(0, color="k", lw=0.6)
axes[1].set(xlabel="d (Å)", ylabel="Error vs. FCI (mHa)", title="Error de CCSD y de VQE (medido)")
for ax in axes:
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
geo_summary.round(3)

#### Decisiones, resultados y limitaciones (#17)

**Geometría.** `h4_geometry_shape(d, shape, aspect)`: `"linear"` (devuelve `h4_geometry(d)` sin cambios), `"square"` (lado $d$) y `"rectangle"` ($d \times a\,d$). No se tocó la celda 0.1: todo vive en esta subsección, con funciones y tablas propias (`scf_shape`, `qubit_hamiltonian_shape`, `references_shape`, `geo_refs`, `geo_opt`, `geo_vqe`) para no pisar `refs`/`vqe`/`params` del desafío central. La grilla es reducida (11 distancias, 0,8–3,0 Å); el VQE del cuadrado empieza en 0,9 Å porque a 0,8 Å el fundamental del sector $S_z = 0$ es un **triplete** (1,5 mHa por debajo del singlete, regla de Hund por la degeneración) y el desafío es el singlete.

**RHF del cuadrado.** Con simetría $D_{4h}$ el determinante cerrado no existe (dos electrones en un par degenerado), así que RHF rompe la simetría y PySCF cae en soluciones **distintas según $d$**, no todas mínimas: sin más cuidado, $E_\text{HF}$ salta (54 mHa más alta a 0,8 Å y 76 mHa a 1,2 Å que la solución estable) y CCSD sobre esa referencia erra +148 y +180 mHa. Por eso `scf_shape` sigue la inestabilidad interna (`mf.stability()`, RHF → RHF) hasta un mínimo en todas las distancias. Se mantienen orbitales RHF (no UHF), igual que en la sección 2 (ADR-0003). El criterio de signo del ADR-0003 sigue funcionando (todo orbital tiene algún coeficiente > 1e-4, se verifica), pero hay dos soluciones RHF equivalentes (rotadas 90°), así que la continuidad entre distancias no está garantizada: el *warm start* vale menos y el multi-start es lo que manda.

**Dónde falla CCSD** (tabla de arriba, error vs. FCI):

| | lineal | cuadrado |
|---|---|---|
| ¿Desde dónde > precisión química? | 1,6 Å (−3,0 mHa) | **0,9 Å** (−2,6 mHa) |
| Peor error | −30,6 mHa (2,4 Å) | −19,6 mHa (2,0 Å) |
| Signo | por debajo de FCI (no variacional) | por debajo de FCI en todas las distancias convergidas |
| Convergencia | siempre | **no converge a 3,0 Å** (el valor depende de cuántas iteraciones se dejen) |
| $\max\lvert t_2\rvert$ | 0,14–0,71 | **≈ 1,0** en todas las distancias |

En el cuadrado CCSD falla en toda la curva, incluso cerca del mínimo, porque el problema no es estirar enlaces sino la casi degeneración: las amplitudes dobles valen ~1 (la referencia HF no domina la función de onda). El barrido del rectángulo a 1,2 Å lo confirma: al alargar un lado ($a$ = 1 → 2) la brecha crece de 26 a 221 mHa, $\max\lvert t_2\rvert$ baja de 0,99 a 0,14 y el error de CCSD pasa de −6,5 a +0,01 mHa.

**Ansatz.** Se usó el elegido en la sección 3 (2-QUpCCGSD podado, 19 parámetros, 465 `cz`), aunque su poda se ajustó sobre la cadena lineal. **Sigue llegando a FCI en el cuadrado**: con 10 arranques el mejor queda a ≤ 0,10 mHa en todas las distancias (statevector, solo para elegir θ). Pero el paisaje es peor: solo 4–9 de 10 arranques llegan a precisión química (en el lineal, 3–9 de 10), mientras que el 2-QUpCCGSD completo (24 parámetros, 590 `cz`) llega con 7–10 de 10 y siempre a ≤ 1e-6 mHa con ⟨S²⟩ ~ 1e-14. El podado da ⟨S²⟩ ≤ 4e-4 en el cuadrado (algo de triplete, que a 3,0 Å está a 0,7 mHa). Conclusión: la poda no rompe la expresividad en el cuadrado, pero lo vuelve más dependiente del multi-start; si hiciera falta robustez, el completo cuesta 125 `cz` más.

**Medición (stand-in de #6).** `qwc_sampler_energy`: grupos QWC de `SparsePauliOp.group_commuting(qubit_wise=True)` (46 grupos en el lineal, 33–35 en el cuadrado), una rotación de base por grupo (X: H; Y: S†H), shots por grupo ∝ $\sum_k \lvert c_k\rvert$ del grupo, $3\times10^6$ shots por punto. $E$ y $\sigma$ salen de los bitstrings de `SamplerV2`: por cada bitstring se evalúa el grupo entero (así entran las covarianzas dentro del grupo), $\sigma^2 = \sum_g \text{Var}_g / N_g$. Validación: con 20 semillas a 1,8 Å (cuadrado, $2\times10^5$ shots) la dispersión es 4,7 mHa y la σ calculada 4,8 mHa; todas las energías medidas quedan a ≤ 2,4 σ del valor statevector de sus θ.

**Resultados** (energías medidas; tabla `geo_summary`):

| | lineal | cuadrado |
|---|---|---|
| $R_e$ FCI / VQE | 0,890 / 0,897 ± 0,008 Å | 1,279 / 1,285 ± 0,012 Å |
| $D_e$ FCI / VQE | 313,2 / 310,7 ± 1,9 mHa | 103,1 / 105,2 ± 1,6 mHa |
| $R_e$ CCSD | 0,890 Å | 1,304 Å |
| $D_e$ CCSD | 290,2 mHa (−23 mHa) | no definido (no converge a 3,0 Å) |
| σ por punto | 1,2–1,7 mHa | 1,1–1,3 mHa |
| máx. \|VQE − FCI\| | 3,4 mHa (2,6 σ) | 2,2 mHa (1,8 σ) |

El error del VQE medido es **ruido de shots**: el error statevector de los mismos θ es ≤ 0,1 mHa, y $\lvert E-E_\text{FCI}\rvert$ queda a ≤ 2,6 σ en todos los puntos. En el cuadrado el VQE le gana a CCSD ($\lvert E_\text{VQE}-E_\text{FCI}\rvert + \sigma < \lvert E_\text{CCSD}-E_\text{FCI}\rvert$) en todas las distancias desde 0,9 Å; en el lineal, solo desde ~1,6 Å, que es donde CCSD empieza a fallar.

**Qué cambia respecto de la cadena lineal y por qué.**
- **Curva más baja y ancha, con el mínimo más lejos** ($R_e$ 1,28 vs. 0,89 Å) y **$D_e$ tres veces menor** (103 vs. 313 mHa). En el cuadrado cada H tiene dos vecinos a $d$ y uno a $\sqrt2\,d$ (más repulsión nuclear), y los 4 electrones llenan un orbital enlazante y la mitad de un par degenerado no enlazante: es el análogo de H₄ del ciclobutadieno antiaromático. El cuadrado es en realidad un estado de transición entre dos rectángulos (dos H₂), como muestra el barrido del rectángulo. A 3,0 Å las dos formas llegan al mismo límite de 4 átomos (−1,867 Ha).
- **CCSD falla en toda la curva**, no solo en la zona estirada, porque la referencia es casi degenerada desde el principio (ver tabla). El ansatz de pares generalizados no depende de que HF domine, así que su error no cambia con la forma.
- **El paisaje del VQE es más difícil** (menos arranques buenos con el ansatz podado), y la brecha singlete–triplete es chica en las dos puntas de la curva (1,5 mHa a 0,8 Å, 0,7 mHa a 3,0 Å), así que hay que mirar ⟨S²⟩ además de la energía.

**Qué no funcionó o descartamos.**
- **Varios PUBs con distinta cantidad de shots en un mismo job de `SamplerV2`** (qiskit-aer 0.17.2): sus muestras salen **correlacionadas** (correlación 0,64 entre los conteos de dos circuitos de 1 qubit con 2000 y 3000 shots; con la misma cantidad de shots no pasa). Con un job por punto, los grupos quedaban correlacionados y la dispersión real era ~1,45 veces la σ calculada (10,6 vs. 7,3 mHa en 60 repeticiones). Se resolvió con **un job por grupo y semilla propia** (`SeedSequence([seed, g])`). **Ojo #6:** el reparto de shots por grupo dispara este problema.
- Reparto óptimo $N_g \propto \sqrt{\text{Var}_g}$ con un piloto de 1000 shots por grupo: lo probamos primero, pero la validación de σ fallaba (en parte por el problema anterior, y porque reusar la semilla del piloto en la corrida principal repite las mismas muestras). Nos quedamos con $\propto \sum\lvert c_k\rvert$, que no necesita piloto; con varianzas exactas, el óptimo ahorraría ~20 % de shots para la misma σ.
- PySCF con los hilos de OpenMP por defecto: con matrices de 4×4 CCSD tardaba ~10 veces más (66 s vs. 5 s para las referencias); se usa un hilo solo en esta celda (`with_omp_threads(1)`).
- `get_int_counts()` sobre 3 millones de shots es un loop de Python (~30 s por punto); se cuenta con `np.bincount` sobre el arreglo de bits.

**Limitaciones.**
- σ ≈ 1,2–1,7 mHa es del orden de la precisión química (1,6 mHa): la precisión química de los puntos medidos se cumple solo en promedio, no con margen. Bajarla a ~0,5 mHa costaría ~10 veces más shots con QWC; la agrupación por rotación de base (`docs/analysis/`) sería el camino.
- Los últimos dígitos dependen de qué celdas corrieron antes (BFGS cae en óptimos levemente distintos y cambian los θ medidos): entre corridas vimos moverse el error máximo medido del lineal entre 3,1 y 3,4 mHa.
- Grilla reducida: $R_e$ sale de un spline sobre 3–4 puntos cerca del mínimo (el de FCI sobre esa misma grilla es la referencia justa); la incertidumbre de $R_e$ y $D_e$ es un bootstrap paramétrico sobre σ.
- Solo se midió el ansatz podado; el completo se evaluó con statevector.
- Las semillas de la medición están fijas, pero BFGS puede caer en óptimos apenas distintos entre corridas (diferencias numéricas mínimas), así que las últimas cifras de la tabla pueden moverse unas décimas de mHa.
- Tiempo: ~3 min la subsección entera (casi todo es muestrear $6\times10^7$ shots).

### Comparación de ansätze (#15)

Comparamos cinco ansätze en **toda la grilla** (19 distancias). Todos arrancan en `HF_BITSTRING`:

| Ansatz | Qué es | Conserva |
|---|---|---|
| **HEA (RY + cz, 4 capas)** | Hardware-efficient tipo Kandala et al. (2017): capas de `ry` y una cadena de `cz` en línea | nada (ni N, ni S_z, ni S²) |
| **UCCSD** | Coupled cluster unitario, un paso de Trotter: dobles y después simples (occ → virt) | N, S_z |
| **k-UpCCGSD** (k = 1, 2, 3) | Lee et al. (2019): por capa, dobles de pares `(pα, pβ) → (qα, qβ)` y simples generalizadas con el mismo θ para α y β | N, S_z, S² |

**Cómo se mide cada cosa**

- **Parámetros (vale statevector):** simulador numpy propio con gradiente analítico (método adjunto) y L-BFGS-B. Los ansätze que conservan N y S_z se simulan en el sector de 36 estados en lugar de 256. Se barre la grilla con warm start en las dos direcciones, con 4 arranques en cada extremo (θ = 0 y 3 al azar), y se queda la menor energía por punto.
- **Control cruzado:** para cada punto, el circuito de Qiskit **transpilado** da la misma energía que el simulador numpy (`assert` a 1e-7 Ha).
- **`cz` y profundidad:** con `transpile_line` de la celda 0.1 (base `cz, sx, x, rz`, conectividad en línea, `optimization_level=3`). Las excitaciones fermiónicas se compilan como `PauliEvolutionGate` de sus cadenas de Jordan-Wigner.
- **Error medido:** `SamplerV2` de `qiskit_aer` con el **mismo presupuesto para todos**: 2²⁰ shots por punto, repartidos entre los grupos QWC en proporción a la norma 1 de sus coeficientes. E y σ salen de los bitstrings.
- **⟨S²⟩:** del estado final. Detecta contaminación con el triplete, que a 3,0 Å está a 0,46 mHa del fundamental.
- **Con ruido:** si #11 define `noise_model` (de `ruido.py`), se mide también con ruido en `d ≥ 1,8 Å` (matriz densidad, 2¹⁸ shots por punto).

El Hamiltoniano es `qubit_hamiltonian(d)` de la sección 2 (#2), y los operadores de escalera son los mismos (`_jw_ladder`).

**Resultados guardados.** Recalcular todo tarda ~8 min (~10 min más con ruido), así que por defecto la celda usa las tablas guardadas en `ANSATZ_CACHE` (de una corrida con estos mismos parámetros) y solo arma el resumen y los gráficos. Para recalcular: `RUN_ANSATZ_COMPARISON = True`.

In [ ]:
# Resultados guardados de la comparación de ansätze (#15). Recalcularlos tarda ~8 min, así que la celda
# siguiente los usa por defecto (RUN_ANSATZ_COMPARISON = False). Salieron de esa misma celda con True;
# para regenerarlos: poner True, correr y pegar acá results/ansatz_summary.csv y results/ansatz_comparison.csv.
ANSATZ_CACHE = {
    "origen": "corrida del 2026-11-21, 2²⁰ shots por punto, sin ruido",
    "summary": """\
ansatz,params,cz,depth,shots/punto,t (s)
"HEA (RY+cz, 4 capas)",40,28,35,1048575,70.8
UCCSD,26,1303,4220,1048575,62.9
1-UpCCGSD,12,474,1451,1048576,48.2
2-UpCCGSD,24,982,2915,1048576,103
3-UpCCGSD,36,1490,4379,1048576,156.8
""",
    "comparison": """\
ansatz,d,err_sv_mHa,err_meas_mHa,sigma_mHa,s2,nfev,err_noisy_mHa,sigma_noisy_mHa
"HEA (RY+cz, 4 capas)",0.6,28.1825,26.8938,2.63636,0.00156709,163,,
"HEA (RY+cz, 4 capas)",0.7,35.1411,29.0508,2.52857,0.00218068,104,,
"HEA (RY+cz, 4 capas)",0.8,43.3454,47.4434,2.45431,0.00288881,116,,
"HEA (RY+cz, 4 capas)",0.9,53.0817,52.3082,2.40897,0.00370608,222,,
"HEA (RY+cz, 4 capas)",1,64.7266,61.7675,2.37955,0.00466501,161,,
"HEA (RY+cz, 4 capas)",1.1,78.6893,79.3273,2.36247,0.00580673,305,,
"HEA (RY+cz, 4 capas)",1.2,93.6281,91.5721,2.41864,0.308266,3165,,
"HEA (RY+cz, 4 capas)",1.3,98.4228,92.647,2.46322,0.786907,125,,
"HEA (RY+cz, 4 capas)",1.4,93.1391,93.6263,2.44505,1.06858,85,,
"HEA (RY+cz, 4 capas)",1.5,82.2726,83.9773,2.40746,1.25899,96,,
"HEA (RY+cz, 4 capas)",1.6,69.2302,63.5915,2.36147,1.4117,62,,
"HEA (RY+cz, 4 capas)",1.7,56.1066,55.504,2.322,1.54195,56,,
"HEA (RY+cz, 4 capas)",1.8,44.1041,41.6511,2.28246,1.65174,49,,
"HEA (RY+cz, 4 capas)",2,25.4069,25.7798,2.23158,1.81061,59,,
"HEA (RY+cz, 4 capas)",2.2,13.7091,20.4713,2.20551,1.90254,22,,
"HEA (RY+cz, 4 capas)",2.4,7.08725,6.62226,2.20529,1.95156,21,,
"HEA (RY+cz, 4 capas)",2.6,3.54889,3.50976,2.21012,1.97657,20,,
"HEA (RY+cz, 4 capas)",2.8,1.72681,0.242384,2.2253,1.98897,20,,
"HEA (RY+cz, 4 capas)",3,0.81643,2.80267,2.23513,1.99494,19,,
UCCSD,0.6,0.00572607,4.41402,2.68983,4.37935e-19,14,,
UCCSD,0.7,0.00733863,-5.5938,2.6074,1.43506e-18,13,,
UCCSD,0.8,0.0152497,0.977777,2.56645,9.63256e-18,13,,
UCCSD,0.9,0.0371981,-7.45098,2.54101,5.41812e-18,13,,
UCCSD,1,0.0852724,4.5116,2.52809,1.16769e-16,17,,
UCCSD,1.1,0.176873,-4.25244,2.52149,2.84358e-16,14,,
UCCSD,1.2,0.334807,3.76587,2.52154,8.80198e-15,14,,
UCCSD,1.3,0.584819,3.70552,2.52206,9.90329e-14,14,,
UCCSD,1.4,0.9492,5.09126,2.51968,2.23873e-13,16,,
UCCSD,1.5,1.43658,2.62862,2.51658,5.80612e-14,18,,
UCCSD,1.6,2.02955,2.32065,2.50346,4.5476e-13,18,,
UCCSD,1.7,2.67152,2.81257,2.4871,1.67786e-12,20,,
UCCSD,1.8,3.25205,6.15402,2.46613,2.72276e-15,26,,
UCCSD,2,3.54357,3.2924,2.41163,3.99049e-13,35,,
UCCSD,2.2,2.29129,0.404783,2.39175,0.112206,28,,
UCCSD,2.4,1.10441,-1.69757,2.38475,0.113269,36,,
UCCSD,2.6,0.49051,3.14259,2.39358,0.0995992,45,,
UCCSD,2.8,0.208136,-0.915683,2.39633,0.0824785,44,,
UCCSD,3,0.0861095,3.52078,2.41919,0.0662304,3,,
1-UpCCGSD,0.6,10.1778,8.57839,2.6574,5.27969e-25,21,,
1-UpCCGSD,0.7,12.0536,8.75324,2.58352,2.45413e-35,10,,
1-UpCCGSD,0.8,13.0704,15.8611,2.53942,2.65713e-19,49,,
1-UpCCGSD,0.9,14.0548,16.2274,2.50001,1.07337e-34,43,,
1-UpCCGSD,1,14.9955,14.9839,2.48542,8.00467e-34,98,,
1-UpCCGSD,1.1,15.8454,19.208,2.47435,5.27653e-21,36,,
1-UpCCGSD,1.2,16.5146,17.0844,2.46904,1.31351e-18,29,,
1-UpCCGSD,1.3,16.88,16.4728,2.46684,1.56144e-18,31,,
1-UpCCGSD,1.4,16.8099,18.3218,2.46443,-3.73341e-18,29,,
1-UpCCGSD,1.5,16.2059,13.9196,2.4643,1.9533e-33,27,,
1-UpCCGSD,1.6,15.0475,18.4423,2.45384,-4.75324e-18,24,,
1-UpCCGSD,1.7,13.4175,13.2529,2.43883,-5.42634e-18,43,,
1-UpCCGSD,1.8,11.4874,14.7318,2.42672,-2.43706e-19,23,,
1-UpCCGSD,2,7.54312,6.70262,2.39407,3.16623e-19,26,,
1-UpCCGSD,2.2,4.41885,2.08338,2.37912,-6.24107e-18,26,,
1-UpCCGSD,2.4,2.39676,2.72364,2.37942,6.08774e-33,28,,
1-UpCCGSD,2.6,1.23267,5.36989,2.39224,4.79978e-33,34,,
1-UpCCGSD,2.8,0.608199,-2.46581,2.39659,1.0543e-32,24,,
1-UpCCGSD,3,0.289347,0.495346,2.41843,1.38387e-17,3,,
2-UpCCGSD,0.6,4.583e-10,1.41951,2.6837,2.95846e-19,3,,
2-UpCCGSD,0.7,4.44977e-10,-5.509,2.60393,-5.03209e-19,299,,
2-UpCCGSD,0.8,6.75016e-10,7.09576,2.55845,-3.04819e-19,336,,
2-UpCCGSD,0.9,4.57412e-10,-1.98067,2.53016,8.9572e-19,174,,
2-UpCCGSD,1,2.53131e-10,2.5444,2.51366,-6.74623e-19,145,,
2-UpCCGSD,1.1,3.29514e-10,-4.22836,2.50446,1.08604e-18,149,,
2-UpCCGSD,1.2,1.03917e-10,1.31878,2.49996,-1.5746e-18,153,,
2-UpCCGSD,1.3,1.56763e-10,0.0527445,2.49404,1.18316e-18,134,,
2-UpCCGSD,1.4,6.44818e-10,0.980463,2.48734,7.23949e-18,125,,
2-UpCCGSD,1.5,1.69198e-10,-1.38094,2.47477,-1.46723e-18,116,,
2-UpCCGSD,1.6,7.37188e-11,-1.32799,2.46238,-7.63897e-18,96,,
2-UpCCGSD,1.7,2.22933e-10,-5.16801,2.43934,3.50295e-18,108,,
2-UpCCGSD,1.8,6.30607e-11,1.62632,2.4186,-2.65691e-18,100,,
2-UpCCGSD,2,5.7554e-10,-1.0067,2.3677,-2.42194e-17,104,,
2-UpCCGSD,2.2,8.9706e-11,-3.65695,2.3337,5.13917e-32,120,,
2-UpCCGSD,2.4,3.4639e-11,-1.82842,2.32428,-1.53576e-17,135,,
2-UpCCGSD,2.6,4.44089e-11,-0.859931,2.32188,-1.98845e-17,125,,
2-UpCCGSD,2.8,9.41025e-10,-0.960527,2.32813,-1.125e-17,114,,
2-UpCCGSD,3,1.10134e-10,-1.15899,2.34193,5.33076e-32,142,,
3-UpCCGSD,0.6,2.35367e-10,-1.43662,2.68337,6.26632e-20,3,,
3-UpCCGSD,0.7,1.66978e-10,-0.33559,2.60912,-3.57219e-19,123,,
3-UpCCGSD,0.8,3.20632e-10,-2.32172,2.5593,-6.37748e-19,115,,
3-UpCCGSD,0.9,3.16192e-10,0.506899,2.52963,6.60693e-32,110,,
3-UpCCGSD,1,4.94715e-10,-4.94887,2.5111,1.09528e-18,97,,
3-UpCCGSD,1.1,4.05898e-10,-1.15193,2.50215,-1.32167e-18,111,,
3-UpCCGSD,1.2,4.64517e-10,-3.17039,2.49887,4.10515e-32,96,,
3-UpCCGSD,1.3,1.30562e-10,-0.970725,2.4921,6.21738e-18,89,,
3-UpCCGSD,1.4,9.23706e-11,-2.13826,2.48931,-2.95894e-18,98,,
3-UpCCGSD,1.5,1.40776e-10,-0.790646,2.47872,-4.86695e-18,77,,
3-UpCCGSD,1.6,2.75335e-11,-0.177602,2.46116,-2.20627e-18,83,,
3-UpCCGSD,1.7,1.77636e-12,-1.38698,2.44142,5.9522e-18,81,,
3-UpCCGSD,1.8,8.83738e-11,-4.34975,2.41389,-6.40586e-18,98,,
3-UpCCGSD,2,4.04121e-10,1.82549,2.37154,-1.41666e-17,83,,
3-UpCCGSD,2.2,1.11022e-11,1.9776,2.33846,1.06538e-17,96,,
3-UpCCGSD,2.4,9.10383e-11,0.15998,2.32505,1.0987e-17,81,,
3-UpCCGSD,2.6,9.32587e-12,1.54444,2.32754,7.58006e-32,75,,
3-UpCCGSD,2.8,1.36779e-10,3.78092,2.33411,1.12503e-17,81,,
3-UpCCGSD,3,1.28786e-11,-0.3995,2.33706,-4.26998e-18,82,,
""",
}

In [ ]:
# Extras · Comparación de ansätze (#15)
# Para cada familia y cada d de la grilla: error vs. FCI (statevector y medido con SamplerV2),
# cz y profundidad tras transpilar en línea, parámetros, evaluaciones del optimizador y <S²>.
# Si existe `noise_model` (#11, ruido.py), también el error medido con ruido.
RUN_ANSATZ_COMPARISON = False  # True: recalcular todo (~8 min). False: usar los resultados guardados en ANSATZ_CACHE
NOISY_SHOTS_PER_POINT = 2 ** 18
import io
import itertools
import time
from functools import reduce

import scipy.sparse as sp
from qiskit.circuit import Parameter
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Statevector
from qiskit_aer.primitives import SamplerV2
from scipy.optimize import minimize

# --- Operadores fermiónicos en Jordan-Wigner -------------------------------------
# Los de la sección 2 (#2): a_j = Z_0 ... Z_{j-1} (X_j + iY_j)/2, qubit p = (p, α), qubit 4 + p = (p, β).
_AD, _A = _jw_ladder(N_QUBITS)


def _s2_operator() -> SparsePauliOp:
    """S² = S_z² + S_z + S₋S₊ (en JW, con la convención de la celda 0.1)."""
    n = lambda q: _AD[q] @ _A[q]
    sz = 0.5 * SparsePauliOp.sum([n(alpha_qubit(p)) - n(beta_qubit(p)) for p in range(N_ORB)])
    s_plus = SparsePauliOp.sum([_AD[alpha_qubit(p)] @ _A[beta_qubit(p)] for p in range(N_ORB)])
    return (sz @ sz + sz + s_plus.adjoint() @ s_plus).simplify(atol=1e-12)


S2_OP = _s2_operator()

# --- Ansätze --------------------------------------------------------------------
# Cada ansatz es HF + una lista de factores exp(θ_k G_k), con G_k antihermítico y real.
#  - "exc": excitación fermiónica G = T − T†. Cumple G³ = −G => exp(θG) = I + sinθ G + (1 − cosθ) G².
#  - "rot": rotación de Pauli G = −iP/2   =>  exp(θG) = cos(θ/2) I + sin(θ/2) (−iP).
# Varios factores pueden compartir parámetro (k-UpCCGSD empareja α y β).


def _excitation(creates: list[int], annihilates: list[int]) -> SparsePauliOp:
    """Generador hermítico i(T − T†) con T = a†_c1 a†_c2 ... a_a2 a_a1."""
    T = reduce(lambda x, y: x @ y, [_AD[c] for c in creates] + [_A[a] for a in reversed(annihilates)])
    return (1j * (T - T.adjoint())).simplify(atol=1e-12)


def _single(p: int, q: int, s: int) -> SparsePauliOp:  # (q, s) -> (p, s)
    return _excitation([_so(p, s)], [_so(q, s)])


def _double(p: int, q: int, r: int, s_: int, s1: int, s2: int) -> SparsePauliOp:  # (r s1, s_ s2) -> (p s1, q s2)
    return _excitation([_so(p, s1), _so(q, s2)], [_so(r, s1), _so(s_, s2)])


OCC, VIRT = range(N_ALPHA), range(N_ALPHA, N_ORB)


def ansatz_uccsd():
    """UCCSD de un paso de Trotter (occ -> virt, conservan S_z). Orden: dobles y después simples."""
    ops = []
    for s in (0, 1):  # dobles con el mismo espín
        ops += [_double(a, b, i, j, s, s) for i, j in itertools.combinations(OCC, 2)
                for a, b in itertools.combinations(VIRT, 2)]
    ops += [_double(a, b, i, j, 0, 1) for i in OCC for j in OCC for a in VIRT for b in VIRT]  # αβ
    ops += [_single(a, i, s) for s in (0, 1) for i in OCC for a in VIRT]
    return [("exc", g, k) for k, g in enumerate(ops)], len(ops)


def ansatz_kupccgsd(k: int):
    """k-UpCCGSD (Lee et al. 2019): por capa, dobles de pares y después simples generalizadas
    (θ compartido entre α y β, así el ansatz conserva S²)."""
    gens, n = [], 0
    for _ in range(k):
        for p, q in itertools.combinations(range(N_ORB), 2):  # par (p α, p β) -> (q α, q β)
            gens.append(("exc", _double(q, q, p, p, 0, 1), n))
            n += 1
        for p, q in itertools.combinations(range(N_ORB), 2):
            gens += [("exc", _single(q, p, 0), n), ("exc", _single(q, p, 1), n)]
            n += 1
    return gens, n


def ansatz_hea(layers: int):
    """Hardware-efficient (Kandala et al. 2017, versión real): capas de RY + cadena de cz en línea."""
    gens, n = [], 0
    for layer in range(layers + 1):
        for q in range(N_QUBITS):
            gens.append(("rot", SparsePauliOp.from_sparse_list([("Y", [q], 1.0)], N_QUBITS), n))
            n += 1
        if layer < layers:
            gens.append(("cz", None, None))
    return gens, n


ANSATZ_FACTORIES = {
    "HEA (RY+cz, 4 capas)": lambda: ansatz_hea(4),
    "UCCSD": ansatz_uccsd,
    "1-UpCCGSD": lambda: ansatz_kupccgsd(1),
    "2-UpCCGSD": lambda: ansatz_kupccgsd(2),
    "3-UpCCGSD": lambda: ansatz_kupccgsd(3),
}

# --- Circuito de Qiskit equivalente (para contar cz y para medir) ---------------


def ansatz_circuit(gens, n_params: int) -> tuple[QuantumCircuit, list[Parameter]]:
    theta = [Parameter(f"θ{k:02d}") for k in range(n_params)]
    qc = QuantumCircuit(N_QUBITS)
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for kind, gen, k in gens:
        if kind == "cz":
            for q in range(N_QUBITS - 1):
                qc.cz(q, q + 1)
        elif kind == "rot":
            qc.ry(theta[k], int(np.flatnonzero(gen.paulis.x[0])[0]))  # exp(−iθY/2)
        else:  # exp(θ G) = exp(−i θ (iG)), y gen ya es iG (hermítico)
            qc.append(PauliEvolutionGate(gen, time=theta[k]), range(N_QUBITS))
    return qc, theta


# --- Simulador numpy con gradiente adjunto (solo para buscar parámetros) -------
_CZ_DIAG = np.array([(-1) ** sum((b >> q) & (b >> (q + 1)) & 1 for q in range(N_QUBITS - 1))
                     for b in range(2 ** N_QUBITS)], dtype=float)


def _real_generator(kind, gen):
    """Matriz real y rala del generador antihermítico G."""
    M = gen.to_matrix(sparse=True)  # = iG (exc) o P (rot)
    G = (-1j * M) if kind == "exc" else (-0.5j * M)
    assert abs(G.imag).max() < 1e-12
    return sp.csr_matrix(G.real)


# SECTOR (sección 2): los 36 estados con N = 4, S_z = 0. Acá solo se usa como conjunto de índices.


class NumpyAnsatz:
    """Simulador del ansatz. Si todos los factores conservan N y S_z, trabaja en el sector de 36 estados."""

    def __init__(self, gens, n_params):
        self.n = n_params
        self.idx = SECTOR if all(kind == "exc" for kind, _, _ in gens) else np.arange(2 ** N_QUBITS)
        self.ops = [(kind, None if kind == "cz" else self.proj(_real_generator(kind, gen)), k)
                    for kind, gen, k in gens]
        self.hf = (self.idx == int(HF_BITSTRING, 2)).astype(float)

    def proj(self, M):
        return sp.csr_matrix(M[self.idx][:, self.idx])

    @staticmethod
    def _apply(kind, G, t, v):
        if kind == "cz":  # solo en el espacio completo (HEA)
            return _CZ_DIAG * v
        Gv = G @ v
        if kind == "exc":
            return v + np.sin(t) * Gv + (1 - np.cos(t)) * (G @ Gv)
        return np.cos(t / 2) * v + 2 * np.sin(t / 2) * Gv

    def state(self, theta):
        v = self.hf
        for kind, G, k in self.ops:
            v = self._apply(kind, G, None if k is None else theta[k], v)
        return v

    def energy_and_grad(self, theta, H):
        psi = self.state(theta)
        lam = H @ psi
        e = psi @ lam
        grad = np.zeros(self.n)
        for kind, G, k in reversed(self.ops):  # dE/dθ = 2 <λ_k| G |ψ_k>
            if kind != "cz":
                grad[k] += 2 * lam @ (G @ psi)
            t = None if k is None else -theta[k]
            psi, lam = self._apply(kind, G, t, psi), self._apply(kind, G, t, lam)
        return e, grad


def _minimize(model, H, x0):
    res = minimize(model.energy_and_grad, x0, args=(H,), jac=True, method="L-BFGS-B",
                   options={"maxiter": 5000, "ftol": 1e-14, "gtol": 1e-9})
    return res.x, res.fun, res.nfev


def optimize_grid(model, Hs, n_random=3, seed=0):
    """Barrido con warm start en las dos direcciones + multi-start en los extremos; por punto, el mejor."""
    rng = np.random.default_rng(seed)
    best = {}
    Hs = {d: model.proj(H) for d, H in Hs.items()}

    def keep(d, x, e, nfev):
        if d not in best or e < best[d][1] - 1e-12:
            best[d] = (x, e, nfev)

    def starts():
        return [np.zeros(model.n)] + [rng.normal(0, 0.1, model.n) for _ in range(n_random)]

    for order in (list(D), list(D)[::-1]):
        x = min((_minimize(model, Hs[order[0]], x0) for x0 in starts()), key=lambda r: r[1])[0]
        for d in order:
            x, e, nfev = _minimize(model, Hs[d], x)
            keep(d, x, e, nfev)
    return best


# --- Medición con SamplerV2 (grupos QWC, shots ∝ peso del grupo) -----------------
SHOTS_PER_POINT = 2 ** 20  # mismo presupuesto por punto para todos los ansätze
# Ojo: con SamplerV2(seed=s), Aer repite semillas entre pubs y las mediciones de distintos grupos
# quedan correlacionadas (la σ calculada subestima el error real). Una semilla independiente por pub lo evita.
_seeds = np.random.default_rng(1234)


def _rotate_to_z(qc: QuantumCircuit, basis_label: str) -> QuantumCircuit:
    m = qc.copy()
    for q, b in enumerate(reversed(basis_label)):
        if b == "X":
            m.h(q)
        elif b == "Y":
            m.sdg(q)
            m.h(q)
    m.measure_all()
    return transpile(m, basis_gates=BASIS_GATES, optimization_level=0)


def measure_energies(tqc, theta, xs, ops_l, shots_per_point=SHOTS_PER_POINT, noise_model=None):
    """E y σ medidos para cada fila de xs (un juego de parámetros por d), sobre el circuito transpilado.

    ops_l: SparsePauliOp ya con el layout del circuito transpilado, uno por d (mismo orden que xs).
    """
    labels = sorted(set().union(*[set(op.paulis.to_labels()) for op in ops_l]) - {"I" * N_QUBITS})
    groups = SparsePauliOp(labels).group_commuting(qubit_wise=True)
    coeff = [dict(zip(op.paulis.to_labels(), op.coeffs.real)) for op in ops_l]
    weight = np.array([np.mean([sum(abs(c.get(l, 0.0)) for l in g.paulis.to_labels()) for c in coeff]) for g in groups])
    shots = np.maximum(100, np.round(shots_per_point * weight / weight.sum())).astype(int)

    pubs = []
    for g, n in zip(groups, shots):
        basis = "".join(max(col, key=lambda ch: ch != "I") for col in zip(*g.paulis.to_labels()))
        circ = _rotate_to_z(tqc, basis)
        pubs.append((circ, np.array([[x[theta.index(p)] for p in circ.parameters] for x in xs]), int(n)))
    options = {} if noise_model is None else {
        "backend_options": {"method": "density_matrix", "noise_model": noise_model}}
    results = [SamplerV2(seed=int(_seeds.integers(2 ** 31)), options=options).run([pub]).result()[0]
               for pub in pubs]

    bits = np.array([[(b >> q) & 1 for q in range(N_QUBITS)] for b in range(2 ** N_QUBITS)])
    E = np.array([c.get("I" * N_QUBITS, 0.0) for c in coeff])
    var = np.zeros(len(xs))
    for g, res in zip(groups, results):
        ls = g.paulis.to_labels()
        masks = np.array([[ch != "I" for ch in reversed(l)] for l in ls])          # (términos, qubits)
        signs = (-1.0) ** (bits @ masks.T.astype(int))                              # (256, términos)
        for j in range(len(xs)):
            v = signs @ np.array([coeff[j].get(l, 0.0) for l in ls])               # valor por shot, por bitstring
            cnt = res.data.meas.get_int_counts(loc=j)
            idx, n_b = np.fromiter(cnt.keys(), int), np.fromiter(cnt.values(), float)
            n = n_b.sum()
            mean = n_b @ v[idx] / n
            E[j] += mean
            var[j] += (n_b @ (v[idx] - mean) ** 2) / (n - 1) / n
    return E, np.sqrt(var), int(shots.sum())


# --- Corrida (o resultados guardados) ----------------------------------------
# Con RUN_ANSATZ_COMPARISON = False se leen las tablas guardadas en ANSATZ_CACHE (celda anterior), que salieron
# de esta misma celda con True. El resumen y los gráficos de abajo se calculan igual en los dos casos.
e_fci = refs.set_index("d").e_fci
if RUN_ANSATZ_COMPARISON:
    t0 = time.time()
    NOISE_MODEL = globals().get("noise_model")  # #11: el NoiseModel de ruido.py (o la función ruido.noise_model)
    if callable(NOISE_MODEL):
        NOISE_MODEL = NOISE_MODEL()
    H_OPS = {d: qubit_hamiltonian(d) for d in D}
    H_MATS = {d: sp.csr_matrix(op.to_matrix(sparse=True).real) for d, (op, _) in H_OPS.items()}
    S2_MAT = sp.csr_matrix(S2_OP.to_matrix(sparse=True).real)

    rows, ansatz_summary = [], []
    for name, factory in ANSATZ_FACTORIES.items():
        t = time.time()
        gens, n_params = factory()
        model = NumpyAnsatz(gens, n_params)
        best = optimize_grid(model, H_MATS)

        qc, theta = ansatz_circuit(gens, n_params)
        tqc = transpile_line(qc)
        ops_l = [H_OPS[d][0].apply_layout(tqc.layout) for d in D]
        xs = [best[d][0] for d in D]

        # Control cruzado: el circuito de Qiskit transpilado da la misma energía que el simulador numpy.
        for d, x, op_l in zip(D, xs, ops_l):
            e_qiskit = Statevector(tqc.assign_parameters(dict(zip(theta, x)))).expectation_value(op_l).real
            assert abs(e_qiskit - best[d][1]) < 1e-7, f"{name}: circuito ≠ modelo en d={d}"

        e_meas, sigma, shots = measure_energies(tqc, theta, xs, ops_l)
        noisy = {}
        if NOISE_MODEL is not None:  # con ruido: solo la zona estirada (matriz densidad, es más lento)
            js = [j for j, d in enumerate(D) if d >= STRETCHED]
            e_n, s_n, _ = measure_energies(tqc, theta, [xs[j] for j in js], [ops_l[j] for j in js],
                                           NOISY_SHOTS_PER_POINT, NOISE_MODEL)
            noisy = {D[j]: (e_n[i], s_n[i]) for i, j in enumerate(js)}
        for j, d in enumerate(D):
            x, e_sv, nfev = best[d]
            psi, s2_mat = model.state(x), model.proj(S2_MAT)
            e_nuc = H_OPS[d][1]
            rows.append({"ansatz": name, "d": d,
                         "err_sv_mHa": 1e3 * (e_sv + e_nuc - e_fci[d]),
                         "err_meas_mHa": 1e3 * (e_meas[j] + e_nuc - e_fci[d]),
                         "sigma_mHa": 1e3 * sigma[j], "s2": psi @ (s2_mat @ psi), "nfev": nfev,
                         "err_noisy_mHa": 1e3 * (noisy[d][0] + e_nuc - e_fci[d]) if d in noisy else np.nan,
                         "sigma_noisy_mHa": 1e3 * noisy[d][1] if d in noisy else np.nan})
        ops = tqc.count_ops()
        ansatz_summary.append({"ansatz": name, "params": n_params, "cz": ops.get("cz", 0), "depth": tqc.depth(),
                        "shots/punto": shots, "t (s)": round(time.time() - t, 1)})
        print(f"{name:22s} listo en {time.time() - t:5.1f} s")

    ansatz_cmp = pd.DataFrame(rows)
    ansatz_base = pd.DataFrame(ansatz_summary).set_index("ansatz")
    os.makedirs("results", exist_ok=True)
    ansatz_cmp.to_csv("results/ansatz_comparison.csv", index=False)
    ansatz_base.to_csv("results/ansatz_summary.csv")
    print(f"Total: {time.time() - t0:.0f} s")
else:
    ansatz_cmp = pd.read_csv(io.StringIO(ANSATZ_CACHE["comparison"]))
    ansatz_base = pd.read_csv(io.StringIO(ANSATZ_CACHE["summary"])).set_index("ansatz")
    print(f"Resultados guardados ({ANSATZ_CACHE['origen']}). Para recalcular: RUN_ANSATZ_COMPARISON = True.")

stretched = ansatz_cmp[ansatz_cmp.d >= STRETCHED]
ansatz_summary = ansatz_base.copy()
ansatz_summary["max err sv (d ≥ 1,8) mHa"] = stretched.groupby("ansatz").err_sv_mHa.max()
ansatz_summary["max err sv (toda) mHa"] = ansatz_cmp.groupby("ansatz").err_sv_mHa.max()
ansatz_summary["max |err med|+σ (d ≥ 1,8) mHa"] = (stretched.err_meas_mHa.abs() + stretched.sigma_mHa).groupby(stretched.ansatz).max()
if ansatz_cmp.err_noisy_mHa.notna().any():
    ansatz_summary["max |err ruido| (d ≥ 1,8) mHa"] = stretched.err_noisy_mHa.abs().groupby(stretched.ansatz).max()
ansatz_summary["max <S²>"] = ansatz_cmp.groupby("ansatz").s2.max()
ansatz_summary["nfev medio"] = ansatz_cmp.groupby("ansatz").nfev.mean().round(0)
ansatz_summary["le gana a CCSD"] = stretched.assign(
    ok=lambda df: df.err_meas_mHa.abs() + df.sigma_mHa < 1e3 * (refs.set_index("d").e_ccsd - e_fci).abs()[df.d].values
).groupby("ansatz").ok.all()
print(ansatz_summary.to_string(float_format=lambda v: f"{v:.3g}"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for name, df in ansatz_cmp.groupby("ansatz", sort=False):
    ax1.semilogy(df.d, df.err_sv_mHa.clip(lower=1e-6), "o-", ms=3, label=f"{name} ({ansatz_summary.loc[name, 'cz']} cz)")
    ax2.errorbar(df.d, df.err_meas_mHa, yerr=df.sigma_mHa, fmt="o-", ms=3, capsize=2, label=name)
ccsd_err = 1e3 * (refs.e_ccsd - refs.e_fci).abs()
ax1.semilogy(refs.d, ccsd_err, "k--", label="|CCSD − FCI|")
for ax in (ax1, ax2):
    ax.axhline(1e3 * CHEM_ACC, color="C2", ls=":", label="precisión química")
    ax.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0)
    ax.set_xlabel("d (Å)")
ax1.set(ylabel="E_VQE − E_FCI (mHa)", title="Error con statevector (piso en 1e-6 mHa = exacto)")
ax2.plot(refs.d, ccsd_err, "k--")
ax2.set(ylabel="E_medida − E_FCI (mHa)", title=f"Error medido con SamplerV2 ({SHOTS_PER_POINT:.0e} shots/punto)")
ax1.legend(fontsize=7)
plt.tight_layout()
plt.show()

#### Resultados

| Ansatz | Parámetros | `cz` | Profundidad | Máx. error sv, d ≥ 1,8 (mHa) | Máx. error sv, toda la curva (mHa) | Máx. \|error medido\| + σ, d ≥ 1,8 (mHa) | Máx. ⟨S²⟩ | ¿Le gana a CCSD? |
|---|---:|---:|---:|---:|---:|---:|---:|:---:|
| HEA (4 capas) | 40 | 28 | 35 | 44,1 | 98,4 | 44–48 | 1,99 | no |
| UCCSD | 26 | 1303 | 4220 | 3,5 | 3,5 | 8,6 | 0,1 | sí (justo) |
| 1-UpCCGSD | 12 | 474 | 1451 | 11,5 | 16,9 | 17,2 | 0 | no |
| **2-UpCCGSD** | 24 | 982 | 2915 | **< 1e-9** | **< 1e-9** | 6,0 | 0 | **sí** |
| 3-UpCCGSD | 36 | 1490 | 4379 | < 1e-9 | < 1e-9 | 6,8 | 0 | sí |

Los números del HEA varían un poco entre corridas: su optimización tiene muchos mínimos locales y el álgebra lineal multihilo no es determinista.

#### Conclusión

- **2-UpCCGSD** es el que conviene si no hay ruido. Llega a FCI en toda la curva con solo 24 parámetros y conserva S² por construcción: no se contamina con el triplete casi degenerado de la zona estirada. 3-UpCCGSD no agrega nada y suma 50 % más de `cz`.
- **UCCSD** se traba en 3,5 mHa entre 1,8 y 2,2 Å. Le gana a CCSD por poco a 1,8 Å (8,6 contra 9,1 mHa) y nunca llega a precisión química en esa zona. **1-UpCCGSD** no alcanza.
- El **HEA** es 35 veces más barato en `cz`, pero se queda en mínimos locales (44 mHa a 1,8 Å) y rompe la simetría de espín (⟨S²⟩ ≈ 2): energía baja no significa estado correcto.
- **Lo que limita al ansatz exacto es la medición**, no el ansatz. Con 2²⁰ shots por punto, σ ≈ 2,3 mHa. Para tener precisión química medida hacen falta ~8 veces más shots (σ ≈ 0,8 mHa) o una medición más eficiente (#6).
- **Con ruido, la elección se invierte.** En una prueba con un modelo sintético igual al del enunciado (depolarizante, 0,1 % por `cz` y 0,005 % por `sx`/`x`), 2-UpCCGSD queda a ~450 mHa de FCI y el HEA a 21–67 mHa. Con ~1000 `cz`, el error por compuerta lo domina todo. Reducir `cz` (excitaciones compiladas como compuertas de Givens o QEB en lugar de cadenas de Pauli, #4) es lo que hace falta para que el ansatz exacto sirva con ruido.

#### Qué no funcionó

- **El orden de los factores importa mucho.** Con simples antes que dobles, 1-UpCCGSD llega a 220 mHa de error a 3,0 Å (con el orden inverso, a 0,3 mHa). UCCSD llega a 5,2 mHa y se contamina con el triplete (⟨S²⟩ hasta 1,9). Por eso todos aplican primero las dobles y después las simples, que actúan como una rotación orbital final.
- **Más capas de HEA no ayudan.** Con 8 capas (72 parámetros) da exactamente el mismo error que con 4 y tarda el doble: es un problema de optimización (mínimos locales), no de expresividad. Lo sacamos de la comparación.
- **`SamplerV2(seed=...)` de Aer repite semillas entre pubs.** Las mediciones de distintos grupos quedaban correlacionadas, la σ subestimaba el error real y aparecían desvíos de 3–4σ. Con una semilla independiente por pub, los z-scores de los ansätze exactos tienen desvío 1,06 en 40 puntos (máx. |z| = 2,8): la σ está bien calibrada. **Ojo #6:** les pasa lo mismo si usan una semilla fija.

### Menos qubits: tapering Z₂ (#14)

El Hamiltoniano de 8 qubits conmuta con 3 cadenas de Z (simetrías Z₂). En el sector del estado HF cada una fija el valor de un qubit, así que el problema se reduce a **5 qubits** sin perder nada: el mínimo del Hamiltoniano reducido es exactamente FCI. La reducción se valida en las 19 distancias, se arma un ansatz de 5 qubits y se compara con el de 8 en términos, `cz`, error y error con ruido.

**Resultados guardados.** La búsqueda de θ del ansatz de 5 qubits (2/3, ~30 s) usa por defecto los θ* guardados en `PARAMS5_CACHE` (a precisión completa, de una corrida de esa celda con `RUN_ANSATZ5 = True`); la tabla `ansatz5_check` y todo 3/3 se recalculan desde esos θ*. Para repetir la búsqueda: `RUN_ANSATZ5 = True`.

In [ ]:
# Extra #14 (1/3): tapering Z₂. El Hamiltoniano de 8 qubits tiene simetrías que son cadenas de Z
# (paridades); en el sector del estado HF cada una fija un qubit, así que se puede sacar.
from qiskit.quantum_info import Z2Symmetries


def hf_eigenvalues(symmetries) -> list[int]:
    """Autovalor (±1) de cada simetría (cadena de Z) sobre el estado HF_BITSTRING."""
    hf = int(HF_BITSTRING, 2)
    return [(-1) ** int(np.sum(s.z & np.array([(hf >> q) & 1 for q in range(N_QUBITS)], bool)))
            for s in symmetries]


def taper(op: SparsePauliOp, symmetries: Z2Symmetries, sector: list[int]) -> SparsePauliOp:
    """Proyecta `op` al sector `sector` (±1 por simetría) y saca los qubits fijados por las simetrías.

    `op` tiene que conmutar con todas las simetrías (el Hamiltoniano y los generadores que las respetan).
    """
    z2 = Z2Symmetries(symmetries.symmetries, symmetries.sq_paulis, symmetries.sq_list, tapering_values=sector)
    return z2.taper(op).simplify(atol=1e-12)


Z2 = Z2Symmetries.find_z2_symmetries(qubit_hamiltonian(STRETCHED)[0])
SECTOR_HF = hf_eigenvalues(Z2.symmetries)
KEEP_QUBITS = [q for q in range(N_QUBITS) if q not in Z2.sq_list]       # qubits que sobreviven
N_TAPERED = len(KEEP_QUBITS)
_hf = int(HF_BITSTRING, 2)
HF_TAPERED = sum(((_hf >> q) & 1) << i for i, q in enumerate(KEEP_QUBITS))  # HF sigue siendo un estado de la base
print("Simetrías (q7…q0):", [s.to_label() for s in Z2.symmetries], "→ autovalores en HF:", SECTOR_HF)
print(f"Se sacan los qubits {Z2.sq_list}; quedan {KEEP_QUBITS} → {N_TAPERED} qubits. "
      f"HF reducido: |{HF_TAPERED:0{N_TAPERED}b}⟩")

taper_rows, H_TAPERED = [], {}
for d in map(float, D):
    op, e_nuc = qubit_hamiltonian(d)
    z2_d = Z2Symmetries.find_z2_symmetries(op)
    assert [s.to_label() for s in z2_d.symmetries] == [s.to_label() for s in Z2.symmetries], f"otras simetrías en d={d}"
    op5 = taper(op, Z2, SECTOR_HF)
    H5 = op5.to_matrix()
    assert np.abs(H5.imag).max() < 1e-12
    H_TAPERED[d] = (op5, e_nuc)
    e_min = np.linalg.eigvalsh(H5.real)[0] + e_nuc
    e_fci = refs.set_index("d").e_fci[d]
    taper_rows.append({"d": d, "terms_8q": len(op), "terms_5q": len(op5),
                       "qwc_8q": len(op.group_commuting(qubit_wise=True)), "qwc_5q": len(op5.group_commuting(qubit_wise=True)),
                       "err_min_vs_fci": e_min - e_fci, "err_hf": H5[HF_TAPERED, HF_TAPERED].real + e_nuc - refs.set_index("d").e_hf[d]})
taper_check = pd.DataFrame(taper_rows)
assert (taper_check.err_min_vs_fci.abs() < 1e-8).all() and (taper_check.err_hf.abs() < 1e-8).all(), taper_check
print(f"Validado en las {len(D)} distancias: mín. del H reducido − FCI ≤ {taper_check.err_min_vs_fci.abs().max():.1e} Ha, "
      f"⟨HF|H|HF⟩ − E_HF ≤ {taper_check.err_hf.abs().max():.1e} Ha. Términos de Pauli: "
      f"{taper_check.terms_8q.max()} → {taper_check.terms_5q.max()}; grupos QWC: {taper_check.qwc_8q.max()} → {taper_check.qwc_5q.max()}.")

In [ ]:
# θ* guardados del ansatz de 5 qubits (#14, celda 2/3). Repetir la búsqueda tarda ~30 s, así que la celda siguiente
# los usa por defecto (RUN_ANSATZ5 = False). Salieron de esa misma celda con True, a precisión completa (17 cifras);
# para regenerarlos: poner True, correr y pegar acá results/params5.csv.
PARAMS5_CACHE = {
    "origen": "corrida del 2026-11-21 con RUN_ANSATZ5 = RUN_MITIGACION = True, qiskit 2.5.2, qiskit-aer 0.17.2, pyscf 2.14.0",
    "params5": """\
d,phi_0,phi_1,phi_2,phi_3,phi_4,phi_5,phi_6,phi_7,phi_8,phi_9,phi_10,phi_11,phi_12,phi_13,phi_14,phi_15,phi_16,phi_17,phi_18,phi_19,phi_20,phi_21,phi_22,phi_23,phi_24,phi_25,phi_26,phi_27,phi_28,phi_29,phi_30,phi_31,phi_32,phi_33,phi_34
0.6,-0.2495166910297941,1.1568608870853738,-0.40409432836646814,0.050933629569185165,-1.4855247266916043,0.2965831298890075,-0.6540937224932328,-0.8730589710245626,0.26318800219263266,-0.01455739274764664,0.06750704327481939,0.8799965008893436,-0.21917772422261272,0.7249558331606666,0.4671345382072146,0.49963220414523646,0.5594973986588874,0.3671518182261652,1.2294579483350685,-0.7803623461232575,0.08647262019083043,0.26412384334456046,1.7663331121725996,0.05255814293682417,-1.2450502751930832,0.40416655281231545,0.6658752754348685,-0.054708954868338024,-0.9915169947838909,-0.15740589260220292,0.3560724334274413,0.010665862888065857,-0.15783612578825348,-1.142025214263661,0.06976832074617759
0.7,-0.09328535912615829,-0.37747689911823534,0.07966505178866745,0.007779354675019224,1.1148349981076928,0.20762558679502602,0.9595150089898249,0.46424084144784866,-1.424218261271436,-0.5930829842159089,-0.4608090381660039,-1.4770504538605693,-1.156283388272017,-1.2045628447545491,0.24198911039531626,-0.5356628760956101,-1.3797085489466083,-1.5126179290986388,1.5622440641763498,0.11950565330359078,0.08882433341289876,2.372453577300662,-1.4621218402406189,-0.14616858851738931,0.4688364706645635,0.36828431391460903,-0.21722638529564892,-0.46398990534245715,0.35190903094491055,0.3705281289764524,0.14918560440253625,0.467384152086513,0.3650467452921551,-0.2505780639473611,-1.1895539625935097
0.8,-0.10498026641380682,1.1992151616998814,-0.38902891258458455,0.19001931545654588,-1.4351282430405143,0.21018383862685072,-0.4432565436970007,-0.5421310279634232,0.05725626937917742,-0.028951914268515018,0.02701414466438053,0.7286471605649586,-0.1320104521586854,0.7031509889979277,0.3662855341217774,0.5811332820353677,0.9870699280476072,0.30619570196796403,1.259684736272809,-0.8542026054794053,0.07970983777498697,0.5861028331525242,1.9864848033563816,-0.2175560034550875,-1.2837366281072202,0.3682828680728985,0.8453118660196877,-0.03256870269256856,-0.9908113545335564,-0.09835219158536256,0.4770090237305729,0.031574709089203366,-0.07570982650955438,-0.9241164199231114,0.07631472749350603
0.9,-0.2517519818666754,-0.46317974144675056,-0.14532730192801815,-0.0559808381302753,0.8037952300212078,-0.047579411580025245,0.8897567839113605,0.6445088002419971,-1.2758303288291204,-1.0049650532881222,-0.21249352632891366,-1.2945170544073723,-1.1125249903129015,-1.1284385608440817,0.24128645746668118,-0.45029712380413667,-1.013126861453507,-1.3725626012801555,1.5279215605406256,0.3449535052692891,-0.06648290903711937,2.468445947927776,-1.4567130772035193,-0.2202684465225797,0.4602996942904249,0.4364331944495965,-0.43461172914259444,-0.46429589337010924,0.6256609654493722,0.5266192860797673,0.01721865491348711,0.6573218187737437,0.42760613803270076,0.2337310353629207,-1.0092952061268237
1.0,-0.2719783534631532,-0.4255905959276936,-0.24993191037851795,-0.18319331191878918,0.6201219720264419,-0.05238864481507536,0.59613435422339,0.7534508573582158,-1.1949076956951838,-1.2601066774031473,-0.27806452300700224,-1.311948685550931,-1.0285771172289533,-1.151640969411077,0.31257362947251854,-0.586288652413655,-0.8043382780440514,-1.3416670132555513,1.603467042623209,0.2707912664985359,-0.057734241036237643,2.2613200815410317,-1.4793689589889483,-0.02244870105783167,0.5001072799357632,0.4631207864661696,-0.5655123126387785,-0.6323596722978864,0.3845223219060368,0.6585141000462197,-0.0668623207042356,0.7710702195998617,0.38446395919889115,0.16464234116119053,-0.9951275614252746
1.1,-1.176387945548666,-2.1643548477191663,0.7399380581690278,-0.2616061357059103,-0.9288360352676038,2.4123240043531657,0.19584246540149355,-0.7713024553363296,1.3644701536271415,0.8843356876890901,-0.11470758483647607,0.9840329244678683,0.2063109886370199,-0.37675025388680927,0.1751751912389655,-0.3737191589504349,0.7381684424574022,-0.056884992518693965,0.7303488694068808,-1.0413892490091703,-0.5653379227437539,-0.6607238076484953,0.3879470477925028,1.011410792907547,0.460322267833745,0.4757456617697549,-1.6612266029665061,1.9322995307560504,-0.2045439904434397,-0.8682236511614171,2.989781694319969,-1.5389792377291038,1.3732535927777278,-1.3940959665889616,0.18010307016392935
1.2,0.36692104561078903,1.2317678511446604,-1.0689878499619023,-0.6052859468179751,1.9429738278088509,0.3909284049297361,-0.7766638594069571,-0.520723730016897,-0.604180932650727,-0.25305766946680847,0.04160429398081556,0.7275310300396219,0.8580351243970223,-2.69603328391907,-0.11148299302573272,0.48476088523353505,0.49286936389542824,0.0007268462004406898,-2.479345237135229,0.6088855461906271,-0.3914139591700859,1.6922000230853802,0.27076792695082624,-0.3901467514719886,0.20028824260492809,0.4035432717469509,0.3781846625451229,-0.21837148090795372,-0.3827450276161359,-0.023889258013329213,0.5009065854561321,1.0573397396925173,-0.7006743063562656,0.8296834350340367,-1.2087748094718316
1.3,-0.5769743996847441,-1.030621391895816,0.17403057247295822,-0.9254176264888323,0.5349430847806121,0.07954211421884153,-0.35304115447604795,0.19733304682180616,-1.0470020156676978,1.0556596636779576,0.260687782657167,-0.7335143145141249,0.4901000856695511,-0.07397057462693475,1.7152647394388636,0.17096296475943354,-0.019387245876193924,1.125351617814821,0.9226189337425504,-0.027740153051967042,0.7364629605087775,-0.3401506246865191,-0.704720525602697,-0.5957614403749261,-0.9566296844904991,0.3714109648461117,0.35667177669608596,-0.6153637374385424,1.398797053536754,1.4613830982996323,-0.06525705976525177,-0.2105193667934262,-0.9731796201418572,1.116360539069534,0.043149891767152566
1.4,-0.4813589607135931,-0.6995874334524409,0.10239710045304466,-0.7158268610557046,0.6145439524743228,0.3458135418517716,-0.14119535520329948,-0.05368099455821645,-1.039859628935629,1.4087003307474202,0.40226811465142753,-0.6509180213536544,0.5312424894637329,-0.22925553412214056,1.3754849378481615,0.12155023769902926,-0.06585664359790531,0.700870893677793,0.794804938163148,0.17739390468609142,0.7442312872516476,-0.29460893355978174,-0.35577376294929774,-0.8524076049158513,-1.0230340175917925,0.3714929615232011,0.15786516096736156,-0.4539149453076708,1.168277033504383,1.3528169946737938,-0.034960618968168794,-0.007175318682666319,-0.4778651122846231,1.0185342603138694,-0.06760607332133095
1.5,-0.43278974147352645,-0.7827154754822034,0.1005413667069363,-0.6665930156128279,0.6391846804666691,0.4993782028863297,-0.12253896633766048,-0.12165293105152533,-1.0637494575043898,1.430654851659708,0.44183889959226574,-0.6685073983630672,0.5878576634821822,-0.16271109849961796,1.3636610621732561,0.07840983527299397,-0.005090175059995424,0.6356911358910329,0.7630669124687036,0.1864610175222126,0.8112429961292151,-0.2723886739708068,-0.3419125735165714,-0.9180544799026772,-1.1243156746190666,0.4727069886130462,0.17474476008210635,-0.5384184585391177,1.1093919557784795,1.2589267878110904,-0.07425462031201968,0.009849998471993561,-0.46560922148248024,1.089657926844027,-0.05286511826039511
1.6,-1.6681131894608725,1.242137242192066,0.4365409881124652,0.3766666794889529,0.3714850491988693,0.34478738395609093,1.2028494303327883,0.8908951153639426,0.06927894149439237,-1.7914353497890927,-0.4865098377066523,0.24453671292447918,0.2698947123110265,-0.17110197429886134,-1.4271442096897453,0.8927493103098699,2.0491574468530165,1.0124304010498093,0.4848855759778051,0.42724665382383165,-1.2962487883328764,-2.3458332416977994,-0.9845029092635468,0.7720103201562798,-0.21153589154730976,0.3174006115043461,0.18702102687863226,1.917946382889125,-0.40550636310783955,1.373729888018741,0.1977333246515132,1.8954726924134724,1.4793828420177568,1.0400704685565678,-1.1757138153001236
1.7,2.1343655357301983,-0.9503165509185784,0.19075504055941714,-0.4390209963935735,1.1089530952395,-0.6029483805226915,1.2863707990398885,-0.5502624216088938,0.6130936020606943,-0.5937030890264082,0.771910193302391,1.0803759557643837,-0.5940063768807501,-1.1966606506518245,3.5426540822302672,1.5029243803746166,0.4064929427408436,-2.6296829322663413,0.22216825563933176,0.4540451860713096,-0.5937288555979847,0.8258146202558144,-0.15029001491455124,-1.612570502578527,2.6186866867385077,0.9311202835209085,-3.283895877956236,-1.322490388604113,1.4964305333634582,-0.5867745105519179,-0.40504364207645344,0.359127093301244,1.4858324534637486,0.4468379887009236,-0.8067111479581448
1.8,1.7842048857391999,-0.8597144042718308,0.10735305527385475,-0.3208498982187234,1.0739370503192853,-0.9268992630738937,1.4612472206189813,-0.6962849663275105,0.5904846042459224,-0.7869393607574056,0.6175753748705431,0.9478049206022497,-0.4282949995480812,-1.2290005752794981,3.4627609067598533,1.5003042200554504,0.34786990530644285,-2.4983237009424246,0.12634139734275796,0.4833982033671481,-0.7021977206784056,0.8067365065168859,-0.08489496821932047,-1.6743092071108616,2.704988131111018,1.0030870334657467,-3.1735131124042875,-1.39417371314792,1.544739580572021,-0.5075334196076677,-0.2935241770508209,0.43340052092202747,1.4534613491555908,0.29584045780823576,-0.6735938705325095
2.0,-0.006688223228779854,-1.4900330767540286,2.4366407832489276,-2.3356779681083313,-1.9027687354719232,-1.137212592562429,0.9946334464820165,-0.09563188750077692,-0.7029706507692671,-0.11283042964479464,-0.4365938325750612,-0.7930844329129009,0.9266913966892529,0.20379018644301658,0.6368612698955761,-1.5595796903798305,-0.6720352928881583,0.7044286316014042,-1.3848941187990478,-1.7598504120233316,0.05211029033807774,0.4598940952659044,-1.0539813402906266,0.922226042118812,-1.6375913678802476,-1.9095125085023266,-0.844127610527385,-2.0955140997236383,1.604996306113643,1.2880428275309317,-2.192394760393979,0.08026518241499357,-0.14452447786371153,-0.12714206891467594,-3.063392311954465
2.2,0.37458872284633277,-1.6636638161211317,0.2246502041794749,1.0351097369614974,-0.16877361355203307,-0.9843622705166963,1.7185220688792362,-1.6591479393733954,0.6618898528061535,0.4598620910880058,0.22640486688515557,-0.013351323553927123,0.233280945781341,0.6526493907269111,0.42436399885792,2.0335008934002015,0.6784718598505356,2.2280909587537443,-1.5840955326411146,0.4761901043575375,1.881308420006957,3.30692373606964,-1.4565440093407052,-1.3608093089985702,1.597944124452511,1.4246249007272054,-0.9779326647785517,1.1970195252009903,-0.186867405303584,-0.16978646365592703,1.864936502247302,1.8045113643278938,0.766530131456784,-0.4117171005059727,-1.4333885370782038
2.4,0.13329460160361792,-1.6414391197740283,0.31613977002503296,0.9706286975936943,-0.10165228307973803,-1.2788126905914807,1.7669219535238696,-1.116722720846024,0.7149912779763574,0.6158194786884115,0.01524056897400425,0.1078847321605446,0.12274061907398458,0.6330717260737848,0.5685776022309755,2.176949429637338,0.6646003356024841,2.3121735895007167,-1.4885159509690018,0.48443610825447336,1.8127653032902578,3.063831981148327,-1.4678828806974409,-1.2188980284684232,1.5749833310369126,0.9284133638642459,-1.0804612863856897,1.2134601712464084,-0.18076140802147228,-0.03441087468083849,1.7280165464548465,1.7083830301952652,0.8431126279643455,-0.35239567642566255,-1.3526771852543438
2.6,-0.3191671967865056,-1.6084178171292303,0.34399308413480756,0.8723100297191355,-0.0716279751152495,-1.7755860735675835,1.5879282040803766,-1.238694208203383,0.6379969086906757,0.5639032428999953,0.3031803958733072,0.0710027169765544,0.09570350499657578,0.6138104088229797,0.6419712540772925,1.7766553507981409,0.6623979577439386,2.5142733825436547,-1.4729614367171293,0.3832178492528552,1.6527967569292383,2.946804280620504,-1.4626528543419712,-1.2747031175223795,1.524655356487389,0.9655723689439178,-1.1147525549037243,1.1182789885886988,-0.10622679792411834,-0.021301073388171506,1.6897124262397518,1.6548171091550266,1.0211450504912432,-0.32653752514301004,-1.2898268805965076
2.8,0.8236083644735409,0.640924154850414,0.46110857096416685,-0.7620470691354185,-0.33585258144730196,0.8950026813718952,1.9779930049131207,1.2086897786774378,-2.6367048746244643,1.9454152116923336,0.37161355871648544,-0.852274383113038,0.3175830730402183,0.9627989470949476,-0.39361403800487077,-0.17339205435317548,2.3145640042842883,0.010660355976465063,0.8791535331025383,-0.9705270099603938,0.7154718765488076,0.547172853012302,-0.818814449803709,-0.22706573814968029,-0.7918618506019797,1.0958767869293708,-0.7595626353541278,-1.111513712849744,-0.7292184047225925,0.0475894605743081,0.04072658289148971,-0.2720474463787877,-0.17834680317677337,-1.8694928657590257,1.2136856134745642
3.0,0.9017929098986363,0.6688022910739302,0.4829999842841391,-0.794168504483266,-0.3003192675038475,0.8980427503030365,2.030113945001536,1.2072696758255446,-2.681108012385585,2.0305282263792703,0.37704812745648186,-0.8374700743573871,0.29504410945645393,0.9883477333101616,-0.42657517449791166,-0.20653171921314772,2.276739484179376,-0.0036917361876312913,0.8861942486031285,-0.9040869831448032,0.6715107475311021,0.5248917860193046,-0.7859893924862925,-0.2703528152614441,-0.7756361066016478,1.1724527198029624,-0.7589983649267987,-1.0763065614437304,-0.6747460468280613,0.1124967103945864,0.033189824432715884,-0.24644364549011985,-0.20952444915173654,-1.9717317858192176,1.1272163714438201
""",
}

In [ ]:
# Extra #14 (2/3): ansatz de 5 qubits. Capas de ry en todos los qubits + cadena de cx en línea
# (RealAmplitudes), arrancando del HF reducido. El H reducido es real, así que alcanzan amplitudes reales.
# Búsqueda de parámetros con statevector (permitido), en el espacio de 2⁵ = 32 amplitudes, con gradiente adjunto.
import io
import time

RUN_ANSATZ5 = False  # True: repetir la búsqueda de θ (~30 s). False: usar los θ* guardados en PARAMS5_CACHE (celda de arriba)

from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize

LAYERS_5Q = 6
_DIM5 = 2 ** N_TAPERED


def _cx_permutation(c: int, t: int) -> np.ndarray:
    return np.array([b ^ (1 << t) if (b >> c) & 1 else b for b in range(_DIM5)])


class RealAmplitudes5:
    """Simulador numpy del ansatz de 5 qubits con gradiente adjunto exacto."""

    def __init__(self, layers: int):
        self.layers, self.n = layers, N_TAPERED * (layers + 1)
        self.hf = np.zeros(_DIM5)
        self.hf[HF_TAPERED] = 1.0
        self.cx = [_cx_permutation(q, q + 1) for q in range(N_TAPERED - 1)]
        self.K = []  # generador de ry: −iY/2 en el qubit q (matriz real)
        for q in range(N_TAPERED):
            K = np.zeros((_DIM5, _DIM5))
            for b in range(_DIM5):
                K[b ^ (1 << q), b] = 0.5 if not (b >> q) & 1 else -0.5
            self.K.append(K)

    def _ops(self):
        k = 0
        for layer in range(self.layers + 1):
            for q in range(N_TAPERED):
                yield q, k
                k += 1
            if layer < self.layers:
                yield None, None  # cadena de cx

    def _ry(self, q, t, v):
        return np.cos(t / 2) * v + 2 * np.sin(t / 2) * (self.K[q] @ v)

    def _chain(self, v, inverse=False):
        for P in (reversed(self.cx) if inverse else self.cx):  # cada cx es una permutación involutiva
            w = np.empty_like(v)
            w[P] = v
            v = w
        return v

    def state(self, x):
        v = self.hf
        for q, k in self._ops():
            v = self._chain(v) if q is None else self._ry(q, x[k], v)
        return v

    def energy_grad(self, x, H):
        ops = list(self._ops())
        v = self.state(x)
        lam = H @ v
        e, g = v @ lam, np.zeros(self.n)
        for q, k in reversed(ops):
            if q is None:
                v, lam = self._chain(v, True), self._chain(lam, True)
            else:
                g[k] = 2 * lam @ (self.K[q] @ v)
                v, lam = self._ry(q, -x[k], v), self._ry(q, -x[k], lam)
        return e, g

    def circuit(self) -> QuantumCircuit:
        th = ParameterVector("φ", self.n)
        qc = QuantumCircuit(N_TAPERED, name=f"RealAmplitudes-5q-{self.layers}")
        for i in range(N_TAPERED):
            if (HF_TAPERED >> i) & 1:
                qc.x(i)
        for q, k in self._ops():
            if q is None:
                for a in range(N_TAPERED - 1):
                    qc.cx(a, a + 1)
            else:
                qc.ry(th[k], q)
        return qc


ansatz5 = RealAmplitudes5(LAYERS_5Q)
ansatz5_circuit = ansatz5.circuit()

# El circuito de Qiskit y el simulador numpy dan el mismo estado; el gradiente adjunto = diferencias finitas.
_x = np.random.default_rng(0).normal(size=ansatz5.n)
assert np.allclose(Statevector(ansatz5_circuit.assign_parameters(_x)).data, ansatz5.state(_x)), "circuito ≠ simulador"
_H = H_TAPERED[STRETCHED][0].to_matrix().real
_fd = [(ansatz5.energy_grad(_x + h, _H)[0] - ansatz5.energy_grad(_x - h, _H)[0]) / 2e-6 for h in 1e-6 * np.eye(ansatz5.n)]
assert np.allclose(ansatz5.energy_grad(_x, _H)[1], _fd, atol=1e-6), "gradiente adjunto ≠ diferencias finitas"

_t = time.perf_counter()
H5_MAT = {d: H_TAPERED[d][0].to_matrix().real for d in H_TAPERED}
if RUN_ANSATZ5:
    # Warm start en las dos direcciones + 3 arranques al azar por punto; se queda el mejor.
    _rng = np.random.default_rng(14)
    best5 = {}
    for order in (list(map(float, D)), list(map(float, D))[::-1]):
        x = _rng.normal(scale=0.7, size=ansatz5.n)
        for d in order:
            starts = [x, x + _rng.normal(scale=0.3, size=x.size)] + [_rng.normal(scale=0.7, size=x.size) for _ in range(2)]
            r = min((minimize(ansatz5.energy_grad, s, args=(H5_MAT[d],), jac=True, method="BFGS", options={"gtol": 1e-7})
                     for s in starts), key=lambda r: r.fun)
            x = r.x
            if d not in best5 or r.fun < best5[d].fun:
                best5[d] = r
    params5 = {d: r.x for d, r in best5.items()}
    os.makedirs("results", exist_ok=True)
    pd.DataFrame([{"d": d, **{f"phi_{k}": v for k, v in enumerate(params5[d])}} for d in params5]).to_csv(
        "results/params5.csv", index=False, float_format="%.17g")
else:
    _p5 = pd.read_csv(io.StringIO(PARAMS5_CACHE["params5"]), float_precision="round_trip")
    params5 = {float(r.d): np.array(r.iloc[1:], dtype=float) for _, r in _p5.iterrows()}
    assert sorted(params5) == sorted(map(float, D)) and all(len(v) == ansatz5.n for v in params5.values()), \
        "PARAMS5_CACHE es de otra grilla u otro ansatz: correr con RUN_ANSATZ5 = True"
    print(f"θ* guardados ({PARAMS5_CACHE['origen']}). Para repetir la búsqueda: RUN_ANSATZ5 = True.")

rows = []
for d in map(float, D):
    op5, e_nuc = H_TAPERED[d]
    w, V = np.linalg.eigh(H5_MAT[d])
    psi = ansatz5.state(params5[d])
    rows.append({"d": d, "err_5q_mHa": (ansatz5.energy_grad(params5[d], H5_MAT[d])[0] + e_nuc - refs.set_index("d").e_fci[d]) * 1e3,
                 "fidelidad": abs(V[:, 0] @ psi) ** 2, "brecha_5q_mHa": (w[1] - w[0]) * 1e3})
ansatz5_check = pd.DataFrame(rows)
print(f"{ansatz5_circuit.name}: {ansatz5.n} parámetros, {cz_count(ansatz5_circuit)} cz (transpilado en línea); "
      f"búsqueda en {time.perf_counter() - _t:.0f} s" + ("" if RUN_ANSATZ5 else " (θ* guardados)"))
print(f"Error vs. FCI (statevector, diagnóstico): máx. {ansatz5_check.err_5q_mHa.abs().max():.2e} mHa en toda la grilla; "
      f"fidelidad con el fundamental del H reducido ≥ {ansatz5_check.fidelidad.min():.6f}")

In [ ]:
# Extra #14 (3/3): 8 vs. 5 qubits. Términos, grupos QWC, cz, profundidad, error vs. FCI y, con el ruido del
# enunciado (el de notebooks/ruido.py, copiado acá para que el notebook sea autocontenido), la energía de
# EstimatorV2 de qiskit_aer con matriz densidad: valor esperado exacto del estado ruidoso, sin ruido de shots.
from itertools import product

from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit_aer.primitives import EstimatorV2 as AerEstimator


def _pauli_error(p: float, n: int):
    others = ["".join(l) for l in product("IXYZ", repeat=n) if set(l) != {"I"}]
    return pauli_error([("I" * n, 1 - p)] + [(l, p / len(others)) for l in others])


NOISE_14 = NoiseModel(basis_gates=BASIS_GATES)
NOISE_14.add_all_qubit_quantum_error(_pauli_error(1e-3, 2), ["cz"])            # 0,1 % por cz
NOISE_14.add_all_qubit_quantum_error(_pauli_error(5e-5, 1), ["sx", "x", "rz"])  # 0,005 % por compuerta de 1 qubit
_estimators = {False: AerEstimator(), True: AerEstimator(options={"backend_options": {"method": "density_matrix",
                                                                                     "noise_model": NOISE_14}})}

t8, t5 = transpile_line(ansatz), transpile_line(ansatz5_circuit)
NOISE_D = (STRETCHED, 2.4, DISSOCIATED)
noise_rows = []
for d in NOISE_D:
    e_fci = refs.set_index("d").e_fci[d]
    for label, tqc, op, e_nuc, theta in (("8 qubits (#4)", t8, *qubit_hamiltonian(d), params[d]),
                                         ("5 qubits (#14)", t5, *H_TAPERED[d], params5[d])):
        obs = op.apply_layout(tqc.layout)
        row = {"d": d, "versión": label}
        for noisy in (False, True):
            ev = float(_estimators[noisy].run([(tqc, obs, theta)]).result()[0].data.evs)
            row["err_con_ruido_mHa" if noisy else "err_sin_ruido_mHa"] = (ev + e_nuc - e_fci) * 1e3
        noise_rows.append(row)
noise_14 = pd.DataFrame(noise_rows)

stretched_8 = opt[opt.d >= STRETCHED].err_mHa.abs().max()
stretched_5 = ansatz5_check[ansatz5_check.d >= STRETCHED].err_5q_mHa.abs().max()
comparison_14 = pd.DataFrame({
    "8 qubits (#4)": [taper_check.terms_8q.max(), taper_check.qwc_8q.max(), ansatz.num_parameters,
                      t8.count_ops().get("cz", 0), t8.depth(), opt.err_mHa.abs().max(), stretched_8],
    "5 qubits (#14)": [taper_check.terms_5q.max(), taper_check.qwc_5q.max(), ansatz5.n,
                       t5.count_ops().get("cz", 0), t5.depth(), ansatz5_check.err_5q_mHa.abs().max(), stretched_5],
}, index=["términos de Pauli", "grupos QWC", "parámetros", "cz (en línea)", "profundidad",
          "máx. |error| sv, toda la grilla (mHa)", "máx. |error| sv, d ≥ 1,8 Å (mHa)"])
print(comparison_14.to_string(float_format=lambda v: f"{v:.3g}"))
print("\nCon el ruido del enunciado (EstimatorV2 de qiskit_aer, mismos θ*):")
print(noise_14.pivot(index="d", columns="versión", values=["err_sin_ruido_mHa", "err_con_ruido_mHa"])
      .to_string(float_format=lambda v: f"{v:.2f}"))

#### Decisiones, resultados y limitaciones (#14)

**Las simetrías.** `Z2Symmetries.find_z2_symmetries` encuentra las mismas 3 en toda la grilla: `ZZZZIIII` (paridad de N_β), `ZIZIZIZI` (reflexión de la cadena: los orbitales 1 y 3 son impares) y `ZIZIIZIZ` (combinada con la paridad de N_α). En el estado HF las tres valen +1, y ese es el sector que usamos. Se sacan los qubits 0, 1 y 4, y el HF queda como el estado de la base |00100⟩ (solo hacen falta compuertas `x`).

**Validación.** En las 19 distancias, el mínimo del Hamiltoniano reducido coincide con FCI (≤ 2e-14 Ha) y ⟨HF|H|HF⟩ con E_HF de PySCF. No hace falta restringir N: el mínimo del espacio reducido (32 estados, con varias N de la misma paridad) ya es el FCI neutro.

**Resultados** (los números salen de la celda de arriba):

| | 8 qubits (#4) | 5 qubits (#14) |
|---|---:|---:|
| Términos de Pauli | 185 | 132 |
| Grupos QWC | 46 | 34 |
| Parámetros | 19 | 35 |
| `cz` en línea | 465 | **24** |
| Profundidad | 905 | 65 |
| Máx. error vs. FCI, statevector | 0,003 mHa | < 1e-10 mHa |
| Error con ruido, 1,8 / 2,4 / 3,0 Å | 320 / 300 / 291 mHa | **26–28 / 24–25 / 24 mHa** |

El ansatz de 5 qubits es un `RealAmplitudes` de 6 capas: `ry` en cada qubit y una cadena de `cx` en línea. Es exacto en toda la curva con 19 veces menos `cz`. Con el ruido del enunciado (0,1 % por `cz`), el error baja de ~300 a 24–28 mHa (varía un poco entre corridas, porque cambian los θ* encontrados). A 2,4 Å, con ruido, queda por debajo del error de CCSD (24–25 contra 31 mHa), algo que la versión de 8 qubits no logra en ningún punto.

**Por qué no el ansatz de #4 reducido.** El ansatz elegido usa a propósito simples entre orbitales de distinta paridad de reflexión (0↔1, 0↔3, 1↔2, 2↔3), con θ* grandes (±π/4 y más), y las compensa entre sí. Esas compuertas sacan el estado del sector, así que no se pueden implementar en 5 qubits.

**Qué no funcionó.**
- Quedarse con los 12 generadores de #4 que respetan las simetrías: 285 mHa de error.
- 1 a 3 capas del pool simétrico (6 dobles de pares + simples 0↔2 y 1↔3), compiladas como `PauliEvolutionGate` de los generadores reducidos: 54–285 mHa de error y 314–930 `cz`. Al reducirlos, los generadores pasan a ser cadenas de Pauli largas y caras.
- `ry` + cadena de `cz` (en vez de `cx`), de 4 a 10 capas: siempre trabado en 55 mHa. No es mala suerte del optimizador: maximizando la fidelidad con un estado real al azar no pasa de 0,68, así que ese circuito no llega a todos los estados reales. Con `cx` llega (fidelidad 1,0000 con 6 capas). Con 4 capas de `cx` el error máximo es 0,9 mHa; por eso usamos 6.

**Limitaciones.**
- El ansatz de 5 qubits no conserva N ni S_z: no hay cantidades físicas que vigilar en el circuito. La confiabilidad sale del principio variacional (el mínimo del espacio reducido es FCI) y de la brecha con el primer excitado del espacio reducido, que es de 0,95 mHa a 3,0 Å. Ahí el triplete casi degenerado del sector completo (0,46 mHa) queda afuera por simetría.
- La búsqueda de parámetros es con statevector (32 amplitudes, gradiente adjunto, 4 arranques por punto en las dos direcciones; ~30 s). En hardware haría falta un optimizador que tolere el ruido de shots.
- La energía con ruido es el valor esperado exacto de `EstimatorV2` con matriz densidad, sin ruido de shots. Para la barra de error de verdad, hay que medir con `SamplerV2` como en #6.

### Ruido (ruido.py) (#11)

Extra de #11. Repetimos la medición de #6 en cuatro distancias representativas (0,9 / 1,8 / 2,4 / 3,0 Å) sobre un simulador
**con ruido**: error despolarizante de 0,1 % por `cz` y 0,005 % por compuerta de un qubit (`sx`, `x`, `rz`), conectividad en
línea (el modelo común del enunciado, `ruido.py`). Medimos con la misma `measure_energy` de #6 (`SamplerV2` de `qiskit_aer`,
shots reales), con los mismos $\theta^*$ y el circuito de `transpile_line`, y comparamos **con ruido vs. sin ruido vs. FCI**.
Como el ansatz exacto tiene cientos de `cz`, agregamos **versiones cortas** (pocos operadores) para ver el compromiso
entre expresividad (sesgo del ansatz) y ruido (crece con los `cz`).

`ruido.py` todavía no está en el repo: si existe `notebooks/ruido.py` (o `ruido.py` junto al notebook) se usa; si no, un
**stand-in** con la especificación del enunciado, que se apaga solo cuando llegue el archivo oficial.

**Resultados guardados.** La medición con ruido (11.3, matriz densidad) tarda ~1,5 min, así que por defecto usa la tabla
guardada en `RUIDO_CACHE` (de una corrida de esa misma celda con `RUN_RUIDO = True`, `SHOTS_SCALE = 1`); las versiones del
ansatz (11.2) y la tabla, los gráficos y los chequeos de 11.4 se recalculan igual. Para volver a medir: `RUN_RUIDO = True`
en 11.3 (también guarda `results/ruido.csv`).

In [ ]:
%%writefile ruido.py
"""Modelo de ruido común de la QuantumJam 2026 (extra "Simulador con ruido").

Lo que pide el enunciado (igual para todos los equipos):
  - compuertas cz, sx, x y rz;
  - qubits conectados en línea (0-1-2-...-7);
  - una probabilidad de error de 0,1 % por cz y de 0,005 % por compuerta de un qubit.

Cómo lo interpretamos: después de cada compuerta, con probabilidad p se aplica un error de Pauli
elegido al azar entre los no triviales (15 para cz, 3 para un qubit). La fidelidad de proceso de
cada compuerta queda en 1 − p. Las rz también llevan error, porque el enunciado no las excluye.
No hay error de lectura: el enunciado no lo menciona.

Sigue el estilo de los notebooks de los workshops (github.com/Computer-Society-ITBA/quantumjam-workshops):
transpilar con generate_preset_pass_manager, mapear el observable con apply_layout y medir con las
primitivas de qiskit_aer. Ejemplo:

    from ruido import transpilar, estimator_ruidoso, sampler_ruidoso, contar_cz

    transpilado = transpilar(circuito)                       # base cz/sx/x/rz, qubits en línea
    H_mapeado = H.apply_layout(transpilado.layout)
    energia = estimator_ruidoso().run([(transpilado, H_mapeado, parametros)]).result()[0].data.evs

    transpilado.measure_all()
    cuentas = sampler_ruidoso(semilla=5).run([(transpilado, parametros)], shots=10_000).result()[0].data.meas.get_counts()

estimator_ruidoso() se puede pasar directo como `estimator=` a la función vqe(...) de los workshops.

Importante: transpilar SIEMPRE con transpilar(). Una compuerta que no esté en COMPUERTAS_BASE
(h, sdg, cx, ...) se simularía sin ruido.
"""
from __future__ import annotations

from itertools import product

from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit_aer.primitives import EstimatorV2 as AerEstimator, SamplerV2 as AerSampler

COMPUERTAS_BASE = ["cz", "sx", "x", "rz"]
ERROR_CZ = 1e-3   # 0,1 % por cz
ERROR_1Q = 5e-5   # 0,005 % por compuerta de un qubit
METODO = "density_matrix"  # con 8 qubits es mucho más rápido que statevector, que corre una trayectoria por shot


def _error_de_pauli(p: float, n_qubits: int):
    """Con probabilidad p, un Pauli no trivial al azar sobre n_qubits; con probabilidad 1 − p, nada."""
    paulis = ["".join(letras) for letras in product("IXYZ", repeat=n_qubits) if set(letras) != {"I"}]
    return pauli_error([("I" * n_qubits, 1 - p)] + [(pauli, p / len(paulis)) for pauli in paulis])


def modelo_de_ruido(error_cz: float = ERROR_CZ, error_1q: float = ERROR_1Q) -> NoiseModel:
    """NoiseModel de qiskit_aer con el ruido del enunciado."""
    modelo = NoiseModel(basis_gates=COMPUERTAS_BASE)
    modelo.add_all_qubit_quantum_error(_error_de_pauli(error_cz, 2), ["cz"])
    modelo.add_all_qubit_quantum_error(_error_de_pauli(error_1q, 1), ["sx", "x", "rz"])
    return modelo


def conectividad(n_qubits: int) -> CouplingMap:
    """Qubits en línea: 0-1-2-...-(n-1), en las dos direcciones."""
    return CouplingMap.from_line(n_qubits)


def transpilar(circuito: QuantumCircuit, optimization_level: int = 3, seed_transpiler: int = 0) -> QuantumCircuit:
    """Adapta el circuito a las compuertas y la conectividad del modelo (igual que transpile_line de la celda 0.1)."""
    pm = generate_preset_pass_manager(optimization_level=optimization_level, basis_gates=COMPUERTAS_BASE,
                                      coupling_map=conectividad(circuito.num_qubits), seed_transpiler=seed_transpiler)
    return pm.run(circuito)


def contar_cz(circuito: QuantumCircuit) -> int:
    """Cuenta las cz que quedan al transpilar el circuito."""
    return transpilar(circuito).count_ops().get("cz", 0)


def opciones_backend(metodo: str = METODO) -> dict:
    """backend_options para las primitivas de qiskit_aer."""
    return {"method": metodo, "noise_model": modelo_de_ruido()}


def backend_ruidoso(metodo: str = METODO) -> AerSimulator:
    """AerSimulator con el ruido, para usar con backend.run(transpilado, shots=...)."""
    return AerSimulator(**opciones_backend(metodo))


def sampler_ruidoso(semilla: int | None = None, metodo: str = METODO) -> AerSampler:
    """SamplerV2 de qiskit_aer con el ruido.

    Ojo con la semilla: con una semilla fija, Aer repite semillas entre los pubs de una misma llamada
    a run(), y las mediciones de distintos pubs quedan correlacionadas. Para medir varios grupos de
    términos, usar un sampler (o una semilla) distinto por pub.
    """
    return AerSampler(seed=semilla, options={"backend_options": opciones_backend(metodo)})


def estimator_ruidoso(precision: float = 0.0, semilla: int | None = None, metodo: str = METODO) -> AerEstimator:
    """EstimatorV2 de qiskit_aer con el ruido.

    Devuelve el valor esperado exacto del estado ruidoso. Con precision > 0, Aer le suma ruido gaussiano
    de ese ancho (no muestrea shots): para la barra de error de verdad, medir con sampler_ruidoso().
    """
    opciones = {"backend_options": opciones_backend(metodo), "default_precision": precision}
    if semilla is not None:
        opciones["run_options"] = {"seed": semilla}
    return AerEstimator(options=opciones)


# Nombres en inglés (los documentados primero en el issue #3).
BASIS_GATES, P_CZ, P_1Q = COMPUERTAS_BASE, ERROR_CZ, ERROR_1Q
noise_model, coupling_map, transpile_noisy = modelo_de_ruido, conectividad, transpilar


def noisy_sampler(seed: int | None = None, method: str = METODO) -> AerSampler:
    return sampler_ruidoso(seed, method)

In [ ]:
# --- 11.1 Modelo de ruido: ruido.py oficial o stand-in con la especificación del enunciado ---------------------
import importlib.util
from contextlib import contextmanager
from qiskit_aer.noise import NoiseModel, depolarizing_error

P_CZ, P_1Q = 1e-3, 5e-5          # especificación del enunciado
_N_U_DECOMP = 5                  # u(θ,φ,λ) = rz sx rz sx rz: la rotación de base de la medición cuesta 5 compuertas de 1 qubit


def _standin_noise_model():
    """STAND-IN de ruido.py: despolarizante 1e-3 en cz y 5e-5 en sx/x/rz. El `u` de rotación de base de la medición no
    está en la base del modelo; se le asigna el error de su descomposición (5 compuertas de 1 qubit)."""
    nm = NoiseModel(basis_gates=["cz", "sx", "x", "rz", "u"])
    nm.add_all_qubit_quantum_error(depolarizing_error(P_CZ, 2), ["cz"])
    nm.add_all_qubit_quantum_error(depolarizing_error(P_1Q, 1), ["sx", "x", "rz"])
    nm.add_all_qubit_quantum_error(depolarizing_error(1 - (1 - P_1Q) ** _N_U_DECOMP, 1), ["u"])
    return nm


def _load_ruido_py():
    """Carga ruido.py (#3: `modelo_de_ruido()`, antes `noise_model()`), escrito por la celda %%writefile de arriba; devuelve un NoiseModel o None."""
    for path in ("notebooks/ruido.py", "ruido.py"):
        if os.path.exists(path):
            spec = importlib.util.spec_from_file_location("ruido", path)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            for name in ("modelo_de_ruido", "get_noise_model", "build_noise_model", "noise_model", "make_noise_model", "NOISE_MODEL"):
                obj = getattr(mod, name, None)
                if obj is not None:
                    return obj() if callable(obj) else obj
    return None


NOISE_MODEL = _load_ruido_py()
if NOISE_MODEL is not None and "u" not in NOISE_MODEL.noise_instructions:
    # ruido.py no incluye `u`, que usamos para rotar la base de medición; sin esto se simularía sin ruido.
    # Le damos el error de su descomposición (5 compuertas de un qubit), igual que el stand-in.
    NOISE_MODEL.add_basis_gates(["u"])
    NOISE_MODEL.add_all_qubit_quantum_error(depolarizing_error(1 - (1 - P_1Q) ** _N_U_DECOMP, 1), ["u"])
if NOISE_MODEL is None:
    _notice_standin("ruido", "ruido.py (#3) no está: uso _standin_noise_model (despolarizante 1e-3 cz, 5e-5 sx/x/rz).")
    NOISE_MODEL = _standin_noise_model()
print("Modelo de ruido:", "stand-in del enunciado" if "ruido" in STANDIN_USED else "ruido.py oficial",
      "| compuertas con error:", sorted(NOISE_MODEL.noise_instructions))


@contextmanager
def con_ruido(nm=None):
    """Dentro del bloque, el SamplerV2 que usa measure_energy (de #6) simula con el modelo de ruido `nm`: se cambian las
    opciones de backend de Aer (`_AER_OPTIONS` de #6). Matriz densidad: el circuito se evoluciona una vez por binding y los
    shots se muestrean del estado final."""
    nm = NOISE_MODEL if nm is None else nm
    g = globals()
    original = g["_AER_OPTIONS"]
    g["_AER_OPTIONS"] = {**original, "method": "density_matrix", "noise_model": nm}
    try:
        yield
    finally:
        g["_AER_OPTIONS"] = original


@contextmanager
def _con_generadores(subset):
    """Cambia temporalmente los generadores del stand-in de #4 (para armar/ajustar una versión corta del ansatz)."""
    g = globals()
    original = g["_STANDIN_GENS"]
    g["_STANDIN_GENS"] = [original[i] for i in subset]
    try:
        yield
    finally:
        g["_STANDIN_GENS"] = original

In [ ]:
# --- 11.2 Versiones del ansatz: completo + cortas (pocos operadores), todas con transpile_line -------------------
DIST_RUIDO = [0.9, 1.8, 2.4, 3.0]
K_CORTAS = [2, 12]                      # operadores de la versión corta (el completo se agrega aparte)
SEED_RUIDO = SEED + 77_000
_t = time.time()

if "build_ansatz" in STANDIN_USED:
    # Ranking de operadores por tamaño del ángulo óptimo (máximo sobre la curva); las cortas conservan los más importantes
    # y se REAJUSTAN por statevector (buscar parámetros vale de cualquier forma).
    _rank = list(np.argsort(-np.max([np.abs(THETA[d]) for d in D], axis=0)))
    VERSIONES = {}
    for k in K_CORTAS + [len(_rank)]:
        sub = sorted(int(i) for i in _rank[:k])
        with _con_generadores(sub):
            qc_k = _standin_build_ansatz()
            if k < len(_rank):
                th_k, _ = _standin_fit_uccsd({d: HAM[d][0] for d in D}, {d: HAM[d][1] for d in D})
            else:
                th_k = THETA
        VERSIONES[k] = (qc_k, th_k)
else:
    # Ansatz real de #4: versión corta = los k operadores de mayor |θ*| (máximo sobre la curva) con el resto en 0 (el
    # transpilador elimina las rotaciones nulas), y los k ángulos REAJUSTADOS por statevector en el sector N=4, Sz=0
    # (`sector_ansatz` y `H_SECTOR` de la sección 4; buscar parámetros vale de cualquier forma).
    from scipy.optimize import minimize as _minimize
    _n = ansatz.num_parameters
    _rank = list(np.argsort(-np.max([np.abs(THETA[d]) for d in D], axis=0)))

    def _theta_corto(sub, d):
        Hs = H_SECTOR[float(d)][0]

        def fg(y):
            x = np.zeros(_n)
            x[sub] = y
            e, g = sector_ansatz.energy_grad(x, Hs)
            return e, g[sub]

        best = min((_minimize(fg, y0, jac=True, method="BFGS", options={"gtol": 1e-8}) for y0 in (THETA[d][sub], np.zeros(len(sub)))),
                   key=lambda r: r.fun)
        x = np.zeros(_n)
        x[sub] = np.round(best.x, 6)
        return x

    VERSIONES = {}
    for k in K_CORTAS + [_n]:
        sub = sorted(int(i) for i in _rank[:k])
        VERSIONES[k] = (ansatz, THETA if k == _n else {d: _theta_corto(sub, d) for d in D})

def _observable_fisico(circ, op):
    """transpile_line elige un layout y puede rutear con swaps: el qubit lógico i termina en el físico f[i]. Para medir el MISMO
    observable sobre el circuito transpilado se lleva cada Pauli del qubit lógico i al físico f[i]."""
    f = circ.layout.final_index_layout(filter_ancillas=True) if circ.layout is not None else list(range(circ.num_qubits))
    return op.apply_layout(list(f), num_qubits=circ.num_qubits)


CIRC_LINEA, CZ_V = {}, {}
for k, (qc_k, th_k) in VERSIONES.items():
    for d in DIST_RUIDO:
        CIRC_LINEA[(k, d)] = transpile_line(qc_k.assign_parameters(th_k[d]))   # cz, sx, x, rz sobre línea
        if k == max(VERSIONES):   # control: el circuito de línea + observable permutado reproduce la energía del ansatz de #6
            _e = float(Statevector(CIRC_LINEA[(k, d)]).expectation_value(_observable_fisico(CIRC_LINEA[(k, d)], HAM[d][0])).real) + HAM[d][1]
            assert abs(_e - E_SV[d]) < 1e-6, f"layout de transpile_line mal manejado en d={d}: {_e} vs {E_SV[d]}"
    CZ_V[k] = CIRC_LINEA[(k, DIST_RUIDO[0])].count_ops().get("cz", 0)
print(f"Versiones del ansatz transpiladas en {time.time() - _t:.1f} s:",
      {f"{k} op": f"{CZ_V[k]} cz" for k in VERSIONES})

In [ ]:
# Resultados guardados de la medición con ruido (#11, celda 11.3). Volver a medirlos tarda ~1,5 min, así que la celda
# siguiente los usa por defecto (RUN_RUIDO = False). Salieron de esa misma celda con True;
# para regenerarlos: poner True, correr y pegar acá results/ruido.csv.
RUIDO_CACHE = {
    "origen": "corrida completa del 2026-11-21 con RUN_* = True, SHOTS_SCALE = 1, qiskit 2.5.2, qiskit-aer 0.17.2, pyscf 2.14.0",
    "ruido": """\
ops,d,cz,n1q,E_sin,s_sin,E_con,s_con,shots,E_fci,E_ccsd,E_sv
2,0.9,104,338,-2.122906435,0.00283598,-1.976134912,0.00293629,822650,-2.180316614,-2.180309411,-2.124259739
2,1.8,104,341,-1.662410722,0.0030292,-1.609659165,0.00311309,567000,-1.924430638,-1.933538276,-1.666867161
2,2.4,104,341,-1.439436872,0.0031657,-1.394161309,0.00313249,562000,-1.874651582,-1.905211588,-1.43645854
2,3,104,341,-1.312309559,0.00328164,-1.28301788,0.00318316,589000,-1.867291372,-1.890304373,-1.313311786
12,0.9,300,998,-2.175963998,0.00299345,-1.800392777,0.00311993,922008,-2.180316614,-2.180309411,-2.177640066
12,1.8,300,990,-1.914318764,0.00296079,-1.689591226,0.00302523,672000,-1.924430638,-1.933538276,-1.919449131
12,2.4,300,981,-1.867746599,0.00308392,-1.661276203,0.00304273,577000,-1.874651582,-1.905211588,-1.862605334
12,3,300,987,-1.868414383,0.00307746,-1.657348449,0.00310407,619000,-1.867291372,-1.890304373,-1.866048432
19,0.9,389,1283,-2.182796969,0.00293632,-1.673718945,0.00290372,1052363,-2.180316614,-2.180309411,-2.180316614
19,1.8,389,1261,-1.922952405,0.00298373,-1.642787792,0.00305294,622000,-1.924430638,-1.933538276,-1.924430638
19,2.4,389,1373,-1.876881234,0.0030242,-1.600982997,0.00321055,527000,-1.874651582,-1.905211588,-1.874650564
19,3,389,1357,-1.872002945,0.00304259,-1.605499851,0.00317708,559000,-1.867291372,-1.890304373,-1.867289157
""",
}

In [ ]:
# --- 11.3 Medición: sin ruido y con ruido (SamplerV2 de qiskit_aer), mismos θ*, mismo circuito ---------------------
import io

RUN_RUIDO = False  # True: volver a medir (~1,5 min). False: usar la tabla guardada en RUIDO_CACHE (celda de arriba)
SIGMA_RUIDO = 3e-3 / np.sqrt(SHOTS_SCALE)   # el ruido sesga mucho más que 3 mHa: no hace falta más precisión
PILOT_RUIDO = _scaled(PILOT_SHOTS_PER_GROUP, floor=200)
# Con ruido cada binding (grupo QWC) es una evolución completa de la matriz densidad: lo caro son los bindings, no los shots.
# Un piso alto de shots por binding deja ~1 binding por grupo (en vez de ~4) y acelera ~3x sin cambiar σ.
MIN_SHOTS_BINDING = 5_000
if RUN_RUIDO:
    _t = time.time()
    _rows = []
    for k in VERSIONES:
        for j, d in enumerate(DIST_RUIDO):
            circ = CIRC_LINEA[(k, d)]
            op, e_nuc = _observable_fisico(circ, HAM[d][0]), HAM[d][1]
            seed = SEED_RUIDO + 1000 * k + j
            m0 = measure_energy(circ, op, e_nuc, target_sigma=SIGMA_RUIDO, seed=seed, pilot_shots_per_group=PILOT_RUIDO, min_group_shots=MIN_SHOTS_BINDING)
            with con_ruido():
                m1 = measure_energy(circ, op, e_nuc, target_sigma=SIGMA_RUIDO, seed=seed, pilot_shots_per_group=PILOT_RUIDO, min_group_shots=MIN_SHOTS_BINDING)
            _rows.append(dict(ops=k, d=d, cz=CZ_V[k], n1q=sum(v for g_, v in circ.count_ops().items() if g_ != "cz"),
                              E_sin=m0["energy"], s_sin=m0["sigma"], E_con=m1["energy"], s_con=m1["sigma"],
                              shots=m0["shots"] + m1["shots"], E_fci=REF[d]["e_fci"], E_ccsd=REF[d]["e_ccsd"],
                              E_sv=float(Statevector(circ).expectation_value(op).real) + e_nuc))
    RUIDO = pd.DataFrame(_rows)
    os.makedirs("results", exist_ok=True)
    RUIDO.to_csv("results/ruido.csv", index=False)
    print(f"{len(RUIDO)} puntos (cada uno sin y con ruido) en {time.time() - _t:.1f} s; shots totales {RUIDO.shots.sum():,}")
else:
    RUIDO = pd.read_csv(io.StringIO(RUIDO_CACHE["ruido"]))
    print(f"Resultados guardados ({RUIDO_CACHE['origen']}). Para volver a medir: RUN_RUIDO = True.")
    if not RUIDO.ops.map(CZ_V).eq(RUIDO.cz).all():
        print("AVISO: los cz de los circuitos de 11.2 no coinciden con los guardados: el caché es de otra versión del "
              "ansatz; correr con RUN_RUIDO = True.")

In [ ]:
# --- 11.4 Tabla: ruido vs. sin ruido vs. FCI (y vs. CCSD) ------------------------------------------------------------
RUIDO["err_sin_mHa"] = (RUIDO.E_sin - RUIDO.E_fci) * 1e3
RUIDO["err_con_mHa"] = (RUIDO.E_con - RUIDO.E_fci) * 1e3
RUIDO["ruido_mHa"] = (RUIDO.E_con - RUIDO.E_sin) * 1e3
RUIDO["sigma_mHa"] = np.hypot(RUIDO.s_sin, RUIDO.s_con) * 1e3
RUIDO["ccsd_mHa"] = (RUIDO.E_ccsd - RUIDO.E_fci).abs() * 1e3   # |E_CCSD - E_FCI|, como el criterio del desafío
print("Errores respecto de FCI (mHa); ruido = E_con - E_sin; sigma = σ de esa diferencia; ccsd = error de CCSD:")
print(RUIDO[["ops", "d", "cz", "err_sin_mHa", "err_con_mHa", "ruido_mHa", "sigma_mHa", "ccsd_mHa"]]
      .to_string(index=False, float_format=lambda v: f"{v:.1f}"))

_full = max(VERSIONES)
_f = RUIDO[RUIDO.ops == _full].set_index("d")
print(f"\nAnsatz completo ({CZ_V[_full]} cz): el ruido suma entre {_f.ruido_mHa.min():.0f} y {_f.ruido_mHa.max():.0f} mHa; "
      f"¿con ruido supera el error de CCSD? " + ", ".join(f"{d}: {'sí' if _f.err_con_mHa[d] > _f.ccsd_mHa[d] else 'no'}"
                                                           for d in DIST_RUIDO if d >= STRETCHED))
_best = RUIDO.loc[RUIDO.groupby("d").err_con_mHa.idxmin(), ["d", "ops", "cz", "err_con_mHa", "ccsd_mHa"]]
print("Mejor versión con ruido por distancia (menor error vs. FCI):")
print(_best.to_string(index=False, float_format=lambda v: f"{v:.1f}"))

# Modelo global de despolarización: E_con ≈ E_sin + (1-F)(E_mezcla - E_sin), F = (1-P_CZ)^cz (1-P_1Q)^n1q y
# E_mezcla = c_I + e_nuc (energía del estado máximamente mezclado). Cota gruesa: el error real es local (2 qubits por cz).
RUIDO["F"] = (1 - P_CZ) ** RUIDO.cz * (1 - P_1Q) ** RUIDO.n1q
_Emix = np.array([float(HAM[d][0].coeffs[~(HAM[d][0].paulis.z | HAM[d][0].paulis.x).any(axis=1)].real.sum()) + HAM[d][1]
                  for d in RUIDO.d])
RUIDO["ruido_modelo_mHa"] = (1 - RUIDO.F) * (_Emix - RUIDO.E_sin) * 1e3

_cols = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4))
for d, c in zip(DIST_RUIDO, _cols):
    g = RUIDO[RUIDO.d == d].sort_values("cz")
    ax1.plot(g.cz, g.ruido_mHa, "o-", color=c, label=f"d = {d} Å")
    ax2.plot(g.cz, g.err_con_mHa, "o-", color=c, label=f"d = {d} Å, con ruido")
    ax2.plot(g.cz, g.err_sin_mHa, "x--", color=c, alpha=0.6, label=f"d = {d} Å, sin ruido")
    if d >= STRETCHED:
        ax2.axhline(float(g.ccsd_mHa.iloc[0]), color=c, ls=":", lw=1)
ax1.set(xlabel="cz del circuito", ylabel="E_con − E_sin  (mHa)", title="Costo del ruido vs. número de cz")
ax1.legend(fontsize=7)
ax2.set(xlabel="cz del circuito", ylabel="E − E_FCI  (mHa)", yscale="symlog", title="Error vs. FCI (punteado: error de CCSD)")
ax2.legend(fontsize=6, ncol=2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(D, [REF[d]["e_fci"] for d in D], "k-", lw=1, label="FCI")
for k, g in RUIDO.groupby("ops"):
    ax.plot(g.d, g.E_con, "o", ms=5, label=f"con ruido, {k} op ({g.cz.iloc[0]} cz)")
g = RUIDO[RUIDO.ops == _full]
ax.plot(g.d, g.E_sin, "k^", mfc="none", ms=7, label="sin ruido, ansatz completo")
ax.set(xlabel="d (Å)", ylabel="E (Ha)", title="Con ruido (cz: 0,1 %) vs. sin ruido vs. FCI")
ax.legend(fontsize=6)
plt.tight_layout()
plt.show()

# Relación con los cz: pendiente por el origen (mHa por cz) y comparación con el modelo global.
_sl = {d: float((g.cz * g.ruido_mHa).sum() / (g.cz ** 2).sum()) for d, g in RUIDO.groupby("d")}
print("Costo del ruido por cz (pendiente por el origen, mHa/cz):", {d: round(v, 3) for d, v in _sl.items()})
print(RUIDO[RUIDO.ops == _full][["d", "cz", "F", "ruido_mHa", "ruido_modelo_mHa"]].to_string(
    index=False, float_format=lambda v: f"{v:.3f}"))
for d, g in RUIDO.groupby("d"):
    g = g.sort_values("cz")
    assert g.ruido_mHa.iloc[-1] > g.ruido_mHa.iloc[0] + 3 * g.sigma_mHa.max(), f"el ruido no crece con los cz en d={d}"
print(f"Stand-ins usados: {list(STANDIN_USED)}")
RUIDO

#### Qué decidimos, por qué y qué no funcionó (#11)

**Qué decidimos**

- **Modelo de ruido:** despolarizante de 1e-3 en `cz` y 5e-5 en `sx`, `x`, `rz`, como pide el enunciado. Se carga `ruido.py`
  si existe; si no, el stand-in. Como el circuito ya sale de `transpile_line` (base `cz, sx, x, rz`, línea), no hace falta pasar
  el mapa de acoplamiento al simulador. Sin error de lectura (el enunciado no lo menciona).
- **Misma medición que #6:** reutilizamos `measure_energy` (`SamplerV2` con shots, grupos QWC) y solo cambiamos el simulador
  dentro de `con_ruido()`. El `EstimatorV2` de Aer no muestrea, así que no sirve para medir con ruido.
- **Las rotaciones de base de la medición también cuestan:** el `u` que agrega `measure_energy` se descompone en 5 compuertas
  de 1 qubit, así que se le asignó ese error (≈ 2,5e-4 por qubit, poco frente al `cz`).
- **Matriz densidad** en Aer: con 8 qubits es mucho más rápido que trayectorias (una evolución por binding y muestreo del final).
- **Versiones cortas:** los operadores más importantes (mayor $|\theta^*|$ sobre la curva), reajustados por statevector, que
  está permitido para buscar parámetros. Se mide siempre con shots.

**Qué vimos**

- El error de ruido crece con el número de `cz`: cada `cz` fallido (1e-3) acerca el estado a uno mezclado y la energía medida a
  $E_\text{mezcla}$ (el término identidad del Hamiltoniano, muy por encima del fundamental). Con ~600-700 `cz` el ruido suma
  unos 300 mHa (con el modelo global esperaríamos 430-500: el ruido real es local y pega menos), muy por encima del margen de CCSD
  (9 mHa a 1,8 Å): con este modelo ningún ansatz que llegue a FCI le gana a CCSD; la pendiente es ≈ 0,45 mHa por `cz`.
- Las versiones cortas pagan sesgo del ansatz pero ganan en ruido; el mejor compromiso depende de la distancia (ver la tabla
  de "mejor versión con ruido por distancia").

**Qué no funcionó / límites**

- **Transpilar en línea cambia el layout:** `transpile_line` elige layout y mete swaps, así que el qubit lógico i termina en otro
  físico; hay que llevar el observable a la misma permutación (`_observable_fisico`), verificado contra la energía de #6.
  Sin eso, las energías salían cientos de mHa mal sin ruido.
- El modelo de despolarización **global** da solo el orden de magnitud: el ruido es local.
- No re-optimizamos con ruido (opcional del issue): con cientos de `cz` cada evaluación de energía es cara.
- El nombre de la función que expone `ruido.py` es una suposición; ajustar `_load_ruido_py` cuando llegue el archivo oficial (#3).

### Ruido de hardware (FakeFez) (#12)

Repetimos **un** punto estirado, $d = 2{,}4$ Å, con el ruido completo de `FakeFez` (backend simulado de IBM, 156 qubits Heron r2):
errores de puertas por qubit/par, T1/T2 y **error de lectura**. Receta:

1. `FakeFez` viene de `qiskit_ibm_runtime.fake_provider`: es una **dependencia nueva** (`qiskit-ibm-runtime`, se instala en la primera celda de este extra, solo si se recalcula).
2. El circuito se transpila **al backend** (base `cz, sx, x, rz, id`, conectividad heavy-hex, con elección de layout y SWAPs) y recién ahí se cuentan `cz` y profundidad.
3. Se mide con `SamplerV2` de `qiskit_aer` con `NoiseModel.from_backend(FakeFez())`, con la lectura incluida. Es el mismo esquema que #6
   (grupos QWC, σ por varianza muestral): se **reusa** `measure_energy`, cambiando solo cómo se arma y se corre el circuito.
4. No usamos `EstimatorV2` para el número reportado: el de Aer calcula la esperanza exacta sin pasar por la medición, así que **esconde** el error de lectura (lo comprobamos abajo con el estado HF).

**Resultados guardados.** Transpilar a FakeFez y medir con su ruido tarda ~1 min y además necesita instalar `qiskit-ibm-runtime`,
así que por defecto este extra usa los resultados guardados en `FEZ_CACHE` (de una corrida con `RUN_FEZ = True`,
`SHOTS_SCALE = 1`): `cz` y profundidad en el backend (12.2), la tabla medida (12.3) y el control con `EstimatorV2` (12.4).
Los circuitos de 12.1, la tabla de errores y el gráfico se recalculan igual. Para recalcular todo: `RUN_FEZ = True` en la
celda del `%pip` de abajo (instala `qiskit-ibm-runtime` y guarda `results/fez_*.csv`).

In [ ]:
# Resultados guardados del extra FakeFez (#12, celdas 12.2 a 12.4). Recalcularlos tarda ~1 min y necesita
# qiskit-ibm-runtime, así que las celdas de abajo los usan por defecto (RUN_FEZ = False). Salieron de esas mismas
# celdas con True; para regenerarlos: poner True, correr y pegar acá results/fez_backend.csv, results/fez.csv y
# el valor de results/fez_control.csv.
FEZ_CACHE = {
    "origen": "corrida completa del 2026-11-21 con RUN_FEZ = True, SHOTS_SCALE = 1, qiskit 2.5.2, qiskit-aer 0.17.2, qiskit-ibm-runtime 0.47.0",
    "backend": """\
circuito,cz_backend,depth_backend,cz_linea,qubits_fisicos
ansatz completo,301,657,415,24 25 26 37 43 44 45 46
corto (2 op.),82,164,149,23 24 25 26 37 44 45 46
corto (1 op.),37,89,88,117 123 124 125 126 127 137 143
HF (0 op.),0,1,0,4 23 84 96
""",
    "fez": """\
circuito,cz,profundidad,E_sv,E_sin_ruido,sigma_sin,E_fez,sigma_fez,shots_fez
ansatz completo,301,657,-1.874650564,-1.873287365,0.0030007,-1.321342917,0.00269274,452216
corto (2 op.),82,164,-1.43645854,-1.43507183,0.00289101,-1.348595275,0.00272501,446392
corto (1 op.),37,89,-1.43645854,-1.434084749,0.00290759,-1.388111913,0.00274003,419482
HF (0 op.),0,1,-1.43645854,-1.434887368,0.00290362,-1.434000271,0.00291746,298215
""",
    "e_estimator_hf": -1.4362274108516653,  # 12.4: EstimatorV2 de Aer con el ruido de FakeFez sobre HF (Ha)
}

In [ ]:
# Dependencia solo de este extra (#12): FakeFez vive en qiskit-ibm-runtime, que no está en nixpkgs.
# Solo hace falta para recalcular (RUN_FEZ = True); con los resultados guardados no se instala ni se importa.
# Por eso no va en la celda %pip principal ni en requirements.txt (los compara `nix flake check`).
# En Colab se instala acá; fuera de Colab, instalarlo a mano en el entorno (pip install qiskit-ibm-runtime==0.47.0).
RUN_FEZ = False  # True: instalar qiskit-ibm-runtime, transpilar a FakeFez y volver a medir (~1 min). False: usar FEZ_CACHE
if RUN_FEZ:
    %pip install -q qiskit-ibm-runtime==0.47.0

In [ ]:
# --- 12.0 Backend, modelo de ruido y puente hacia measure_energy ---------------------------------------
import contextlib
from qiskit import ClassicalRegister
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import SamplerV2 as _AerSamplerV2

D_FEZ = float(D[int(np.argmin(np.abs(np.asarray(D) - 2.4)))])  # 2,4 Å
SEED_FEZ = SEED + 12_000
if RUN_FEZ:
    try:
        from qiskit_ibm_runtime.fake_provider import FakeFez
    except ModuleNotFoundError:  # p. ej. en `nix develop`, donde el %pip de arriba no puede instalar
        print("AVISO: RUN_FEZ = True pero falta qiskit-ibm-runtime (pip install qiskit-ibm-runtime==0.47.0); "
              "sigo con los resultados guardados.")
        RUN_FEZ = False
if RUN_FEZ:
    FEZ = FakeFez()
    FEZ_NOISE = NoiseModel.from_backend(FEZ)  # incluye errores de puerta, relajación térmica y lectura
    # method="density_matrix": el circuito usa 8 de los 156 qubits (Aer descarta los inactivos) y 8 qubits en matriz densidad
    # son baratos; así los shots salen de la distribución exacta del circuito ruidoso, sin trayectorias.
    FEZ_BACKEND_OPTIONS = {"method": "density_matrix", "noise_model": FEZ_NOISE}

    _t = FEZ.target
    _med = lambda name: float(np.median([p.error for p in _t[name].values() if p is not None and p.error is not None]))
    print(f"FakeFez: {FEZ.num_qubits} qubits, base {sorted(_t.operation_names)}.")
    print(f"Errores medianos del backend: cz {_med('cz'):.2e}, sx {_med('sx'):.2e}, lectura {_med('measure'):.2e}.")
else:
    print(f"RUN_FEZ = False: uso los resultados guardados ({FEZ_CACHE['origen']}); FakeFez no se carga.")


def to_backend(circ, seeds=(0, 1, 2)):
    """Transpila `circ` (8 qubits) a FakeFez (opt. 3, layout incluido). Devuelve (circuito, cz, profundidad, seed).

    Elige la mejor de varias semillas por número de cz (el transpilador es estocástico)."""
    best = None
    for s in seeds:
        tc = transpile(circ, FEZ, optimization_level=3, seed_transpiler=s)
        cz = tc.count_ops().get("cz", 0)
        if best is None or cz < best[1]:
            best = (tc, cz, tc.depth(), s)
    return best


class _Proxy:
    """Plantilla para measure_energy: expone 8 qubits lógicos pero lleva el circuito real (156 qubits físicos)."""
    def __init__(self, qc):
        self.qc, self.num_qubits, self.num_parameters = qc, 8, qc.num_parameters


class _NoisySampler:
    """Reemplazo de SamplerV2 con el ruido de FakeFez; desenvuelve la plantilla y deja el resto igual."""
    def __init__(self, seed=None, options=None):  # `options` (statevector sin ruido de #6) se ignora: acá va el ruido de FakeFez
        self._s = _AerSamplerV2(seed=seed, options={"backend_options": FEZ_BACKEND_OPTIONS})

    def run(self, pubs):
        (proxy, vals, shots), = pubs
        return self._s.run([(proxy.qc, vals, shots)])


_FEZ_SEED = {}  # semilla de transpilación elegida para cada circuito (se calcula una vez y se reusa en la plantilla)


def _fez_template(circuit_bound):
    """Plantilla de medición en el backend: circuito + U(θ_q,0,λ_q) por qubit (la base de cada grupo) + lectura de 8 bits."""
    key = id(circuit_bound)
    if key not in _FEZ_SEED:
        _FEZ_SEED[key] = to_backend(circuit_bound)[3]
    th, la = ParameterVector("_rot_theta", 8), ParameterVector("_rot_lambda", 8)
    qc = QuantumCircuit(8)
    qc.compose(circuit_bound, inplace=True)
    for q in range(8):
        qc.u(th[q], 0.0, la[q], q)
    qc.add_register(ClassicalRegister(8, "meas"))
    qc.measure(range(8), range(8))
    tc = transpile(qc, FEZ, optimization_level=3, seed_transpiler=_FEZ_SEED[key])
    order = {p: i for i, p in enumerate(tc.parameters)}
    return _Proxy(tc), [order[p] for p in th], [order[p] for p in la]


@contextlib.contextmanager
def con_ruido_fez():
    """Dentro del bloque, measure_energy (de #6) transpila al backend y muestrea con el ruido de FakeFez."""
    g = globals()
    old = g["_measurement_template"], g["SamplerV2"], g["UNITS_PER_GROUP"]
    # Con matriz densidad cada asignación del PUB es una simulación completa del circuito ruidoso (los shots son casi gratis):
    # 1 paquete por grupo en vez de ~2 reduce 2 veces el costo sin cambiar la estadística (σ sigue saliendo de la varianza por grupo).
    g["_measurement_template"], g["SamplerV2"], g["UNITS_PER_GROUP"] = _fez_template, _NoisySampler, 1
    try:
        yield
    finally:
        g["_measurement_template"], g["SamplerV2"], g["UNITS_PER_GROUP"] = old


def medir_fez(circ, op, e_nuc, sigma, seed):
    """Energía ± σ con SamplerV2 + FakeFez (lectura incluida). σ objetivo relajado por SHOTS_SCALE como en #6."""
    with con_ruido_fez():
        return measure_energy(circ, op, e_nuc, target_sigma=sigma / np.sqrt(SHOTS_SCALE), seed=seed,
                              pilot_shots_per_group=PILOT_SHOTS, max_shots=4_000_000)

#### Circuitos: ansatz completo, versiones cortas (1 y 2 operadores) y HF

El ansatz completo tiene cientos de `cz`: con ~4 × 10⁻³ por `cz` el circuito pierde casi toda la señal. Por eso armamos versiones cortas:
nos quedamos con los 1 y 2 parámetros de mayor $|\theta^*|$ (los operadores que más aportan) y los **re-ajustamos por statevector sin ruido**
(buscar parámetros vale de cualquier forma; la energía reportada igual sale de medir). El estado HF (0 operadores, 0 `cz`) aísla el efecto
de la **lectura**. Nota: `restringir` conserva el prefijo sin parámetros (los `X` del HF) y las puertas parametrizadas elegidas.

In [ ]:
# --- 12.1 Circuitos del punto d = 2,4 Å -----------------------------------------------------------------
from scipy.optimize import minimize

op_f, e_nuc_f = HAM[D_FEZ]
th_full = THETA[D_FEZ]
_P = list(ansatz.parameters)


def _params_de(inst):
    ps = set()
    for p in inst.operation.params:
        ps |= set(getattr(p, "parameters", ()))
    t = getattr(inst.operation, "time", None)
    return ps | set(getattr(t, "parameters", ()))


def restringir(qc, keep, values):
    """Circuito con solo los parámetros `keep` (ya asignados a `values`); el resto de las puertas parametrizadas se descarta.

    Las puertas fijas (X inicial, los CX de cada rotación de Givens) se conservan: sin sus rotaciones, los CX de cada
    excitación descartada se cancelan entre sí (identidad) y el transpilador los elimina."""
    keep = set(keep)
    out = QuantumCircuit(*qc.qregs, name="corto")
    for inst in qc.data:
        ps = _params_de(inst)
        if ps and not ps <= keep:
            continue
        out.append(inst.operation, inst.qubits, inst.clbits)
    return out.assign_parameters(dict(zip(sorted(keep, key=lambda p: _P.index(p)), values)))


def circuito_corto(n_ops):
    idx = sorted(np.argsort(-np.abs(th_full))[:n_ops])
    keep = [_P[i] for i in idx]
    Hm = op_f.to_matrix()

    def f(x):
        sv = Statevector(restringir(ansatz, keep, x)).data
        return float(np.real(np.vdot(sv, Hm @ sv)))

    r = minimize(f, th_full[idx], method="BFGS")  # statevector sin ruido: solo búsqueda de parámetros
    return restringir(ansatz, keep, np.round(r.x, 6)), idx


_t0 = time.time()
CIRCS_FEZ = {}
CIRCS_FEZ["ansatz completo"] = ansatz.assign_parameters(th_full)
for n_ops in (2, 1):
    CIRCS_FEZ[f"corto ({n_ops} op.)"], _idx = circuito_corto(n_ops)
    print(f"  corto ({n_ops} op.): parámetros {[int(i) for i in _idx]} de {len(_P)}")
hf = QuantumCircuit(8, name="HF")
for q, bit in enumerate(reversed(HF_BITSTRING)):
    if bit == "1":
        hf.x(q)
CIRCS_FEZ["HF (0 op.)"] = hf

E_SV_FEZ = {k: float(Statevector(c).expectation_value(op_f).real) + e_nuc_f for k, c in CIRCS_FEZ.items()}  # solo control
print(f"Circuitos listos en {time.time() - _t0:.1f} s.")

In [ ]:
# --- 12.2 Transpilación al backend: cz y profundidad ----------------------------------------------------
import io

if RUN_FEZ:
    _t0 = time.time()
    BACKEND_STATS = {}
    for k, c in CIRCS_FEZ.items():
        tc, cz, depth, seed = to_backend(c)
        used = sorted({tc.find_bit(q).index for inst in tc.data for q in inst.qubits})
        BACKEND_STATS[k] = dict(cz_backend=cz, depth_backend=depth, cz_linea=cz_count(c), qubits_fisicos=used)
    os.makedirs("results", exist_ok=True)
    (pd.DataFrame(BACKEND_STATS).T.rename_axis("circuito")
     .assign(qubits_fisicos=lambda t: t.qubits_fisicos.map(lambda q: " ".join(map(str, q))))
     .to_csv("results/fez_backend.csv"))
    print(f"Transpilación en {time.time() - _t0:.1f} s.")
else:
    BACKEND_STATS = {r.circuito: dict(cz_backend=int(r.cz_backend), depth_backend=int(r.depth_backend), cz_linea=int(r.cz_linea),
                                      qubits_fisicos=[int(q) for q in str(r.qubits_fisicos).split()])
                     for r in pd.read_csv(io.StringIO(FEZ_CACHE["backend"])).itertuples()}
    print(f"Resultados guardados ({FEZ_CACHE['origen']}). Para recalcular: RUN_FEZ = True.")
print(pd.DataFrame(BACKEND_STATS).T[["cz_linea", "cz_backend", "depth_backend"]].to_string())
print(f"(layout de 'ansatz completo': qubits físicos {BACKEND_STATS['ansatz completo']['qubits_fisicos']})")

In [ ]:
# --- 12.3 Medición con SamplerV2: sin ruido y con FakeFez -----------------------------------------------
SIGMA_FEZ = 3.0e-3  # σ objetivo (Ha) en cada medición; con SHOTS_SCALE < 1 se relaja como en #6
e_fci_f = REF[D_FEZ]["e_fci"]
if RUN_FEZ:
    _t0 = time.time()
    rows = []
    for j, (k, c) in enumerate(CIRCS_FEZ.items()):
        m0 = measure_energy(c, op_f, e_nuc_f, target_sigma=SIGMA_FEZ / np.sqrt(SHOTS_SCALE), seed=SEED_FEZ + 10 * j,
                            pilot_shots_per_group=PILOT_SHOTS)  # SamplerV2 sin ruido (mismo esquema que #6)
        m1 = medir_fez(c, op_f, e_nuc_f, SIGMA_FEZ, SEED_FEZ + 10 * j + 1)
        rows.append(dict(circuito=k, cz=BACKEND_STATS[k]["cz_backend"], profundidad=BACKEND_STATS[k]["depth_backend"],
                         E_sv=E_SV_FEZ[k], E_sin_ruido=m0["energy"], sigma_sin=m0["sigma"],
                         E_fez=m1["energy"], sigma_fez=m1["sigma"], shots_fez=m1["shots"]))
        print(f"  {k}: {time.time() - _t0:.0f} s")
    fez = pd.DataFrame(rows)
    fez.to_csv("results/fez.csv", index=False)
    print(f"Medición con FakeFez en {time.time() - _t0:.1f} s.")
else:
    fez = pd.read_csv(io.StringIO(FEZ_CACHE["fez"]))
    print(f"Resultados guardados ({FEZ_CACHE['origen']}). Para volver a medir: RUN_FEZ = True.")
fez["err_sin_ruido_mHa"] = (fez.E_sin_ruido - e_fci_f) * 1e3
fez["err_fez_mHa"] = (fez.E_fez - e_fci_f) * 1e3
fez["ruido_mHa"] = (fez.E_fez - fez.E_sin_ruido) * 1e3
_ccsd_f = (REF[D_FEZ]["e_ccsd"] - e_fci_f) * 1e3
print(f"d = {D_FEZ} Å: E_FCI = {e_fci_f:.6f} Ha, E_CCSD - E_FCI = {_ccsd_f:.2f} mHa.")
print(fez[["circuito", "cz", "profundidad", "E_sin_ruido", "sigma_sin", "E_fez", "sigma_fez", "err_sin_ruido_mHa",
           "err_fez_mHa", "ruido_mHa", "shots_fez"]].round({"E_sin_ruido": 6, "E_fez": 6, "sigma_sin": 5, "sigma_fez": 5,
           "err_sin_ruido_mHa": 2, "err_fez_mHa": 1, "ruido_mHa": 1}).to_string(index=False))

In [ ]:
# --- 12.4 Control: EstimatorV2 de Aer esconde el error de lectura (solo comparación, no se reporta) --------
if RUN_FEZ:
    hf_tc, _, _, _ = to_backend(CIRCS_FEZ["HF (0 op.)"])
    op_phys = op_f.apply_layout(hf_tc.layout, hf_tc.num_qubits)
    est_fez = EstimatorV2(options={"backend_options": FEZ_BACKEND_OPTIONS, "run_options": {"seed": SEED_FEZ}})
    e_est = float(est_fez.run([(hf_tc, op_phys)]).result()[0].data.evs) + e_nuc_f
    pd.DataFrame({"e_estimator_hf": [e_est]}).to_csv("results/fez_control.csv", index=False)
else:
    e_est = float(FEZ_CACHE["e_estimator_hf"])
e_smp = float(fez.loc[fez.circuito == "HF (0 op.)", "E_fez"].iloc[0])
print(f"HF con ruido FakeFez, d = {D_FEZ} Å:  EstimatorV2 -> {e_est:.5f} Ha   SamplerV2 -> {e_smp:.5f} Ha   "
      f"(sin ruido {E_SV_FEZ['HF (0 op.)']:.5f})")
print(f"EstimatorV2 mueve el HF {abs(e_est - E_SV_FEZ['HF (0 op.)']) * 1e3:.1f} mHa; SamplerV2 {abs(e_smp - E_SV_FEZ['HF (0 op.)']) * 1e3:.1f} mHa.")

In [ ]:
# --- 12.5 Gráfico: error contra FCI, con y sin ruido, según los cz del backend --------------------------
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(fez))
w = 0.38
ax.bar(x - w / 2, np.abs(fez.err_sin_ruido_mHa).clip(lower=1e-1), w, yerr=fez.sigma_sin * 1e3, label="SamplerV2 sin ruido")
ax.bar(x + w / 2, np.abs(fez.err_fez_mHa), w, yerr=fez.sigma_fez * 1e3, label="SamplerV2 + FakeFez")
ax.axhline(abs(_ccsd_f), color="k", ls="--", lw=1, label=f"|E_CCSD - E_FCI| = {abs(_ccsd_f):.1f} mHa")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels([f"{c}\n{z} cz" for c, z in zip(fez.circuito, fez.cz)], fontsize=8)
ax.set_ylabel("|E - E_FCI| (mHa)")
ax.set_title(f"H4 a d = {D_FEZ} Å: ruido de FakeFez")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

#### Cómo se lee, y la diferencia con `ruido.py` (#11)

* **Qué mide la tabla.** `err_sin_ruido` es el sesgo del circuito (sin ruido físico; queda el σ de los shots) y `err_fez` suma el efecto de FakeFez.
  `ruido_mHa` es la diferencia. Crece con los `cz` **del backend** y no con los de la conectividad lineal: en heavy-hex hacen falta SWAPs extra.
* **Por qué el ansatz exacto no sirve acá.** Con $\epsilon_{cz} \sim 4\times10^{-3}$ y cientos de `cz`, la fidelidad del circuito es
  $\sim e^{-N_{cz}\epsilon}$, de unas pocas decenas de por ciento o menos: el estado se parece a uno mezclado y la energía cae lejos de FCI.
  Los circuitos cortos pierden menos por ruido (menos `cz`), pero con 1-2 operadores dejan cientos de mHa de sesgo sin ruido: en este punto ninguno supera al estado HF
  medido con el mismo ruido. Con este hardware simulado el compromiso expresividad/ruido se resuelve a favor de HF; el ansatz exacto queda peor que HF.
* **Lectura.** Incluso con 0 `cz` (HF) hay error: la lectura mediana (~0,8 %) por qubit voltea bits del bitstring. Se ve con `SamplerV2`, y **no** con
  `EstimatorV2` de Aer (celda 12.4).
* **Diferencia con el modelo común (`ruido.py`, #11).** `ruido.py` (no está en `main` todavía, por eso no lo corrimos) es, según el enunciado, un
  modelo **sintético y uniforme**: depolarizante $10^{-3}$ en `cz` y $5\times10^{-5}$ en `sx`, `x`, `rz`, conectividad lineal, sin T1/T2 ni lectura.
  FakeFez es el ruido **calibrado** de un chip real: `cz` ~4× peor (y distinto en cada par), relajación durante las esperas, error de lectura
  por qubit y conectividad heavy-hex (más `cz` por SWAPs). Cuando llegue #11 hay que correr el mismo circuito con los dos modelos y poner los números lado a lado.

#### Qué decidimos, por qué y qué no funcionó (#12)

* **SamplerV2 con matriz densidad.** Los shots salen de la distribución exacta del circuito ruidoso (`method="density_matrix"`); es más rápido que trayectorias
  y no cambia la estadística de muestreo. σ es el error estándar de los shots, no incluye variación entre calibraciones.
* **Reuso de `measure_energy` (#6).** En vez de copiarlo, dentro de `con_ruido_fez()` se reemplazan la plantilla de medición (se transpila a FakeFez con la mejor
  de 3 semillas y se mide solo en 8 bits) y el sampler. Así los grupos QWC, el reparto de shots y σ son los de #6.
* **cz y profundidad.** Se reportan del circuito **sin** rotaciones de medición, transpilado al backend con opt. 3 y la mejor de 3 semillas.
* **Circuitos cortos.** Se eligen por $|\theta^*|$ y se reajustan sin ruido por statevector. Más simple que ADAPT; si #4 cambia la estructura del ansatz hay que revisar `restringir`.
* **EstimatorV2.** Solo como control con HF: esconde la lectura y por eso queda sub-estimado el error.
* **No funcionó / límites.** No probamos mitigación (extrapolación a ruido cero, TREX), que sería el siguiente paso.

### Esquemas de medición (#19)

La energía reportada sale de #6: grupos QWC (*sorted insertion*) medidos con `SamplerV2`. Acá buscamos **gastar menos
shots para la misma σ** y lo cuantificamos en tres distancias estiradas (1,8 / 2,4 / 3,0 Å) sobre el mismo estado
$|\psi(\theta^*)\rangle$ que mide #6. Comparamos cuatro esquemas:

| Esquema | Qué es | Circuito de medición |
|---|---|---|
| **QWC #6** | línea de base: *sorted insertion* por $\lvert c_k\rvert$ | `H` / `S†H` por qubit (0 `cz`) |
| **(a) QWC mejorado** | mismo tipo de grupos, pero elegimos la partición que minimiza $K=\sum_g\sqrt{\operatorname{Var}_g}$ entre muchas candidatas (órdenes aleatorios + coloreo de grafos) | igual que #6 (0 `cz`) |
| **(b) Conmutantes (GC)** | grupos de Paulis que conmutan (no necesariamente qubit a qubit) | Clifford que diagonaliza el grupo (sí suma `cz`) |
| **(c) Rotación de base (DF)** | factorización doble (Huggins et al., npj QI 7, 23 (2021)): $H = F_0 + \sum_\ell \tfrac{\lambda_\ell}{2}\big(\sum_{pq} L^{\ell}_{pq} E_{pq}\big)^2$; cada fragmento es diagonal en ocupaciones tras una rotación de orbitales | red de Givens por espín |

**Cómo comparamos.** Para un esquema con fragmentos $g$ (cada uno: un circuito de cambio de base $V_g$ y una tabla
$h_g(x)$ con el valor por shot), con reparto óptimo $N_g\propto\sqrt{\operatorname{Var}_g}$ vale $\sigma = K/\sqrt N$, así que
los **shots para σ = 1 mHa** son $N = (K/10^{-3})^2$. $\operatorname{Var}_g$ es la varianza del valor por shot del fragmento
entero (incluye covarianzas), igual que en #6. Para *diseñar* (elegir particiones, repartir shots y llenar la columna de
shots) usamos las varianzas exactas sobre el statevector: es una decisión de diseño, no una energía. La **energía** de
cada esquema sale de medir con `SamplerV2` y se compara con la de #6.

**Cómo validamos cada esquema.** No confiamos en convenciones de signos: para cada fragmento construimos la matriz
$V_g O_g V_g^\dagger$ (256×256) y exigimos que sea diagonal (error $<10^{-9}$); la tabla $h_g(x)$ es esa diagonal. Además
$\sum_g O_g$ tiene que reproducir el Hamiltoniano de #2 como operador.

In [ ]:
# --- 19.1 Datos comunes: distancias, estado, términos de Pauli -------------------------------------
import time
import numpy as np
import pandas as pd
import scipy.linalg
import rustworkx as rx
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, SparsePauliOp, StabilizerState, Statevector
from qiskit_aer.primitives import SamplerV2
from pyscf import ao2mo, gto, scf

_T0_MS = time.time()
MS_D = [float(D[np.argmin(np.abs(D - x))]) for x in (1.8, 2.4, 3.0)]  # distancias estiradas de la grilla
MS_SIGMA = 1.0e-3            # Ha: σ de referencia para la columna "shots para σ = 1 mHa"
MS_SIGMA_RUN = 2.0e-3        # Ha: σ objetivo de la corrida con SamplerV2 (a escala completa; se relaja con SHOTS_SCALE)
MS_RANDOM_ORDERS = 100       # órdenes aleatorios probados por (a) y (b)
MS_SEED = SEED + 19_000
_MS_DKEY = {d: D[np.argmin(np.abs(D - d))] for d in MS_D}  # clave exacta en HAM/CIRCUIT/MEAS
_NQ = N_QUBITS
_DIM = 2 ** _NQ
_MS_SECTOR_MASK = np.zeros(_DIM, bool)
_MS_SECTOR_MASK[_SECTOR] = True  # N = 4, Sz = 0 (de #6)
_MS_NA = np.array([sum(x >> alpha_qubit(p) & 1 for p in range(N_QUBITS // 2)) for x in range(_DIM)])  # N_α por bitstring
_MS_NB = np.array([sum(x >> beta_qubit(p) & 1 for p in range(N_QUBITS // 2)) for x in range(_DIM)])


def _ms_pauli_data(d):
    """Términos no identidad de H(d): códigos IXYZ, matrices de compatibilidad QWC/GC y momentos exactos sobre ψ(θ*)."""
    op, e_nuc = HAM[_MS_DKEY[d]]
    coeffs = np.asarray(op.coeffs).real
    xs, zs = op.paulis.x, op.paulis.z
    keep = ~((~xs) & (~zs)).all(axis=1)
    c_id = float(coeffs[~keep].sum())
    idx = np.flatnonzero(keep)
    x, z = xs[idx].astype(int), zs[idx].astype(int)
    codes = np.where(x & z, 2, np.where(x == 1, 1, np.where(z == 1, 3, 0)))  # 0=I 1=X 2=Y 3=Z (como #6)
    qwc = ((codes[:, None, :] == 0) | (codes[None, :, :] == 0) | (codes[:, None, :] == codes[None, :, :])).all(axis=2)
    anti = (x @ z.T + z @ x.T) % 2
    comm = anti == 0
    psi = Statevector(CIRCUIT[_MS_DKEY[d]]).data
    phi = np.array([op.paulis[k].to_matrix(sparse=True) @ psi for k in idx])  # P_k |ψ>
    C = (phi.conj() @ phi.T).real            # <P_j P_k> (real para pares que conmutan)
    m = (phi.conj() @ psi).real              # <P_k>
    return dict(op=op, e_nuc=e_nuc, c_id=c_id, idx=idx, coeffs=coeffs[idx], codes=codes, qwc=qwc, comm=comm,
                C=C, m=m, psi=psi)


def _ms_K(P, groups):
    """K = sum_g sqrt(Var_g) exacto (covarianzas incluidas) para una partición en índices locales."""
    K = 0.0
    for g in groups:
        c = P["coeffs"][g]
        K += np.sqrt(max(c @ P["C"][np.ix_(g, g)] @ c - (c @ P["m"][g]) ** 2, 0.0))
    return K


def _ms_insertion(compat, order):
    """Inserción greedy: cada término va al primer grupo con el que es compatible con todos sus miembros."""
    groups, allowed = [], []  # allowed[i][k]: k es compatible con todos los miembros del grupo i
    for k in order:
        for g, ok in zip(groups, allowed):
            if ok[k]:
                g.append(int(k))
                ok &= compat[k]
                break
        else:
            groups.append([int(k)])
            allowed.append(compat[k].copy())
    return groups


def _ms_coloring(compat, strategy):
    """Partición por coloreo greedy del grafo de conflictos (arista = términos incompatibles) con rustworkx."""
    n = compat.shape[0]
    graph = rx.PyGraph()
    graph.add_nodes_from(range(n))
    ii, jj = np.nonzero(np.triu(~compat, 1))
    graph.add_edges_from_no_data(list(zip(ii.tolist(), jj.tolist())))
    col = rx.graph_greedy_color(graph, strategy=strategy)
    groups = {}
    for node, cidx in col.items():
        groups.setdefault(cidx, []).append(int(node))
    return list(groups.values())


def _ms_best_partition(P, compat, seed):
    """Candidatas: sorted insertion, coloreos de rustworkx y órdenes aleatorios. Devuelve (mejor, tabla de candidatas)."""
    rng = np.random.default_rng(seed)
    cands = {"sorted insertion": _ms_insertion(compat, np.argsort(-np.abs(P["coeffs"]), kind="stable"))}
    for name in ("Degree", "Saturation", "IndependentSet"):
        if hasattr(rx.ColoringStrategy, name):
            cands[f"coloreo {name}"] = _ms_coloring(compat, getattr(rx.ColoringStrategy, name))
    best_rand = None
    for _ in range(MS_RANDOM_ORDERS):
        g = _ms_insertion(compat, rng.permutation(len(P["coeffs"])))
        k = _ms_K(P, g)
        if best_rand is None or k < best_rand[0]:
            best_rand = (k, g)
    cands[f"mejor de {MS_RANDOM_ORDERS} órdenes aleatorios"] = best_rand[1]
    rows = [(name, len(g), _ms_K(P, g)) for name, g in cands.items()]
    name = min(rows, key=lambda r: r[2])[0]
    return cands[name], name, rows


_t = time.time()
MS_P = {d: _ms_pauli_data(d) for d in MS_D}
print(f"Datos de Pauli y momentos exactos en {MS_D} Å: {len(MS_P[MS_D[0]]['coeffs'])} términos no identidad "
      f"({time.time() - _t:.1f} s).")

In [ ]:
# --- 19.2 Fragmentos medibles: (circuito de cambio de base, tabla por shot) y su validación ----------


def _ms_fragment(circ, O, label):
    """Fragmento medible: V = circ, tabla = diag(V O V†). Exige que V O V† sea diagonal."""
    Vm = Operator(circ).data
    M = Vm @ O @ Vm.conj().T
    off = np.abs(M - np.diag(np.diag(M))).max()
    assert off < 1e-9, f"{label}: V O V† no es diagonal (off = {off:.2e})"
    # ¿El circuito de medición conserva N_α y N_β? Entonces cada shot sigue en el sector N=4, Sz=0 y se puede post-seleccionar.
    keeps = all(np.abs(Vm * (n[:, None] - n[None, :])).max() < 1e-9 for n in (_MS_NA, _MS_NB))
    return dict(circ=circ, table=np.diag(M).real.copy(), cz=cz_count(circ) if circ.size() else 0, keeps_N=keeps)


def _ms_pauli_sum_matrix(P, g):
    return SparsePauliOp(P["op"].paulis[P["idx"][g]], P["coeffs"][g]).to_matrix()


def ms_qwc_fragments(P, groups):
    """Grupos QWC -> fragmentos con rotaciones de un qubit (X: H, Y: S†H)."""
    frags = []
    for g in groups:
        basis = P["codes"][g].max(axis=0)  # en un grupo QWC cada qubit tiene a lo sumo un Pauli no trivial
        circ = QuantumCircuit(_NQ)
        for q, b in enumerate(basis):
            if b == 2:
                circ.sdg(q)
            if b in (1, 2):
                circ.h(q)
        frags.append(_ms_fragment(circ, _ms_pauli_sum_matrix(P, g), "QWC"))
    return frags


def _ms_independent(P, g):
    """Subconjunto independiente (sobre GF(2), vectores simplécticos) de un grupo conmutante."""
    op = P["op"].paulis[P["idx"][g]]
    rows = np.hstack([op.x, op.z]).astype(np.uint8)
    basis, chosen = [], []
    for i, r in enumerate(rows):
        v = r.copy()
        for piv, b in basis:
            if v[piv]:
                v ^= b
        if v.any():
            piv = int(np.flatnonzero(v)[0])
            basis = [(p, b ^ v if b[piv] else b) for p, b in basis] + [(piv, v)]
            chosen.append(i)
    return [op[i].to_label().lstrip("-i") for i in chosen]


def ms_gc_fragments(P, groups):
    """Grupos conmutantes -> Clifford C cuyo estado estabilizador tiene a los generadores del grupo; medimos tras C†."""
    frags = []
    for g in groups:
        gens = _ms_independent(P, g)
        cliff = StabilizerState.from_stabilizer_list(gens, allow_underconstrained=True).clifford
        circ = cliff.to_circuit().inverse()
        frags.append(_ms_fragment(circ, _ms_pauli_sum_matrix(P, g), "GC"))
    return frags


# ---- (c) Factorización doble ----
def _ms_annihilators():
    """a_j densos (256×256) en Jordan-Wigner, qubit j = bit j del índice (little endian, como Qiskit)."""
    out = []
    for j in range(_NQ):
        a = np.zeros((_DIM, _DIM))
        for x in range(_DIM):
            if x >> j & 1:
                a[x ^ (1 << j), x] = (-1) ** bin(x & ((1 << j) - 1)).count("1")
        out.append(a)
    return out


_MS_A = _ms_annihilators()
_MS_NORB = N_QUBITS // 2
# E_pq = sum_σ a†_pσ a_qσ con la convención del contrato (α: alpha_qubit(p), β: beta_qubit(p))
_MS_E = np.array([[sum(_MS_A[f(p)].T @ _MS_A[f(q)] for f in (alpha_qubit, beta_qubit))
                   for q in range(_MS_NORB)] for p in range(_MS_NORB)])


def _ms_one_body(A):
    return np.tensordot(A, _MS_E, axes=([0, 1], [0, 1]))


def _ms_integrals(d):
    """h_pq y (pq|rs) en la base MO de RHF/STO-3G (PySCF), con los signos de MO elegidos para reproducir H de #2."""
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-10
    mf.run()
    H_ref = HAM[_MS_DKEY[d]][0].to_matrix().real
    for signs in np.array(np.meshgrid(*[[1, -1]] * (_MS_NORB - 1))).T.reshape(-1, _MS_NORB - 1):
        Cmo = mf.mo_coeff * np.concatenate([[1], signs])
        h = Cmo.T @ mf.get_hcore() @ Cmo
        eri = ao2mo.restore(1, ao2mo.kernel(mol, Cmo), _MS_NORB)
        Hf = _ms_one_body(h) - 0.5 * _ms_one_body(np.einsum("pqqs->ps", eri))
        for p in range(_MS_NORB):
            for q in range(_MS_NORB):
                Hf += 0.5 * _MS_E[p, q] @ _ms_one_body(eri[p, q])
        if np.abs(Hf - H_ref).max() < 1e-8:
            return h, eri
    print(f"AVISO d={d}: las integrales de PySCF no reproducen el Hamiltoniano de #2 (otra convención); "
          "salteamos DF en esta distancia.")
    return None


def _ms_givens_gate(theta):
    """exp(θ (a†_lo a_hi − a†_hi a_lo)) sobre dos modos adyacentes en JW (sin cuerda de Z entre ellos)."""
    a_lo = np.array([[0, 1], [0, 0]])
    lo, hi = np.kron(np.eye(2), a_lo), np.kron(a_lo, np.diag([1, -1]))  # índice = 2*b_hi + b_lo
    return UnitaryGate(scipy.linalg.expm(theta * (lo.T @ hi - hi.T @ lo)), label="G")


def _ms_orbital_rotation(U):
    """Circuito de R(U) (R a†_q R† = sum_p U_pq a†_p) para cada espín: red de Givens adyacentes + signos (Z)."""
    n = U.shape[0]
    W, factors = U.copy(), []
    for j in range(n - 1):
        for i in range(n - 1, j, -1):
            a, b = W[i - 1, j], W[i, j]
            r = np.hypot(a, b)
            if r < 1e-14:
                continue
            c, s = a / r, b / r
            G = np.eye(n)
            G[i - 1, i - 1], G[i - 1, i], G[i, i - 1], G[i, i] = c, s, -s, c
            W = G @ W
            factors.append((i - 1, np.arctan2(-s, c)))  # G^T = exp(θ A_{i-1,i})
    diag = np.sign(np.diag(W))  # U = G_1^T ... G_m^T D
    circ = QuantumCircuit(_NQ)
    for f in (alpha_qubit, beta_qubit):
        for p in range(n):
            if diag[p] < 0:
                circ.z(f(p))
        for p, th in reversed(factors):
            circ.append(_ms_givens_gate(th), [f(p), f(p + 1)])
    return circ


def ms_df_fragments(d):
    """Factorización doble exacta (sin truncar): F_0 = e_nuc + h', F_ℓ = λ_ℓ/2 (Σ L^ℓ_pq E_pq)^2."""
    ints = _ms_integrals(d)
    if ints is None:
        return None
    h, eri = ints
    e_nuc = HAM[_MS_DKEY[d]][1]
    n = _MS_NORB
    h1 = h - 0.5 * np.einsum("pqqs->ps", eri)
    lam, vec = np.linalg.eigh(eri.reshape(n * n, n * n))
    keep = lam > 1e-10
    parts = [(h1, None)] + [(vec[:, l].reshape(n, n), lam[l]) for l in np.flatnonzero(keep)]
    frags, total = [], np.zeros((_DIM, _DIM))
    for A, lam_l in parts:
        A = 0.5 * (A + A.T)
        eps, U = np.linalg.eigh(A)
        O1 = _ms_one_body(A)
        O = O1 + e_nuc * np.eye(_DIM) if lam_l is None else 0.5 * lam_l * O1 @ O1
        total += O
        circ = _ms_orbital_rotation(U).inverse()  # V = R(U)†: lleva Σ ε_k b†_k b_k a Σ ε_k n_k
        frags.append(_ms_fragment(circ, O, "DF"))
    H_full = HAM[_MS_DKEY[d]][0].to_matrix().real + e_nuc * np.eye(_DIM)
    assert np.abs(total - H_full).max() < 1e-8, "la suma de fragmentos DF no reproduce H"
    return frags

In [ ]:
# --- 19.3 Construcción de los cuatro esquemas en 1,8 / 2,4 / 3,0 Å ----------------------------------
_t = time.time()
MS_SCHEMES, MS_CANDS = {}, []
for i, d in enumerate(MS_D):
    P = MS_P[d]
    pos = {int(k): j for j, k in enumerate(P["idx"])}
    base_groups = [[pos[k] for k in g["terms"]] for g in qwc_groups(P["op"])[1]]  # la partición exacta de #6
    qwc_best, qwc_name, rows_a = _ms_best_partition(P, P["qwc"], MS_SEED + 10 * i)
    gc_best, gc_name, rows_b = _ms_best_partition(P, P["comm"], MS_SEED + 10 * i + 1)
    MS_CANDS += [dict(d=d, esquema="(a) QWC", candidata=r[0], grupos=r[1], K_Ha=r[2]) for r in rows_a]
    MS_CANDS += [dict(d=d, esquema="(b) GC", candidata=r[0], grupos=r[1], K_Ha=r[2]) for r in rows_b]
    MS_SCHEMES[d] = {
        "QWC #6": (P["e_nuc"] + P["c_id"], ms_qwc_fragments(P, base_groups), "—"),
        "(a) QWC mejorado": (P["e_nuc"] + P["c_id"], ms_qwc_fragments(P, qwc_best), qwc_name),
        "(b) conmutantes + Clifford": (P["e_nuc"] + P["c_id"], ms_gc_fragments(P, gc_best), gc_name),
    }
    _df = ms_df_fragments(d)
    if _df is not None:
        MS_SCHEMES[d]["(c) rotación de base (DF)"] = (0.0, _df, "factorización doble exacta")
print(f"Esquemas construidos y validados (V O V† diagonal, suma = H) en {time.time() - _t:.1f} s.")
print(pd.DataFrame(MS_CANDS).assign(K_Ha=lambda t: t.K_Ha.round(4)).to_string(index=False))

In [ ]:
# --- 19.4 Shots para σ = 1 mHa (exacto) y energía medida con SamplerV2 -------------------------------


def _ms_exact(psi, frags):
    """(media, varianza por shot) exactas de cada fragmento sobre ψ, y fracción de probabilidad en el sector N=4, Sz=0."""
    out = []
    for f in frags:
        p = np.abs(Statevector(psi).evolve(f["circ"]).data) ** 2
        mu = p @ f["table"]
        out.append((mu, p @ (f["table"] - mu) ** 2))
    return np.array(out)


def ms_measure(circuit_bound, c0, frags, var_exact, total_shots, seed, postselect=False):
    """Energía con SamplerV2: N_g ∝ sqrt(Var_g) shots para el fragmento g, una llamada de SamplerV2 por fragmento con
    semilla propia (sacada de SeedSequence) para que los fragmentos sean independientes. Con postselect descarta shots
    fuera de N=4, Sz=0 (solo tiene sentido si el circuito de medición conserva N_α y N_β, como en DF)."""
    sq = np.sqrt(np.maximum(var_exact, 1e-16))
    n_g = np.maximum(MIN_GROUP_SHOTS, np.round(total_shots * sq / sq.sum())).astype(int)
    seeds = np.random.SeedSequence(int(seed)).generate_state(len(frags)) % (2 ** 31 - 1)
    counts = np.zeros((len(frags), _DIM))
    for g, f in enumerate(frags):
        qc = circuit_bound.compose(f["circ"])
        qc.measure_all()  # sin transpilar: Aer ejecuta h, s, cx, swap y unitary nativamente
        arr = SamplerV2(seed=int(seeds[g])).run([(qc, None, int(n_g[g]))]).result()[0].data.meas.array.astype(np.int64)
        ints = np.zeros(arr.shape[0], dtype=np.int64)
        for byte in range(arr.shape[1]):
            ints = (ints << 8) | arr[:, byte]
        counts[g] = np.bincount(ints, minlength=_DIM)
    raw = counts.sum()
    if postselect:
        counts[:, ~_MS_SECTOR_MASK] = 0
    n = counts.sum(axis=1)
    tables = np.array([f["table"] for f in frags])
    means = (counts * tables).sum(axis=1) / n
    vars_ = (counts * (tables - means[:, None]) ** 2).sum(axis=1) / (n - 1)
    return dict(energy=float(c0 + means.sum()), sigma=float(np.sqrt((vars_ / n).sum())), shots=int(raw),
                kept=float(n.sum() / raw))


_t = time.time()
_sig_run = MS_SIGMA_RUN / np.sqrt(SHOTS_SCALE)
MS_ROWS = []
for i, d in enumerate(MS_D):
    dk = _MS_DKEY[d]
    for j, (name, (c0, frags, _)) in enumerate(MS_SCHEMES[d].items()):
        ex = _ms_exact(MS_P[d]["psi"], frags)
        K = float(np.sqrt(ex[:, 1]).sum())
        e_exact = c0 + ex[:, 0].sum()
        assert abs(e_exact - E_SV[dk]) < 1e-8, f"{name} d={d}: la suma de fragmentos no da <H>"
        n_keep = sum(f["keeps_N"] for f in frags)
        if name == "QWC #6":  # la línea de base ya está medida en #6: no la repetimos
            r = dict(energy=MEAS[dk]["energy"], sigma=MEAS[dk]["sigma"], shots=MEAS[dk]["shots"], kept=np.nan)
        else:
            r = ms_measure(CIRCUIT[dk], c0, frags, ex[:, 1], int(np.ceil((K / _sig_run) ** 2)), MS_SEED + 100 * i + j,
                           postselect=n_keep == len(frags))
        MS_ROWS.append(dict(
            d=d, esquema=name, circuitos=len(frags), K_Ha=K, shots_1mHa=(K / MS_SIGMA) ** 2,
            cz_med_max=max(f["cz"] for f in frags), cz_med_total=sum(f["cz"] for f in frags),
            postsel_N_Sz="sí (todos)" if n_keep == len(frags) else f"{n_keep}/{len(frags)} circuitos",
            E=r["energy"], sigma=r["sigma"], shots=r["shots"], kept=r["kept"],
            z_vs_6=(r["energy"] - MEAS[dk]["energy"]) / np.hypot(r["sigma"], MEAS[dk]["sigma"]),
            z_vs_sv=(r["energy"] - E_SV[dk]) / r["sigma"]))
ms = pd.DataFrame(MS_ROWS)
ms["ahorro_vs_6"] = [ms.shots_1mHa[(ms.d == d) & (ms.esquema == "QWC #6")].iloc[0] / s for d, s in zip(ms.d, ms.shots_1mHa)]
print(f"Corridas con SamplerV2 en {time.time() - _t:.1f} s (σ objetivo de la corrida = {_sig_run * 1e3:.2f} mHa).")
print(ms[["d", "esquema", "circuitos", "shots_1mHa", "ahorro_vs_6", "cz_med_max", "cz_med_total", "postsel_N_Sz"]]
      .assign(shots_1mHa=lambda t: t.shots_1mHa.map(lambda v: f"{v:.2e}"), ahorro_vs_6=lambda t: t.ahorro_vs_6.round(2))
      .to_string(index=False))
print()
print(ms[["d", "esquema", "E", "sigma", "shots", "kept", "z_vs_6", "z_vs_sv"]]
      .assign(sigma_mHa=lambda t: (t.sigma * 1e3).round(3), E=lambda t: t.E.round(5),
              z_vs_6=lambda t: t.z_vs_6.round(2), z_vs_sv=lambda t: t.z_vs_sv.round(2))
      .drop(columns="sigma").to_string(index=False))
print(f"#6 en esas distancias: " + ", ".join(f"{d}: {MEAS[_MS_DKEY[d]]['energy']:.5f} ± {MEAS[_MS_DKEY[d]]['sigma'] * 1e3:.2f} mHa"
                                            for d in MS_D))
assert (ms.z_vs_6.abs() < 3).all(), "algún esquema se aparta de #6 en más de 3σ"

In [ ]:
# --- 19.5 Figura: shots para σ = 1 mHa por esquema ---------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 3.6))
names = list(dict.fromkeys(n for d in MS_D for n in MS_SCHEMES[d]))
w = 0.8 / len(names)
for j, name in enumerate(names):
    vals = [ms.shots_1mHa[(ms.d == d) & (ms.esquema == name)].max() for d in MS_D]  # NaN si el esquema falta
    ax.bar(np.arange(len(MS_D)) + (j - (len(names) - 1) / 2) * w, vals, w, label=name)
ax.set_yscale("log")
ax.set_xticks(range(len(MS_D)), [f"{d} Å" for d in MS_D])
ax.set_ylabel("shots para σ = 1 mHa")
ax.set_title("Esquemas de medición sobre ψ(θ*)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"Tiempo total de la sección de #19: {time.time() - _T0_MS:.0f} s")

#### Conclusión

Sobre el mismo estado que mide #6 (los shots para σ = 1 mHa no dependen de `SHOTS_SCALE`):

- **(c) Rotación de base (DF)** es el gran ganador: 10 circuitos y $3{,}7\times10^5$ shots a 1,8 Å (**6×** menos que los
  $2{,}3\times10^6$ de QWC #6; coincide con los ~$3{,}4\times10^5$ del análisis de la rama de referencia), y el ahorro crece
  al estirar (~37× a 2,4 Å, ~300× a 3,0 Å). Cuesta como mucho **24 `cz`** extra por circuito (red de Givens, 6 por espín) y,
  como la rotación conserva $N_\alpha$ y $N_\beta$, **permite post-seleccionar** por N y $S_z$ (útil para #13).
- **(b) Conmutantes + Clifford**: 9–13 circuitos y 2,2–3,4× menos shots, pero hasta 69–81 `cz` por circuito y sin
  post-selección. Con ruido no conviene frente a DF.
- **(a) QWC mejorado**: entre ~100 particiones candidatas la mejor apenas gana 0–5 %: *sorted insertion* ya está cerca del óptimo.
- Las energías medidas con (a), (b) y (c) coinciden con la de #6 dentro de 3σ combinada (los valores de $z$
  de la tabla de 19.4 cambian con `SHOTS_SCALE` y la semilla; los apartamientos más grandes vinieron del propio punto de #6) y todas
  están a < 2σ de $E_\text{sv}$.

**Recomendación:** proponer DF para el resultado principal de #6 (mismo σ con 6× menos shots en 1,8 Å, donde el margen
es más chico). Ojo: sus `cz` de medición hay que reportarlos aparte de los del ansatz, y con ruido pesan.

#### Qué decidimos, por qué y qué no funcionó

- **Varianzas exactas para diseñar, shots para medir.** La partición de (a)/(b), el reparto $N_g\propto\sqrt{\operatorname{Var}_g}$
  y la columna "shots para σ = 1 mHa" usan la varianza exacta del valor por shot de cada fragmento sobre el statevector
  (es diseño, como buscar parámetros). La energía de cada esquema sale de `SamplerV2` con shots reales, y la σ medida se
  estima con las varianzas muestrales.
- **Validación sin convenciones.** Cada fragmento se acepta solo si $V_g O_g V_g^\dagger$ es diagonal; la tabla por shot es
  esa diagonal. Así los signos de Pauli, el orden de bits y la dirección de la rotación de orbitales quedan verificados
  numéricamente (además, $\sum_g O_g = H$ y $\sum_g \langle O_g\rangle = E_\text{sv}$).
- **Integrales para DF.** Las calculamos con PySCF (RHF/STO-3G) y elegimos los signos de los orbitales que reproducen el
  Hamiltoniano de #2 como operador; si #2 usa otra convención, la búsqueda de signos lo absorbe (y si no alcanza, avisa
  y saltea DF en esa distancia en lugar de cortar el notebook). La factorización es exacta (no truncamos autovalores > 1e-10), así que no hay sesgo.
- **Rotaciones de Givens.** Descomponemos $U$ en rotaciones de modos adyacentes (que en JW no tienen cuerda de Z dentro
  del bloque α o β) más signos (`Z`); cada rotación es un `UnitaryGate` de 2 qubits que `transpile_line` deja en 2 `cz`.
- **Diagonalización Clifford.** Generadores independientes del grupo (eliminación en GF(2)) → `StabilizerState.from_stabilizer_list`
  (completa los estabilizadores que faltan) → medimos tras $C^\dagger$. Es la síntesis genérica de Qiskit: no optimizamos
  esos `cz` (una síntesis a medida bajaría algo, pero no cambia la conclusión frente a DF).
- **Una llamada de `SamplerV2` por fragmento, con semilla propia.** Juntar todos los circuitos en un job (como #6 con los
  bindings) obligaba a repetir PUBs y costaba ~40 ms por PUB; con PUBs de distinto número de shots Aer reutiliza la
  semilla y correlaciona los fragmentos (lo mismo que vio #6). Tampoco transpilamos para Aer: armar el `target` del
  backend en cada `transpile` dominaba el tiempo.
- **σ de la corrida = 2 mHa** (a escala completa) y no 1 mHa, para que la sección sea rápida: la comprobación "dentro de 2σ
  de #6" no necesita más, y la relación $\sigma = K/\sqrt N$ ya la validó #6.
- **No funcionó / no ayudó:** los coloreos greedy de grafos (Degree, Saturation, IndependentSet) dan menos grupos en
  algunos casos pero **más** $K$ que *sorted insertion*: minimizar el número de grupos no es minimizar los shots.
  La línea de base QWC #6 no se vuelve a medir: usamos directamente la energía y σ de #6.

### Mitigación de errores (#13)

¿Cuánto recupera la mitigación que funcionaría en una computadora real? Medimos con el modelo de ruido común (`ruido.py`: 0,1 % por `cz`, 0,005 % por compuerta de un qubit, qubits en línea) en un punto estirado, $d = 1{,}8$ Å, y aplicamos tres técnicas:

- **Post-selección por simetría** (Bonet-Monroig et al., PRA 98, 062339): el estado tiene $N_\alpha = N_\beta = 2$, así que en el grupo que se mide en base Z se descartan los shots con otra ocupación. Es gratis en circuitos y solo cuesta los shots descartados.
- **Zero Noise Extrapolation** (Temme, Bravyi y Gambetta, PRL 119, 180509): se amplifica el ruido plegando cada `cz` (`cz → cz cz cz`, la misma unidad con el triple de ruido), se mide con factores λ = 1, 3, 5 y se extrapola a λ = 0. Probamos tres modelos: lineal, Richardson (cuadrática por los 3 puntos) y exponencial $E(\lambda) = E_\infty - C\,g^{(\lambda-1)/2}$, que respeta que el ruido satura hacia el estado mezclado.
- **Corrección de lectura**: `ruido.py` no tiene error de lectura, así que para probarla le sumamos un 1 % por qubit, calibramos la matriz de confusión tensorizada con dos circuitos ($|0\cdots0\rangle$ y $|1\cdots1\rangle$) y corregimos el valor de cada bitstring con $(A^{-1})^\top$.

Comparamos dos circuitos: el **ansatz completo de #4** (465 `cz`, el que usamos para la curva) y una **versión corta de una sola doble** (30 `cz`), para ver el régimen de pocas compuertas. Reportamos dos errores:

- `err_FCI`: contra la respuesta exacta;
- `sesgo_ruido`: contra el **mismo circuito sin ruido**, que es lo único que la mitigación puede corregir (la versión corta está lejos de FCI aunque no haya ruido).

Usa el `ruido.py` que escribe la celda `%%writefile` de #11 (el mismo archivo que `notebooks/ruido.py` del repo).

**Resultados guardados.** Medir con ruido todas las variantes tarda ~1 min, así que por defecto la celda usa las tablas guardadas en `MITIGACION_CACHE` (de una corrida de esa misma celda con `RUN_MITIGACION = True`): `mitigacion`, `lectura`, los puntos de la extrapolación y la calibración de lectura. Las tablas impresas y el gráfico salen de ahí con el mismo código. Para volver a medir: `RUN_MITIGACION = True`.

In [ ]:
# Resultados guardados de la mitigación (#13). Volver a medirlos tarda ~1 min, así que la celda siguiente los usa
# por defecto (RUN_MITIGACION = False). Salieron de esa misma celda con True; para regenerarlos: poner True, correr y
# pegar acá results/mitigacion.csv, results/mitigacion_lectura.csv, results/mitigacion_zne.csv y
# results/mitigacion_calibracion.csv.
MITIGACION_CACHE = {
    "origen": "corrida del 2026-11-21 con RUN_ANSATZ5 = RUN_MITIGACION = True, qiskit 2.5.2, qiskit-aer 0.17.2, pyscf 2.14.0",
    "mitigacion": """\
circuito,cz,d,método,shots,err_FCI_mHa,sigma_mHa,sesgo_ruido_mHa,err_sv_mHa,ccsd_mHa,aceptados_PS
ansatz completo (#4),465,1.8,sin mitigar,131071,317.271,6.9088,317.271,2.68674e-10,-9.10764,1
ansatz completo (#4),465,1.8,post-selección,131071,207.87,6.8617,207.87,2.68674e-10,-9.10764,0.718245
ansatz completo (#4),465,1.8,ZNE lineal,393213,235.006,8.37119,235.006,2.68674e-10,-9.10764,1
ansatz completo (#4),465,1.8,ZNE Richardson,393213,97.9144,15.8534,97.9144,2.68674e-10,-9.10764,1
ansatz completo (#4),465,1.8,ZNE exponencial,393213,35.316,29.4907,35.316,2.68674e-10,-9.10764,1
ansatz completo (#4),465,1.8,ZNE lineal + PS,393213,145.614,8.32187,145.614,2.68674e-10,-9.10764,0.718245
ansatz completo (#4),465,1.8,ZNE Richardson + PS,393213,57.7721,15.7586,57.7721,2.68674e-10,-9.10764,0.718245
ansatz completo (#4),465,1.8,ZNE exponencial + PS,393213,24.1177,26.6512,24.1177,2.68674e-10,-9.10764,0.718245
corto: 1 doble,30,1.8,sin mitigar,2097151,203.48,1.72871,6.57687,196.903,-9.10764,1
corto: 1 doble,30,1.8,post-selección,2097151,200.148,1.72862,3.24513,196.903,-9.10764,0.9818
corto: 1 doble,30,1.8,ZNE lineal,6291453,196.897,2.08802,-0.00588838,196.903,-9.10764,1
corto: 1 doble,30,1.8,ZNE Richardson,6291453,195.368,3.9493,-1.53491,196.903,-9.10764,1
corto: 1 doble,30,1.8,ZNE exponencial,6291453,195.279,4.29228,-1.6241,196.903,-9.10764,1
corto: 1 doble,30,1.8,ZNE lineal + PS,6291453,196.177,2.08786,-0.72541,196.903,-9.10764,0.9818
corto: 1 doble,30,1.8,ZNE Richardson + PS,6291453,194.904,3.94899,-1.99871,196.903,-9.10764,0.9818
corto: 1 doble,30,1.8,ZNE exponencial + PS,6291453,194.801,4.42986,-2.10152,196.903,-9.10764,0.9818
""",
    "lectura": """\
circuito,d,método,err_FCI_mHa,sigma_mHa,sesgo_ruido_mHa
corto: 1 doble,1.8,ruido.py (sin error de lectura),203.48,1.72871,6.57687
corto: 1 doble,1.8,"+ lectura, sin corregir",225.881,1.73028,28.9786
corto: 1 doble,1.8,"+ lectura, post-selección",205.843,1.72947,8.9401
corto: 1 doble,1.8,"+ lectura, corrección de lectura",200.878,1.92777,3.97552
""",
    "zne": """\
circuito,d,serie,lam_1,lam_3,lam_5
ansatz completo (#4),1.8,crudo,317.271,625.181,758.684
ansatz completo (#4),1.8,PS,207.87,424.107,528.399
corto: 1 doble,1.8,crudo,203.48,218.254,231.097
corto: 1 doble,1.8,PS,200.148,209.429,217.101
""",
    "calibracion": """\
q,p1_si_0,p1_si_1
0,0.0100403,0.990143
1,0.00976562,0.988983
2,0.0101624,0.98938
3,0.00973511,0.989868
4,0.0102539,0.989563
5,0.00979614,0.989929
6,0.00967407,0.990479
7,0.00946045,0.990082
""",
}

In [ ]:
# Extras · Mitigación de errores (#13): post-selección por simetría, ZNE por plegado de cz y corrección de lectura.
# Todo se mide con SamplerV2 de qiskit_aer sobre el modelo de ruido.py (celda anterior). El statevector solo se usa
# para buscar parámetros y como referencia "sin ruido" del mismo circuito (diagnóstico, no resultado).
import importlib
import io
import time

import ruido
from qiskit.circuit import ParameterVector
from qiskit_aer.noise import ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler

importlib.reload(ruido)  # la versión que escribió la celda anterior
_t13 = time.perf_counter()
RUN_MITIGACION = False  # True: volver a medir (~1 min). False: usar las tablas guardadas en MITIGACION_CACHE (celda de arriba)

D_MIT = (1.8,)       # punto estirado donde CCSD deja menos margen (con 2,4 Å las conclusiones son las mismas)
FACTORES = (1, 3, 5)  # plegado de cz: cz -> cz cz cz (= cz · cz† · cz), el ruido de cada cz se multiplica por λ
# Shots por punto y por factor, repartidos entre los grupos según la norma 1 de sus coeficientes. El circuito corto
# lleva más: su sesgo de ruido es de ~20 mHa y hace falta σ de pocos mHa para ver la extrapolación.
SHOTS_MIT = {"ansatz completo (#4)": 2 ** 17, "corto: 1 doble": 2 ** 21}
P_LECTURA = 0.01      # error de lectura agregado SOLO para probar su corrección: ruido.py no tiene error de lectura
SHOTS_CALIB = 2 ** 15
_rng13 = np.random.default_rng(13)


# --- Circuitos: el ansatz de #4 y una versión corta de una sola doble (para ver el régimen de pocas cz) -------------
def circuito_de(ops) -> QuantumCircuit:
    """Mismo constructor que build_ansatz (#4), con otra lista de excitaciones."""
    th = ParameterVector("θ", len(ops))
    qc = QuantumCircuit(N_QUBITS)
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for t, (_, kind, p, q) in zip(th, ops):
        if kind == "D":
            givens_double(qc, t, alpha_qubit(p), beta_qubit(p), alpha_qubit(q), beta_qubit(q))
        else:
            givens_single(qc, t, alpha_qubit(p), alpha_qubit(q))
            givens_single(qc, t, beta_qubit(p), beta_qubit(q))
    return qc


OPS_CORTO = [(0, "D", 1, 2)]  # la doble que más baja la energía sola (selección greedy, ver markdown)
_corto = SectorAnsatz(OPS_CORTO)
theta_corto = {d: min((minimize(_corto.energy_grad, x0, args=(H_SECTOR[d][0],), jac=True, method="BFGS")
                       for x0 in (np.zeros(1), np.full(1, 0.5), np.full(1, -0.5))), key=lambda r: r.fun).x
               for d in D_MIT}
CIRCUITOS = {"ansatz completo (#4)": (transpile_line(ansatz), lambda d: params[d]),
             "corto: 1 doble": (transpile_line(circuito_de(OPS_CORTO)), lambda d: theta_corto[d])}


def plegar(qc: QuantumCircuit, factor: int) -> QuantumCircuit:
    """Cada cz se repite `factor` veces (factor impar): misma unidad, `factor` veces el ruido de las cz."""
    out = qc.copy_empty_like()
    for inst in qc.data:
        for _ in range(factor if inst.operation.name == "cz" else 1):
            out.append(inst)
    return out


# --- Medición por grupos QWC con SamplerV2 (una semilla por pub: con semilla fija Aer las repite entre pubs) --------
_BITS = (np.arange(2 ** N_QUBITS)[:, None] >> np.arange(N_QUBITS)) & 1  # bit del qubit q en cada bitstring


def _base_del_grupo(g: SparsePauliOp) -> str:
    return "".join(max(col, key=lambda ch: ch != "I") for col in zip(*g.paulis.to_labels()))


def _valores(g: SparsePauliOp) -> np.ndarray:
    """Valor del grupo (suma de sus términos) para cada bitstring medido en la base del grupo."""
    masks = np.array([[ch != "I" for ch in reversed(l)] for l in g.paulis.to_labels()], dtype=int)
    return ((-1.0) ** (_BITS @ masks.T)) @ g.coeffs.real


def medir(circ: QuantumCircuit, op_l: SparsePauliOp, noise_model, shots: int) -> list:
    """Cuentas por grupo. circ: transpilado y con parámetros asignados; op_l: observable con su layout.
    Los términos diagonales (solo Z) van juntos en un grupo propio: es el único que se puede post-seleccionar, y si
    se mezclan con X/Y en otros qubits (lo que hace el agrupamiento QWC greedy) no queda ningún grupo en base Z."""
    labels = op_l.paulis.to_labels()
    diag = [i for i, l in enumerate(labels) if set(l) <= {"I", "Z"}]
    resto = [i for i in range(len(labels)) if i not in diag]
    grupos = [op_l[diag]] + op_l[resto].group_commuting(qubit_wise=True)
    peso = np.array([np.abs(g.coeffs).sum() for g in grupos])
    n = np.maximum(200, np.round(shots * peso / peso.sum())).astype(int)
    opciones = {"backend_options": {"method": "density_matrix", "noise_model": noise_model}}
    out = []
    for g, ng in zip(grupos, n):
        m = circ.copy()
        for q, b in enumerate(reversed(_base_del_grupo(g))):
            if b == "X":
                m.h(q)
            elif b == "Y":
                m.sdg(q)
                m.h(q)
        m.measure_all()
        m = transpile(m, basis_gates=ruido.COMPUERTAS_BASE, optimization_level=0)  # nivel 0: no deshace el plegado
        cnt = AerSampler(seed=int(_rng13.integers(2 ** 31)), options=opciones).run([m], shots=int(ng)).result()[0]
        out.append((g, cnt.data.meas.get_int_counts()))
    return out


def estimar(medidas: list, e_nuc: float, filtro=None, correccion=None) -> tuple[float, float, int, float]:
    """E y σ desde las cuentas. filtro: máscara de bitstrings aceptados (post-selección, solo en grupos en base Z).
    correccion: matriz (A⁻¹)ᵀ de lectura; corrige el valor por bitstring, así E sigue siendo un promedio por shot.
    Devuelve (E, σ, shots usados, fracción aceptada en el grupo en base Z)."""
    E, var, n_usados, aceptados = e_nuc, 0.0, 0, 1.0
    for g, cnt in medidas:
        v = _valores(g) if correccion is None else correccion @ _valores(g)
        idx, nb = np.fromiter(cnt.keys(), int), np.fromiter(cnt.values(), float)
        if filtro is not None and set(_base_del_grupo(g)) <= {"Z", "I"}:
            aceptados = nb[filtro[idx]].sum() / nb.sum()
            idx, nb = idx[filtro[idx]], nb[filtro[idx]]
        N = nb.sum()
        media = nb @ v[idx] / N
        E += media
        var += (nb @ (v[idx] - media) ** 2) / (N - 1) / N
        n_usados += int(N)
    return E, np.sqrt(var), n_usados, aceptados


def filtro_simetria(tqc: QuantumCircuit) -> np.ndarray:
    """Bitstrings (qubits físicos del circuito transpilado) con N_α = 2 y N_β = 2."""
    fis = tqc.layout.final_index_layout()  # qubit virtual -> qubit físico al final del circuito
    occ = _BITS[:, fis]
    return (occ[:, :N_ORB].sum(1) == N_ALPHA) & (occ[:, N_ORB:].sum(1) == N_BETA)


# --- Extrapolación a ruido cero, con σ propagada ------------------------------------------------------------------
def zne(E: np.ndarray, S: np.ndarray, lam=np.array(FACTORES, float)) -> dict:
    """Lineal (mínimos cuadrados pesados), Richardson (cuadrática exacta por los 3 puntos) y exponencial
    E(λ) = E∞ − C·g^((λ−1)/2), que respeta que el ruido satura hacia el estado mezclado."""
    W = np.diag(1 / S ** 2)
    X = np.vstack([np.ones_like(lam), lam]).T
    cov = np.linalg.inv(X.T @ W @ X)
    lin = (cov @ X.T @ W @ E)[0]
    gam = np.array([np.prod([lj / (lj - li) for lj in lam if lj != li]) for li in lam])  # Lagrange en λ = 0

    def exp_fit(y):
        g = (y[2] - y[1]) / (y[1] - y[0])
        if not 0 < g < 1:
            return np.nan
        C = (y[1] - y[0]) / (1 - g)
        return y[0] + C - C * g ** -0.5

    e_exp = exp_fit(E)
    J = np.array([(exp_fit(E + h) - exp_fit(E - h)) / (2 * 1e-6) for h in 1e-6 * np.eye(3)])  # método delta
    return {"ZNE lineal": (lin, np.sqrt(cov[0, 0])), "ZNE Richardson": (gam @ E, np.sqrt(gam ** 2 @ S ** 2)),
            "ZNE exponencial": (e_exp, np.sqrt(J ** 2 @ S ** 2))}


# --- 1) Post-selección y ZNE con el ruido de ruido.py ---------------------------------------------------------------
if RUN_MITIGACION:
    NM = ruido.modelo_de_ruido()
    e_fci_m, e_ccsd_m = refs.set_index("d").e_fci, refs.set_index("d").e_ccsd
    filas, zne_puntos, medidas_l1 = [], {}, {}
    for nombre, (tqc, theta_de) in CIRCUITOS.items():
        filtro = filtro_simetria(tqc)
        cz = tqc.count_ops()["cz"]
        for d in D_MIT:
            op, e_nuc = qubit_hamiltonian(d)
            op_l = op.apply_layout(tqc.layout)
            circ = tqc.assign_parameters(theta_de(d))
            e_sv = Statevector(circ).expectation_value(op_l).real + e_nuc  # mismo circuito sin ruido (diagnóstico)
            res = {lam: medir(plegar(circ, lam), op_l, NM, SHOTS_MIT[nombre]) for lam in FACTORES}
            medidas_l1[nombre, d] = res[1]  # se reusa como referencia en la parte de lectura
            crudo = {lam: estimar(res[lam], e_nuc) for lam in FACTORES}
            ps = {lam: estimar(res[lam], e_nuc, filtro=filtro) for lam in FACTORES}
            shots_1 = crudo[1][2]
            zne_puntos[nombre, d] = {k: (np.array([t[l][0] for l in FACTORES]) - e_fci_m[d]) * 1e3 for k, t in (("crudo", crudo), ("PS", ps))}
            metodos = {"sin mitigar": crudo[1][:2], "post-selección": ps[1][:2]}
            for base, tabla in (("", crudo), (" + PS", ps)):
                for k, v in zne(np.array([tabla[l][0] for l in FACTORES]), np.array([tabla[l][1] for l in FACTORES])).items():
                    metodos[k + base] = v
            for metodo, (E, s) in metodos.items():
                filas.append({"circuito": nombre, "cz": cz, "d": d, "método": metodo,
                              "shots": shots_1 * (len(FACTORES) if metodo.startswith("ZNE") else 1),
                              "err_FCI_mHa": (E - e_fci_m[d]) * 1e3, "sigma_mHa": s * 1e3,
                              "sesgo_ruido_mHa": (E - e_sv) * 1e3, "err_sv_mHa": (e_sv - e_fci_m[d]) * 1e3,
                              "ccsd_mHa": (e_ccsd_m[d] - e_fci_m[d]) * 1e3,
                              "aceptados_PS": ps[1][3] if "PS" in metodo or metodo == "post-selección" else 1.0})
    mitigacion = pd.DataFrame(filas)

    # --- 2) Lectura: ruido.py + 1 % de error de lectura, calibración tensorizada por qubit y corrección -----------------
    NM_LECTURA = ruido.modelo_de_ruido()
    NM_LECTURA.add_all_qubit_readout_error(ReadoutError([[1 - P_LECTURA, P_LECTURA], [P_LECTURA, 1 - P_LECTURA]]))


    def calibrar_lectura(noise_model) -> np.ndarray:
        """(A⁻¹)ᵀ con A = ⊗_q A_q, estimada preparando |0…0⟩ y |1…1⟩ (2 circuitos, el mínimo para la versión por qubit)."""
        marg = []
        for todos_uno in (False, True):
            c = QuantumCircuit(N_QUBITS)
            if todos_uno:
                c.x(range(N_QUBITS))
            c.measure_all()
            cnt = AerSampler(seed=int(_rng13.integers(2 ** 31)), options={"backend_options": {
                "method": "density_matrix", "noise_model": noise_model}}).run(
                [transpile(c, basis_gates=ruido.COMPUERTAS_BASE, optimization_level=0)], shots=SHOTS_CALIB).result()[0]
            idx, nb = np.fromiter((cc := cnt.data.meas.get_int_counts()).keys(), int), np.fromiter(cc.values(), float)
            marg.append((nb @ _BITS[idx]) / nb.sum())  # P(leer 1) por qubit
        A_inv = np.ones((1, 1))
        for q in range(N_QUBITS):  # kron(A_7, …, A_0): el qubit 0 es el bit menos significativo
            e0, e1 = marg[0][q], 1 - marg[1][q]  # P(1|0), P(0|1)
            A_inv = np.kron(np.linalg.inv(np.array([[1 - e0, e1], [e0, 1 - e1]])), A_inv)
        return A_inv.T, marg


    CORR_LECTURA, _marg = calibrar_lectura(NM_LECTURA)
    filas_l = []
    for nombre in ["corto: 1 doble"]:  # el error de lectura no depende de la profundidad: alcanza con el circuito corto
        tqc, theta_de = CIRCUITOS[nombre]
        filtro = filtro_simetria(tqc)
        for d in D_MIT:
            op, e_nuc = qubit_hamiltonian(d)
            op_l = op.apply_layout(tqc.layout)
            circ = tqc.assign_parameters(theta_de(d))
            e_sv = Statevector(circ).expectation_value(op_l).real + e_nuc
            r_sin = medidas_l1[nombre, d]                              # ruido.py solo, λ = 1 (lo que habría que recuperar)
            r_con = medir(circ, op_l, NM_LECTURA, SHOTS_MIT[nombre])   # ruido.py + lectura
            for metodo, (E, s, _, _) in {"ruido.py (sin error de lectura)": estimar(r_sin, e_nuc),
                                      "+ lectura, sin corregir": estimar(r_con, e_nuc),
                                      "+ lectura, post-selección": estimar(r_con, e_nuc, filtro=filtro),
                                      "+ lectura, corrección de lectura": estimar(r_con, e_nuc, correccion=CORR_LECTURA)}.items():
                filas_l.append({"circuito": nombre, "d": d, "método": metodo, "err_FCI_mHa": (E - e_fci_m[d]) * 1e3,
                                "sigma_mHa": s * 1e3, "sesgo_ruido_mHa": (E - e_sv) * 1e3})
    lectura = pd.DataFrame(filas_l)
    os.makedirs("results", exist_ok=True)
    mitigacion.to_csv("results/mitigacion.csv", index=False)
    lectura.to_csv("results/mitigacion_lectura.csv", index=False)
    pd.DataFrame([{"circuito": n, "d": d, "serie": k, **{f"lam_{l}": v for l, v in zip(FACTORES, arr)}}
                  for (n, d), series in zne_puntos.items() for k, arr in series.items()]).to_csv(
        "results/mitigacion_zne.csv", index=False)
    pd.DataFrame({"q": range(N_QUBITS), "p1_si_0": _marg[0], "p1_si_1": _marg[1]}).to_csv(
        "results/mitigacion_calibracion.csv", index=False)
else:
    mitigacion = pd.read_csv(io.StringIO(MITIGACION_CACHE["mitigacion"]))
    lectura = pd.read_csv(io.StringIO(MITIGACION_CACHE["lectura"]))
    _z = pd.read_csv(io.StringIO(MITIGACION_CACHE["zne"]))
    zne_puntos = {}
    for _, r in _z.iterrows():
        zne_puntos.setdefault((r.circuito, float(r.d)), {})[r.serie] = np.array([r[f"lam_{l}"] for l in FACTORES], float)
    _c = pd.read_csv(io.StringIO(MITIGACION_CACHE["calibracion"]))
    _marg = [_c.p1_si_0.to_numpy(), _c.p1_si_1.to_numpy()]
    print(f"Resultados guardados ({MITIGACION_CACHE['origen']}). Para volver a medir: RUN_MITIGACION = True.")

print(f"Mitigación (#13) en {time.perf_counter() - _t13:.0f} s. Error de lectura calibrado por qubit: "
      f"P(1|0) = {_marg[0].min():.4f}–{_marg[0].max():.4f}, P(0|1) = {1 - _marg[1].max():.4f}–{1 - _marg[1].min():.4f} "
      f"(real: {P_LECTURA})")
_cols = ["circuito", "cz", "d", "método", "err_FCI_mHa", "sigma_mHa", "sesgo_ruido_mHa", "shots", "aceptados_PS"]
print("\nPost-selección y ZNE (ruido.py). sesgo_ruido = E − E(mismo circuito sin ruido); err_FCI = E − E_FCI (mHa):")
print(mitigacion[_cols].to_string(index=False, formatters={"aceptados_PS": "{:.2f}".format}, float_format=lambda v: f"{v:.1f}"))
print("\nLectura (ruido.py + 1 % de error de lectura):")
print(lectura.to_string(index=False, float_format=lambda v: f"{v:.1f}"))

# Gráfico: energía vs. factor de ruido λ y las tres extrapolaciones, con post-selección, para cada circuito a 1,8 Å.
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, nombre in zip(axs, CIRCUITOS):
    t = mitigacion[(mitigacion.circuito == nombre) & (mitigacion.d == D_MIT[0])].set_index("método")
    ref = t.err_sv_mHa.iloc[0]
    ax.axhline(0, color="k", lw=1, label="FCI")
    ax.axhline(ref, color="0.5", ls=":", label="mismo circuito sin ruido")
    ax.axhline(t.ccsd_mHa.iloc[0], color="C3", ls="--", label="CCSD")
    for k, c in (("crudo", "0.3"), ("PS", "C4")):
        ax.plot(FACTORES, zne_puntos[nombre, D_MIT[0]][k], "s-", color=c, label=f"medido ({'sin mitigar' if k == 'crudo' else 'post-selección'})")
    for k, (m, c) in enumerate((("ZNE lineal + PS", "C0"), ("ZNE Richardson + PS", "C1"), ("ZNE exponencial + PS", "C2"))):
        ax.errorbar(-0.15 + 0.15 * k, t.err_FCI_mHa[m], yerr=t.sigma_mHa[m], fmt="o", color=c, capsize=3, label=m)
    ax.set(xlabel="factor de ruido λ (cz plegadas)", ylabel="E − E_FCI (mHa)", xticks=[0, 1, 3, 5],
           title=f"{nombre}, {t.cz.iloc[0]} cz, d = {D_MIT[0]} Å")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

#### Resultados y decisiones (#13)

**Con el ansatz completo (465 `cz`) ninguna mitigación alcanza para ganarle a CCSD.** El ruido de `ruido.py` suma ~320 mHa a 1,8 Å, y lo mejor que conseguimos lo deja en ~24 ± 27 mHa, lejos del −9,1 mHa de CCSD y de la precisión química. Con 465 `cz` y 0,1 % por `cz`, más de un tercio de las corridas tiene al menos un error, y extrapolar desde ahí es frágil.

| $d = 1{,}8$ Å | Ansatz completo, 465 `cz` (mHa vs. FCI) | Corto, 30 `cz` (sesgo de ruido, mHa) | Shots |
|---|---:|---:|---|
| Sin mitigar | 317 ± 7 | 6,6 ± 1,7 | 1× |
| Post-selección (N, S_z) | 208 ± 7 | 3,2 ± 1,7 | 1× |
| ZNE lineal + PS | 146 ± 8 | −0,7 ± 2,1 | 3× |
| ZNE Richardson + PS | 58 ± 16 | −2,0 ± 3,9 | 3× |
| ZNE exponencial + PS | 24 ± 27 | −2,1 ± 4,4 | 3× |
| *CCSD* | *−9,1* | | |

El ansatz completo sin ruido está a 0,003 mHa de FCI, así que ahí error y sesgo coinciden. La versión corta está a ~197 mHa de FCI aun sin ruido: para ella reportamos solo el sesgo, que es lo que la mitigación puede corregir.

Números de una corrida (semillas fijas). Cambian dentro de σ entre corridas.

**Qué aporta cada técnica**

- **La post-selección es la más rentable.** No agrega circuitos y en el ansatz completo saca ~110 mHa de 320. Solo actúa sobre los términos diagonales: el grupo en base Z acepta ~70 % de los shots con λ = 1 (menos al plegar). Los grupos con X/Y no se pueden filtrar con mediciones QWC; para filtrar todo haría falta medir en la base rotada de #19.
- **La ZNE depende de qué tan profundo es el circuito.**
  - Con 30 `cz` el ruido crece casi lineal con λ y la extrapolación lineal elimina el sesgo (6,6 → −0,7 ± 2,1 mHa).
  - Con 465 `cz` la energía satura hacia el estado mezclado y la lineal subestima el ruido. La exponencial es la que mejor sigue esa forma, pero con la σ más grande.
- **La corrección de lectura funciona.** Con 1 % de error de lectura por qubit (agregado a `ruido.py` solo para esta prueba), el sesgo del circuito corto pasa de 6,6 a 29,0 mHa. La corrección tensorizada lo devuelve a 4,0 ± 1,9; la post-selección sola, a 8,9, porque también descarta bitstrings con un bit mal leído.

**Costo en shots.** La ZNE mide 3 circuitos por punto (3× shots) y además agranda la σ: con los mismos shots por factor, la Richardson la multiplica por √((15/8)² + (10/8)² + (3/8)²) ≈ 2,3 y la exponencial por ~2,5–4. Para la misma σ que sin mitigar hacen falta entre 3 × 2,3² ≈ 16 y ~50 veces más shots. La post-selección cuesta los shots descartados (~30 % del grupo diagonal). La corrección de lectura cuesta 2 circuitos de calibración y agranda la σ ~10 %.

**Qué no funcionó**

- **Post-selección con el agrupamiento QWC greedy.** Mezcla los términos diagonales con términos que tienen X/Y en otros qubits, y así no queda ningún grupo en base Z para filtrar: la primera versión no filtraba nada. Ahora los términos diagonales van en un grupo propio.
- **Versiones cortas del ansatz para la zona estirada.** Tomamos las k excitaciones de mayor |θ| o las elegimos de forma greedy, y con 1 a 6 excitaciones (30–170 `cz`) quedan a 115–345 mHa de FCI aun sin ruido. A 1,8 Å el estado es multirreferencia y no hay atajo barato. Por eso la versión corta solo sirve para estudiar la mitigación, no como resultado.
- **Medir con el método `statevector` de Aer.** Con ruido corre una trayectoria por shot y es 10–80 veces más lento que `density_matrix`.

**Limitaciones**

- El plegado multiplica solo el ruido de las `cz`. Las compuertas de un qubit no se pliegan, y con 0,005 % de error eso no importa.
- La ZNE supone que el ruido escala con λ: vale para este modelo (Pauli por compuerta) y es menos exacto en hardware real.
- La σ de la corrección de lectura no incluye la incertidumbre de la calibración (2¹⁵ shots por circuito, despreciable acá).
- Un solo punto estirado. Con 2,4 Å las conclusiones son las mismas: en una corrida previa, el ansatz completo pasó de ~300 a ~33 mHa con la exponencial.

### Hardware real de IBM (#22)

**Fuera del desafío central** (las reglas piden `qiskit_aer` para los resultados juzgados). Un punto estirado, $d = 2{,}4$ Å, corrido en el procesador **`ibm_marrakesh`** (IBM Heron r2, 156 qubits) el 21/11/2026, *job* `dc0urtmq4uuc73a3895g`, ~51 s de tiempo cuántico.

**Cómo se hizo.** El ansatz completo (465 `cz`) en hardware solo da ruido, así que armamos uno corto con los bloques de la sección 3: selección *greedy* tipo ADAPT de las excitaciones distintas, con los ángulos ajustados con statevector. Antes de gastar tiempo de QPU, **emulamos localmente** con el modelo de ruido de la calibración actual de `ibm_marrakesh` (`AerSimulator.from_backend`) y barrimos el tamaño del ansatz: con más operadores el ruido de los `cz` extra pesa más que lo que gana el ansatz, y el óptimo fue **2 operadores (28 `cz`)**. Los mismos circuitos ya transpilados se emularon y se mandaron al equipo, y se guardaron los grupos medidos junto con el *job*. Medición con `SamplerV2` (de `qiskit-ibm-runtime`) en 46 grupos que conmutan qubit a qubit, 4000 shots por grupo; $E$ y $\sigma$ salen de los bitstrings, como en #6.

La celda de abajo muestra los resultados guardados de esa corrida (no necesita credenciales). Para repetirla con una cuenta propia: `python scripts/ibm_hardware.py emulate|submit|fetch` (los *counts* crudos están en `docs/analysis/data/`).

In [ ]:
# Resultados guardados de la corrida en hardware real (#22). No hace llamadas a IBM.
E_FCI_24, E_HF_24 = -1.8746515824858947, -1.43645853998968            # Ha, d = 2,4 Å (de PySCF, misma convención que el contrato)
HW = dict(backend="ibm_marrakesh", job_id="dc0urtmq4uuc73a3895g", n_ops=2, cz=28,
          e_noiseless=-1.5887065113014929, e_hw=-1.4850326678157548, sigma_hw=0.005864600742728037,
          e_emulated=-1.5312892379165925, sigma_emulated=0.005761403777613794)  # emulación de los MISMOS circuitos
SCAN = [{"n_ops": 2, "cz": 28, "noiseless_mHa": 285.95, "emulated_mHa": 342.25, "sigma_mHa": 5.76}, {"n_ops": 4, "cz": 60, "noiseless_mHa": 242.09, "emulated_mHa": 361.23, "sigma_mHa": 6.18}, {"n_ops": 6, "cz": 114, "noiseless_mHa": 234.26, "emulated_mHa": 431.06, "sigma_mHa": 6.52}]

print(f"Emulación con el ruido de {HW['backend']} (error vs. FCI, mHa):")
print(pd.DataFrame(SCAN).to_string(index=False))
tabla_hw = pd.DataFrame([
    ("Hartree-Fock", E_HF_24, None),
    (f"{HW['n_ops']} operadores, sin ruido (statevector)", HW["e_noiseless"], None),
    (f"{HW['n_ops']} operadores, ruido emulado de {HW['backend']}", HW["e_emulated"], HW["sigma_emulated"]),
    (f"{HW['n_ops']} operadores, hardware real {HW['backend']}", HW["e_hw"], HW["sigma_hw"]),
    ("FCI (exacto)", E_FCI_24, None),
], columns=["fuente", "E (Ha)", "σ (Ha)"])
tabla_hw["error vs. FCI (mHa)"] = (tabla_hw["E (Ha)"] - E_FCI_24) * 1e3
print(f"\nd = 2,4 Å, job {HW['job_id']}, {HW['cz']} cz:")
print(tabla_hw.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
_below_hf = (E_HF_24 - HW["e_hw"]) / HW["sigma_hw"]
print(f"\nHardware {(E_HF_24 - HW['e_hw']) * 1e3:.1f} mHa por debajo de Hartree-Fock ({_below_hf:.0f} σ): captura correlación real.")
_gap = HW["e_hw"] - HW["e_emulated"]; _sg = (HW["sigma_hw"] ** 2 + HW["sigma_emulated"] ** 2) ** 0.5
print(f"Hardware vs. su emulación: {_gap * 1e3:+.1f} mHa ({_gap / _sg:.1f} σ combinadas).")


#### Qué decidimos y qué aprendimos (#22)

- **Emular primero, correr después.** El modelo de ruido de la calibración actual permite elegir el tamaño del ansatz sin gastar tiempo de QPU: el barrido mostró que más operadores empeoran el resultado con ruido.
- **El hardware captura correlación real:** la energía medida queda ~49 mHa por debajo de Hartree-Fock (unos 8 σ).
- **El hardware es más ruidoso que su emulación:** ~46 mHa peor (unos 6 σ combinadas). El modelo de `from_backend` se arma con la calibración publicada y no ve *crosstalk*, deriva ni errores coherentes.
- **Lejos de FCI** (~390 mHa): con el ruido actual el ansatz útil es muy corto. Para acercarse harían falta menos qubits (#14) y mitigación (#13), o hardware con menos error por `cz`.
- **Reproducibilidad.** Una primera corrida (job `dc0uo5qnj2vs73del9s0`) no guardó los grupos medidos y el análisis los recalculaba; el mismo conteo dio energías distintas (≈ 7 mHa) según el proceso. Se corrigió: PySCF a un hilo, emular y mandar los mismos circuitos en un solo proceso, y guardar los grupos con el *job*. Los números de arriba son de la segunda corrida.

#### Repeticiones y mitigación de errores en hardware (#22, #13)

**Repeticiones.** Los mismos circuitos se corrieron 4 veces en `ibm_marrakesh` (jobs separados). La dispersión entre corridas (5.4 mHa) es la del ruido de shots ($\chi^2$ = 2.5 con 3 gl): no hay deriva medible entre *jobs*. Promediar baja la barra de error, pero no mueve la energía: el sesgo lo pone el ruido del equipo, no los shots.

**Mitigación** (emulada primero, después en el equipo; job `dc0v1als2omc7389ued0`):

- **Corrección de readout:** dos circuitos de calibración (todo 0, todo 1) en los qubits físicos que medimos dan $P(1\mid 0)$ y $P(0\mid 1)$ por qubit (~1.0 % de error medio); cada distribución se corrige con la inversa tensorial.
- **ZNE:** cada `cz` del circuito ya transpilado pasa a `cz·cz·cz` (misma operación, ~3 veces el ruido de `cz`; 28 → 84 `cz`), y $E(0) = 1{,}5\,E(1) - 0{,}5\,E(3)$. El punto x1 son las 4 corridas sin plegar.

In [ ]:
# Mitigación en hardware real (#22, #13): resultados guardados (no hace llamadas a IBM). Error vs. FCI en mHa.
MIT_HW = {"raw_x1": [-1.49249684, 0.00292806], "raw_x3": [-1.46172784, 0.0059328], "ro_x1": [-1.52783842, 0.0029039], "ro_x3": [-1.49389644, 0.00588879], "zne_raw": [-1.50788134, 0.00529999], "zne_readout": [-1.54480941, 0.00525765]}
MIT_EM = {"raw_x1": [-1.52495785, 0.00573658], "raw_x3": [-1.50359392, 0.00578257], "ro_x1": [-1.56052599, 0.00569383], "ro_x3": [-1.53659646, 0.00573629], "zne_raw": [-1.53563981, 0.00907763], "zne_readout": [-1.57249075, 0.00900947]}
NOMBRES = {"raw_x1": "crudo, ruido x1", "raw_x3": "crudo, cz plegadas x3", "ro_x1": "readout corregido, x1", "ro_x3": "readout corregido, x3", "zne_raw": "ZNE (sin readout)", "zne_readout": "readout + ZNE"}
tabla_mit = pd.DataFrame([(NOMBRES[k], (MIT_EM[k][0] - E_FCI_24) * 1e3, (MIT_HW[k][0] - E_FCI_24) * 1e3, MIT_HW[k][1] * 1e3)
                          for k in NOMBRES], columns=["paso", "emulado (mHa)", "hardware (mHa)", "σ hardware (mHa)"])
print(f"d = 2,4 Å | sin ruido {(HW['e_noiseless'] - E_FCI_24) * 1e3:+.1f} mHa | Hartree-Fock {(E_HF_24 - E_FCI_24) * 1e3:+.1f} mHa")
print(tabla_mit.to_string(index=False, float_format=lambda x: f"{x:+.1f}"))
_crudo, _mit = MIT_HW["raw_x1"][0], MIT_HW["zne_readout"][0]
print(f"\nLa mitigación saca {(_crudo - _mit) * 1e3:.0f} mHa: {100 * (_crudo - _mit) / (_crudo - HW['e_noiseless']):.0f} % del aporte del ruido del equipo.")


**Qué muestra.** La corrección de readout hace la mayor parte (~35 mHa) y ZNE suma ~17 mHa: en total se recupera ~54 % del aporte del ruido (de +382 a +330 mHa, contra +286 sin ruido). Lo que queda (~44 mHa) es ruido que la extrapolación lineal con dos puntos no captura (errores coherentes, no linealidad al plegar); un tercer punto (x5) permitiría un ajuste no lineal. Mitigado, el equipo real queda tan cerca de FCI como el equipo *emulado* sin mitigar. El límite de fondo sigue siendo el ansatz corto (+286 mHa): para acercarse a FCI en hardware harían falta menos qubits (#14) y más profundidad con menos error por `cz`.

#### Correr en un equipo real de IBM (#22)

Esta celda repite el experimento de arriba en un equipo real de IBM. **Está apagada por defecto** (`RUN_IBM = False`): el notebook corre entero en Colab sin credenciales y sin gastar tiempo de QPU.

**Credenciales (nunca en el notebook):**

1. En [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com): crear una instancia (plan Open, gratis, ~10 min de QPU por mes), copiar su **CRN** y crear una **API key**.
2. En Colab, panel **Secretos** (🔑): agregar `IBM_QUANTUM_TOKEN` (la API key) e `IBM_QUANTUM_CRN` (el CRN) y habilitarles el acceso para este notebook. Fuera de Colab: variables de entorno con esos nombres, o una cuenta guardada con `QiskitRuntimeService.save_account`.
3. Necesita `qiskit-ibm-runtime`, que instala la celda `%pip` de la subsección de FakeFez (#12): correr esa celda antes.

**Uso** (cambiar las variables de la celda y correrla):

| `IBM_MODE` | Qué hace | Tiempo de QPU |
|---|---|---|
| `"emulate"` | Arma el ansatz corto, transpila al layout del equipo y simula **localmente con el ruido de su calibración actual** (`AerSimulator.from_backend`) | 0 |
| `"submit"` | Manda **los mismos circuitos** al equipo real y guarda el job id y los grupos medidos en `results/ibm_run.json` | ~1 min (x1) / ~2 min con ZNE |
| `"fetch"` | Trae el job y calcula E ± σ crudo, con corrección de readout y con ZNE | 0 |

Siempre conviene `emulate` antes de `submit`. Con `IBM_ZNE = True` el mismo job lleva los circuitos con `cz` plegadas x3 y 2 circuitos de calibración de readout. La versión en scripts (más opciones: barrido de tamaños, repeticiones) está en `scripts/ibm_hardware.py`, `ibm_hardware_repeat.py` e `ibm_mitigation.py`.

In [ ]:
# Correr en hardware real de IBM (#22): emular -> mandar -> traer. APAGADO por defecto (no gasta tiempo de QPU).
# Credenciales: en Colab, en Secretos (🔑) con los nombres IBM_QUANTUM_TOKEN e IBM_QUANTUM_CRN y acceso habilitado
# para este notebook; fuera de Colab, variables de entorno con esos nombres o una cuenta guardada con
# QiskitRuntimeService.save_account. La clave nunca se imprime ni se guarda en el notebook.
RUN_IBM = False          # True para usar IBM (necesita credenciales)
IBM_MODE = "emulate"     # "emulate": ruido del equipo, local, sin QPU | "submit": manda el job | "fetch": trae y analiza
IBM_BACKEND = None       # None = el equipo real menos ocupado; o un nombre, p. ej. "ibm_marrakesh"
IBM_D = 2.4              # Å, punto estirado
IBM_N_OPS = 2            # operadores del ansatz corto (con más, el ruido de los cz extra empeora: ver la tabla de arriba)
IBM_SHOTS = 4000         # shots por grupo de medición
IBM_ZNE = True           # agrega readout (2 circuitos de calibración) y los circuitos con cz plegadas x3 para ZNE
IBM_JOB_ID = ""          # para "fetch": el job id que imprime "submit" (o se lee de results/ibm_run.json)
IBM_RUN_FILE = os.path.join("results", "ibm_run.json")
# Los equipos de IBM piden el plugin de traducción "ibm_dynamic_circuits" de qiskit-ibm-runtime. Si el paquete se instaló
# en esta misma sesión (abajo), Qiskit ya armó su lista de plugins y no lo ve: usamos el traductor estándar, que da
# circuitos válidos para el equipo (mismas compuertas cz, rz, sx, x) sin tener que reiniciar la sesión.
_IBM_TRANSLATION = "translator"

# qiskit-ibm-runtime no está en la celda %pip principal (no hay versión en nixpkgs) y el %pip del extra de FakeFez
# solo corre con RUN_FEZ = True: con RUN_IBM = True se instala acá si falta (en Colab anda; en nix, instalarlo a mano).
if RUN_IBM:
    try:
        import qiskit_ibm_runtime  # noqa: F401
    except ModuleNotFoundError:
        import subprocess, sys
        if subprocess.call([sys.executable, "-m", "pip", "install", "-q", "qiskit-ibm-runtime==0.47.0"]) != 0:
            raise ModuleNotFoundError("Falta qiskit-ibm-runtime y no se pudo instalar: pip install qiskit-ibm-runtime==0.47.0")

import json as _json
import os
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer.primitives import SamplerV2 as _AerSamplerV2
from scipy.optimize import minimize
SEED = globals().get("SEED", 2026)
from qiskit import ClassicalRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager


def _ibm_service():
    """QiskitRuntimeService desde los Secretos de Colab, variables de entorno o una cuenta guardada (en ese orden)."""
    from qiskit_ibm_runtime import QiskitRuntimeService
    token = crn = None
    try:
        from google.colab import userdata
        token, crn = userdata.get("IBM_QUANTUM_TOKEN"), userdata.get("IBM_QUANTUM_CRN")
    except Exception:
        token, crn = os.environ.get("IBM_QUANTUM_TOKEN"), os.environ.get("IBM_QUANTUM_CRN")
    if token:
        return QiskitRuntimeService(channel="ibm_quantum_platform", token=token, instance=crn)
    return QiskitRuntimeService()  # cuenta guardada en esta máquina


def _ibm_short_circuit(ops, th):
    qc = QuantumCircuit(N_QUBITS)
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for t, (_, kind, p, q) in zip(th, ops):
        if kind == "D":
            givens_double(qc, t, alpha_qubit(p), beta_qubit(p), alpha_qubit(q), beta_qubit(q))
        else:
            givens_single(qc, t, alpha_qubit(p), alpha_qubit(q))
            givens_single(qc, t, beta_qubit(p), beta_qubit(q))
    return qc


def _ibm_problem(d, n_ops):
    """H total (electrónico + e_nuc) y un ansatz corto: selección greedy tipo ADAPT sobre las excitaciones distintas de
    ANSATZ_OPS, ángulos ajustados con statevector (permitido para buscar parámetros)."""
    op, e_nuc = qubit_hamiltonian(d)
    H = (op + SparsePauliOp("I" * N_QUBITS, e_nuc)).simplify()
    Hm = H.to_matrix()
    energy = lambda ops, th: float(np.real(np.conj(v := Statevector(_ibm_short_circuit(ops, th)).data) @ (Hm @ v)))
    pool = list({(k, p, q): (0, k, p, q) for (_, k, p, q) in ANSATZ_OPS}.values())
    chosen, th, h = [], np.zeros(0), 1e-4
    while len(chosen) < n_ops:
        grads = [abs(energy(chosen + [o], np.append(th, h)) - energy(chosen + [o], np.append(th, -h))) / (2 * h)
                 if o not in chosen else -1 for o in pool]
        chosen.append(pool[int(np.argmax(grads))])
        th = minimize(lambda t: energy(chosen, t), np.append(th, 0.0), method="BFGS").x
    ref = references(d)
    return H, chosen, th, dict(d=d, n_ops=n_ops, e_fci=float(ref["e_fci"]), e_hf=float(ref["e_hf"]),
                               e_noiseless=energy(chosen, th), ops=[list(map(str, o)) for o in chosen])


def _ibm_meas_circuits(qc, groups):
    circs = []
    for g in groups:
        basis = ["I"] * N_QUBITS
        for lab in g.paulis.to_labels():
            for k, ch in enumerate(reversed(lab)):
                if ch != "I":
                    basis[k] = ch
        c = qc.copy()
        for k, ch in enumerate(basis):
            if ch == "X":
                c.h(k)
            elif ch == "Y":
                c.sdg(k)
                c.h(k)
        c.add_register(ClassicalRegister(N_QUBITS, "m"))
        c.measure(range(N_QUBITS), range(N_QUBITS))
        circs.append(c)
    return circs


def _ibm_phys(circ):
    m = {circ.find_bit(i.clbits[0]).index: circ.find_bit(i.qubits[0]).index for i in circ.data if i.operation.name == "measure"}
    return [m[k] for k in range(len(m))]


def _ibm_fold(circ, factor=3):
    out = circ.copy_empty_like()
    for i in circ.data:
        for _ in range(factor if i.operation.name == "cz" else 1):
            out.append(i.operation, i.qubits, i.clbits)
    return out


def _ibm_cal(backend, phys):
    pm0 = generate_preset_pass_manager(optimization_level=0, backend=backend, initial_layout=list(range(backend.num_qubits)),
                                       translation_method=_IBM_TRANSLATION)
    out = []
    for flip in (False, True):
        qc = QuantumCircuit(backend.num_qubits, len(phys))
        for j, q in enumerate(phys):
            if flip:
                qc.x(q)
            qc.measure(q, j)
        out.append(pm0.run(qc))
    return out


def _ibm_energy(groups, counts_list, maps=None, conf=None):
    """E y σ desde los bitstrings; con `conf` corrige readout (inversa tensorial por qubit físico)."""
    e, var = 0.0, 0.0
    x = np.arange(2 ** N_QUBITS)
    for k, (g, counts) in enumerate(zip(groups, counts_list)):
        p = np.zeros(2 ** N_QUBITS)
        for b, c in counts.items():
            p[int(b, 2)] += c
        N = p.sum()
        q = p / N
        if conf is not None:
            t = q.reshape([2] * N_QUBITS)
            for j, pq in enumerate(maps[k]):
                p10, p01 = conf[pq]
                A_inv = np.linalg.inv(np.array([[1 - p10, p01], [p10, 1 - p01]]))
                t = np.moveaxis(np.tensordot(A_inv, t, axes=([1], [N_QUBITS - 1 - j])), 0, N_QUBITS - 1 - j)
            q = t.reshape(-1)
        vals = np.zeros(len(x))
        for lab, c in zip(g.paulis.to_labels(), g.coeffs.real):
            m = sum(1 << j for j, ch in enumerate(reversed(lab)) if ch != "I")
            vals += c * (1 - 2 * (np.array([bin(v & m).count("1") for v in x]) & 1))
        mean = float((q * vals).sum())
        w = np.abs(q) / np.abs(q).sum()
        e += mean
        var += float((w * (vals - mean) ** 2).sum()) / N
    return e, float(np.sqrt(var))


def _ibm_conf(cal_counts, phys):
    c0, c1 = cal_counts
    n0, n1 = sum(c0.values()), sum(c1.values())
    return {q: (sum(n for b, n in c0.items() if (int(b, 2) >> j) & 1) / n0,
                sum(n for b, n in c1.items() if not (int(b, 2) >> j) & 1) / n1) for j, q in enumerate(phys)}


def _ibm_report(info, groups, maps, c1, c3=None, cal=None, label=""):
    fci_ = info["e_fci"]
    rows = [("Hartree-Fock", info["e_hf"], None), (f"{info['n_ops']} operadores, sin ruido", info["e_noiseless"], None)]
    rows.append(("crudo x1", *_ibm_energy(groups, c1)))
    if c3 is not None and cal is not None:
        conf = _ibm_conf(cal, sorted({q for m in maps for q in m}))
        e1r, s1r = _ibm_energy(groups, c1, maps, conf)
        e3r, s3r = _ibm_energy(groups, c3, maps, conf)
        rows += [("crudo x3 (cz plegadas)", *_ibm_energy(groups, c3)), ("readout corregido x1", e1r, s1r),
                 ("readout + ZNE", 1.5 * e1r - 0.5 * e3r, float(np.hypot(1.5 * s1r, 0.5 * s3r)))]
    t = pd.DataFrame([(n, e, (e - fci_) * 1e3, None if s is None else s * 1e3) for n, e, s in rows],
                     columns=["fuente", "E (Ha)", "error vs. FCI (mHa)", "σ (mHa)"])
    print(f"{label} | d = {info['d']} Å")
    print(t.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    return t


if not RUN_IBM:
    print("Hardware de IBM apagado (RUN_IBM = False). Para usarlo: credenciales en Secretos y RUN_IBM = True.")
elif IBM_MODE in ("emulate", "submit"):
    _svc = _ibm_service()
    _be = _svc.backend(IBM_BACKEND) if IBM_BACKEND else _svc.least_busy(operational=True, simulator=False)
    _H, _ops, _th, _info = _ibm_problem(IBM_D, IBM_N_OPS)
    _groups = _H.group_commuting(qubit_wise=True)
    _pm = generate_preset_pass_manager(optimization_level=3, backend=_be, seed_transpiler=SEED,
                                       translation_method=_IBM_TRANSLATION)
    _isa = _pm.run(_ibm_meas_circuits(_ibm_short_circuit(_ops, _th), _groups))
    _maps = [_ibm_phys(c) for c in _isa]
    _extra = [_ibm_fold(c) for c in _isa] + _ibm_cal(_be, sorted({q for m in _maps for q in m})) if IBM_ZNE else []
    _pubs = [(c, None, IBM_SHOTS) for c in _isa + _extra]
    print(f"{_be.name}: {len(_groups)} grupos, {max(c.count_ops().get('cz', 0) for c in _isa)} cz por circuito, "
          f"{len(_pubs)} circuitos x {IBM_SHOTS} shots")
    if IBM_MODE == "emulate":
        from qiskit_aer import AerSimulator
        _nm = AerSimulator.from_backend(_be).options.noise_model
        _res = _AerSamplerV2(seed=SEED, options={"backend_options": {"method": "density_matrix", "noise_model": _nm}}).run(_pubs).result()
    else:
        from qiskit_ibm_runtime import SamplerV2 as RuntimeSampler
        _job = RuntimeSampler(mode=_be).run(_pubs)
        os.makedirs("results", exist_ok=True)
        _json.dump(dict(job_id=_job.job_id(), backend=_be.name, info=_info, zne=IBM_ZNE, n_groups=len(_groups), maps=_maps,
                        groups=[[g.paulis.to_labels(), g.coeffs.real.tolist()] for g in _groups]), open(IBM_RUN_FILE, "w"))
        print(f"Job mandado: {_job.job_id()} (guardado en {IBM_RUN_FILE}). Para analizarlo: IBM_MODE = 'fetch'.")
        _res = None
    if _res is not None:
        _n = len(_groups)
        _cnt = [r.data.m.get_counts() for r in list(_res)[:_n]]
        _c3 = [r.data.m.get_counts() for r in list(_res)[_n:2 * _n]] if IBM_ZNE else None
        _cal = [r.data.c.get_counts() for r in list(_res)[2 * _n:]] if IBM_ZNE else None
        tabla_ibm = _ibm_report(_info, _groups, _maps, _cnt, _c3, _cal, f"EMULADO {_be.name}")
elif IBM_MODE == "fetch":
    _run = _json.load(open(IBM_RUN_FILE))
    _jid = IBM_JOB_ID or _run["job_id"]
    _res = list(_ibm_service().job(_jid).result())
    _groups = [SparsePauliOp(l, c) for l, c in _run["groups"]]  # los grupos exactos de los circuitos mandados
    _n = _run["n_groups"]
    _cnt = [r.data.m.get_counts() for r in _res[:_n]]
    _c3 = [r.data.m.get_counts() for r in _res[_n:2 * _n]] if _run["zne"] else None
    _cal = [r.data.c.get_counts() for r in _res[2 * _n:]] if _run["zne"] else None
    tabla_ibm = _ibm_report(_run["info"], _groups, _run["maps"], _cnt, _c3, _cal, f"HARDWARE {_run['backend']} (job {_jid})")
